# Kani Advanced, multilingual

Kani Advanced continued in nine languages: English, Korean, German, French, Spanish, Portuguese, Japanese, and
Simplified and Traditional Chinese. Still **from scratch**, with no pretrained weights.

- **Kani's multilingual tokenizer** (32,768 tokens, BPE pattern version 3): about 4x fewer tokens than the English one
  for Korean, Japanese and Chinese, and 1.7-1.9x fewer for German, French, Spanish and Portuguese.
- **A continuation, not a restart:** the run began from the English Kani Advanced, moved to this tokenizer by
  `kani_advanced.py retokenize` (Modal does this once; see `kaggle/modal_kani_advanced.py`).
- **Text per session:** 40% English (FineWeb-Edu, Cosmopedia v2) and 7.5% of each other language (FineWeb-2), every
  document checked to be in its language's script. Validation holds out text in every language.

## Every session

**Add Input** with the run's latest checkpoint: this notebook's own latest output, or a dataset with the
`kani-multilingual` folder (checkpoint.pt and progress.json) downloaded from Modal. Then **Save & Run All**.
Settings: **GPU T4 x2** and **Internet on**. `continue_from_previous` is True: with no checkpoint attached it stops
within a minute instead of training from zero.

## Output

- `kani-multilingual/kani-advanced.npz`: the model, in Kani's checkpoint format
- `kani-multilingual/samples.json`: a prompt in each language, to see it improve
- `kani-multilingual/checkpoint.pt` and `progress.json`: for continuing

In [ ]:
SETTINGS = {
    'run_dir': '/kaggle/working/kani-multilingual',
    'tokenizer': 'kani_tokenizer_multilingual.json',
    'width': 1024,
    'layers': 16,
    'heads': 16,
    'context': 1024,
    'target_tokens': 4_500_000_000,  # the whole run; the learning rate winds down over the last 20%
    'hours': 10.0,                   # training time per session (Kaggle stops sessions at 12 hours)
    'micro_batch': 4,                # sequences per GPU at once; lower it if you see "CUDA out of memory"
    'continue_from_previous': True,  # stop at once when the run's checkpoint is not attached
    'sources': [  # 40% English, 7.5% of each other language; FineWeb-2 documents are kept only in their script
        {'repo': 'HuggingFaceFW/fineweb-edu', 'prefix': 'sample/10BT/', 'per_session': 1, 'share': 0.30},
        {'repo': 'HuggingFaceTB/smollm-corpus', 'prefix': 'cosmopedia-v2/', 'per_session': 1, 'share': 0.10},
    ] + [{'repo': 'HuggingFaceFW/fineweb-2', 'prefix': f'data/{code}/train/', 'validation_file': f'data/{code}/test/000_00000.parquet',
          'per_session': 1, 'share': 0.075, **extra}
         for code, extra in [('kor_Hang', {'filter': 'hangul'}), ('deu_Latn', {}), ('fra_Latn', {}), ('spa_Latn', {}), ('por_Latn', {}),
                             ('jpn_Jpan', {'filter': 'kana'}), ('cmn_Hani', {'filter': 'simplified'}),
                             ('cmn_Hani', {'filter': 'traditional', 'offset': 185})]],
}

import json
json.dump(SETTINGS, open('settings.json', 'w'), indent=1)
print(json.dumps(SETTINGS, indent=1))

In [ ]:
import torch, urllib.request
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print('PyTorch', torch.__version__, '| GPUs:', gpus or 'none: set Settings → Accelerator → GPU T4 x2')
try:
    urllib.request.urlopen('https://huggingface.co', timeout=15)
    print('Internet: on')
except OSError as error:
    raise SystemExit(f'Cannot reach huggingface.co ({error}). Is Settings → Internet on (Session options panel)?')

Kani's multilingual tokenizer (the same file as `kaggle/kani_tokenizer_multilingual.json` in the KelusModelB1 repository) and the training script (`kaggle/kani_advanced.py`).

In [ ]:
%%writefile kani_tokenizer_multilingual.json
{"kind":"bpe","merges":[[227,129],[227,130],[239,188],[227,128],[227,131],[228,184],[32,236],[101,110],[32,100],[101,114],[263,140],[101,115],[32,97],[105,110],[231,154],[275,132],[111,110],[32,116],[236,157],[32,235],[32,115],[264,130],[32,112],[228,186],[32,99],[230,156],[97,110],[111,114],[97,114],[264,129],[32,108],[229,143],[114,101],[269,101],[229,133],[105,115],[228,187],[229,164],[32,109],[116,105],[237,149],[32,101],[226,128],[97,108],[97,115],[261,174],[111,115],[229,174],[32,234],[117,110],[99,104],[105,116],[229,156],[229,136],[228,189],[235,139],[111,117],[231,148],[261,132],[32,102],[116,101],[230,152],[229,144],[195,169],[113,117],[104,101],[229,176],[230,136],[230,150],[111,109],[32,110],[261,167],[266,128],[32,111],[268,116],[32,98],[261,151],[236,151],[108,101],[261,171],[32,119],[233,128],[261,166],[97,100],[97,116],[229,138],[322,175],[261,168],[229,190],[32,274],[261,153],[316,164],[101,108],[261,170],[228,185],[99,105],[279,180],[229,185],[232,191],[261,159],[114,111],[32,237],[235,138],[273,110],[262,139],[32,325],[261,140],[32,118],[278,326],[274,103],[230,151],[234,179],[229,141],[230,137],[266,141],[284,186],[261,190],[114,97],[229,173],[229,155],[117,115],[363,148],[261,175],[262,146],[313,168],[286,137],[232,166],[32,104],[233,129],[105,99],[265,188],[232,161],[32,310],[229,184],[105,108],[117,114],[105,101],[233,135],[284,134],[300,277],[32,83],[32,103],[229,183],[97,109],[228,188],[229,165],[338,144],[236,160],[234,176],[229,186],[232,128],[236,132],[236,158],[230,157],[105,100],[231,155],[279,132],[131,189],[278,111],[301,152],[298,167],[101,100],[32,65],[229,188],[111,108],[229,135],[236,167],[105,109],[235,161],[266,173],[291,97],[261,139],[230,149],[32,293],[32,121],[230,173],[272,116],[229,175],[366,101],[364,100],[229,191],[101,109],[230,128],[232,174],[229,128],[267,158],[32,268],[261,163],[231,137],[279,152],[101,116],[285,277],[262,130],[334,102],[232,135],[229,134],[265,179],[231,187],[229,140],[230,176],[228,190],[117,109],[266,138],[267,157],[105,114],[229,177],[105,103],[262,138],[232,168],[230,153],[50,48],[32,67],[236,139],[230,138],[429,156],[235,165],[297,165],[231,156],[301,156],[372,160],[136,352],[230,148],[233,155],[285,330],[32,80],[328,145],[261,134],[184,176],[233,153],[358,180],[229,147],[236,154],[229,146],[262,140],[32,77],[318,168],[228,191],[230,142],[32,114],[233,157],[232,175],[303,156],[281,101],[230,179],[387,129],[233,150],[262,137],[32,84],[116,270],[230,132],[303,157],[130,172],[229,137],[32,69],[278,104],[283,289],[235,143],[261,147],[105,311],[229,189],[117,108],[426,186],[427,128],[32,49],[236,156],[267,160],[230,160],[195,161],[232,181],[493,140],[318,159],[32,66],[116,97],[229,145],[236,149],[230,131],[99,101],[270,115],[235,166],[303,153],[232,418],[195,179],[232,167],[292,175],[409,128],[231,167],[231,153],[32,68],[233,171],[231,180],[32,73],[236,138],[314,176],[230,178],[229,139],[261,143],[229,142],[118,101],[279,128],[263,154],[235,147],[392,140],[231,130],[287,100],[232,170],[476,188],[234,178],[231,142],[231,144],[317,114],[371,165],[117,116],[262,185],[195,163],[412,156],[230,175],[231,171],[272,115],[114,105],[329,185],[309,176],[315,156],[262,147],[97,98],[377,351],[283,361],[298,154],[391,265],[455,170],[230,155],[230,140],[574,111],[266,139],[231,173],[314,134],[32,76],[230,181],[195,173],[308,182],[230,158],[231,132],[273,108],[280,176],[233,161],[231,181],[32,195],[328,144],[287,116],[231,149],[267,149],[234,488],[140,128],[327,177],[291,101],[313,176],[296,116],[233,130],[263,129],[230,172],[268,100],[97,103],[261,141],[297,150],[236,150],[32,70],[111,100],[231,168],[97,400],[233,151],[264,140],[269,272],[264,141],[235,130],[101,99],[232,136],[230,184],[195,167],[235,167],[337,343],[262,164],[136,152],[301,180],[517,132],[261,149],[232,178],[355,159],[48,48],[470,130],[232,129],[329,176],[232,180],[111,116],[229,166],[230,139],[261,130],[309,178],[230,177],[286,172],[332,351],[230,143],[539,172],[229,160],[230,180],[231,190],[232,179],[464,180],[405,154],[234,181],[32,272],[302,108],[229,149],[32,71],[379,144],[230,165],[549,152],[316,481],[448,343],[233,160],[323,136],[229,159],[265,136],[406,189],[115,116],[32,82],[281,117],[280,167],[233,154],[97,99],[231,182],[232,130],[284,139],[111,119],[569,134],[229,129],[261,160],[279,184],[235,169],[491,129],[337,360],[380,189],[403,165],[320,288],[265,131],[295,172],[295,168],[195,188],[236,155],[267,139],[108,121],[267,151],[327,143],[393,100],[525,188],[263,137],[235,160],[346,155],[232,190],[232,163],[32,87],[266,186],[302,120],[263,136],[97,121],[441,131],[513,141],[32,106],[269,111],[342,153],[32,296],[261,138],[233,131],[231,186],[308,154],[231,174],[229,187],[231,164],[414,165],[32,437],[232,182],[233,163],[233,418],[179,180],[500,162],[32,277],[336,101],[232,183],[310,103],[286,128],[232,169],[230,168],[233,162],[473,156],[32,72],[283,114],[232,137],[273,115],[302,116],[231,189],[374,139],[624,180],[443,167],[269,105],[231,159],[118,270],[195,164],[298,169],[235,157],[267,132],[349,151],[262,136],[233,149],[116,104],[295,165],[261,145],[235,176],[323,140],[235,182],[236,131],[413,144],[411,133],[274,101],[267,167],[116,268],[32,78],[359,153],[471,49],[263,159],[236,512],[286,136],[355,139],[32,107],[231,169],[394,130],[266,170],[504,149],[552,164],[273,117],[233,152],[563,186],[261,157],[302,109],[232,189],[446,136],[105,97],[309,179],[100,101],[231,172],[516,97],[374,128],[767,188],[410,166],[286,131],[262,175],[331,111],[731,168],[97,112],[461,134],[346,160],[445,139],[315,160],[163,188],[459,150],[183,184],[268,321],[235,172],[266,187],[371,182],[606,160],[103,101],[115,101],[233,187],[265,171],[379,166],[594,137],[269,270],[411,140],[112,108],[561,164],[32,288],[32,324],[542,110],[265,170],[116,114],[265,169],[280,139],[267,641],[713,475],[653,130],[237,150],[344,111],[270,101],[32,462],[229,150],[389,147],[478,139],[111,112],[111,99],[298,150],[262,132],[231,143],[266,154],[229,167],[32,122],[235,158],[413,165],[229,163],[341,104],[299,97],[32,75],[269,397],[372,188],[116,117],[281,111],[267,512],[536,133],[287,115],[292,145],[237,153],[261,169],[236,152],[280,130],[32,428],[601,182],[448,360],[112,114],[262,162],[342,154],[111,103],[236,178],[105,277],[32,86],[398,145],[267,152],[116,115],[674,173],[299,101],[105,122],[265,187],[231,157],[664,142],[109,335],[104,114],[300,99],[398,141],[232,173],[32,50],[416,174],[518,348],[510,159],[555,149],[354,319],[310,100],[660,144],[315,191],[435,317],[231,165],[262,129],[567,140],[407,575],[510,143],[97,117],[105,118],[105,270],[351,365],[506,147],[590,180],[109,97],[460,180],[278,114],[349,140],[283,108],[229,162],[329,135],[261,136],[233,133],[102,102],[231,136],[449,169],[98,270],[497,157],[284,142],[230,186],[32,79],[283,288],[233,156],[349,136],[416,184],[233,166],[292,138],[408,149],[232,187],[317,115],[230,159],[232,144],[235,159],[295,182],[231,179],[32,85],[317,116],[268,115],[273,99],[107,101],[331,101],[303,166],[432,507],[32,74],[515,345],[291,272],[229,131],[274,100],[232,186],[286,159],[230,182],[235,144],[438,185],[315,147],[116,111],[697,180],[392,168],[285,104],[112,288],[261,161],[633,152],[280,612],[529,183],[389,142],[97,356],[325,101],[323,142],[313,139],[112,101],[662,180],[300,118],[308,137],[110,101],[269,97],[322,142],[237,152],[102,288],[235,612],[230,129],[235,178],[236,134],[600,156],[261,164],[297,172],[637,592],[231,183],[230,169],[267,156],[305,115],[230,147],[497,161],[380,160],[280,182],[398,143],[557,187],[32,301],[619,161],[230,183],[707,144],[280,170],[235,141],[302,274],[10,10],[312,104],[237,138],[261,155],[536,179],[466,149],[314,169],[232,145],[236,153],[267,131],[315,134],[663,187],[380,158],[109,101],[195,168],[229,157],[97,321],[506,139],[117,101],[395,108],[576,148],[97,311],[309,823],[449,185],[323,141],[232,188],[229,130],[544,477],[230,130],[309,488],[315,141],[231,178],[971,171],[262,191],[262,183],[408,129],[297,138],[492,148],[424,143],[232,139],[394,184],[617,163],[236,185],[266,137],[309,181],[297,163],[304,108],[105,98],[232,138],[735,186],[436,163],[235,133],[195,170],[112,270],[291,307],[552,181],[231,129],[860,190],[279,188],[32,312],[535,132],[185,132],[117,99],[105,102],[261,191],[292,176],[231,160],[314,182],[807,172],[104,116],[581,128],[32,51],[456,133],[232,162],[235,170],[105,268],[734,161],[319,365],[763,165],[265,172],[778,129],[358,179],[777,128],[327,141],[292,184],[445,145],[280,147],[671,134],[353,108],[408,156],[492,169],[443,157],[551,110],[755,178],[262,143],[32,32],[406,179],[420,382],[97,118],[547,188],[32,326],[230,133],[499,272],[1009,159],[638,140],[656,131],[591,129],[154,140],[273,98],[1027,184],[234,184],[346,168],[899,128],[627,139],[235,742],[359,135],[457,265],[105,111],[543,163],[436,164],[231,151],[232,153],[230,161],[344,97],[341,1026],[105,356],[195,182],[957,187],[269,117],[283,270],[415,344],[433,153],[281,270],[284,155],[568,176],[324,115],[180,128],[100,270],[782,128],[412,177],[359,155],[424,128],[230,174],[229,132],[233,186],[236,821],[387,139],[232,171],[1080,676],[230,188],[478,159],[460,145],[273,116],[520,147],[377,699],[233,159],[328,150],[601,161],[498,165],[237,131],[312,101],[308,158],[329,153],[458,143],[708,156],[280,143],[736,190],[286,141],[292,151],[577,139],[239,189],[108,108],[32,381],[771,183],[320,114],[327,145],[526,132],[262,184],[280,172],[230,141],[117,100],[262,171],[704,1109],[286,186],[105,293],[265,173],[294,108],[235,175],[273,293],[610,132],[265,151],[231,150],[319,585],[527,188],[284,167],[285,287],[297,182],[295,131],[161,176],[235,179],[267,150],[563,185],[565,170],[597,183],[230,187],[265,149],[878,148],[483,187],[362,153],[635,135],[262,179],[273,112],[229,158],[323,145],[1098,171],[108,519],[306,332],[292,170],[403,177],[408,132],[694,154],[32,420],[32,335],[280,742],[164,145],[292,150],[284,140],[295,136],[474,128],[97,296],[237,132],[406,185],[292,163],[233,164],[1033,128],[267,154],[232,177],[236,641],[277,116],[342,178],[680,186],[706,114],[231,162],[284,164],[232,176],[49,48],[330,101],[289,100],[98,108],[557,159],[266,142],[315,149],[403,178],[231,166],[96,96],[112,339],[318,177],[589,1243],[115,111],[232,131],[269,296],[520,177],[231,177],[32,479],[233,169],[561,156],[482,191],[597,129],[416,180],[103,268],[617,132],[348,319],[564,111],[237,1132],[262,181],[526,156],[402,114],[668,173],[267,134],[262,176],[410,151],[746,175],[312,121],[283,293],[845,164],[668,172],[232,132],[388,97],[32,471],[273,114],[945,128],[230,135],[621,101],[341,101],[234,177],[736,186],[517,153],[323,132],[398,140],[308,140],[689,147],[374,147],[283,101],[433,176],[411,131],[469,128],[291,305],[297,187],[308,185],[236,182],[265,161],[501,180],[338,172],[359,145],[232,133],[379,184],[308,162],[526,149],[1255,176],[278,101],[405,129],[105,335],[657,130],[288,101],[461,155],[263,155],[283,570],[229,153],[373,129],[273,100],[825,184],[442,335],[287,103],[319,360],[295,179],[469,173],[446,144],[758,145],[456,141],[951,138],[981,116],[32,317],[329,188],[117,112],[338,133],[390,97],[655,365],[1032,151],[420,480],[770,487],[231,133],[285,108],[411,129],[105,272],[546,145],[340,354],[32,786],[32,270],[281,396],[261,148],[701,582],[549,148],[281,116],[831,158],[311,101],[1237,139],[266,150],[368,270],[313,186],[482,190],[473,160],[232,143],[466,177],[230,144],[269,305],[342,160],[295,137],[740,159],[99,108],[299,287],[792,186],[591,135],[498,168],[459,133],[301,160],[655,468],[359,152],[262,173],[278,378],[644,494],[863,117],[232,165],[916,158],[771,191],[236,176],[355,136],[117,293],[98,101],[265,158],[452,116],[237,148],[1076,132],[134,181],[269,268],[273,109],[318,181],[327,134],[112,289],[261,176],[32,105],[303,152],[230,162],[346,161],[332,622],[108,100],[356,97],[362,152],[299,117],[295,183],[320,97],[267,176],[483,134],[433,180],[441,133],[455,179],[267,821],[32,52],[500,158],[231,139],[319,343],[665,135],[105,112],[304,709],[492,176],[231,131],[697,176],[489,162],[32,53],[529,132],[264,139],[444,190],[504,168],[484,111],[423,110],[1003,140],[264,138],[438,166],[265,134],[231,184],[467,1094],[287,321],[303,147],[233,148],[629,180],[230,163],[229,148],[235,1086],[269,324],[1296,487],[482,182],[235,163],[299,312],[316,136],[1056,190],[409,132],[393,97],[100,111],[435,570],[528,115],[285,111],[287,537],[236,130],[280,144],[274,115],[400,115],[230,167],[459,151],[318,162],[508,326],[230,146],[344,307],[474,149],[609,140],[277,115],[555,153],[232,141],[105,300],[261,152],[320,101],[914,496],[98,293],[554,146],[1010,132],[1416,191],[689,178],[955,172],[460,151],[234,185],[195,186],[291,111],[576,143],[118,353],[328,191],[609,182],[285,289],[233,140],[350,509],[671,143],[109,111],[749,177],[1067,152],[287,110],[464,184],[482,175],[100,268],[114,272],[627,174],[374,141],[281,104],[489,144],[656,189],[195,177],[569,131],[577,153],[1111,177],[1059,265],[441,171],[283,305],[295,167],[415,101],[935,141],[267,182],[459,186],[336,121],[469,136],[229,161],[389,184],[467,110],[270,110],[283,117],[230,154],[283,307],[662,177],[358,182],[452,115],[332,383],[534,179],[99,330],[547,189],[1352,168],[327,135],[1022,168],[231,175],[474,138],[535,188],[486,1006],[237,140],[112,307],[230,191],[232,140],[292,139],[99,111],[262,141],[544,541],[491,161],[973,136],[265,144],[119,101],[311,268],[381,116],[424,149],[368,277],[852,97],[690,178],[1001,175],[309,177],[629,156],[413,133],[104,270],[262,163],[393,101],[324,114],[288,272],[231,134],[482,185],[372,181],[231,188],[461,191],[372,132],[520,162],[446,481],[433,136],[267,178],[237,143],[32,616],[613,347],[232,151],[427,132],[111,107],[449,135],[464,188],[868,296],[598,97],[295,177],[280,179],[277,101],[231,191],[231,161],[584,339],[333,794],[636,133],[554,187],[169,676],[862,139],[1023,176],[109,270],[974,152],[805,181],[297,128],[265,129],[97,1007],[117,103],[356,304],[231,135],[410,148],[32,307],[469,152],[99,277],[146,136],[116,396],[519,116],[1091,148],[881,152],[987,841],[554,161],[942,144],[366,105],[261,173],[108,111],[280,141],[628,115],[703,681],[849,137],[296,115],[100,1087],[969,143],[798,178],[298,170],[99,272],[340,383],[97,274],[958,110],[797,102],[849,481],[605,144],[1020,177],[267,1249],[534,152],[1147,136],[115,321],[303,148],[97,107],[546,141],[357,352],[32,90],[292,136],[638,136],[619,190],[265,135],[412,184],[269,876],[315,143],[262,170],[320,1266],[97,381],[288,100],[444,147],[1073,177],[373,151],[380,155],[680,142],[265,150],[346,159],[527,161],[102,270],[231,141],[32,89],[335,111],[280,178],[322,147],[321,100],[1199,330],[534,138],[425,108],[299,111],[117,304],[999,109],[331,97],[590,184],[305,101],[529,176],[665,170],[283,538],[800,306],[739,133],[49,57],[394,131],[262,135],[272,101],[324,101],[342,159],[932,158],[678,173],[514,110],[286,170],[105,339],[332,453],[268,300],[396,115],[294,109],[103,114],[1496,176],[402,101],[1099,133],[1034,129],[803,382],[515,296],[231,138],[1015,128],[267,138],[297,142],[604,140],[715,164],[739,179],[424,183],[535,136],[758,174],[229,168],[1316,137],[568,135],[354,879],[315,142],[444,161],[98,339],[233,170],[233,173],[802,166],[518,306],[232,142],[265,137],[346,169],[436,165],[329,189],[414,144],[312,111],[314,135],[889,180],[273,381],[371,160],[684,101],[108,272],[356,841],[473,157],[341,305],[195,159],[261,154],[112,115],[292,141],[644,583],[229,169],[278,442],[345,270],[428,101],[430,700],[524,57],[1303,140],[636,184],[234,1161],[378,109],[277,103],[231,185],[105,353],[645,187],[262,166],[296,101],[1225,180],[489,164],[648,925],[377,1028],[118,97],[262,168],[1522,140],[297,152],[264,144],[750,163],[231,170],[264,145],[703,813],[1135,136],[155,132],[486,1110],[1169,188],[103,117],[317,103],[948,150],[825,180],[309,1161],[99,116],[913,1349],[233,134],[344,101],[534,168],[104,97],[334,98],[550,160],[402,111],[647,48],[805,160],[301,153],[492,180],[1101,693],[149,140],[466,128],[954,189],[99,400],[100,105],[289,105],[102,116],[305,116],[362,148],[755,175],[118,268],[288,115],[404,98],[333,819],[610,138],[568,139],[195,181],[234,823],[932,131],[49,50],[362,140],[739,138],[331,592],[1145,133],[316,168],[229,178],[389,139],[280,175],[194,183],[751,168],[1312,558],[527,183],[281,277],[117,120],[362,143],[331,652],[341,288],[51,48],[286,155],[237,158],[373,179],[307,116],[277,272],[1180,179],[236,1223],[508,104],[1862,272],[1053,168],[478,188],[291,105],[1010,160],[413,172],[266,148],[266,135],[230,185],[278,270],[1813,583],[1375,167],[424,186],[349,158],[409,156],[1319,176],[581,156],[232,172],[1212,184],[32,642],[299,289],[510,155],[751,152],[233,138],[373,128],[419,100],[121,772],[525,132],[32,54],[308,131],[297,139],[565,191],[308,164],[337,319],[866,171],[715,165],[596,101],[318,183],[49,49],[104,111],[262,187],[414,177],[1167,181],[710,172],[181,156],[777,132],[484,112],[668,144],[304,101],[314,165],[284,172],[234,183],[103,111],[236,166],[953,165],[107,115],[265,139],[268,537],[686,136],[445,153],[283,626],[778,157],[710,134],[116,293],[320,111],[734,151],[237,1653],[415,111],[387,150],[974,156],[265,145],[602,108],[527,185],[567,189],[362,150],[267,155],[280,148],[881,180],[848,1004],[565,141],[893,164],[235,132],[629,174],[1133,959],[404,101],[97,537],[233,146],[750,161],[379,152],[388,584],[279,140],[1315,505],[112,111],[100,117],[267,162],[285,114],[281,311],[284,148],[115,311],[547,190],[342,129],[425,888],[323,131],[395,100],[497,174],[232,150],[1727,277],[390,111],[116,272],[233,147],[686,140],[494,365],[314,157],[1334,156],[97,105],[295,169],[289,101],[110,268],[460,163],[499,324],[525,160],[371,169],[267,185],[1885,136],[1034,157],[588,171],[109,268],[1008,154],[236,136],[237,129],[506,128],[1267,186],[269,307],[880,164],[550,132],[331,307],[412,160],[280,1086],[401,397],[531,1036],[230,145],[449,140],[289,116],[265,163],[281,1271],[341,465],[285,101],[230,189],[690,161],[122,101],[732,191],[473,164],[267,1223],[633,180],[300,558],[461,139],[329,175],[864,152],[286,175],[1279,96],[295,141],[546,129],[281,519],[474,165],[708,160],[640,457],[948,153],[768,160],[379,151],[392,147],[862,148],[1828,104],[232,134],[338,176],[629,168],[231,146],[232,158],[309,184],[297,183],[112,293],[1204,174],[718,133],[1732,487],[266,164],[1174,148],[324,110],[433,184],[645,183],[498,146],[373,135],[266,166],[588,136],[434,108],[104,108],[300,108],[308,159],[880,129],[274,97],[267,130],[634,116],[498,167],[53,48],[342,163],[524,48],[280,1846],[931,381],[266,147],[476,180],[318,187],[145,156],[101,120],[449,136],[478,129],[521,116],[649,148],[285,117],[390,304],[744,101],[710,176],[485,361],[390,101],[1064,177],[118,111],[885,272],[273,572],[375,347],[444,169],[292,130],[32,378],[302,339],[262,131],[695,774],[459,187],[337,585],[371,133],[105,558],[470,175],[1391,1507],[651,168],[707,148],[1069,172],[281,97],[335,115],[792,182],[267,153],[226,150],[603,148],[605,166],[262,186],[276,376],[265,160],[468,585],[317,1439],[494,343],[458,147],[328,183],[600,151],[99,578],[415,97],[109,824],[50,49],[362,1427],[657,186],[389,148],[445,188],[908,367],[322,159],[1754,164],[314,155],[32,55],[624,145],[1212,188],[373,138],[401,116],[312,115],[1261,161],[119,119],[1051,153],[394,171],[1345,861],[369,465],[285,288],[32,57],[337,2074],[740,142],[118,538],[316,185],[603,169],[803,1173],[304,272],[99,107],[718,161],[353,339],[534,189],[320,310],[1120,1120],[687,155],[265,165],[265,159],[112,116],[32,48],[285,425],[455,180],[299,626],[114,121],[458,159],[438,188],[605,132],[233,190],[1081,171],[354,556],[320,274],[114,305],[573,681],[867,345],[265,147],[768,156],[114,307],[1659,386],[746,159],[145,144],[527,135],[464,152],[312,324],[969,185],[651,185],[101,289],[457,1304],[285,305],[413,132],[1247,293],[281,425],[269,396],[349,128],[1155,101],[280,133],[324,109],[349,174],[571,658],[1168,170],[342,137],[373,149],[554,185],[401,311],[350,100],[302,781],[680,159],[548,101],[226,151],[809,477],[327,142],[1002,149],[877,1030],[453,306],[825,188],[299,1493],[404,824],[605,177],[105,988],[298,177],[402,268],[523,1129],[610,136],[694,165],[236,133],[112,104],[277,100],[288,116],[298,180],[532,101],[547,186],[1203,180],[81,117],[807,145],[1195,158],[1230,262],[434,103],[288,97],[300,103],[299,274],[1260,176],[114,287],[111,121],[604,185],[338,136],[102,101],[112,361],[394,136],[603,156],[265,138],[298,135],[747,268],[338,173],[603,143],[423,108],[32,56],[295,139],[853,156],[105,378],[229,170],[604,158],[360,917],[237,142],[32,959],[117,311],[350,115],[325,272],[644,319],[230,164],[413,145],[659,367],[236,161],[390,107],[458,153],[429,157],[472,104],[688,107],[404,307],[274,116],[500,146],[838,357],[237,133],[235,148],[414,191],[405,160],[114,312],[1287,179],[103,287],[283,353],[377,332],[438,190],[300,109],[99,521],[363,165],[543,132],[337,377],[853,174],[1516,386],[401,101],[749,166],[117,98],[32,361],[686,144],[1452,1065],[636,175],[270,118],[754,272],[359,144],[104,268],[265,157],[1873,184],[775,169],[800,494],[304,897],[410,149],[261,185],[313,159],[99,293],[229,179],[49,53],[281,112],[314,151],[369,121],[1602,676],[69,83],[331,1655],[588,137],[460,148],[509,110],[229,152],[349,183],[97,902],[471,48],[443,187],[328,152],[97,106],[267,1936],[269,277],[1485,140],[543,134],[594,150],[665,188],[864,140],[104,345],[270,97],[935,146],[1146,149],[289,121],[298,132],[73,110],[105,120],[348,639],[516,300],[1550,167],[904,505],[637,1892],[998,184],[388,305],[768,184],[379,169],[434,99],[349,133],[261,187],[458,132],[105,304],[410,183],[276,333],[344,305],[1069,173],[285,293],[772,270],[1190,153],[797,311],[1752,182],[292,152],[423,114],[421,599],[327,136],[362,131],[979,1320],[334,112],[568,169],[1918,277],[1190,132],[292,183],[194,187],[341,1044],[1060,696],[237,151],[609,153],[726,101],[443,142],[410,156],[1203,184],[1643,1420],[732,167],[620,111],[109,274],[273,558],[1184,128],[635,172],[299,1348],[409,129],[994,705],[654,137],[414,145],[1156,1598],[266,156],[1093,333],[444,176],[399,333],[370,115],[98,268],[441,181],[102,390],[768,177],[32,395],[577,160],[1192,1437],[763,179],[373,161],[461,157],[1497,152],[467,104],[283,111],[528,579],[268,110],[1165,562],[49,56],[403,158],[529,155],[723,115],[295,154],[262,177],[547,187],[782,132],[407,918],[281,442],[800,487],[97,962],[164,128],[518,494],[297,189],[368,101],[1608,177],[470,174],[2068,141],[301,1637],[954,165],[444,138],[484,109],[231,140],[321,108],[1656,2135],[105,107],[905,176],[295,133],[1172,139],[374,190],[134,160],[362,138],[1269,131],[436,162],[751,134],[893,136],[1278,143],[283,425],[718,157],[103,272],[194,171],[118,425],[1015,132],[470,186],[1664,148],[1112,698],[307,101],[314,171],[231,176],[798,159],[514,115],[376,276],[1500,152],[424,181],[692,110],[929,876],[50,50],[891,270],[358,178],[102,108],[152,144],[1410,585],[99,97],[280,132],[646,347],[473,172],[477,949],[893,129],[395,339],[934,365],[1419,168],[357,810],[394,166],[457,681],[1125,108],[1041,100],[946,587],[1039,115],[432,884],[387,143],[741,347],[265,141],[263,145],[762,293],[237,1427],[117,105],[436,187],[373,176],[1127,112],[286,168],[392,128],[543,146],[686,142],[514,108],[750,153],[494,1433],[403,174],[285,304],[112,97],[394,182],[116,324],[116,378],[281,109],[466,139],[299,277],[414,142],[237,139],[336,451],[513,181],[335,272],[683,114],[916,168],[483,163],[309,183],[815,595],[122,97],[236,1249],[583,354],[2430,99],[84,326],[1050,131],[1031,496],[368,952],[360,319],[368,296],[1023,152],[872,100],[479,352],[109,115],[898,898],[1035,347],[110,111],[121,115],[580,795],[233,168],[1480,174],[118,272],[1473,189],[1425,132],[281,121],[816,112],[1051,179],[889,156],[2067,176],[341,270],[765,110],[358,191],[708,164],[345,370],[1532,1756],[405,152],[284,136],[565,158],[265,128],[311,110],[433,133],[276,347],[308,179],[285,326],[430,724],[1099,168],[117,270],[954,172],[501,157],[1103,854],[308,152],[291,857],[1039,293],[738,528],[444,164],[296,321],[182,140],[312,270],[420,1337],[472,330],[331,462],[867,519],[284,171],[849,136],[233,165],[426,134],[664,164],[2021,172],[49,54],[1313,48],[346,158],[715,136],[476,184],[1069,144],[389,169],[393,115],[323,167],[501,129],[120,116],[451,104],[397,100],[98,111],[485,289],[267,180],[1077,110],[420,918],[1287,187],[299,422],[521,289],[770,468],[633,156],[310,116],[373,131],[495,289],[195,180],[762,936],[280,134],[387,186],[501,165],[416,138],[374,191],[556,695],[747,115],[103,270],[775,156],[362,129],[375,776],[1600,176],[265,153],[283,1011],[236,162],[703,1215],[292,164],[596,97],[233,167],[414,161],[1013,1599],[1816,523],[934,343],[394,140],[827,1478],[298,156],[338,481],[579,98],[1172,150],[1438,365],[288,121],[118,415],[318,179],[329,173],[261,146],[1345,895],[1654,110],[230,166],[477,463],[905,183],[275,174],[942,150],[438,134],[682,276],[284,146],[719,101],[281,381],[873,1515],[103,1803],[110,270],[371,143],[639,1102],[281,105],[657,159],[336,117],[608,115],[466,133],[674,189],[1002,136],[438,140],[506,162],[853,132],[721,837],[1513,353],[265,175],[292,178],[568,175],[274,107],[295,146],[690,137],[1618,172],[189,148],[310,101],[447,99],[948,172],[580,743],[1126,139],[345,97],[410,167],[398,135],[278,1797],[381,115],[350,1403],[749,178],[527,184],[111,98],[117,361],[489,141],[793,1393],[889,173],[1064,165],[117,102],[491,170],[591,137],[1003,141],[1895,962],[637,97],[356,112],[414,131],[446,165],[1974,1914],[483,162],[586,102],[1258,138],[285,816],[345,104],[229,151],[261,184],[856,339],[49,51],[602,1283],[550,154],[298,137],[391,573],[605,130],[273,111],[334,1162],[826,505],[268,101],[553,399],[373,142],[581,153],[802,172],[866,176],[1091,180],[677,1102],[237,2101],[234,187],[265,162],[408,144],[598,115],[608,272],[992,679],[117,437],[296,104],[389,138],[273,103],[717,190],[261,179],[1632,174],[1073,130],[469,177],[309,185],[1258,144],[107,116],[717,131],[407,382],[105,287],[750,130],[387,170],[379,163],[421,276],[1060,847],[775,152],[281,428],[1968,176],[318,176],[341,397],[2563,1071],[718,189],[756,115],[340,1005],[882,1365],[1577,128],[340,453],[550,176],[456,183],[1631,146],[358,190],[1794,154],[640,681],[50,52],[289,110],[678,133],[99,296],[278,1858],[323,171],[606,188],[377,360],[610,140],[649,183],[264,133],[292,166],[2677,2299],[577,175],[1282,276],[285,270],[580,1063],[688,116],[281,330],[265,143],[365,348],[2605,1539],[1820,129],[306,383],[520,169],[1072,339],[578,101],[355,166],[408,128],[638,164],[1219,698],[656,176],[802,137],[32,1368],[262,172],[313,152],[347,333],[451,321],[581,149],[456,179],[98,114],[1442,311],[237,130],[368,304],[874,1115],[1319,180],[105,307],[831,145],[299,272],[423,115],[195,162],[1151,110],[505,1347],[809,386],[956,623],[226,152],[1192,1509],[543,128],[105,288],[2020,160],[391,1202],[1014,799],[299,723],[535,189],[281,970],[101,112],[380,190],[104,110],[117,378],[669,321],[106,101],[543,129],[344,288],[543,130],[50,53],[492,184],[669,99],[280,169],[1015,136],[456,134],[576,141],[50,51],[232,152],[1967,144],[355,160],[99,270],[323,166],[108,97],[1396,175],[226,134],[446,172],[984,479],[513,135],[388,692],[444,174],[273,118],[281,991],[698,2033],[117,272],[312,97],[323,184],[230,170],[1448,505],[298,171],[1081,163],[354,583],[1168,191],[49,52],[10,2187],[313,150],[458,173],[588,160],[494,360],[546,187],[609,165],[264,142],[313,146],[442,307],[1327,2780],[2314,148],[451,122],[623,1006],[1441,132],[65,82],[324,272],[264,143],[373,154],[489,133],[1233,149],[2227,184],[1224,184],[905,152],[281,397],[281,843],[287,829],[576,155],[874,2374],[872,293],[283,104],[380,162],[237,1824],[600,132],[550,162],[589,595],[1459,173],[231,158],[336,108],[1414,343],[356,277],[534,162],[105,400],[791,110],[299,305],[49,55],[1583,128],[280,2521],[702,109],[686,164],[623,1110],[102,114],[1040,135],[763,173],[785,794],[1711,317],[356,100],[65,67],[1253,2050],[603,155],[805,132],[296,533],[533,274],[232,147],[551,116],[745,105],[398,142],[1097,1333],[368,538],[685,112],[561,160],[541,298],[749,149],[1556,148],[355,133],[444,186],[333,733],[432,354],[355,144],[328,176],[430,276],[1107,1343],[603,176],[287,97],[663,151],[905,137],[102,274],[651,163],[177,133],[1018,160],[362,142],[274,111],[281,288],[499,437],[717,166],[345,396],[649,175],[283,378],[342,143],[336,572],[388,616],[1500,139],[119,270],[541,716],[1157,118],[890,115],[446,145],[2420,152],[324,100],[2391,673],[672,114],[907,725],[420,611],[504,129],[406,135],[1135,137],[1986,139],[1291,2424],[265,148],[281,1263],[626,101],[32,117],[321,1196],[535,152],[2849,939],[362,1824],[333,2432],[427,129],[744,709],[233,172],[278,268],[933,822],[1517,642],[388,270],[262,167],[636,169],[978,575],[2799,152],[336,293],[426,160],[105,841],[328,182],[1545,1458],[514,114],[1648,159],[1360,1559],[302,936],[526,129],[550,185],[1289,151],[474,164],[678,152],[791,101],[1769,140],[998,149],[649,158],[69,110],[501,183],[664,169],[548,397],[118,335],[472,277],[456,153],[233,189],[467,268],[136,172],[312,304],[1082,385],[235,184],[694,156],[1154,109],[1566,115],[336,114],[237,2491],[1191,667],[283,97],[316,152],[503,117],[298,141],[526,128],[798,191],[349,139],[295,173],[267,166],[104,304],[2535,112],[288,109],[617,138],[1834,171],[889,152],[1058,391],[424,160],[1584,182],[295,171],[857,1077],[320,108],[880,128],[106,97],[231,147],[1289,172],[286,139],[342,128],[353,102],[937,177],[1084,115],[1854,132],[263,146],[768,164],[1477,128],[359,189],[649,178],[430,990],[375,541],[76,97],[448,348],[689,154],[297,164],[446,133],[887,1140],[602,103],[501,149],[570,115],[603,152],[445,159],[302,764],[236,184],[69,82],[649,138],[333,1158],[415,270],[231,145],[314,187],[319,319],[2686,133],[262,189],[285,404],[322,165],[441,151],[180,145],[597,142],[845,185],[567,169],[405,188],[231,163],[680,185],[345,272],[2509,161],[456,181],[304,107],[721,1078],[608,111],[354,507],[715,168],[515,114],[236,2472],[270,103],[350,977],[265,156],[469,138],[236,180],[412,157],[232,146],[114,268],[234,2632],[404,395],[262,128],[105,361],[963,422],[1880,172],[285,335],[336,111],[917,343],[339,100],[926,587],[1707,541],[97,993],[499,1474],[1882,107],[98,304],[848,1276],[301,169],[404,1712],[524,56],[586,1669],[583,332],[905,166],[1396,156],[982,2122],[732,140],[456,155],[405,145],[2255,2397],[498,162],[581,144],[1229,191],[99,579],[860,173],[602,115],[292,131],[694,135],[609,170],[513,175],[1056,137],[654,172],[299,268],[308,163],[280,138],[546,176],[1374,340],[2695,1884],[867,111],[270,111],[1502,698],[98,289],[620,101],[232,155],[687,138],[1012,154],[1301,564],[345,111],[97,122],[436,166],[486,276],[304,115],[604,141],[470,154],[501,134],[409,149],[654,191],[515,1733],[409,144],[1111,164],[784,101],[649,140],[1032,137],[1436,157],[230,134],[297,149],[955,176],[1067,180],[463,276],[922,538],[728,819],[263,144],[362,1132],[855,553],[1036,910],[1466,920],[1379,1253],[388,345],[528,110],[328,143],[1273,390],[2833,1454],[443,129],[299,330],[998,132],[504,162],[654,141],[3204,2063],[265,132],[674,181],[1346,111],[373,187],[1061,490],[281,533],[401,111],[112,117],[269,616],[265,155],[2492,185],[567,160],[32,415],[3151,1585],[687,190],[2660,1440],[438,159],[1485,168],[54,48],[427,136],[443,165],[403,180],[404,1280],[845,168],[52,48],[567,176],[671,138],[337,432],[1180,191],[348,354],[465,272],[341,289],[1504,173],[1564,422],[465,683],[1155,272],[412,164],[110,97],[590,190],[508,101],[2385,457],[623,276],[1090,2947],[2567,176],[272,400],[1002,132],[262,182],[692,270],[838,450],[308,157],[522,737],[353,121],[281,304],[235,2212],[299,345],[775,148],[368,288],[403,166],[392,163],[539,176],[287,99],[762,272],[322,190],[594,148],[685,98],[265,154],[700,599],[525,161],[1264,562],[280,185],[1284,140],[334,2402],[331,1046],[352,382],[555,162],[321,109],[285,521],[543,137],[1915,128],[388,462],[355,157],[311,97],[102,521],[405,151],[1018,132],[336,1100],[332,699],[230,190],[687,143],[484,1280],[1324,607],[1643,1826],[436,140],[838,559],[269,1011],[346,191],[836,1898],[867,268],[515,287],[318,163],[1145,155],[600,144],[366,304],[319,1179],[685,97],[1866,140],[386,1544],[1556,156],[1773,183],[65,110],[367,1371],[2153,642],[314,146],[1050,137],[318,154],[1293,1183],[1018,153],[286,157],[547,130],[717,188],[1146,159],[866,178],[293,101],[1277,1188],[105,305],[679,582],[314,164],[546,128],[538,116],[792,141],[265,142],[774,343],[296,1593],[1584,131],[554,179],[1511,272],[281,2587],[289,97],[937,182],[287,122],[797,120],[586,2720],[1025,10],[539,189],[503,103],[56,48],[881,168],[844,640],[49,647],[108,564],[233,188],[875,1567],[701,1138],[912,111],[345,115],[337,556],[1445,184],[233,141],[458,180],[485,108],[97,572],[300,872],[524,53],[1014,1144],[524,50],[340,770],[353,115],[500,169],[403,157],[1253,2061],[369,109],[429,160],[451,115],[660,155],[837,390],[1070,979],[434,1457],[633,168],[302,325],[236,188],[2186,1849],[1141,137],[550,133],[1050,149],[304,772],[646,375],[314,153],[508,114],[987,277],[32,570],[1152,2197],[373,148],[273,102],[281,274],[1374,354],[389,149],[117,321],[1088,390],[657,189],[406,151],[128,357],[296,311],[359,142],[1445,168],[112,634],[689,173],[515,851],[287,111],[600,129],[306,432],[732,166],[765,116],[32,519],[754,274],[903,97],[678,144],[1018,168],[833,531],[672,101],[438,182],[327,148],[1023,148],[73,78],[739,163],[296,109],[432,468],[331,634],[1012,141],[1408,3254],[1216,171],[292,179],[636,172],[79,78],[584,101],[361,119],[685,311],[1014,720],[1188,3190],[2272,661],[331,952],[1526,856],[1042,1638],[308,136],[355,176],[339,103],[336,634],[441,156],[687,156],[333,985],[278,959],[312,116],[373,136],[1945,157],[119,111],[717,185],[645,184],[775,176],[1265,658],[901,115],[1106,1090],[1145,135],[334,99],[348,383],[281,321],[1581,140],[842,391],[1064,166],[281,1562],[280,136],[368,3145],[993,111],[751,132],[2275,907],[332,2175],[620,270],[491,141],[793,992],[635,136],[687,168],[589,883],[878,152],[312,117],[619,138],[1124,1030],[295,187],[791,2894],[301,168],[1465,1767],[399,1143],[1121,761],[416,145],[695,884],[320,395],[336,272],[1538,1131],[359,176],[784,111],[1956,105],[628,304],[774,365],[1003,161],[1479,128],[298,143],[361,912],[267,187],[317,112],[573,1058],[1425,188],[1184,156],[229,171],[539,132],[921,120],[2037,100],[605,140],[108,781],[735,163],[1172,135],[2252,270],[312,289],[1024,268],[368,1851],[118,296],[945,178],[461,161],[953,147],[635,170],[300,102],[266,131],[489,169],[108,307],[32,578],[116,121],[2256,152],[389,135],[654,133],[805,188],[654,155],[401,112],[331,692],[734,177],[1600,180],[651,173],[349,169],[1036,1136],[1412,2035],[376,1176],[229,154],[380,186],[466,158],[839,2319],[973,178],[3072,2086],[706,110],[1260,148],[284,174],[336,305],[233,144],[237,157],[112,2414],[232,148],[382,1639],[341,608],[32,1122],[122,270],[297,141],[356,305],[636,155],[1159,385],[1443,386],[578,890],[236,148],[261,144],[590,178],[337,1179],[1233,156],[324,99],[341,851],[226,153],[410,173],[436,161],[827,1951],[1909,155],[270,268],[97,395],[76,101],[306,730],[312,268],[557,134],[1074,1465],[299,1887],[445,146],[509,115],[1229,161],[1510,137],[879,487],[368,111],[546,175],[1126,162],[652,104],[1943,188],[298,174],[444,184],[656,140],[233,139],[620,268],[1203,188],[1530,162],[261,178],[820,276],[999,927],[69,108],[103,108],[1801,3302],[359,158],[474,151],[1380,365],[449,155],[732,162],[913,675],[97,312],[621,272],[1224,181],[355,142],[280,168],[261,150],[269,289],[334,764],[1398,173],[389,160],[118,304],[514,120],[1130,357],[483,168],[104,293],[2885,164],[56,56],[291,922],[295,184],[229,180],[466,164],[1096,48],[339,99],[1891,296],[195,160],[306,367],[2016,2434],[65,78],[875,3198],[501,132],[615,580],[358,184],[267,136],[293,2708],[297,133],[497,130],[357,1134],[866,147],[79,83],[627,142],[232,184],[1076,184],[261,142],[1705,793],[235,150],[710,173],[356,111],[392,151],[604,175],[935,184],[364,121],[414,190],[1216,145],[503,274],[394,173],[478,190],[345,1421],[103,97],[350,1721],[951,141],[1024,270],[1501,1942],[1184,157],[2051,3669],[1278,187],[120,428],[678,187],[482,187],[942,171],[299,395],[663,178],[109,272],[296,1533],[353,116],[557,139],[1197,101],[792,191],[270,116],[427,145],[645,160],[335,97],[388,330],[472,108],[99,121],[893,172],[740,175],[374,185],[183,168],[1710,178],[452,102],[2350,182],[1966,137],[284,154],[1344,2615],[2058,2336],[645,168],[650,276],[32,1088],[1028,343],[1053,163],[236,187],[287,300],[333,1293],[373,180],[953,144],[606,186],[283,856],[582,698],[578,111],[921,102],[83,69],[648,1951],[2096,1355],[57,48],[281,317],[485,114],[520,188],[661,480],[600,182],[327,132],[597,139],[738,97],[753,101],[115,115],[137,180],[579,118],[734,128],[1699,101],[1980,385],[609,171],[649,189],[1277,1325],[467,101],[534,188],[104,288],[278,117],[609,176],[1258,168],[1472,1604],[458,134],[857,107],[1045,2581],[299,451],[1224,132],[637,111],[782,129],[1945,136],[1664,176],[103,451],[476,152],[420,523],[1093,1251],[791,1536],[342,130],[292,140],[937,190],[1332,1276],[1033,184],[106,451],[913,2081],[955,137],[819,376],[1778,1187],[362,157],[956,430],[177,132],[713,1460],[873,1647],[299,121],[477,593],[595,3412],[1315,1347],[754,428],[1424,274],[80,76],[707,140],[1770,148],[355,137],[368,425],[533,103],[304,2151],[333,1019],[518,1587],[786,57],[341,1796],[269,437],[802,189],[844,457],[32,88],[278,1113],[2595,147],[863,462],[373,143],[864,145],[106,111],[1099,169],[416,159],[556,354],[375,505],[1260,180],[1762,129],[380,176],[274,272],[294,118],[785,1877],[654,169],[665,189],[577,159],[2168,119],[3417,1207],[586,2149],[728,1819],[690,140],[982,507],[352,480],[1324,705],[1978,177],[607,1707],[625,114],[2655,370],[536,160],[802,187],[281,99],[320,970],[2627,304],[280,2212],[776,827],[301,173],[353,97],[483,153],[537,112],[515,268],[586,1768],[114,1185],[1453,172],[293,100],[456,156],[284,137],[663,190],[484,885],[273,119],[340,730],[402,378],[2778,142],[100,97],[69,78],[261,182],[1216,151],[561,177],[299,381],[419,1491],[702,927],[1024,442],[1274,1014],[273,1546],[423,98],[838,417],[1917,307],[268,1440],[423,381],[1061,766],[285,1071],[236,2743],[285,579],[316,172],[524,49],[2218,311],[359,148],[466,130],[402,451],[555,157],[2861,3949],[833,1708],[232,164],[740,178],[1175,276],[103,289],[489,132],[112,277],[292,171],[280,184],[426,187],[324,108],[358,133],[1288,1302],[3527,3088],[433,163],[776,648],[262,180],[604,152],[1008,168],[109,289],[354,367],[303,158],[277,110],[639,1217],[389,191],[344,1607],[293,345],[806,102],[947,1359],[1214,1487],[1531,110],[343,453],[1095,2325],[401,117],[662,130],[1611,186],[1840,626],[454,102],[900,700],[428,335],[1127,2105],[99,422],[2482,310],[785,1158],[97,102],[346,185],[421,833],[2232,3935],[460,184],[695,348],[498,136],[611,566],[880,136],[423,100],[1506,1463],[284,145],[643,2306],[333,593],[380,173],[551,108],[1295,347],[786,55],[32,1554],[233,132],[2475,122],[32,2264],[329,183],[299,304],[2291,146],[117,400],[462,335],[2291,189],[490,276],[2267,161],[908,384],[2726,1995],[291,1804],[718,156],[50,54],[458,136],[1488,1417],[288,110],[740,155],[266,190],[584,395],[1608,159],[3425,1449],[999,101],[501,173],[576,146],[524,54],[322,160],[388,296],[239,190],[651,159],[73,67],[2098,861],[864,128],[1299,588],[284,191],[497,131],[1300,1610],[2671,270],[582,496],[333,1776],[710,180],[863,396],[320,3345],[227,133],[315,179],[315,153],[619,162],[1040,154],[500,160],[735,182],[2092,1092],[3596,2009],[410,138],[1246,983],[514,683],[678,130],[285,396],[281,592],[987,1888],[2604,1864],[1286,3339],[692,115],[386,333],[923,116],[102,706],[408,136],[963,119],[1166,1638],[1453,128],[109,98],[373,160],[715,185],[328,178],[55,48],[107,268],[109,287],[296,296],[2814,128],[1242,347],[235,168],[1040,144],[1072,395],[513,155],[410,143],[837,272],[2636,939],[336,381],[1854,188],[645,172],[272,311],[483,170],[447,118],[316,180],[1181,780],[2200,141],[700,2970],[591,145],[1629,128],[311,116],[1362,1548],[845,172],[1012,135],[2944,162],[103,1240],[404,335],[1152,2249],[116,361],[368,397],[1900,150],[416,155],[2847,2228],[266,165],[273,325],[291,289],[705,1547],[1333,1714],[1915,183],[1152,2353],[114,390],[1810,3774],[657,160],[833,2927],[565,178],[1414,360],[666,1115],[237,154],[958,83],[1769,168],[103,304],[458,135],[1123,97],[105,2077],[3395,1062],[388,851],[324,579],[562,861],[1088,101],[1314,116],[1414,365],[635,158],[285,2880],[359,156],[2069,136],[103,378],[483,150],[433,145],[669,991],[283,317],[388,2601],[892,2478],[1684,109],[434,2345],[649,154],[1286,3586],[754,330],[931,287],[1034,136],[1611,150],[2423,1420],[388,111],[728,1158],[32,465],[651,167],[2303,177],[483,182],[1481,796],[443,149],[660,180],[65,83],[1530,175],[2294,184],[1197,422],[299,521],[320,3215],[1510,175],[1022,169],[85,110],[446,152],[456,160],[1268,1714],[2200,153],[3156,475],[109,307],[1380,677],[263,147],[1736,818],[1811,399],[607,720],[614,2844],[746,145],[586,321],[489,189],[551,109],[645,161],[1111,156],[235,185],[331,2805],[1288,352],[598,110],[622,585],[955,188],[281,272],[410,131],[568,137],[122,268],[496,2147],[786,56],[164,132],[1291,2372],[1981,268],[117,268],[868,107],[50,56],[867,1740],[1022,187],[1195,151],[2457,111],[65,108],[922,415],[483,132],[582,720],[1506,1455],[748,930],[2224,311],[3099,1922],[583,695],[278,437],[405,180],[403,168],[937,134],[289,1142],[1444,312],[1581,144],[758,151],[1973,164],[814,311],[276,421],[1616,1004],[393,509],[2090,180],[303,187],[308,133],[327,188],[1051,183],[359,183],[1125,3495],[2827,810],[332,1371],[308,156],[368,652],[425,100],[801,2115],[1252,531],[1592,268],[755,186],[554,153],[1077,1038],[867,851],[873,356],[1233,148],[627,177],[115,105],[1299,2842],[106,288],[751,145],[1022,133],[347,2929],[516,116],[831,132],[1438,585],[311,270],[2119,818],[1629,184],[232,156],[1040,128],[1880,137],[119,288],[491,136],[3022,159],[1693,1604],[116,116],[798,179],[567,131],[1411,347],[233,143],[291,1357],[296,1389],[354,1241],[262,178],[577,165],[899,163],[409,157],[276,691],[375,986],[635,137],[285,3071],[57,57],[320,293],[358,171],[1561,4168],[286,171],[745,268],[829,98],[68,101],[268,97],[453,337],[1882,1439],[50,55],[295,180],[975,941],[273,939],[557,154],[312,307],[2320,111],[99,289],[625,288],[1020,183],[1581,133],[452,99],[1479,174],[331,330],[373,177],[532,108],[435,2218],[943,114],[980,289],[668,176],[1328,2072],[1242,386],[233,185],[380,180],[367,2522],[269,2515],[2002,838],[1497,156],[859,351],[524,52],[878,169],[2568,4320],[472,111],[542,115],[1210,1304],[2709,130],[2898,133],[2300,141],[3175,121],[321,116],[73,83],[327,129],[919,382],[526,144],[148,352],[1008,138],[524,55],[434,325],[775,149],[106,634],[278,2471],[402,289],[507,319],[539,188],[849,165],[896,902],[336,437],[283,339],[1425,140],[2738,1864],[284,158],[361,108],[831,168],[1529,101],[1621,143],[2672,146],[402,117],[524,647],[984,642],[278,1100],[396,272],[3119,140],[1151,114],[1564,3303],[269,723],[420,1460],[284,178],[573,1472],[961,993],[100,344],[524,51],[868,962],[1909,157],[491,165],[969,133],[2294,152],[275,135],[4022,507],[754,324],[1679,1004],[1587,487],[484,1038],[485,288],[105,289],[3439,564],[586,1985],[79,82],[104,851],[514,2758],[323,172],[331,396],[319,556],[489,179],[1952,101],[444,173],[98,538],[235,129],[357,983],[531,1219],[882,112],[1442,1779],[285,1148],[483,178],[345,116],[1917,723],[3043,2196],[32,81],[395,115],[669,112],[378,103],[1232,1534],[314,184],[514,274],[2005,865],[1075,385],[112,620],[116,304],[1309,2648],[402,108],[478,140],[350,683],[121,112],[474,152],[1519,1548],[2014,180],[327,189],[390,305],[98,115],[356,1888],[917,365],[323,151],[348,432],[748,1338],[394,174],[413,136],[280,129],[342,177],[685,1078],[2553,559],[376,1682],[2467,1667],[872,378],[278,317],[495,101],[1710,168],[270,99],[1358,695],[929,344],[548,272],[633,160],[2994,692],[386,1158],[466,172],[595,2638],[346,170],[659,1662],[900,990],[2136,184],[1641,382],[314,186],[842,886],[1465,2282],[358,149],[1617,173],[3524,2769],[415,307],[288,103],[348,3912],[65,84],[1973,136],[430,3711],[687,170],[308,161],[586,112],[774,879],[888,378],[2001,182],[235,131],[263,142],[1309,2162],[521,894],[268,830],[342,148],[667,1191],[908,332],[341,2144],[1993,121],[3008,2356],[579,97],[580,1683],[686,158],[265,164],[295,181],[391,813],[663,139],[1058,640],[345,122],[434,109],[966,101],[903,304],[1094,116],[362,3116],[943,110],[3904,4571],[390,307],[678,136],[1404,121],[1632,172],[546,152],[472,289],[1233,152],[287,107],[550,141],[796,1134],[336,296],[32,120],[280,161],[330,109],[434,304],[912,115],[1180,147],[581,132],[110,1976],[2659,136],[262,174],[3266,139],[434,102],[744,116],[1617,152],[1167,186],[390,272],[445,141],[705,700],[278,361],[1163,2666],[2504,541],[1550,163],[1850,400],[73,68],[678,134],[868,274],[236,3039],[624,181],[1061,1037],[407,643],[3538,582],[444,178],[484,101],[1621,157],[391,457],[1694,1062],[328,166],[336,353],[455,186],[1997,166],[83,101],[1583,159],[360,982],[843,307],[300,114],[1154,111],[1311,177],[53,647],[350,102],[97,120],[660,161],[386,1616],[518,3943],[345,288],[498,137],[32,528],[468,306],[358,188],[235,180],[466,138],[546,139],[810,480],[1167,139],[2395,4196],[501,190],[456,176],[640,832],[800,639],[887,986],[746,157],[97,1209],[456,172],[1214,357],[109,305],[327,154],[285,2144],[878,156],[1008,180],[1225,145],[104,2018],[2455,894],[364,115],[275,134],[565,173],[626,121],[50,57],[2939,356],[67,72],[651,162],[1189,2301],[2425,1114],[546,159],[108,270],[1517,479],[1968,148],[424,159],[355,179],[98,395],[371,162],[474,147],[550,128],[236,163],[565,172],[1285,99],[287,637],[985,817],[68,397],[907,276],[880,168],[287,105],[308,174],[362,151],[468,1179],[552,185],[281,3592],[302,117],[1400,607],[2793,559],[103,353],[3210,136],[602,339],[732,170],[4104,335],[619,167],[710,133],[276,711],[452,830],[263,143],[294,3551],[1278,129],[726,528],[1130,1688],[782,145],[289,528],[368,2150],[375,812],[448,261],[1344,2654],[586,118],[831,131],[105,2298],[280,166],[240,159],[516,321],[764,110],[2236,4285],[411,144],[1066,1420],[1410,1902],[283,3838],[339,116],[599,3689],[929,121],[1295,386],[348,306],[1229,145],[341,396],[631,806],[706,2182],[638,144],[590,191],[765,114],[1986,133],[778,137],[610,189],[1095,865],[908,383],[48,53],[306,2302],[516,903],[638,157],[621,110],[436,183],[234,3189],[336,2308],[416,164],[1523,110],[708,172],[73,84],[346,131],[790,725],[2090,176],[507,494],[740,158],[233,142],[596,272],[2070,1218],[550,167],[3389,562],[705,743],[105,2628],[604,167],[625,108],[269,114],[420,978],[111,118],[605,149],[503,1827],[100,114],[112,112],[235,134],[422,270],[616,305],[749,179],[903,111],[1407,4638],[318,178],[368,105],[469,130],[923,268],[342,144],[504,176],[1138,811],[507,2003],[661,382],[1353,595],[280,157],[302,118],[591,175],[1240,268],[1376,1011],[283,1675],[408,184],[840,2561],[735,160],[957,150],[4538,716],[406,182],[970,97],[350,99],[472,114],[1866,144],[2130,1016],[116,370],[354,2539],[606,170],[414,175],[803,760],[1631,187],[98,2367],[522,399],[906,53],[1979,1063],[1565,151],[677,453],[402,270],[463,1228],[304,116],[962,312],[2039,174],[396,110],[82,101],[748,650],[336,289],[420,352],[426,143],[924,348],[1012,138],[520,185],[414,159],[719,2375],[443,170],[233,174],[449,153],[794,376],[1186,1159],[3323,1742],[299,353],[452,843],[307,115],[715,135],[117,353],[532,114],[710,144],[1135,184],[1018,1637],[501,187],[1569,3180],[1652,330],[291,428],[299,442],[1341,2147],[933,2449],[786,54],[444,128],[1617,180],[460,143],[853,157],[802,175],[945,135],[226,145],[947,1601],[438,171],[513,178],[1001,144],[1204,162],[278,3199],[522,1159],[1074,1194],[329,151],[292,142],[1050,170],[102,1150],[278,3564],[292,165],[441,152],[104,105],[745,4089],[1242,505],[639,730],[2313,140],[2320,97],[111,425],[1093,1068],[306,659],[2240,143],[392,155],[1061,571],[348,453],[104,1456],[619,178],[807,166],[2036,1630],[3178,766],[233,145],[937,178],[104,289],[350,321],[1240,101],[194,191],[233,158],[1092,811],[2115,335],[3020,1365],[69,115],[514,109],[642,1579],[1124,1082],[1048,1941],[727,272],[653,185],[339,118],[1278,174],[1029,505],[51,53],[581,129],[848,1144],[1796,2396],[1805,148],[69,68],[953,187],[102,2363],[452,533],[666,1105],[1434,902],[1530,178],[323,137],[726,381],[424,177],[1169,187],[484,98],[1565,180],[1168,132],[65,76],[116,288],[458,167],[233,184],[635,185],[458,156],[652,111],[897,101],[1445,190],[551,102],[610,158],[622,1179],[751,157],[926,818],[1189,1489],[717,178],[2590,898],[283,652],[375,544],[588,151],[622,354],[748,675],[508,2471],[231,152],[1130,847],[1281,941],[813,842],[1668,890],[116,277],[378,274],[565,176],[483,183],[2394,879],[3041,184],[469,142],[291,397],[1303,144],[1398,156],[1621,165],[1742,304],[621,268],[961,99],[4563,479],[687,142],[1565,150],[2673,769],[447,115],[672,111],[1543,2144],[452,2571],[990,599],[908,453],[4316,293],[100,1835],[325,270],[435,97],[758,170],[3200,1100],[109,108],[498,161],[520,149],[550,153],[622,360],[436,178],[810,382],[1139,352],[404,112],[604,187],[607,799],[944,988],[1272,115],[4211,883],[2052,1807],[391,306],[646,386],[853,182],[3238,583],[3282,760],[469,170],[402,272],[1964,2072],[495,287],[322,168],[2023,1971],[388,688],[67,330],[471,2152],[589,1281],[267,2743],[1782,4328],[2532,480],[324,103],[1291,1636],[2350,176],[990,2188],[1576,3021],[1444,1209],[471,471],[1215,1210],[332,306],[3170,310],[557,191],[373,152],[1713,136],[3176,160],[3288,691],[347,385],[777,136],[1456,111],[406,150],[416,163],[694,180],[287,1673],[447,103],[3499,304],[4740,172],[327,138],[1334,148],[603,157],[759,1208],[1308,109],[1299,640],[377,468],[928,1106],[966,2363],[48,51],[1127,4564],[721,112],[899,161],[381,101],[498,140],[620,293],[485,293],[390,115],[901,111],[313,141],[937,173],[762,1709],[299,296],[489,184],[817,276],[862,144],[903,115],[1542,541],[283,345],[614,396],[1614,607],[2932,147],[236,189],[321,396],[1197,117],[119,115],[268,103],[594,145],[2102,3333],[319,1005],[281,451],[2327,487],[1361,1395],[589,1057],[746,179],[687,148],[97,300],[483,133],[1952,270],[2160,650],[345,268],[677,1217],[1424,378],[1739,737],[1831,3203],[2568,509],[523,382],[790,333],[2208,4232],[3306,335],[312,272],[313,136],[331,1043],[1651,691],[273,885],[388,344],[455,137],[304,296],[447,100],[790,989],[298,162],[3134,157],[109,109],[287,101],[48,52],[118,422],[1174,171],[625,2273],[311,111],[603,149],[118,397],[2300,156],[350,3720],[388,1272],[1334,149],[1303,141],[117,122],[1325,4640],[785,1687],[1028,365],[1215,1104],[579,620],[926,650],[2549,1508],[1130,559],[1045,2817],[2681,172],[284,166],[688,509],[1182,1187],[750,147],[1477,153],[314,128],[577,158],[300,116],[498,170],[875,3064],[950,1139],[877,1159],[1205,111],[1907,983],[147,573],[378,400],[2373,773],[3970,723],[98,97],[299,578],[362,2101],[276,3691],[345,307],[754,542],[269,467],[551,115],[1269,183],[320,304],[390,287],[1151,4170],[1455,1187],[294,1879],[2267,169],[2847,4198],[374,167],[557,130],[281,1718],[660,143],[2126,1801],[1609,3478],[664,133],[229,172],[281,584],[299,288],[346,135],[531,276],[961,872],[1163,2284],[428,1712],[353,111],[339,120],[2014,152],[697,148],[408,145],[1130,417],[1543,2259],[1872,910],[3657,180],[605,161],[665,163],[299,4323],[800,2578],[1441,184],[1770,154],[1590,1807],[1713,149],[269,330],[299,3464],[777,129],[980,1995],[548,305],[584,1785],[3662,897],[940,1131],[350,2349],[700,1097],[405,138],[1219,861],[4100,5189],[115,3153],[1183,1193],[236,1936],[495,277],[101,764],[296,300],[308,191],[452,110],[2114,465],[128,475],[931,1977],[563,142],[1169,151],[1590,2217],[1908,696],[1339,166],[770,1662],[1289,154],[1055,1761],[299,1044],[374,152],[594,134],[1197,370],[2754,117],[661,566],[52,53],[233,178],[362,154],[533,312],[748,421],[932,151],[1551,1735],[1752,175],[2189,391],[434,344],[4861,174],[303,162],[1459,136],[522,276],[616,115],[65,80],[769,553],[1072,116],[1463,1660],[1641,760],[655,884],[104,307],[510,191],[754,1160],[1113,272],[121,110],[69,76],[415,305],[624,184],[1191,812],[267,3817],[331,317],[2751,496],[265,167],[499,296],[518,487],[758,178],[761,276],[1689,101],[1256,276],[877,1731],[1362,4967],[531,1502],[699,319],[1206,2358],[726,570],[762,2177],[1141,151],[1193,553],[1442,121],[1646,289],[281,1185],[411,179],[1356,115],[105,824],[2659,157],[1250,769],[273,1814],[3147,352],[289,311],[442,837],[645,169],[674,154],[1360,2169],[627,179],[1000,475],[1342,1139],[122,117],[1553,1333],[107,451],[956,1471],[381,843],[627,191],[656,128],[98,378],[119,110],[1327,117],[2409,895],[2925,173],[341,1113],[302,1856],[349,129],[441,153],[790,930],[289,804],[333,1819],[355,190],[405,184],[2657,1623],[48,55],[320,572],[729,1283],[785,333],[1577,135],[2124,1461],[319,911],[405,157],[428,274],[972,925],[1413,4838],[300,927],[403,167],[708,157],[273,936],[903,305],[2477,887],[261,181],[277,97],[489,136],[550,175],[4815,188],[1382,97],[4192,141],[645,188],[3842,407],[128,417],[1834,146],[741,386],[296,830],[362,158],[373,171],[405,176],[1413,3229],[1468,100],[3390,287],[114,397],[336,277],[635,146],[2971,696],[65,77],[912,268],[1629,181],[1261,134],[2136,132],[3113,1175],[110,578],[688,272],[65,115],[629,187],[357,523],[314,183],[963,289],[103,116],[350,118],[1381,55],[1477,174],[712,4468],[1512,2122],[1676,509],[1763,2755],[314,154],[375,553],[331,105],[454,783],[517,133],[278,330],[323,155],[1054,779],[1960,3796],[281,960],[285,272],[464,140],[931,723],[1560,148],[189,883],[289,109],[388,288],[269,2749],[308,151],[434,537],[48,49],[495,274],[1268,1292],[2920,148],[1126,167],[1246,480],[843,305],[364,110],[721,404],[2179,2243],[336,442],[587,1200],[2032,169],[289,115],[318,156],[985,737],[1177,324],[1405,925],[51,54],[116,307],[611,643],[1208,2984],[406,165],[474,177],[1081,181],[226,148],[446,132],[495,312],[767,189],[1224,145],[598,305],[291,888],[1900,175],[111,311],[720,399],[752,1489],[1050,184],[1140,1138],[65,68],[270,558],[105,462],[386,276],[785,347],[795,3132],[2616,691],[3829,780],[141,188],[274,509],[4834,1955],[572,111],[694,143],[1618,137],[765,902],[951,159],[1261,170],[388,564],[392,165],[653,187],[285,1699],[465,277],[788,566],[864,156],[268,305],[3126,2802],[375,2705],[522,1082],[394,157],[281,353],[405,164],[906,52],[424,130],[597,170],[605,178],[648,1070],[1216,190],[1166,1030],[2303,140],[3257,144],[51,49],[310,977],[388,2455],[733,276],[4431,1724],[475,1844],[873,4603],[423,1196],[584,97],[361,1196],[416,161],[586,1582],[1398,141],[97,396],[580,1238],[1402,698],[732,179],[1195,165],[2754,1205],[3903,117],[442,939],[986,1017],[2834,608],[3668,172],[261,156],[523,480],[1462,48],[1230,1604],[1628,776],[1380,2143],[2289,3690],[3660,897],[1411,386],[2854,858],[232,149],[397,109],[2423,1826],[375,667],[484,310],[594,139],[639,453],[3022,160],[3419,769],[583,659],[3561,339],[2647,357],[554,137],[107,108],[940,3558],[281,542],[2347,343],[680,183],[335,114],[433,162],[598,1526],[1225,184],[1546,111],[2779,306],[281,356],[621,442],[3181,158],[236,179],[262,134],[103,274],[287,533],[438,169],[315,155],[1832,288],[631,99],[548,270],[838,2468],[535,133],[458,157],[391,2140],[3051,288],[728,1541],[929,2538],[1265,892],[2224,370],[291,616],[1039,1038],[1163,1021],[53,53],[229,512],[237,3116],[378,321],[144,352],[266,129],[818,773],[320,2860],[389,173],[993,97],[331,305],[455,168],[631,1562],[518,518],[1012,148],[1606,397],[1805,188],[410,171],[728,347],[4600,156],[1117,115],[2441,780],[951,162],[3338,337],[607,658],[294,102],[434,2708],[500,136],[2380,693],[3836,521],[309,3189],[3579,132],[195,174],[356,108],[1429,98],[1632,128],[3038,1332],[416,152],[691,895],[744,3625],[281,107],[453,487],[652,1603],[657,161],[1524,1623],[384,730],[445,164],[78,111],[963,2654],[1933,1942],[690,175],[3048,99],[1053,170],[513,167],[2303,180],[3835,108],[261,165],[1098,139],[1265,5644],[118,105],[1091,129],[693,1143],[293,519],[484,1432],[577,182],[1020,187],[1652,843],[1943,156],[396,311],[1773,171],[375,1200],[893,168],[267,3895],[272,321],[1436,176],[2630,720],[3344,101],[327,164],[920,1571],[1482,99],[572,272],[877,531],[1300,1958],[280,150],[331,3046],[394,129],[423,99],[582,780],[1997,132],[525,168],[1403,381],[67,277],[320,378],[708,182],[4352,117],[564,101],[1143,2882],[3368,3541],[3476,677],[266,180],[326,312],[423,73],[296,509],[297,153],[726,1594],[888,339],[1523,1273],[1710,142],[375,769],[436,144],[274,1837],[327,190],[1530,132],[345,296],[561,157],[584,288],[1268,887],[32,538],[263,133],[973,137],[2413,599],[285,1720],[364,428],[438,132],[3688,170],[334,1078],[1384,3735],[1767,3423],[2681,176],[3089,1686],[267,2472],[350,2597],[526,145],[937,184],[2137,178],[102,268],[308,135],[1560,145],[274,578],[368,465],[1143,2154],[1048,1118],[1492,5254],[1224,128],[2137,160],[291,781],[1762,152],[2574,188],[109,422],[278,293],[451,270],[485,2414],[423,1640],[1327,572],[48,50],[263,141],[642,575],[1215,842],[1123,288],[330,2018],[363,152],[3540,875],[366,3355],[3667,181],[503,101],[362,133],[3537,578],[283,272],[467,274],[1789,268],[2390,1613],[97,558],[117,312],[263,149],[84,2375],[743,276],[705,815],[3681,168],[80,104],[263,148],[267,3039],[306,340],[371,151],[779,545],[1644,2035],[2691,1221],[2826,1280],[845,180],[102,111],[783,115],[1379,833],[3566,1692],[302,1046],[335,361],[404,3313],[1893,276],[645,180],[800,518],[861,1437],[1339,179],[276,650],[567,172],[762,97],[607,1194],[551,902],[750,185],[341,2458],[273,4827],[3074,352],[434,1046],[504,161],[1308,2846],[1765,306],[1777,701],[2384,2769],[886,640],[312,422],[376,1589],[717,147],[728,333],[2216,1218],[499,268],[319,487],[412,188],[2377,1305],[589,910],[1871,129],[360,468],[503,381],[2213,2641],[69,84],[1718,692],[1283,356],[285,570],[355,161],[396,103],[555,164],[576,142],[1032,163],[1235,457],[302,1669],[1341,3092],[273,3614],[423,109],[533,400],[754,1797],[1190,140],[2649,3378],[1190,133],[84,104],[295,176],[373,150],[2057,681],[388,723],[388,4347],[885,578],[982,1732],[447,3206],[775,177],[786,53],[1820,151],[368,1806],[770,556],[611,407],[85,83],[375,1140],[579,111],[1326,1400],[4047,1004],[391,3351],[300,1196],[348,622],[391,1058],[1793,306],[291,1072],[1020,161],[48,54],[523,566],[865,417],[671,166],[285,465],[984,2798],[285,317],[1040,151],[2276,1173],[3269,101],[119,2008],[1066,1158],[1117,2360],[1649,2674],[614,344],[1480,175],[2963,116],[280,160],[289,277],[895,1509],[1556,169],[398,157],[572,115],[864,168],[1075,1065],[507,911],[515,2507],[672,270],[3984,307],[269,404],[281,876],[419,111],[993,307],[2845,907],[349,146],[351,351],[368,922],[615,1557],[665,158],[2145,1102],[1047,3847],[1074,5115],[3537,272],[532,270],[550,171],[972,2393],[1149,274],[2154,733],[291,5571],[339,519],[425,115],[500,153],[1164,417],[1308,1432],[1453,151],[1735,811],[99,339],[433,143],[503,99],[1609,2733],[2891,1533],[283,304],[1318,98],[267,147],[291,4203],[232,185],[235,3854],[377,695],[1966,136],[291,2546],[799,399],[1381,54],[2020,156],[934,1433],[916,157],[2558,3223],[109,312],[362,146],[1713,132],[2371,4331],[2378,693],[235,136],[386,376],[662,133],[1448,5029],[3246,2191],[159,129],[457,573],[3194,168],[100,390],[276,3841],[2121,99],[2293,340],[2893,110],[350,2520],[354,365],[1600,184],[2014,156],[99,353],[274,268],[277,321],[2064,156],[496,941],[535,160],[815,2066],[1719,118],[80,361],[421,1016],[4278,272],[2932,146],[531,2185],[598,2324],[572,122],[1854,140],[1954,382],[1001,169],[76,272],[305,104],[473,173],[1340,2839],[111,105],[693,1281],[2322,3557],[401,104],[1422,312],[2713,2169],[341,723],[652,115],[4372,494],[293,103],[379,149],[467,270],[639,1089],[366,270],[1047,1520],[1441,149],[1311,130],[268,122],[387,179],[1822,573],[1929,761],[1419,176],[266,176],[727,99],[1636,3429],[48,57],[233,179],[1284,142],[336,2790],[331,277],[353,100],[490,811],[703,1644],[718,143],[1261,144],[320,5169],[496,276],[866,129],[980,2977],[2267,167],[321,901],[1076,128],[434,100],[891,288],[283,3855],[427,184],[474,189],[886,832],[264,156],[283,816],[862,191],[2998,382],[79,77],[4744,1720],[281,108],[594,190],[3818,769],[320,1043],[674,188],[753,692],[1135,128],[2457,307],[2756,724],[761,541],[1236,112],[2199,4947],[312,288],[718,129],[2925,135],[4213,1647],[466,143],[482,157],[501,162],[469,188],[779,382],[1232,673],[114,288],[300,537],[860,160],[887,1065],[716,276],[2006,120],[593,276],[1497,160],[2574,156],[3734,185],[521,111],[929,317],[1175,1075],[2001,129],[2765,1903],[51,51],[156,180],[345,1603],[810,696],[2410,128],[423,2763],[670,339],[1381,57],[2600,4217],[544,347],[649,150],[745,104],[1851,307],[477,725],[951,189],[1871,184],[528,108],[580,1612],[1935,1520],[364,307],[422,101],[883,989],[1956,912],[2208,2918],[5194,1317],[236,5971],[85,82],[503,120],[958,109],[1202,2035],[5920,917],[80,83],[436,169],[573,813],[909,2673],[1576,3364],[273,106],[387,134],[1054,2902],[2203,304],[102,1046],[3762,156],[281,1742],[855,855],[1448,4589],[2898,134],[3461,147],[5080,998],[356,307],[538,277],[1444,3674],[805,179],[900,1222],[1090,1501],[2495,2198],[2813,587],[3398,179],[97,2046],[364,300],[2342,1302],[267,186],[366,353],[462,101],[942,128],[117,339],[425,97],[606,137],[1473,163],[2495,2238],[267,137],[284,149],[284,161],[1445,189],[504,137],[792,169],[974,160],[1676,3292],[881,160],[1706,844],[320,987],[690,165],[1285,101],[291,5329],[377,1790],[737,1336],[1182,795],[2553,357],[112,287],[1736,1418],[4755,806],[485,425],[2682,276],[388,923],[532,2204],[2341,5861],[3119,148],[283,960],[543,166],[818,701],[1166,877],[4003,135],[775,155],[1346,272],[1978,162],[2064,154],[87,101],[597,174],[1927,417],[1961,352],[341,111],[1117,100],[1550,165],[1863,5908],[760,1062],[2990,1102],[3140,4267],[3503,509],[115,121],[350,116],[1064,143],[1838,115],[1873,188],[2994,2515],[307,783],[998,145],[48,56],[98,683],[653,136],[980,111],[1228,858],[1342,643],[865,559],[1503,121],[3714,188],[338,134],[3097,634],[404,97],[99,305],[377,2175],[485,97],[299,3059],[495,422],[986,1138],[992,809],[1092,582],[458,188],[3509,1771],[1966,481],[388,287],[822,276],[1141,134],[102,425],[296,628],[357,480],[1188,2480],[1236,2278],[265,166],[423,112],[527,170],[1245,682],[107,370],[226,512],[344,272],[1284,156],[1614,1063],[119,312],[342,131],[620,97],[801,112],[1011,268],[1146,5443],[2313,156],[233,1653],[299,1835],[564,115],[1057,976],[366,1297],[1591,640],[444,175],[1436,133],[940,2711],[1075,1063],[1165,773],[1359,5424],[1380,884],[1918,370],[231,128],[300,988],[395,121],[498,155],[1008,146],[108,397],[331,3137],[782,136],[458,141],[625,293],[706,116],[1365,310],[1917,305],[1998,110],[106,270],[1169,181],[2120,112],[281,462],[3815,1305],[267,4274],[913,522],[3556,3857],[567,168],[748,989],[2854,333],[4964,2953],[69,67],[292,172],[384,1384],[947,1013],[99,109],[320,270],[746,140],[759,1128],[896,4330],[916,150],[291,451],[344,114],[500,156],[3067,559],[5281,168],[535,161],[611,545],[4584,112],[232,157],[457,306],[493,150],[1922,3099],[2224,1475],[3579,136],[2220,111],[289,272],[318,152],[355,151],[955,189],[109,304],[521,97],[683,268],[684,272],[765,108],[966,111],[1269,162],[1969,1115],[4239,300],[501,137],[970,270],[515,2739],[691,1221],[969,141],[97,2085],[306,453],[354,360],[1560,158],[3713,2221],[532,451],[1446,2982],[4096,3038],[281,2631],[371,129],[376,1542],[1424,4435],[3620,140],[4929,139],[104,121],[105,538],[279,181],[771,135],[886,2687],[1089,2584],[319,659],[362,2491],[401,121],[874,1369],[1017,490],[1971,799],[1978,159],[2498,598],[3468,181],[119,4471],[348,4033],[712,270],[1339,144],[2865,883],[601,166],[736,188],[1794,166],[374,182],[1294,3813],[376,1521],[4390,115],[117,107],[311,272],[326,100],[333,1877],[802,174],[929,1594],[1908,2680],[2126,562],[273,1745],[953,148],[1311,184],[735,179],[1311,145],[753,270],[1087,107],[1157,528],[1525,766],[1706,4408],[3195,4478],[457,813],[469,147],[3306,1712],[438,146],[2559,1546],[360,695],[2233,1055],[274,270],[336,3709],[391,1206],[434,1572],[1071,268],[1451,48],[716,1016],[1269,136],[1441,145],[336,1724],[458,131],[519,270],[658,933],[801,270],[1943,185],[2524,136],[105,962],[402,3545],[654,165],[1577,132],[364,304],[378,829],[695,965],[268,270],[302,289],[358,163],[4700,160],[527,145],[554,136],[280,146],[353,381],[2829,1162],[269,1536],[289,404],[312,1160],[334,108],[2123,332],[104,305],[644,507],[1406,522],[195,137],[261,158],[321,395],[651,181],[1681,1062],[233,136],[426,161],[678,129],[874,1298],[1103,3277],[4010,148],[666,1579],[719,111],[1284,161],[2807,4139],[3698,2737],[3929,2088],[265,146],[5703,1317],[604,181],[650,3106],[677,1358],[1424,2699],[2651,270],[532,289],[273,121],[740,190],[1359,1152],[1721,628],[312,305],[1150,841],[503,902],[1360,733],[104,2009],[1116,1029],[117,118],[299,4946],[613,1052],[713,978],[2492,184],[80,67],[283,3358],[472,902],[3080,188],[4031,810],[634,400],[818,1904],[269,317],[1162,110],[426,157],[1494,310],[1359,941],[2110,1310],[3788,628],[4560,562],[296,99],[444,191],[785,1323],[288,114],[689,160],[1012,129],[294,3037],[526,136],[587,276],[1789,101],[235,1846],[1430,673],[555,135],[604,186],[950,417],[1966,165],[3667,152],[458,191],[5046,270],[657,135],[1018,173],[1860,382],[4158,5508],[281,467],[421,1340],[639,1358],[285,307],[644,3828],[1381,56],[281,692],[468,340],[497,132],[278,952],[357,1343],[1805,129],[4443,101],[474,181],[779,450],[55,53],[295,188],[614,1526],[625,101],[1311,154],[4685,2446],[267,161],[1048,2508],[1479,146],[276,675],[1618,176],[1843,1939],[2981,2593],[456,178],[2289,4113],[458,169],[489,163],[515,2443],[529,164],[604,190],[1201,1355],[655,6353],[741,375],[1204,159],[103,437],[286,151],[1167,138],[4215,268],[1108,1367],[302,100],[721,3333],[1466,456],[1922,6290],[550,148],[1119,774],[106,2091],[1396,140],[1842,4358],[2106,1152],[105,764],[749,162],[1039,1540],[3615,164],[269,361],[2429,616],[2532,382],[746,168],[576,171],[499,1160],[2264,3355],[1523,115],[396,116],[1242,541],[1287,141],[1510,183],[1654,304],[2488,595],[363,144],[402,361],[465,109],[857,111],[1504,174],[1522,128],[310,107],[765,311],[896,287],[934,360],[1009,139],[1436,166],[1484,773],[1496,180],[357,382],[940,3101],[4748,305],[672,117],[784,1046],[1124,531],[1577,137],[4517,1775],[586,2703],[2059,3792],[118,1160],[704,1042],[1054,1369],[2361,180],[3447,121],[1799,57],[294,509],[472,287],[556,365],[278,1686],[279,145],[283,579],[323,175],[469,186],[798,187],[1231,566],[4026,2052],[294,98],[433,151],[534,136],[907,2555],[1513,119],[831,152],[1064,151],[2338,335],[433,181],[546,146],[814,118],[1322,1687],[104,109],[532,111],[860,141],[342,134],[737,276],[1339,191],[1713,160],[4146,184],[105,626],[666,2730],[717,155],[170,189],[339,901],[470,182],[663,165],[2220,97],[281,326],[327,130],[405,153],[575,382],[924,2157],[4010,140],[121,991],[317,830],[728,1826],[1359,1466],[2001,190],[4351,161],[112,1806],[308,129],[320,5840],[571,1065],[803,417],[1127,3996],[1749,4404],[3255,307],[980,1357],[1013,496],[2361,137],[269,465],[285,274],[457,1202],[786,52],[1002,140],[335,709],[906,49],[429,128],[105,1007],[320,5767],[469,180],[556,4426],[1840,4743],[2106,1362],[2410,156],[299,324],[1473,190],[1549,417],[617,166],[721,616],[3578,3508],[266,157],[368,804],[558,296],[636,136],[745,370],[1648,131],[267,189],[442,101],[695,4464],[1053,137],[4387,1515],[4559,2257],[591,165],[907,2213],[311,397],[1027,185],[1030,1320],[2107,1297],[327,186],[573,1423],[588,158],[51,52],[73,80],[122,111],[1649,496],[2405,2748],[3159,2596],[389,141],[434,1709],[737,1660],[836,347],[1067,168],[1147,131],[1301,317],[2173,293],[83,3996],[265,152],[391,384],[423,117],[582,799],[658,1178],[51,56],[593,3916],[599,2722],[774,2556],[1018,169],[112,572],[191,144],[368,97],[562,910],[572,307],[586,3888],[548,69],[4866,3864],[5398,152],[1455,3209],[1956,626],[701,1443],[774,360],[1119,1028],[2690,133],[844,573],[338,180],[1950,1982],[291,652],[1126,174],[320,390],[979,1082],[2759,1766],[437,97],[1045,2462],[114,528],[1453,130],[1807,496],[2229,1013],[2834,1475],[319,432],[328,170],[1074,1360],[269,2463],[292,182],[667,2937],[933,1221],[1795,1984],[5902,136],[1644,3085],[2280,1953],[504,155],[970,111],[1077,115],[1387,289],[276,1385],[436,179],[621,1356],[3634,1218],[1147,140],[66,101],[691,875],[817,1227],[976,1152],[1316,130],[1405,1478],[719,465],[289,830],[638,137],[5317,312],[2195,1563],[99,335],[299,1474],[546,166],[646,1052],[69,77],[108,305],[353,110],[1569,462],[522,667],[1174,143],[1236,361],[5588,144],[115,268],[1831,1613],[685,99],[719,288],[869,277],[1538,1817],[1784,103],[274,1931],[1128,541],[1165,4106],[371,167],[542,114],[684,111],[1583,131],[1480,144],[2524,152],[377,348],[402,5296],[485,104],[489,134],[1133,115],[1214,316],[2466,4954],[4735,177],[379,148],[387,154],[726,422],[2998,479],[604,148],[615,4341],[652,307],[1131,2953],[355,152],[1285,4008],[2490,553],[3376,2903],[419,4500],[1480,177],[1827,97],[195,187],[833,1461],[2075,794],[2384,114],[402,1663],[461,181],[555,159],[718,184],[883,836],[1419,189],[911,348],[195,185],[1413,1814],[80,288],[288,1378],[614,103],[653,153],[3134,144],[278,5562],[375,1758],[2138,475],[380,178],[839,5777],[1154,1721],[109,112],[278,4505],[663,155],[698,1726],[1118,276],[4318,97],[274,628],[319,1587],[262,188],[994,2080],[1081,175],[1264,773],[414,128],[522,817],[932,168],[1588,761],[52,56],[1335,842],[97,936],[929,345],[3089,2777],[385,728],[882,1640],[890,272],[300,272],[1232,1286],[1987,4143],[3830,164],[278,289],[333,490],[448,3536],[663,158],[687,187],[2542,1666],[474,156],[1381,53],[1977,5367],[495,305],[295,132],[349,161],[565,160],[610,160],[762,1733],[1625,2973],[100,442],[278,3201],[472,288],[816,101],[1964,658],[5686,956],[579,272],[1235,2190],[838,871],[2524,164],[2966,4686],[1770,133],[121,101],[1309,1982],[2323,272],[341,2317],[832,306],[2874,871],[100,307],[233,137],[331,310],[537,2191],[1105,417],[1186,2970],[1648,136],[2348,615],[114,345],[320,912],[472,425],[1168,128],[1636,3977],[375,2625],[981,321],[1519,1306],[4922,1924],[268,1673],[327,139],[542,103],[611,382],[840,108],[3121,451],[1103,2974],[402,293],[452,1850],[532,121],[607,1198],[738,1205],[1339,185],[266,185],[350,1150],[604,182],[1840,370],[280,180],[353,781],[376,531],[570,1673],[582,895],[671,161],[2311,490],[3476,365],[111,321],[111,936],[434,1985],[1309,752],[1318,2633],[2369,806],[2389,533],[396,121],[798,182],[1260,169],[1419,169],[195,151],[312,3522],[447,5675],[493,168],[666,760],[965,306],[1313,471],[1654,272],[2176,3639],[3400,120],[51,50],[1267,188],[1964,3268],[1322,1541],[1386,3100],[1560,138],[51,55],[690,164],[1285,1582],[2116,288],[278,5070],[285,105],[341,3963],[2655,626],[4186,164],[5865,3165],[326,121],[425,272],[426,128],[2186,2991],[333,1780],[334,5354],[1092,1400],[97,3341],[827,1070],[3212,153],[1316,183],[666,3289],[1564,1273],[4869,918],[420,3087],[495,111],[1443,976],[3373,1320],[112,268],[338,148],[108,274],[477,858],[168,760],[317,118],[359,171],[375,3764],[672,272],[3027,505],[5183,1777],[287,2768],[297,129],[900,1571],[3840,6957],[106,2274],[302,4877],[1891,1733],[738,289],[2965,1897],[65,114],[434,1652],[757,99],[6143,4418],[296,272],[388,528],[556,306],[750,170],[1181,347],[1900,191],[1342,1107],[1126,182],[1187,693],[1354,118],[293,116],[118,370],[299,4529],[333,811],[375,4068],[581,150],[1126,163],[1881,293],[3014,139],[6328,7004],[335,304],[374,163],[401,321],[626,111],[874,2043],[3878,1603],[304,100],[419,7111],[556,468],[906,50],[1076,144],[1340,531],[364,2000],[2067,131],[2209,115],[79,76],[428,272],[433,172],[803,3896],[1335,2534],[1651,780],[51,647],[105,325],[320,442],[474,188],[501,141],[724,276],[1354,100],[4768,2664],[444,162],[888,111],[910,2100],[1444,122],[2732,3765],[6241,679],[97,465],[1015,129],[1144,858],[1324,2733],[1641,480],[2504,5453],[4887,1221],[97,104],[320,688],[73,73],[274,307],[554,150],[1156,833],[54,53],[285,584],[947,3627],[1178,883],[452,118],[4101,1222],[101,107],[654,156],[1163,1823],[1247,114],[1407,522],[2116,330],[3399,1137],[402,287],[1174,162],[1988,277],[2548,2167],[3380,160],[283,422],[506,137],[669,1078],[687,177],[1081,189],[2681,180],[2943,1595],[4065,137],[1246,2253],[1407,1165],[2027,115],[3948,1495],[68,272],[423,116],[278,304],[141,676],[835,110],[1413,274],[1829,1252],[4070,976],[236,3895],[438,184],[3049,986],[3584,432],[300,107],[504,165],[703,1774],[3135,783],[291,108],[318,186],[368,688],[2555,2713],[99,396],[294,2908],[356,1491],[463,793],[660,174],[1177,378],[5976,843],[384,924],[472,1508],[666,2983],[780,276],[2369,99],[675,276],[3283,119],[3707,3519],[4636,397],[100,467],[415,824],[116,305],[309,2632],[673,276],[857,97],[2240,134],[3388,769],[233,177],[346,179],[360,556],[3856,2233],[343,1102],[627,133],[2988,173],[273,307],[333,3251],[1128,812],[2764,1158],[232,154],[3770,143],[374,149],[428,97],[909,3388],[1029,553],[1339,176],[1392,121],[1680,773],[300,1038],[305,422],[2047,848],[818,3926],[920,2974],[419,112],[531,938],[1612,1031],[1900,190],[305,272],[317,100],[1601,3765],[1679,3052],[2146,1004],[5832,1343],[638,129],[657,151],[803,4058],[297,178],[177,3029],[451,121],[903,307],[1241,383],[2626,580],[410,147],[545,382],[788,382],[974,1637],[1301,1368],[5083,2140],[87,2375],[263,138],[269,578],[330,98],[1919,1021],[3467,1337],[818,463],[340,2665],[485,101],[711,648],[1473,174],[1580,276],[1621,164],[1681,1302],[6189,395],[548,114],[650,5417],[715,184],[348,487],[506,177],[1014,1839],[518,2578],[848,1061],[882,621],[1001,139],[3080,134],[104,995],[336,4683],[568,187],[904,1226],[1590,1599],[3612,1946],[551,3744],[1430,1286],[289,111],[296,97],[503,1652],[576,149],[600,154],[779,696],[994,724],[351,3682],[441,141],[810,357],[880,172],[1528,883],[3509,1364],[98,121],[406,145],[588,150],[706,902],[706,3853],[1284,158],[1498,105],[350,2502],[1104,573],[2716,6713],[346,137],[368,330],[752,382],[2119,1329],[5787,1630],[237,5495],[344,116],[657,137],[721,2562],[1629,188],[1806,100],[1903,1624],[2639,1173],[108,370],[328,180],[355,177],[359,133],[396,307],[596,111],[1002,160],[1301,287],[1432,116],[56,53],[386,1359],[679,693],[1045,848],[2745,1348],[4920,679],[105,952],[110,451],[801,111],[1150,277],[2473,383],[3051,2702],[349,144],[1067,156],[1074,2487],[1164,357],[1446,430],[97,2387],[389,186],[719,2340],[786,51],[896,876],[2640,174],[4223,5532],[82,69],[405,143],[520,136],[972,276],[434,4931],[452,326],[1795,1791],[57,53],[558,122],[1726,1016],[2069,184],[98,288],[300,1294],[392,161],[401,105],[636,161],[368,268],[616,111],[788,523],[1363,1019],[2070,2156],[726,117],[1567,1717],[1998,304],[2316,2197],[2988,172],[6447,164],[105,991],[948,134],[1447,693],[1555,679],[2007,819],[116,620],[269,415],[1568,1898],[2313,148],[607,399],[887,1103],[1074,2937],[2041,312],[116,119],[83,84],[105,542],[402,652],[1502,895],[261,129],[266,189],[562,1291],[685,305],[1621,141],[724,693],[784,288],[1693,3357],[2231,3950],[447,7077],[565,164],[807,148],[1186,1777],[1204,183],[6431,6389],[492,177],[745,921],[889,160],[1002,148],[3738,1555],[4402,319],[70,101],[261,188],[874,357],[1024,6136],[1106,3003],[1635,5341],[98,287],[378,312],[402,2077],[472,305],[489,186],[1082,992],[114,277],[639,319],[790,1478],[501,166],[729,115],[2223,117],[2858,3013],[191,160],[613,812],[885,3680],[1047,3289],[293,105],[548,111],[283,5715],[1267,142],[416,150],[613,541],[956,1186],[299,105],[320,404],[349,170],[489,176],[625,378],[4114,1131],[345,304],[588,132],[1141,129],[1398,186],[2483,174],[4155,1332],[334,6890],[765,115],[796,6267],[2040,2572],[660,146],[878,149],[973,168],[3219,3597],[278,620],[291,598],[1150,806],[1871,161],[2087,2188],[2976,1087],[3427,1953],[299,465],[372,179],[599,276],[1138,4150],[1284,184],[2774,3680],[69,120],[333,376],[359,170],[503,764],[3149,182],[634,270],[2039,148],[306,3505],[718,130],[4131,902],[283,397],[331,3786],[513,130],[2544,99],[294,2846],[367,1410],[368,2006],[539,180],[1225,188],[1866,168],[2193,339],[3685,1321],[276,1534],[281,1959],[478,160],[613,385],[1693,5881],[434,356],[634,977],[1538,4234],[392,157],[728,1019],[4472,991],[5948,176],[111,102],[654,147],[826,1332],[863,1592],[995,4678],[1998,305],[3170,462],[236,3817],[395,289],[452,321],[1154,102],[661,545],[746,181],[1939,3378],[2156,769],[3778,1320],[6897,1984],[278,288],[340,1371],[476,160],[577,185],[1005,306],[1556,149],[1933,1518],[1233,141],[2334,1678],[385,2042],[79,80],[187,918],[296,270],[635,138],[736,142],[943,115],[2400,2045],[2469,2071],[54,57],[283,621],[2254,1244],[325,923],[415,268],[2064,160],[5625,3239],[119,289],[387,167],[465,307],[678,131],[1081,176],[2456,2310],[294,99],[495,381],[570,537],[666,2012],[1783,795],[2439,1518],[5545,370],[121,108],[266,136],[402,4028],[438,167],[1370,1007],[4400,760],[4401,352],[51,57],[276,3760],[351,3772],[533,1768],[1048,6577],[1051,181],[6582,140],[116,289],[331,288],[597,180],[730,468],[1408,2775],[1954,352],[309,188],[1066,1826],[788,1369],[1325,3100],[346,141],[410,134],[411,151],[670,97],[1146,142],[2146,2984],[2470,659],[4802,2837],[501,154],[316,181],[402,572],[65,73],[104,425],[471,2517],[1265,7575],[1381,52],[1494,2667],[2198,2194],[749,158],[1697,457],[3197,2328],[678,188],[1754,185],[2658,97],[111,465],[281,5879],[548,1378],[436,184],[2052,2217],[1335,2017],[337,911],[394,176],[405,175],[3490,128],[274,305],[353,370],[603,177],[883,930],[300,829],[378,1852],[542,109],[544,1913],[1377,2279],[3246,112],[89,317],[507,2145],[2084,467],[3631,952],[85,78],[447,300],[319,306],[336,1663],[693,2445],[330,268],[377,377],[519,2270],[553,2346],[655,2556],[906,51],[1159,1393],[1463,892],[1988,4911],[105,404],[371,166],[596,1088],[882,2278],[1696,4308],[5523,1869],[109,370],[283,277],[631,268],[639,556],[899,155],[2482,277],[3284,861],[397,116],[1749,1958],[4210,927],[5659,733],[5739,1481],[6359,927],[68,270],[98,361],[362,155],[507,2955],[607,1004],[671,159],[320,425],[1253,541],[1313,2152],[3122,2868],[274,110],[490,725],[1146,145],[1770,130],[514,116],[7306,7119],[528,1526],[564,270],[651,171],[4518,382],[336,361],[357,796],[563,184],[1156,2279],[308,143],[350,2571],[421,587],[2055,1717],[5693,304],[141,168],[666,1910],[976,3104],[1248,480],[1525,333],[1576,1080],[404,109],[756,598],[1054,2325],[1586,764],[704,1628],[1684,1196],[236,6672],[291,578],[541,1660],[756,5841],[276,2624],[281,324],[453,982],[508,111],[943,995],[1886,544],[2240,139],[2638,1193],[3685,1115],[4248,6208],[5339,111],[32,396],[291,3555],[336,851],[443,128],[460,167],[1269,129],[376,1926],[998,188],[1617,156],[2386,276],[3318,296],[359,157],[853,148],[927,122],[973,130],[2545,4138],[5510,1290],[730,6126],[1144,1019],[5774,140],[99,287],[237,134],[285,339],[313,147],[1838,2913],[2256,148],[313,179],[462,112],[543,184],[772,538],[790,858],[904,650],[432,6013],[508,270],[859,677],[1729,1140],[1739,817],[2227,188],[5735,1217],[444,168],[542,4572],[1197,111],[1209,100],[1387,564],[1398,158],[356,5096],[494,585],[834,834],[933,2060],[1787,417],[115,270],[296,6394],[314,152],[830,6380],[1189,865],[1679,2841],[2339,1136],[2614,640],[116,326],[235,2521],[269,1263],[617,128],[1381,51],[1235,588],[1319,184],[1430,2624],[1700,115],[2146,679],[99,307],[374,169],[486,700],[539,173],[1117,509],[1511,558],[3167,376],[4783,174],[54,54],[310,1150],[375,496],[478,137],[555,158],[83,111],[314,138],[485,270],[548,117],[549,174],[1207,276],[1236,5511],[1684,2589],[2704,3108],[98,1540],[1113,268],[307,112],[320,570],[323,185],[456,146],[1121,673],[1680,7786],[5847,691],[477,737],[557,133],[856,101],[7837,112],[336,105],[1141,144],[1184,132],[1510,163],[3193,3733],[3712,587],[73,116],[300,101],[340,348],[424,185],[521,108],[586,100],[1130,2355],[2650,679],[108,1853],[289,5536],[484,1582],[336,397],[554,191],[604,143],[3372,270],[1044,115],[1131,3239],[2808,1675],[4585,779],[267,163],[297,170],[970,4128],[1521,1131],[3041,152],[79,84],[361,415],[404,272],[432,365],[1149,959],[1159,1070],[3176,181],[285,1858],[296,277],[573,2341],[593,3562],[611,1369],[788,545],[1886,914],[4343,107],[80,3448],[483,158],[520,146],[567,128],[690,169],[757,119],[1923,1447],[2483,171],[4519,157],[278,305],[495,960],[687,134],[1541,2292],[1595,985],[3447,2134],[820,1110],[828,115],[1188,3104],[1434,3744],[2611,3406],[3836,2664],[483,156],[525,181],[1747,119],[2297,3472],[2870,2441],[3024,545],[6456,340],[80,82],[373,147],[840,5282],[874,1622],[291,467],[694,182],[1293,3343],[3158,1748],[52,52],[276,2386],[4355,2954],[278,570],[356,114],[1057,1386],[2574,128],[798,133],[940,2653],[1570,911],[5612,103],[920,1116],[1125,344],[3014,157],[320,857],[1381,50],[1402,938],[2124,3977],[308,139],[601,188],[2090,188],[4255,370],[4365,1775],[105,592],[466,165],[1051,168],[553,276],[976,1598],[1301,3616],[1344,120],[2886,288],[3648,1806],[432,340],[542,108],[752,1085],[532,572],[55,56],[269,1627],[298,160],[1375,153],[1438,911],[2062,1589],[2640,176],[1860,480],[2032,132],[2496,950],[790,925],[872,304],[1920,48],[263,150],[496,3092],[499,344],[614,118],[675,1708],[895,780],[2412,268],[2982,2510],[426,166],[993,272],[3761,2497],[121,114],[194,169],[327,191],[564,97],[973,166],[1056,146],[1332,1437],[1747,1294],[2373,938],[5244,3213],[341,2463],[495,1254],[562,720],[651,166],[1047,2049],[2032,184],[2240,135],[6542,397],[68,305],[376,1006],[607,595],[627,141],[897,422],[1921,276],[2641,2282],[121,339],[402,437],[919,643],[1216,178],[1412,2219],[6175,108],[598,287],[1616,1764],[423,572],[457,1058],[463,333],[617,174],[1141,128],[1560,171],[1831,661],[2098,599],[99,117],[388,1123],[401,533],[920,858],[999,116],[4829,4143],[1917,97],[2709,174],[5158,119],[105,1499],[306,1371],[358,185],[380,179],[401,304],[631,5488],[1008,169],[1180,169],[4340,156],[6436,386],[373,183],[441,160],[562,276],[583,879],[602,1944],[2085,268],[6518,386],[390,1457],[400,304],[593,817],[726,111],[1938,287],[2032,152],[3197,5064],[4321,2050],[371,139],[785,1420],[2137,179],[3454,676],[3461,187],[232,159],[276,682],[368,307],[984,1105],[78,101],[285,1113],[497,151],[854,1306],[1119,468],[2920,180],[403,161],[508,378],[637,307],[658,1057],[748,2004],[1407,1136],[1545,1114],[1981,101],[2326,1979],[3257,152],[232,160],[287,356],[717,145],[908,340],[1730,57],[299,7041],[415,335],[868,2901],[1042,1124],[5198,3795],[100,723],[107,321],[360,507],[573,306],[813,681],[844,2057],[1611,147],[2078,3268],[394,179],[398,139],[447,102],[947,759],[1871,155],[421,1390],[536,138],[114,335],[405,141],[596,117],[701,861],[1184,129],[1763,496],[5340,3432],[6535,901],[375,682],[393,111],[1113,111],[1446,2878],[2174,3746],[2809,553],[3249,818],[886,813],[278,312],[516,101],[664,189],[3044,5540],[3269,272],[3489,1456],[278,5187],[285,1718],[981,894],[2704,2489],[340,432],[1037,3374],[1174,184],[114,304],[1545,3058],[2672,141],[4035,7499],[268,356],[596,307],[1150,304],[2223,111],[2366,2957],[4531,2993],[194,176],[744,806],[974,5613],[1154,6669],[2554,7813],[263,152],[532,272],[609,162],[451,97],[926,347],[1104,2207],[7687,7000],[562,895],[801,3694],[1311,171],[2133,415],[4186,7572],[5280,156],[190,184],[401,270],[402,922],[1899,1222],[2400,545],[349,180],[364,111],[367,911],[1103,1049],[3652,3554],[293,97],[1973,144],[549,170],[568,132],[753,111],[1071,115],[55,55],[268,111],[299,621],[443,146],[593,743],[644,377],[890,304],[330,115],[357,1460],[704,1323],[1523,1640],[2271,5689],[307,97],[393,3292],[615,1916],[984,7775],[2401,1041],[71,111],[375,1101],[402,1976],[501,138],[596,564],[687,144],[73,3543],[862,147],[3875,5228],[1676,100],[7228,188],[803,2276],[906,56],[1121,376],[1156,4052],[1366,4642],[1984,296],[297,148],[391,367],[103,115],[106,570],[294,2382],[320,2382],[464,128],[548,105],[622,2584],[651,180],[1987,324],[2055,854],[4771,2737],[503,97],[605,182],[614,1743],[705,976],[798,181],[73,77],[102,1977],[119,620],[654,146],[681,844],[748,2786],[922,395],[2327,884],[2948,343],[3449,2923],[5011,1087],[226,136],[320,2622],[923,747],[1410,367],[4543,2997],[689,156],[2100,1670],[2944,148],[5142,2538],[73,109],[414,168],[495,626],[615,276],[615,2238],[863,101],[1208,667],[3686,110],[269,378],[752,978],[995,121],[1040,166],[1129,1021],[1537,1343],[1543,692],[2565,806],[3718,348],[108,578],[285,361],[327,150],[335,293],[562,799],[1031,2675],[1617,171],[2326,733],[3544,105],[4657,137],[503,3148],[507,1790],[514,339],[591,186],[820,1006],[837,2274],[1170,450],[1831,2526],[5635,115],[317,108],[508,86],[786,50],[939,116],[979,1159],[346,145],[572,1990],[616,268],[934,585],[1706,832],[314,145],[321,1743],[333,1388],[1506,773],[2403,1760],[491,166],[3322,1173],[6261,509],[116,608],[346,177],[432,348],[446,161],[963,103],[1001,162],[1053,178],[1298,450],[2502,4071],[406,136],[416,146],[434,927],[672,4156],[2905,288],[119,1272],[269,274],[1044,272],[1273,272],[1510,144],[1956,296],[338,135],[457,384],[663,129],[731,142],[1328,2060],[1380,468],[122,116],[278,6683],[1188,1140],[4240,121],[5740,2270],[237,146],[738,592],[979,743],[4073,4073],[52,57],[336,287],[427,149],[1338,490],[2011,103],[362,130],[634,110],[778,181],[980,97],[1546,307],[606,156],[765,102],[1023,184],[1151,115],[1899,2922],[4668,2773],[114,324],[1325,3095],[2021,188],[2734,288],[358,187],[5336,312],[6165,189],[276,2859],[368,289],[1017,399],[263,153],[341,353],[7021,6077],[97,1852],[273,7331],[1001,146],[1463,1219],[6433,1830],[114,465],[320,277],[331,1148],[937,189],[1008,143],[1753,1937],[1894,7695],[73,86],[106,307],[281,952],[291,1857],[305,830],[330,287],[386,399],[588,169],[737,589],[1003,144],[1292,1138],[1760,1691],[263,158],[281,1976],[855,1137],[280,156],[350,806],[547,140],[1042,5610],[1174,130],[1487,1816],[2293,306],[273,107],[388,467],[531,3305],[610,188],[642,1170],[2160,861],[3011,5391],[114,467],[319,384],[499,310],[869,108],[1057,463],[4411,938],[5114,772],[6268,975],[278,396],[651,184],[862,134],[2851,1757],[438,172],[737,1867],[856,104],[3103,781],[3287,182],[285,97],[404,115],[452,683],[1828,1094],[2119,1275],[625,274],[741,812],[969,149],[1549,450],[1581,168],[2519,815],[3103,2908],[3319,2699],[352,978],[1729,986],[2886,1043],[4710,136],[341,3199],[474,145],[775,128],[813,844],[7314,100],[79,75],[874,3923],[1587,583],[2832,428],[3468,139],[3835,110],[3877,6546],[333,599],[370,101],[586,98],[1002,164],[49,1841],[661,1085],[1205,97],[1867,8446],[2195,353],[3194,129],[83,105],[788,450],[1205,344],[402,3450],[416,136],[490,1070],[909,553],[110,296],[235,145],[438,187],[1044,97],[2990,1217],[285,1887],[401,634],[1324,1447],[1341,2513],[2706,365],[5900,333],[67,104],[319,348],[415,115],[748,4111],[1344,1046],[1394,385],[269,5802],[276,826],[368,598],[545,523],[1357,270],[1527,615],[276,724],[336,378],[411,182],[415,272],[906,55],[1040,161],[1077,321],[1135,180],[7433,2057],[73,76],[673,1349],[1141,135],[4153,5747],[7877,4606],[489,183],[946,682],[4213,356],[387,189],[909,3635],[1224,188],[1382,6081],[2956,910],[107,270],[368,1554],[675,811],[904,421],[1077,1955],[1265,615],[1997,172],[3316,321],[4359,875],[4725,1827],[6990,1499],[52,54],[337,6933],[370,268],[903,339],[1069,176],[1848,1186],[414,130],[474,158],[3498,876],[273,3485],[278,579],[479,1302],[686,155],[763,191],[917,360],[1566,116],[269,1043],[1455,1660],[1912,5348],[2480,3159],[83,83],[111,356],[336,465],[550,155],[619,160],[664,138],[4701,997],[375,4458],[388,1271],[394,175],[651,183],[1973,183],[2084,1665],[3256,2316],[3984,111],[4632,2061],[5112,1457],[267,184],[284,130],[865,357],[4792,2739],[7015,1934],[345,311],[421,376],[654,188],[1354,816],[2919,425],[3440,3312],[230,171],[445,171],[588,134],[619,163],[866,158],[1128,3052],[1251,1353],[1648,165],[7751,8474],[112,709],[555,146],[640,2684],[1089,340],[1691,1134],[2533,737],[4970,1085],[6557,180],[459,175],[477,989],[499,117],[1094,5020],[1236,3485],[1794,134],[2585,312],[302,305],[330,97],[354,6765],[535,153],[548,296],[631,7878],[961,2149],[1230,681],[5021,496],[5099,2570],[378,121],[471,647],[1206,844],[305,277],[721,99],[1471,1082],[1765,384],[2062,1682],[6213,311],[119,2463],[281,1123],[653,135],[949,827],[1743,307],[2405,3064],[3102,2238],[6107,1123],[299,528],[313,163],[376,701],[741,385],[816,115],[1054,1000],[2113,2613],[6064,5258],[300,2446],[442,1493],[622,343],[661,796],[2938,608],[5401,1931],[106,3172],[507,1925],[515,370],[586,1078],[636,144],[3098,1115],[84,69],[104,1740],[294,7770],[368,312],[1544,1092],[1567,2242],[3224,276],[3392,1852],[7351,1735],[277,1162],[472,101],[598,108],[728,2183],[1043,1944],[1852,5895],[2734,1675],[347,375],[404,2018],[621,270],[901,307],[909,1193],[2509,189],[3658,733],[4086,1307],[6720,608],[401,97],[711,827],[6754,1791],[584,108],[906,57],[1424,293],[72,101],[331,572],[336,304],[345,2911],[851,519],[966,816],[1387,2034],[2193,288],[3651,3278],[595,6377],[658,376],[766,883],[1455,892],[2754,381],[3149,166],[52,647],[428,307],[717,163],[719,289],[884,468],[990,1553],[1981,312],[5525,352],[686,137],[1286,1624],[3444,4397],[386,1601],[434,118],[460,155],[1423,457],[1929,376],[5277,806],[56,54],[79,110],[371,168],[498,131],[643,566],[1091,130],[4830,2872],[1229,133],[2367,101],[5486,430],[533,829],[696,357],[801,3264],[1354,6595],[731,181],[906,54],[991,300],[1151,311],[1592,1809],[347,2117],[401,2337],[863,381],[1146,171],[1497,1637],[2236,390],[3149,156],[522,1257],[522,2103],[634,117],[1245,553],[1381,49],[1459,159],[1471,2188],[3381,2128],[83,116],[510,136],[947,1528],[6027,2731],[7098,850],[613,505],[639,2855],[891,538],[1535,566],[1560,181],[2544,1832],[404,274],[1407,8346],[2016,1746],[2297,112],[888,110],[1629,132],[2440,1019],[4999,2985],[435,111],[498,178],[543,136],[631,101],[1330,3285],[1419,172],[1993,2108],[4180,829],[4183,2596],[4806,425],[4865,102],[6886,3178],[266,161],[274,2009],[306,3259],[333,6286],[430,711],[593,1017],[779,566],[783,400],[1429,467],[1657,2431],[2256,168],[381,272],[423,107],[593,737],[819,789],[1745,578],[1860,559],[2882,733],[5858,296],[283,2463],[409,136],[754,578],[788,1298],[933,997],[2313,144],[3114,144],[5199,3117],[119,5032],[406,143],[485,67],[856,115],[939,103],[1267,176],[1281,1367],[1327,300],[2159,2431],[2202,354],[2505,7441],[104,564],[331,324],[434,116],[484,270],[554,162],[589,1136],[1361,1321],[1670,347],[2373,2079],[5951,796],[398,138],[707,128],[798,170],[896,305],[896,451],[1077,109],[1229,139],[1671,2494],[1701,376],[1800,1906],[2036,6961],[2602,2249],[2629,110],[344,2269],[401,287],[32,2869],[265,168],[421,667],[536,156],[882,109],[2776,3117],[4007,1338],[5209,562],[115,104],[489,170],[666,3621],[2179,1355],[3322,352],[3387,2980],[66,65],[277,330],[320,528],[362,6097],[438,186],[472,1263],[961,7120],[4120,3127],[334,7789],[402,305],[593,333],[1952,442],[2111,1985],[2326,3423],[2454,382],[73,65],[334,4642],[536,161],[1112,861],[1429,1804],[2132,2872],[2263,8348],[263,151],[348,699],[425,6984],[556,343],[803,480],[805,168],[998,156],[1082,1070],[1795,1078],[2646,704],[377,365],[436,175],[748,682],[803,2698],[931,608],[1057,2474],[1229,180],[2339,2480],[2368,480],[4385,1563],[285,564],[654,134],[745,428],[826,3907],[971,141],[1682,701],[3081,270],[4926,1745],[6216,2337],[102,598],[103,335],[183,176],[374,142],[384,1005],[542,1533],[748,2823],[1119,2003],[1146,154],[1625,2825],[2535,1791],[4263,361],[80,114],[266,184],[836,1330],[3073,116],[71,66],[117,465],[303,149],[354,337],[380,163],[607,376],[813,1210],[1360,1761],[1390,276],[5009,5714],[5239,780],[278,8281],[338,188],[1050,169],[1258,152],[1911,2178],[2068,186],[3023,3634],[388,274],[495,97],[1568,375],[2425,1231],[3191,1395],[3243,2985],[4938,110],[104,2269],[319,340],[452,783],[846,643],[953,177],[1612,4315],[2080,1049],[3455,396],[6638,2429],[429,175],[470,180],[280,145],[424,132],[921,100],[1381,48],[1447,819],[1700,370],[1810,1757],[277,437],[299,3802],[1561,8559],[1719,1498],[1954,357],[3675,883],[98,272],[99,274],[117,1422],[277,121],[320,289],[587,490],[859,468],[926,682],[1168,178],[1325,986],[1398,172],[1967,148],[6157,2298],[286,177],[351,1384],[523,3771],[613,667],[1312,537],[1544,733],[103,370],[320,396],[443,150],[501,175],[695,367],[868,4330],[980,6422],[1512,343],[1560,151],[1601,6379],[2406,822],[405,159],[485,6145],[536,145],[586,7700],[646,946],[791,109],[886,5766],[1024,272],[2341,2083],[3221,1623],[4561,144],[56,57],[105,2428],[372,161],[812,2487],[843,121],[1070,693],[1344,339],[1377,2170],[2709,156],[2989,888],[3934,276],[4215,121],[766,276],[1520,479],[2684,832],[3624,3920],[5866,121],[73,71],[80,108],[84,101],[299,519],[606,167],[678,147],[796,566],[900,1333],[3057,1369],[4579,1215],[5181,3967],[135,188],[324,901],[355,143],[521,272],[936,268],[1008,163],[1289,190],[2107,3810],[3055,1150],[4195,1356],[320,467],[666,566],[1629,145],[3067,1085],[4594,1965],[649,152],[1762,7267],[8368,164],[660,154],[1128,5683],[1793,367],[1997,170],[3641,97],[84,86],[281,688],[299,270],[299,1011],[345,361],[839,1162],[1396,178],[1970,820],[2132,1164],[2368,3710],[4526,321],[6019,983],[421,1780],[703,6207],[720,2426],[896,117],[912,97],[1054,3205],[1056,151],[1194,1886],[1611,169],[2099,978],[4612,1982],[233,180],[364,5651],[486,1029],[555,137],[1107,1937],[1839,1778],[2106,679],[493,177],[741,667],[750,169],[1213,97],[1436,129],[1436,175],[1935,1667],[1963,692],[7918,1593],[344,293],[657,129],[1808,3085],[8611,2534],[390,361],[681,1104],[958,114],[3286,1221],[99,1263],[302,102],[304,97],[323,138],[406,139],[648,972],[701,895],[811,276],[1054,2306],[1248,1337],[1340,1708],[1673,111],[4727,901],[9054,97],[79,67],[350,1511],[386,809],[390,628],[651,165],[1048,8191],[4702,2753],[6678,2022],[65,71],[73,108],[121,370],[349,185],[395,116],[685,296],[1608,138],[5962,141],[65,99],[104,404],[285,353],[439,109],[786,49],[1175,347],[2347,622],[3276,463],[6806,1822],[7693,2502],[376,1257],[593,2759],[2039,164],[2040,2474],[2453,883],[69,109],[278,4849],[291,2633],[310,509],[312,122],[324,378],[365,2157],[401,843],[484,901],[532,293],[1095,523],[1615,382],[4526,8665],[5310,558],[299,1594],[313,134],[375,399],[420,2253],[653,150],[1012,166],[1555,1761],[4171,307],[32,467],[236,137],[432,774],[738,850],[900,1121],[1200,1121],[1583,149],[3321,2015],[5445,305],[276,1066],[470,168],[590,188],[868,311],[1189,7491],[1423,640],[2164,490],[68,70],[395,307],[625,2363],[976,3920],[983,382],[1669,400],[1774,844],[2600,110],[2878,3267],[3372,442],[4996,2700],[300,270],[483,145],[1021,357],[1095,3192],[2892,115],[5092,4649],[6861,140],[6958,716],[70,288],[265,180],[274,894],[302,3111],[661,407],[672,289],[994,2073],[1346,307],[2137,188],[3644,553],[79,68],[376,938],[666,642],[694,183],[1455,1218],[5526,633],[453,1371],[602,2360],[748,4081],[1911,3336],[2123,879],[2493,1447],[3249,1234],[74,101],[310,97],[717,134],[1227,997],[2670,110],[3740,117],[4278,370],[4301,1062],[119,3443],[272,1499],[331,8212],[428,3313],[595,6529],[909,4456],[910,938],[2592,118],[2690,149],[5638,628],[194,186],[291,4940],[319,332],[463,399],[499,2977],[864,181],[1016,276],[1267,179],[1298,566],[1557,2238],[2569,650],[4480,188],[815,2022],[1300,2301],[3028,468],[4425,3831],[226,132],[391,383],[980,345],[1304,832],[1376,428],[1648,146],[1971,2974],[2368,1000],[3734,184],[486,347],[792,180],[1408,2848],[1564,296],[3344,268],[5863,112],[8270,679],[101,101],[376,925],[2322,2372],[3073,268],[3490,156],[3902,270],[5761,5203],[52,55],[102,115],[109,2071],[289,321],[310,272],[351,2556],[588,138],[732,183],[932,153],[1216,188],[1225,187],[1543,2008],[1825,276],[2039,156],[3556,1271],[5120,450],[102,293],[236,129],[515,3433],[5378,111],[115,112],[235,512],[334,7304],[339,2908],[487,982],[702,404],[810,643],[892,3275],[940,7457],[1012,180],[3046,272],[273,5262],[604,191],[635,141],[2242,2194],[2457,97],[2865,2796],[384,639],[452,2177],[1157,97],[1212,5263],[2113,933],[3413,5016],[3636,1405],[472,304],[793,1176],[1219,531],[2540,1979],[3962,109],[4179,105],[5674,818],[103,1043],[281,370],[401,425],[509,2901],[655,5084],[752,3745],[762,305],[840,9070],[2023,940],[121,97],[442,1515],[727,296],[920,1171],[972,1478],[1022,184],[2636,4479],[2831,1400],[3055,4614],[426,140],[691,2616],[879,2578],[1140,1017],[1329,7716],[3024,2603],[4737,1554],[602,1857],[675,1735],[753,564],[840,112],[1053,169],[1254,115],[1313,2517],[1323,276],[3645,1178],[4439,559],[332,730],[334,107],[346,180],[350,103],[693,1029],[2640,173],[109,3313],[278,4028],[298,159],[563,146],[1041,103],[4110,139],[313,167],[331,1976],[362,139],[1479,132],[1935,1164],[69,683],[485,570],[590,156],[655,677],[935,183],[2069,128],[2126,2807],[2277,2290],[3158,1535],[4318,305],[5343,733],[6842,2149],[103,396],[1948,2189],[2280,2798],[2607,1717],[3666,422],[3937,883],[76,307],[291,1474],[401,991],[402,824],[485,304],[645,140],[721,1955],[1449,1200],[269,1160],[294,558],[355,131],[909,875],[1329,6786],[3160,1347],[8589,161],[274,99],[580,276],[620,370],[1116,1329],[1146,155],[4950,268],[79,115],[269,519],[278,8369],[375,1103],[383,4651],[739,149],[1030,1042],[1377,376],[3011,2766],[5516,6222],[114,1779],[331,1123],[621,111],[791,970],[989,276],[1126,142],[1208,1092],[1929,673],[6322,871],[307,843],[320,584],[1206,1202],[2214,1910],[5268,5185],[117,533],[270,370],[306,1089],[333,1687],[494,911],[867,121],[2493,1946],[3055,105],[3175,6281],[4221,111],[5426,911],[6288,304],[6883,3229],[98,277],[291,850],[344,268],[375,1363],[404,268],[513,169],[938,1292],[2639,2586],[465,305],[877,6733],[1114,475],[1147,165],[1570,2202],[2612,2026],[3677,1200],[4584,1791],[53,56],[307,111],[696,1639],[1167,191],[1617,160],[2216,2156],[2612,733],[4409,2445],[646,812],[1375,174],[2268,616],[308,189],[310,533],[520,187],[522,1017],[591,150],[732,184],[879,518],[1022,135],[1186,716],[5160,1786],[5897,451],[109,451],[586,121],[1046,116],[2635,109],[2690,144],[3690,3513],[5425,370],[8268,7296],[67,67],[114,1368],[386,1388],[452,100],[579,270],[682,1670],[741,541],[744,537],[771,156],[1198,972],[1290,382],[1430,2371],[1943,160],[1952,114],[3209,1599],[3804,716],[98,960],[349,132],[529,162],[572,268],[1001,182],[1338,1106],[6727,152],[7447,7658],[513,138],[788,3766],[797,100],[1838,4423],[1889,2820],[5730,7582],[401,2587],[485,117],[521,115],[1454,1102],[1656,877],[7513,304],[100,305],[688,400],[721,1582],[730,2143],[1444,537],[1485,185],[2503,3211],[2665,453],[3176,161],[3256,1731],[322,140],[337,1374],[402,353],[663,170],[1527,2564],[2207,1202],[328,187],[726,317],[1494,1623],[77,289],[112,102],[197,147],[428,111],[711,276],[1032,161],[1097,815],[1459,143],[117,509],[121,111],[263,134],[311,324],[393,2250],[474,185],[3126,300],[339,121],[581,136],[596,397],[710,132],[940,3639],[997,3373],[1045,2047],[1176,826],[1569,3472],[2972,1927],[118,288],[281,723],[501,186],[551,6491],[904,1388],[65,66],[273,465],[337,2122],[356,335],[462,1712],[2498,1511],[2606,100],[3340,365],[77,101],[112,305],[1104,640],[1204,149],[79,86],[99,1947],[331,451],[401,274],[1232,1960],[1636,693],[2611,822],[2909,1624],[226,156],[285,345],[324,2561],[437,270],[874,3766],[1072,1785],[4102,1165],[4263,378],[4459,5418],[320,117],[589,486],[1990,8392],[2236,1594],[4178,904],[105,117],[280,3854],[288,422],[715,140],[237,612],[317,772],[1031,1764],[2328,1055],[236,186],[2540,733],[2831,582],[3809,4155],[55,57],[107,114],[285,3172],[565,140],[705,990],[829,2444],[831,142],[1447,794],[1459,164],[1506,5351],[2271,616],[3437,2705],[4548,304],[289,422],[339,1832],[414,156],[470,147],[704,1731],[873,312],[2385,2687],[4693,7317],[891,304],[1233,136],[1374,659],[1406,2198],[1875,4555],[2478,679],[3114,138],[3812,2332],[5906,1108],[311,115],[333,1055],[672,108],[2831,698],[2884,111],[6254,1063],[6891,395],[7276,111],[78,83],[336,270],[654,148],[778,136],[1324,900],[4110,142],[4799,141],[114,289],[118,305],[182,129],[276,375],[381,104],[434,885],[957,149],[993,305],[1932,1666],[2469,3034],[3814,642],[5578,562],[7600,2946],[287,307],[293,122],[323,144],[357,760],[464,189],[517,188],[1184,136],[428,109],[598,112],[696,450],[1020,168],[2372,2957],[2465,1666],[2951,2187],[54,56],[67,69],[112,960],[299,608],[328,184],[453,3165],[607,276],[1199,5350],[2338,7221],[4504,304],[80,570],[105,936],[324,325],[438,191],[752,2025],[2552,1118],[3941,276],[4146,188],[5217,270],[100,272],[296,903],[449,162],[645,191],[719,270],[916,173],[1011,270],[1040,143],[1074,6628],[1398,182],[2271,1837],[300,1533],[427,156],[2296,116],[388,2269],[440,3693],[682,399],[1167,150],[1176,2722],[1463,1187],[5966,132],[108,289],[112,2258],[297,191],[331,1474],[340,343],[599,1443],[636,160],[707,5394],[1202,3085],[3708,3708],[7329,268],[361,112],[751,156],[1093,1704],[1525,490],[4581,5547],[236,7733],[366,422],[732,175],[1061,930],[1346,115],[1986,128],[2307,2089],[2658,296],[6031,165],[299,960],[388,428],[685,922],[2225,2225],[2670,116],[2811,940],[2975,9315],[32,1044],[406,167],[735,137],[755,190],[1172,184],[2405,3305],[2793,357],[366,324],[489,182],[616,272],[644,1028],[840,99],[1218,276],[1335,2903],[1404,2376],[1749,3619],[2099,382],[111,120],[310,988],[315,136],[1166,1394],[1502,531],[1960,3209],[2528,1231],[347,1634],[388,304],[573,1335],[745,107],[1073,175],[291,564],[319,367],[388,121],[554,159],[699,507],[3210,157],[324,116],[396,370],[801,8306],[869,101],[1200,490],[1546,97],[1896,523],[80,65],[174,164],[798,180],[920,3277],[1504,158],[5551,3718],[140,147],[299,397],[323,180],[637,305],[739,139],[1037,553],[1170,382],[1447,976],[1717,6346],[2113,700],[276,759],[368,378],[589,826],[620,115],[654,159],[711,1227],[738,307],[756,112],[1376,307],[1482,98],[1758,1547],[1901,2429],[3226,97],[3601,4357],[3960,400],[7871,677],[293,1072],[299,6601],[301,145],[1296,884],[1453,188],[1891,851],[2090,182],[4663,1138],[8918,415],[302,2758],[441,189],[669,101],[725,2225],[741,793],[958,902],[454,293],[504,181],[573,384],[816,509],[1528,648],[1679,679],[2019,463],[3212,184],[4032,451],[99,1499],[268,995],[268,1647],[285,462],[288,107],[366,1100],[375,994],[826,1914],[879,453],[923,2592],[947,1045],[2020,153],[2547,6167],[5914,312],[6578,422],[333,815],[366,9265],[675,4286],[2344,383],[2483,155],[288,804],[317,764],[357,2698],[420,3147],[472,404],[998,129],[3295,1021],[3615,168],[4800,304],[53,57],[110,370],[383,1790],[464,145],[1119,2145],[1855,2715],[2323,109],[7773,277],[52,50],[103,305],[239,185],[299,2753],[514,1498],[790,430],[826,3784],[1134,566],[1284,131],[1301,345],[3670,161],[405,166],[588,170],[6230,357],[72,692],[110,304],[278,7122],[293,344],[333,1049],[385,615],[391,3357],[423,3002],[562,1136],[591,130],[670,272],[754,528],[4070,1386],[4199,479],[5982,1310],[76,69],[573,1412],[617,170],[645,157],[690,157],[721,2631],[731,173],[961,5985],[1033,149],[1483,908],[1591,391],[2287,48],[2307,289],[2400,2163],[2600,7848],[3350,2615],[274,321],[283,528],[410,132],[620,608],[791,4708],[1589,701],[1614,2167],[276,2244],[625,395],[947,1178],[3740,2941],[103,293],[114,519],[315,144],[375,4236],[390,521],[588,145],[864,7187],[1406,2242],[2091,307],[2155,553],[5842,830],[584,268],[674,138],[803,713],[1050,175],[1230,1104],[1292,1016],[1442,6296],[1872,496],[2100,743],[3668,165],[5692,160],[6156,144],[6920,1666],[7888,189],[745,5590],[886,1774],[1205,289],[1311,134],[1606,1370],[2217,496],[2537,376],[3010,1085],[3320,2954],[105,1789],[267,148],[309,189],[335,270],[1186,7556],[1355,1231],[1358,2125],[1523,806],[1856,400],[3195,3595],[3703,1758],[4502,344],[508,626],[534,128],[658,1019],[1114,566],[1559,1957],[1802,661],[4854,1200],[73,70],[285,109],[291,692],[336,2367],[831,146],[962,3697],[1012,142],[2498,105],[2552,811],[454,1709],[646,1620],[913,5395],[1095,5552],[2021,180],[2038,339],[2747,276],[7958,381],[299,970],[320,305],[449,167],[490,3114],[699,468],[1274,658],[3000,1458],[3975,84],[5474,128],[6856,917],[6923,115],[267,188],[453,2522],[662,134],[704,2959],[1059,4451],[1189,2163],[1334,169],[1642,1906],[333,1226],[467,116],[490,3951],[607,6566],[882,1582],[1261,138],[1447,679],[3641,5001],[115,1389],[374,171],[1625,1107],[2300,133],[3574,112],[4351,136],[5863,1791],[7039,8491],[285,3857],[317,321],[404,422],[629,173],[762,2703],[237,155],[278,287],[375,1696],[651,175],[690,186],[719,437],[726,888],[965,911],[2056,2761],[2745,117],[5005,1934],[5198,112],[6028,4072],[7472,1999],[397,1743],[495,304],[646,541],[669,111],[1429,837],[2369,110],[3212,130],[4783,150],[5855,1085],[100,330],[273,872],[281,9336],[347,2426],[385,1144],[413,161],[472,326],[486,613],[499,519],[598,116],[604,134],[1047,4349],[2271,428],[3589,1372],[6003,9755],[6039,2731],[52,51],[53,54],[78,65],[331,289],[341,268],[551,311],[644,5167],[856,112],[984,1843],[1465,1761],[2324,289],[3953,268],[4965,161],[66,83],[302,2703],[934,911],[973,181],[1123,272],[1280,901],[1408,2499],[4780,1369],[306,1384],[506,131],[881,156],[1403,289],[1882,1947],[2839,7102],[9748,2609],[263,190],[292,148],[453,507],[1042,2262],[1880,173],[5728,949],[87,3963],[310,806],[565,149],[625,3215],[679,776],[796,2431],[1630,121],[1987,117],[3664,1310],[347,386],[351,5577],[379,153],[746,161],[922,97],[1236,991],[1571,854],[3927,365],[295,129],[333,7686],[372,168],[940,3036],[973,188],[1054,1489],[1214,2049],[3516,1131],[4483,2860],[262,169],[299,564],[333,1541],[421,2575],[429,173],[442,1768],[457,3141],[620,422],[966,521],[1012,186],[3139,1461],[4797,578],[484,837],[597,184],[706,939],[752,407],[844,2142],[1056,152],[2851,1085],[3578,1088],[3628,304],[3883,679],[6341,116],[53,55],[77,83],[401,592],[436,180],[666,9046],[1963,270],[3560,272],[3823,4570],[565,137],[602,1804],[613,1670],[884,507],[1021,417],[1376,6198],[1429,288],[485,378],[599,376],[691,1509],[817,1660],[1136,716],[1217,367],[1406,733],[1531,578],[1726,1532],[2116,1540],[2585,1720],[2886,2941],[3571,2405],[274,289],[438,157],[462,97],[496,698],[1021,559],[1418,972],[1578,4649],[3318,1160],[52,49],[265,182],[445,137],[513,145],[677,365],[1090,430],[1163,2766],[1869,4091],[2023,733],[106,289],[114,395],[301,132],[421,5101],[646,505],[683,378],[2268,521],[3340,2143],[3646,950],[77,65],[110,272],[114,344],[285,2226],[308,171],[323,179],[333,1934],[336,312],[680,139],[738,111],[1098,150],[1119,365],[1141,145],[1329,3171],[2103,658],[5660,903],[10089,2142],[112,324],[121,270],[276,2772],[634,107],[717,186],[897,289],[963,1043],[994,2499],[1919,566],[1993,397],[2502,117],[3024,1622],[280,163],[519,894],[523,2284],[1232,5269],[1413,6632],[276,3758],[448,7114],[575,2918],[922,353],[1090,7256],[1789,270],[2327,2175],[2454,760],[4093,2100],[6469,611],[7951,4875],[298,149],[331,2337],[597,153],[653,179],[688,99],[696,545],[1322,1055],[1380,4711],[1675,116],[1793,383],[2651,851],[2891,272],[3716,3482],[97,781],[786,48],[1479,139],[2184,901],[2407,1422],[2503,1579],[2738,940],[627,188],[929,272],[999,1038],[3102,2242],[7326,2026],[67,108],[1000,566],[1053,159],[1284,182],[1902,367],[2322,2167],[2962,760],[4392,5176],[104,4446],[665,147],[997,1573],[1236,289],[2500,311],[5652,3635],[7886,683],[56,647],[67,80],[281,2428],[334,4173],[900,1402],[1358,2123],[2949,4356],[3504,537],[80,289],[289,6009],[354,730],[791,339],[815,1848],[896,1142],[920,2914],[1379,1340],[1510,146],[1668,268],[1762,131],[2560,817],[5267,780],[106,3754],[108,105],[613,1957],[649,180],[776,920],[1176,3785],[1641,523],[2421,1330],[3962,927],[65,112],[103,339],[235,156],[236,164],[337,1089],[357,2730],[589,562],[942,157],[1050,155],[1829,1019],[4012,324],[4189,1138],[379,164],[468,367],[762,122],[947,776],[1117,6486],[1444,572],[1672,587],[2660,3676],[6612,1286],[283,396],[285,3958],[437,115],[685,936],[733,972],[759,2061],[1385,276],[1988,110],[2280,1244],[2314,169],[99,3616],[299,5204],[311,289],[423,102],[579,558],[684,97],[784,592],[1188,1065],[1336,997],[1559,4778],[1871,169],[2646,795],[3823,4739],[4979,5001],[145,184],[645,147],[840,533],[1972,5769],[2202,343],[2574,185],[4269,1717],[4617,296],[6843,1439],[395,272],[468,384],[826,376],[2081,2427],[2244,276],[2528,4428],[2784,926],[3533,69],[7020,289],[87,465],[98,312],[342,188],[963,1944],[1927,357],[2246,417],[2671,1606],[2730,2388],[4174,339],[5846,571],[8584,4918],[281,621],[322,148],[333,1306],[672,2127],[689,153],[818,1735],[2239,8316],[6563,2108],[54,647],[340,639],[654,150],[818,1551],[865,919],[961,330],[1051,190],[2874,950],[3057,1958],[5294,80],[106,305],[108,747],[135,180],[280,6768],[414,176],[491,173],[668,180],[931,688],[1168,159],[5037,895],[6442,709],[373,181],[389,151],[421,4409],[653,157],[794,789],[1073,157],[1494,425],[2161,48],[85,109],[102,783],[300,1491],[421,2784],[595,2508],[726,97],[791,4601],[885,1809],[1718,268],[2268,1272],[2548,811],[383,730],[817,589],[922,111],[1000,2005],[1042,1394],[1361,2065],[1488,2867],[2815,6403],[3387,6606],[3420,100],[7297,4883],[98,5844],[522,3784],[923,321],[945,184],[1128,1574],[1419,184],[1895,939],[2645,56],[3174,1692],[3228,370],[98,921],[285,856],[375,385],[397,339],[450,566],[532,268],[564,330],[643,475],[855,399],[1044,270],[1342,4758],[1713,140],[2082,1898],[2412,2323],[3055,2112],[98,422],[291,425],[362,132],[665,180],[694,137],[741,1052],[1398,150],[1561,759],[1609,2481],[2111,102],[4192,168],[65,100],[269,312],[285,1724],[294,4499],[326,2402],[336,2204],[586,4043],[719,1026],[738,114],[826,3109],[866,189],[929,305],[1092,733],[1119,3828],[1213,305],[2623,1222],[2642,3394],[2734,1348],[3218,115],[3350,395],[117,97],[379,159],[477,817],[519,321],[529,181],[555,181],[627,128],[2447,865],[946,421],[997,2811],[1021,450],[2333,339],[7784,3581],[98,116],[291,277],[299,1263],[321,709],[733,1055],[946,1551],[1088,437],[1178,827],[1354,4621],[2595,176],[3244,788],[5216,137],[283,1317],[285,626],[406,137],[434,1533],[504,179],[556,348],[602,2228],[753,1699],[1245,276],[1255,184],[1512,365],[1681,2526],[1762,156],[2094,5233],[2466,992],[2852,895],[3120,2494],[3135,98],[4342,545],[1565,171],[1815,1210],[1831,479],[3045,795],[3134,137],[4121,850],[8707,287],[304,1307],[342,146],[401,428],[402,2622],[420,1757],[684,451],[743,1108],[887,1013],[1738,3722],[1895,558],[2364,56],[3962,4672],[7547,105],[103,307],[105,7339],[334,6293],[376,2292],[1314,311],[1747,1094],[1938,277],[2827,9105],[4113,3513],[4908,2143],[420,4634],[446,160],[515,270],[538,101],[651,157],[789,2474],[1126,149],[1184,149],[1459,167],[1795,981],[4620,3311],[235,8929],[320,7345],[333,766],[341,2168],[388,987],[474,171],[602,99],[634,1715],[738,305],[758,169],[868,1593],[1169,166],[1201,1781],[1531,272],[3734,188],[325,296],[335,378],[335,1240],[391,391],[408,157],[471,2924],[486,385],[550,139],[1047,382],[1248,2388],[1891,345],[2672,147],[3767,716],[117,806],[278,4648],[652,116],[1504,131],[2540,4778],[2595,142],[3540,6655],[4617,112],[397,1162],[402,5934],[409,153],[421,4013],[1020,152],[1322,615],[1525,1019],[3455,3464],[3626,1717],[4064,97],[7913,305],[98,305],[269,390],[278,274],[374,172],[466,134],[608,305],[611,475],[685,103],[1088,116],[1950,3203],[6952,347],[297,176],[333,976],[421,8849],[646,385],[832,384],[2041,519],[3122,4148],[5457,4151],[84,83],[281,2349],[281,5722],[320,2258],[503,2349],[613,386],[719,608],[2039,155],[2245,384],[3775,752],[9266,111],[99,425],[294,112],[421,711],[571,725],[1750,1639],[3103,121],[3223,829],[104,272],[105,960],[520,176],[1642,1489],[1697,842],[1916,2238],[4135,3064],[6420,391],[105,381],[927,395],[946,386],[1096,49],[1347,376],[2159,8613],[4443,268],[4559,102],[109,277],[131,313],[281,335],[299,9791],[495,519],[1595,10063],[1683,1221],[1991,811],[2174,48],[2321,306],[2384,356],[2939,1647],[98,451],[273,4931],[410,159],[503,5338],[774,911],[1199,4703],[1237,10780],[1353,490],[1860,3074],[3139,3429],[3369,1192],[3612,3850],[3749,142],[6691,370],[7463,2941],[10796,1254],[10797,190],[108,765],[278,1456],[366,564],[621,995],[937,186],[940,1980],[1767,1957],[2456,296],[5369,1421],[263,156],[283,6611],[390,2105],[397,102],[402,442],[741,505],[954,168],[966,272],[1049,2060],[1314,114],[1482,3551],[1517,1535],[3321,3873],[5399,531],[8823,5066],[194,161],[374,177],[402,2127],[452,3386],[469,151],[743,975],[1651,8449],[1691,7399],[2016,4675],[3191,752],[3427,1939],[5664,2323],[54,55],[294,8235],[485,82],[513,170],[679,795],[718,149],[1034,137],[2811,733],[3659,941],[4164,1231],[5110,156],[523,1021],[635,151],[2447,1610],[2483,156],[2774,272],[4344,4344],[9110,1759],[53,50],[300,2298],[319,383],[350,4173],[357,407],[455,173],[1126,176],[1770,143],[2057,1058],[3775,1409],[5140,2114],[8004,1474],[368,2274],[385,3559],[393,2086],[431,7392],[528,112],[578,270],[611,1802],[625,6939],[726,1646],[737,855],[924,908],[1657,2025],[1954,1173],[122,335],[281,872],[588,187],[598,6723],[704,1063],[998,128],[1101,854],[1349,1649],[1719,296],[3365,509],[4396,191],[105,1607],[269,8299],[276,928],[302,106],[330,305],[370,117],[487,348],[652,288],[942,170],[1565,145],[1671,928],[1871,151],[2448,115],[3601,1101],[4923,145],[299,7862],[638,481],[678,145],[753,97],[853,183],[979,875],[1124,979],[2339,562],[2611,3438],[8878,5642],[79,85],[280,131],[503,1432],[504,138],[517,140],[740,189],[795,1461],[796,475],[922,304],[999,1162],[1353,1251],[1462,647],[1628,2638],[1838,783],[7762,2166],[9787,854],[104,2546],[285,1594],[362,612],[388,105],[843,272],[874,2888],[1311,136],[2069,137],[2569,1400],[2728,1176],[2956,1136],[76,451],[278,335],[283,8642],[432,1119],[651,136],[658,1548],[944,116],[1447,1386],[1844,1639],[2826,7552],[3471,1540],[5649,7884],[5978,1440],[6058,115],[336,330],[364,321],[452,4621],[461,167],[721,3515],[838,643],[942,182],[1684,344],[2193,3752],[6059,692],[7282,1298],[7674,626],[276,1048],[280,159],[445,134],[497,138],[966,97],[1049,276],[1268,3484],[1289,182],[1701,486],[6974,4171],[98,1995],[326,927],[400,268],[434,4337],[688,121],[980,404],[1338,571],[1482,10396],[2295,361],[2549,2258],[2614,1808],[6508,9037],[6967,2523],[735,154],[954,132],[1147,163],[1236,1432],[1248,978],[1484,1446],[2554,339],[4003,178],[6941,4694],[8602,1604],[115,117],[364,829],[389,181],[625,270],[1073,184],[1177,533],[2801,417],[4041,307],[4218,289],[434,843],[490,3494],[532,353],[659,432],[784,5076],[1414,585],[1538,1504],[1790,340],[2604,940],[4497,8415],[5226,2370],[5313,107],[9400,723],[86,270],[332,1410],[347,486],[432,306],[943,98],[1187,892],[1366,5455],[1836,490],[2195,111],[2645,57],[2732,2094],[4224,8106],[6462,479],[277,390],[288,293],[304,121],[406,148],[484,335],[593,3424],[1215,844],[1407,910],[1689,381],[1710,187],[2165,7341],[3469,3101],[4915,531],[281,3153],[302,1652],[561,175],[600,157],[684,634],[1255,180],[9589,2542],[285,121],[312,628],[329,145],[388,6602],[1237,130],[1352,180],[1535,382],[2001,186],[2116,293],[2352,324],[2712,6200],[6459,494],[83,321],[98,353],[302,112],[646,667],[748,1766],[1433,3536],[1576,8375],[1856,465],[2601,121],[2690,155],[9598,11000],[82,65],[276,615],[293,404],[434,4940],[514,100],[625,3137],[640,9826],[759,1934],[784,3609],[864,171],[960,890],[1002,189],[1222,1839],[1591,2842],[2717,98],[4300,2048],[4656,2740],[8641,667],[83,311],[111,558],[285,1307],[339,109],[813,1104],[1611,152],[2896,276],[331,3609],[353,2597],[1141,173],[1783,3026],[2076,522],[2160,1400],[2344,11075],[5585,1949],[5961,2484],[421,1031],[573,3376],[1429,111],[1676,5825],[1717,6710],[2410,132],[3749,169],[6251,1046],[7462,3869],[101,122],[287,1440],[344,442],[371,186],[391,340],[523,545],[1318,312],[1527,673],[1877,276],[1923,4949],[2084,3499],[2394,348],[2465,886],[2712,7035],[3077,276],[3452,7691],[4044,1307],[54,52],[119,465],[120,120],[308,153],[309,191],[376,385],[672,378],[953,156],[1800,1481],[3053,116],[4102,1264],[5011,117],[5019,537],[5310,1526],[7230,1793],[83,104],[102,97],[274,422],[373,146],[529,182],[1054,2973],[1450,1062],[4490,1043],[84,65],[107,97],[268,2768],[456,138],[463,3562],[1036,496],[1342,4625],[1909,130],[2326,3444],[3662,296],[3666,390],[4169,1946],[6421,109],[6901,304],[8239,100],[49,2093],[67,68],[119,723],[269,5197],[283,2414],[463,743],[469,179],[589,1013],[1054,2825],[1178,615],[2136,156],[2991,304],[3135,116],[5149,545],[6553,115],[6904,272],[267,133],[281,5624],[367,5299],[406,174],[430,2931],[677,319],[1292,562],[1379,1656],[2412,6293],[2529,1743],[2867,2572],[3014,174],[3186,112],[8144,1932],[602,8643],[644,384],[1908,5256],[2044,1823],[5212,3606],[56,55],[452,1856],[623,385],[1041,396],[1711,570],[1869,1461],[4781,585],[269,5304],[310,356],[324,533],[325,105],[762,118],[1053,160],[1075,347],[1392,98],[1713,148],[2800,1693],[5117,453],[7442,10436],[66,108],[236,4274],[445,186],[465,927],[1073,179],[1293,553],[2065,1310],[2457,305],[3033,1458],[3576,2140],[4135,2748],[102,277],[107,4708],[296,268],[307,977],[320,1071],[376,1110],[411,128],[452,1669],[753,289],[1127,425],[1268,1585],[1542,347],[2231,3385],[2646,693],[3917,8033],[3971,1454],[5159,2985],[84,67],[281,1254],[357,545],[375,587],[433,183],[672,6441],[1245,522],[1290,566],[1308,1709],[1908,2028],[2952,1624],[4169,3542],[7852,3117],[8371,988],[32,542],[112,1011],[311,287],[401,1271],[404,901],[452,764],[611,1489],[703,1304],[1095,4230],[1383,8255],[1946,855],[2667,307],[5082,923],[105,850],[288,1491],[442,115],[458,149],[660,173],[721,5864],[1049,725],[3245,1758],[3706,465],[3915,688],[9476,3441],[53,49],[72,69],[116,2248],[283,723],[288,981],[320,105],[386,587],[423,885],[760,983],[953,132],[1404,101],[1736,1904],[2027,1283],[4685,272],[334,5455],[375,1029],[465,312],[533,1498],[655,348],[983,810],[1172,190],[2193,857],[4511,1185],[4586,3625],[114,270],[268,3813],[288,2396],[325,324],[424,131],[477,930],[495,288],[503,781],[669,300],[826,1131],[892,3659],[1139,480],[1566,105],[1656,1124],[1748,417],[1945,152],[1967,185],[2461,793],[5336,395],[6739,7193],[8200,2484],[10114,136],[276,2031],[306,1235],[666,1114],[1360,1777],[5044,571],[5062,115],[6033,102],[6170,733],[416,134],[475,1863],[578,288],[705,650],[725,276],[735,162],[776,833],[817,1867],[1609,2406],[2663,390],[268,104],[285,451],[518,800],[529,154],[582,607],[586,981],[869,6347],[897,628],[937,183],[3300,321],[643,545],[760,575],[853,138],[886,457],[916,165],[1262,566],[1683,2047],[1738,103],[2138,10010],[2576,2085],[2607,2439],[2706,343],[2840,3733],[4085,562],[5660,6082],[8373,939],[9704,4567],[11141,8067],[434,1669],[499,10554],[621,370],[650,1390],[1174,178],[1746,1369],[2203,8105],[2464,1682],[2598,2249],[5680,6099],[320,3433],[344,270],[693,3026],[738,8699],[863,9264],[984,9111],[1375,164],[3398,164],[4755,1539],[606,150],[659,5270],[666,3328],[733,1092],[1133,339],[1361,2012],[1450,1355],[2344,384],[3057,1164],[4722,3800],[7455,1606],[8219,5714],[53,52],[281,287],[483,149],[590,137],[696,523],[726,404],[727,312],[1328,1016],[1901,109],[6438,2593],[56,52],[114,765],[336,339],[377,884],[396,304],[548,353],[548,1209],[717,176],[1052,385],[1339,148],[3281,417],[7270,911],[348,924],[389,163],[498,152],[1010,161],[1342,1535],[1863,1520],[2262,531],[2394,487],[3045,704],[119,2304],[276,725],[366,3831],[463,3606],[491,178],[578,268],[696,417],[717,158],[1054,5663],[1055,941],[1867,11366],[2276,2586],[2619,109],[2694,1102],[2745,1263],[3846,1121],[5288,121],[67,79],[274,370],[452,2520],[463,593],[826,1756],[1216,134],[1285,7742],[1565,174],[5854,3217],[6735,1539],[362,1653],[395,97],[499,620],[610,133],[930,817],[1121,2391],[1175,1186],[1323,743],[1948,5287],[2352,1977],[2524,163],[2751,1250],[2851,480],[3588,3534],[5075,781],[5232,904],[6712,901],[7520,537],[66,67],[73,311],[726,5901],[1175,376],[8420,4499],[328,146],[396,305],[607,1418],[623,376],[845,181],[1081,140],[1321,417],[1339,190],[1477,129],[1810,6608],[4209,523],[6415,97],[102,310],[375,910],[856,10873],[1026,270],[1125,5156],[1182,2700],[1190,136],[1195,156],[1548,385],[1773,137],[3049,997],[5979,117],[6682,7776],[414,134],[484,1640],[865,450],[869,330],[1024,122],[1147,134],[1456,112],[1480,135],[2407,116],[3235,1102],[3747,370],[6412,4676],[119,2375],[340,924],[495,723],[991,122],[1538,7241],[3001,276],[3234,400],[5086,268],[319,1241],[425,111],[485,1540],[677,1089],[684,422],[761,530],[846,3205],[1202,5312],[1244,2015],[2097,10128],[4612,661],[5996,111],[6295,272],[9457,1993],[53,51],[69,114],[107,111],[296,1038],[331,10327],[597,166],[764,2592],[824,2151],[927,296],[1042,1030],[1050,148],[1197,289],[1819,276],[1965,1114],[2179,1787],[2619,2182],[320,307],[366,289],[430,1628],[466,136],[607,1682],[707,3511],[931,5415],[956,1921],[1206,1808],[1425,189],[1635,7516],[2087,1777],[3632,890],[83,5970],[118,952],[280,173],[281,921],[357,566],[416,190],[514,85],[542,5875],[748,2715],[885,2310],[1033,132],[1525,794],[1615,480],[1948,1822],[3522,122],[7502,2980],[10485,3460],[337,306],[354,3520],[392,176],[596,305],[621,960],[837,723],[1033,148],[1253,5380],[2041,1646],[2245,659],[2384,101],[4799,147],[55,49],[305,370],[421,716],[469,187],[588,149],[1878,6506],[2734,390],[2943,1484],[4364,325],[6247,103],[273,829],[289,370],[296,111],[522,333],[662,161],[1529,5176],[1906,2131],[1969,545],[2268,2664],[2400,1114],[6179,383],[98,397],[102,1999],[276,858],[306,908],[386,2537],[485,111],[579,988],[752,480],[753,616],[1174,134],[1400,1390],[1595,496],[1860,417],[2263,2370],[3902,2376],[8359,141],[99,1271],[278,592],[333,580],[406,180],[815,2564],[997,3778],[1484,2675],[1752,172],[1792,4093],[2110,3336],[10337,2725],[98,117],[98,4683],[278,1633],[515,2063],[624,188],[1246,523],[1744,564],[1748,450],[4604,990],[8574,772],[72,68],[110,462],[337,4461],[892,4149],[2626,1362],[2742,5066],[3304,383],[4214,1586],[4645,2978],[6641,272],[110,307],[315,169],[329,156],[613,375],[666,7057],[803,2253],[1472,8178],[2146,4858],[2240,142],[2464,276],[3530,3530],[5427,7548],[267,7466],[281,325],[291,3358],[349,145],[743,3707],[1285,830],[1361,1694],[1631,188],[2194,716],[2262,979],[2420,136],[2745,317],[4226,2913],[5263,6704],[6540,4530],[107,109],[239,184],[311,307],[456,135],[495,395],[532,305],[565,152],[881,174],[920,3109],[1233,169],[2964,276],[4032,3609],[4490,1720],[4654,8196],[5302,335],[333,2194],[349,147],[432,453],[467,4847],[669,97],[753,425],[1018,181],[1106,580],[1214,1830],[1375,169],[1612,347],[1823,417],[1911,3887],[2173,1540],[2364,57],[2969,1926],[5286,5018],[452,3960],[790,593],[888,268],[1053,148],[1387,7954],[1500,189],[2165,564],[6679,4294],[563,174],[569,180],[912,272],[1112,895],[1680,3182],[2606,103],[3161,1889],[4042,1505],[4723,1830],[5106,2360],[336,274],[336,1853],[352,545],[485,2034],[791,330],[929,1043],[947,2716],[953,143],[1106,4637],[1251,376],[1465,1777],[3090,794],[7182,97],[235,6768],[287,305],[299,9474],[421,691],[475,3032],[842,2219],[1029,795],[1204,144],[1405,2359],[1657,382],[1755,836],[1873,191],[1952,2960],[2342,2049],[2599,1134],[3236,3327],[3241,1599],[6172,1307],[9475,4185],[10653,2806],[76,4948],[273,783],[501,151],[514,289],[588,144],[654,179],[666,8482],[1048,733],[2383,383],[3123,370],[3310,959],[4064,305],[276,2789],[281,395],[320,1072],[382,352],[812,386],[1327,988],[1525,571],[4197,11597],[4819,272],[6342,2910],[10352,2166],[80,80],[295,144],[548,97],[597,169],[679,1218],[931,2727],[1095,10958],[1303,148],[1525,819],[1601,1466],[1621,137],[1982,417],[3512,270],[3604,321],[5968,1131],[8703,579],[345,509],[401,396],[459,185],[683,6292],[788,407],[842,6411],[923,277],[1126,136],[1143,4701],[1339,184],[1471,385],[1559,3859],[1748,357],[2014,168],[2099,918],[4239,1603],[339,1294],[401,395],[506,146],[756,3515],[855,1372],[869,5409],[914,769],[1045,3936],[1188,693],[1232,4187],[1260,184],[1672,4258],[1773,138],[1833,399],[2002,983],[3842,1688],[3964,1194],[3966,2437],[84,111],[104,1699],[299,2382],[341,1266],[341,7676],[368,287],[391,3141],[451,111],[503,105],[548,287],[814,2114],[837,688],[1022,176],[1066,794],[1245,399],[1264,4106],[2220,307],[2548,5419],[4676,2839],[6287,1789],[7200,404],[8398,1849],[80,293],[269,5479],[273,993],[273,3037],[285,8260],[420,5188],[474,162],[610,177],[1083,417],[1199,824],[1387,345],[1479,137],[1586,2667],[2729,766],[2773,2841],[3122,5581],[5742,5742],[5789,2221],[84,77],[99,370],[276,3983],[308,180],[455,163],[495,688],[596,274],[1067,153],[1188,3698],[1226,4011],[1321,1760],[1403,272],[2558,2324],[2657,120],[2802,5383],[3922,667],[8423,3665],[68,65],[110,5397],[302,345],[354,468],[791,6448],[799,3402],[1089,306],[1332,1509],[1386,1152],[1519,5233],[2639,480],[105,6592],[548,634],[910,582],[923,783],[951,184],[966,117],[1500,141],[1512,917],[2619,122],[3007,97],[6435,772],[8747,378],[486,646],[506,163],[598,3776],[932,138],[1046,101],[1303,161],[1628,386],[2392,1700],[2671,10819],[3490,3511],[3893,172],[4796,465],[107,121],[499,10878],[551,78],[721,5699],[926,386],[1082,892],[1273,467],[1449,1061],[2281,391],[2465,2785],[2663,804],[2979,357],[3922,2937],[4624,3013],[5640,1310],[8385,918],[276,3324],[310,390],[452,6968],[629,178],[1911,1579],[2261,105],[2347,365],[3047,679],[4951,1832],[6643,288],[299,7106],[340,1384],[470,176],[675,1616],[738,4197],[784,652],[1244,407],[1327,5182],[1339,149],[1438,1902],[1463,658],[1912,2523],[3542,2331],[5962,128],[107,272],[276,875],[281,598],[291,3754],[690,170],[897,1645],[950,1107],[1642,2012],[1759,1923],[3121,1185],[3234,116],[3568,276],[4240,4467],[4961,1222],[6021,12013],[451,272],[462,115],[548,5128],[913,725],[1071,270],[1265,941],[1609,1400],[3603,109],[4728,2494],[6860,425],[84,114],[272,993],[276,1274],[294,3007],[357,1290],[381,890],[448,982],[529,158],[587,1019],[593,399],[607,4087],[625,304],[795,376],[1711,9915],[194,180],[326,108],[368,620],[401,109],[442,1280],[514,1684],[556,2003],[586,116],[622,384],[731,145],[815,5315],[1210,457],[1534,1756],[1642,1321],[2481,822],[3249,1329],[3478,2185],[6177,1724],[269,428],[324,5282],[339,1849],[639,383],[840,325],[869,289],[943,112],[950,479],[1075,2023],[1123,305],[1305,10856],[1642,2065],[2776,688],[3019,11485],[4618,642],[57,56],[101,274],[276,2610],[367,468],[419,8826],[423,103],[431,119],[433,166],[442,97],[507,306],[597,156],[818,1759],[866,135],[868,3795],[1361,1781],[1618,173],[7639,6953],[73,82],[195,175],[462,2134],[752,611],[1039,995],[1459,152],[1484,553],[1725,399],[1727,3515],[2113,4286],[3863,2943],[5673,5098],[5745,11150],[83,80],[99,304],[532,381],[603,128],[605,181],[650,4245],[677,2327],[711,724],[770,319],[920,276],[947,975],[1068,794],[8046,370],[55,54],[111,274],[283,121],[322,173],[356,1949],[389,174],[513,164],[532,2317],[579,936],[622,306],[758,144],[1040,170],[1068,490],[1141,132],[1354,10224],[3174,3717],[3372,272],[4768,521],[5136,566],[6238,430],[6966,132],[97,353],[109,1536],[112,3386],[535,140],[653,134],[1040,138],[1363,811],[1668,1356],[2789,347],[2989,3870],[4154,642],[4209,7930],[5160,2387],[7315,861],[278,105],[283,395],[320,1209],[536,175],[772,2063],[1089,384],[1236,4407],[1299,9030],[1468,103],[1483,2866],[2464,2062],[3591,1965],[3637,115],[4087,6730],[6033,4901],[8427,1603],[99,977],[108,2753],[404,270],[523,6825],[548,289],[602,1785],[631,10271],[785,1019],[848,593],[869,970],[1047,4869],[1105,357],[1196,268],[1407,2607],[1923,1946],[2158,972],[3080,165],[9290,131],[99,1368],[276,430],[406,170],[895,1507],[918,2221],[1230,832],[1267,151],[1308,103],[1477,139],[1565,130],[1700,274],[1921,1006],[2249,1559],[2649,1262],[5082,4789],[9865,3406],[10336,952],[329,164],[339,1724],[349,181],[354,306],[717,133],[993,3362],[1285,1283],[1834,172],[2686,140],[3061,912],[4561,165],[4790,1341],[8287,8583],[8616,1897],[117,1593],[263,157],[495,2034],[503,312],[1029,1670],[1037,599],[1889,3029],[2635,111],[3086,811],[4723,1781],[432,319],[463,2081],[591,153],[776,1045],[913,1320],[953,179],[1191,1557],[1369,1781],[1519,1585],[2263,1870],[2645,55],[2670,339],[7880,4413],[345,1857],[354,3028],[380,137],[434,310],[505,1839],[514,1669],[535,148],[548,277],[734,173],[954,140],[1105,450],[1105,1958],[2565,772],[4703,305],[69,116],[119,1113],[273,4072],[283,268],[322,134],[331,442],[335,268],[643,450],[1027,188],[1269,147],[1290,8082],[1313,647],[1361,611],[2600,109],[3455,9100],[4532,1562],[5047,111],[5565,390],[6235,6091],[111,764],[291,528],[291,6265],[342,155],[353,1931],[521,628],[525,164],[635,140],[963,4277],[1119,494],[1130,643],[1943,128],[2415,752],[2539,659],[2773,1042],[3272,1574],[4956,325],[5579,304],[368,1940],[606,164],[640,1774],[796,2643],[826,1136],[839,100],[1206,832],[1210,4633],[1308,118],[1537,1489],[1545,611],[1942,741],[10698,1134],[283,287],[297,147],[341,10578],[343,2855],[347,1452],[493,172],[571,972],[683,564],[719,7890],[779,475],[891,425],[922,422],[943,108],[1106,7394],[1338,972],[1855,682],[1870,1781],[2334,3681],[2774,4989],[5065,6050],[6040,115],[6298,415],[7882,1118],[8188,268],[9017,2500],[85,69],[273,428],[326,428],[336,268],[484,8750],[961,584],[1153,305],[2281,832],[2338,824],[3660,296],[3966,861],[4109,4606],[10232,1370],[291,1796],[548,442],[1577,174],[1860,1173],[2409,599],[3027,276],[4306,1085],[4340,130],[8018,575],[10579,1740],[11347,1633],[105,1495],[584,121],[594,146],[685,1185],[836,385],[961,1572],[995,97],[1008,160],[1549,357],[1981,1627],[83,85],[320,2802],[523,2666],[573,9839],[878,141],[1252,1165],[1588,347],[1618,188],[1924,1097],[2837,7330],[2850,383],[3727,2741],[8600,131],[278,4083],[299,8751],[438,130],[447,2063],[452,106],[465,100],[545,2325],[727,311],[737,399],[1012,172],[1020,139],[1040,155],[1154,100],[1496,188],[1613,566],[4065,173],[4631,637],[6544,11102],[55,50],[67,289],[73,79],[370,270],[426,141],[534,181],[741,946],[920,3907],[1151,108],[1169,189],[1482,102],[1731,1847],[2630,1175],[2776,987],[3759,3026],[3868,2219],[5195,3307],[6481,270],[83,68],[364,6281],[368,390],[374,174],[652,370],[670,3640],[724,430],[1196,111],[1671,2092],[2007,376],[4034,479],[4206,1090],[6223,6223],[6902,1495],[79,995],[115,533],[401,1955],[434,2278],[443,168],[477,720],[667,1589],[682,682],[958,995],[1047,450],[1392,5015],[1553,276],[1901,2317],[1938,321],[2262,3432],[3940,111],[107,103],[109,5204],[110,692],[285,3544],[402,3483],[559,1667],[702,116],[704,1166],[1405,2010],[1904,2493],[2412,274],[2808,2269],[4064,304],[7406,925],[110,652],[236,147],[404,116],[467,97],[483,176],[625,1266],[754,296],[834,587],[857,378],[1127,7002],[1221,691],[1284,150],[2127,921],[2333,98],[2630,3277],[2653,4495],[2881,1231],[5854,2290],[102,1266],[320,7328],[346,163],[499,4679],[563,173],[615,593],[634,272],[715,181],[902,2063],[1104,703],[1238,463],[1354,8660],[1361,2370],[1719,105],[2032,134],[2599,475],[3549,382],[10433,1113],[10693,2871],[12467,6900],[98,425],[320,1853],[343,365],[662,170],[721,2860],[921,270],[1469,1426],[1719,993],[2923,2821],[12500,1521],[84,82],[117,289],[162,176],[276,2594],[323,168],[388,5444],[645,171],[788,1622],[1100,537],[1214,3078],[1261,172],[1972,2983],[2029,8977],[2795,1889],[2979,2312],[3489,5070],[106,112],[107,105],[319,1902],[403,187],[800,453],[897,97],[1957,347],[2976,1804],[3220,7014],[4795,1108],[6454,2202],[9365,339],[10747,102],[83,79],[104,1072],[269,1505],[285,8932],[288,894],[319,859],[640,813],[761,1218],[873,8685],[904,2371],[980,270],[1298,788],[1324,399],[1592,103],[1967,140],[3226,272],[3979,390],[4066,272],[4147,385],[4231,2596],[76,305],[401,353],[523,643],[611,2902],[665,128],[719,3803],[900,276],[2736,4577],[3160,818],[4828,11309],[4985,164],[6252,9968],[7432,772],[11433,1254],[104,584],[117,1078],[117,1457],[117,2086],[155,176],[356,837],[359,185],[426,173],[440,851],[455,130],[467,111],[508,621],[537,117],[635,176],[796,1343],[979,2076],[1150,1888],[1675,272],[1792,1763],[1945,140],[7917,107],[401,277],[463,1017],[552,136],[666,4349],[749,169],[1376,5333],[1441,188],[1479,154],[1738,3206],[2059,1262],[3220,8158],[4295,421],[235,12513],[368,335],[381,533],[388,2274],[401,99],[402,4156],[434,2115],[544,4115],[557,179],[703,1932],[704,1717],[1096,647],[1208,5683],[1325,693],[1469,566],[3112,830],[3502,1775],[5824,1472],[7528,1691],[7619,173],[8788,3424],[72,105],[114,850],[306,1335],[324,5190],[379,171],[424,129],[717,168],[732,150],[818,421],[844,813],[940,2421],[1059,3085],[1093,1989],[1511,103],[2611,2130],[2728,883],[3146,117],[3155,3153],[4035,1078],[4528,1194],[6206,804],[54,51],[102,970],[272,97],[390,2209],[472,816],[562,1138],[1247,592],[1387,2702],[2459,399],[2511,533],[2826,4151],[2881,1890],[2927,9559],[5726,783],[227,142],[229,182],[474,150],[507,556],[531,2748],[615,1055],[646,613],[654,140],[661,450],[701,4321],[925,3506],[949,956],[1089,2578],[1127,616],[1241,2175],[1504,164],[1566,7833],[1880,144],[1984,2310],[2239,381],[4178,1218],[6378,3525],[273,6735],[278,608],[281,4565],[435,451],[457,1932],[602,533],[665,162],[691,2845],[711,4224],[766,593],[1050,157],[1169,165],[1482,108],[2165,3196],[3747,293],[4534,10762],[4972,847],[49,471],[56,50],[285,1272],[326,3217],[345,1266],[383,468],[496,920],[614,4195],[707,131],[743,1573],[1066,819],[1334,152],[1341,1192],[1875,1218],[2251,6650],[2383,453],[4719,109],[4751,370],[11160,339],[104,1892],[294,5974],[342,184],[350,5798],[499,578],[551,73],[619,186],[642,1139],[651,143],[719,2458],[829,119],[1250,1590],[1872,5037],[6863,468],[273,8968],[307,533],[336,467],[354,5551],[375,1672],[601,138],[862,187],[1306,463],[1450,865],[1566,2993],[2103,2770],[2245,306],[2792,720],[2981,9215],[4889,160],[9418,571],[107,400],[122,105],[304,804],[336,317],[529,180],[882,837],[973,175],[1059,11248],[1258,133],[1679,4858],[1842,1757],[2814,168],[3261,339],[3617,703],[4361,3045],[8442,1714],[10915,2514],[296,6689],[426,175],[430,933],[503,3752],[563,188],[588,128],[666,382],[764,1317],[1112,1437],[1154,2277],[1376,2507],[1754,172],[2331,3920],[3236,580],[4325,422],[457,367],[474,172],[508,396],[691,861],[835,101],[945,156],[997,2023],[1112,599],[1289,177],[2017,1774],[2608,752],[3596,110],[5634,781],[6728,1498],[12248,432],[78,69],[283,572],[300,268],[334,1439],[334,2562],[388,289],[390,121],[504,189],[662,181],[945,141],[1118,2867],[1407,1264],[2337,422],[2475,104],[2840,819],[4163,2366],[5699,890],[6289,1355],[8419,187],[8787,747],[108,339],[117,1789],[285,9409],[386,1049],[397,3642],[428,1640],[2073,2282],[3257,165],[4421,1209],[5474,129],[324,118],[366,2304],[571,673],[575,566],[595,1941],[721,4529],[1037,1484],[1362,1585],[2254,2045],[2344,367],[2361,155],[2784,822],[3220,361],[5040,1532],[5440,5598],[6086,111],[7617,1519],[12624,2142],[281,1357],[447,325],[633,184],[705,716],[726,3172],[791,4024],[815,1055],[982,354],[985,399],[1178,725],[1538,815],[2011,5145],[2028,417],[2079,1250],[2425,450],[2470,354],[2569,582],[2672,128],[3524,356],[5280,129],[8570,370],[9170,118],[268,578],[281,3042],[326,344],[347,1859],[355,153],[374,173],[606,129],[653,138],[765,100],[788,3957],[926,2715],[1096,53],[1202,2189],[1565,162],[1601,1367],[2683,2049],[2884,307],[2885,185],[4553,523],[5623,724],[138,559],[280,158],[299,1796],[312,1633],[589,1636],[596,345],[648,276],[796,382],[965,383],[1082,2167],[1240,270],[2567,128],[2786,1367],[4163,3030],[4242,2798],[4383,679],[5456,2967],[5942,909],[6616,370],[6697,272],[99,324],[274,304],[296,370],[375,5179],[475,1290],[552,168],[765,2085],[784,451],[983,1062],[1466,907],[1792,836],[6644,1136],[9655,579],[65,4798],[108,921],[276,5184],[294,1422],[594,185],[597,133],[812,1589],[897,370],[1085,1062],[1438,360],[1524,558],[1704,3479],[2136,149],[3458,1706],[3588,1571],[4478,6428],[7188,923],[56,51],[65,109],[73,102],[310,2816],[495,345],[607,6716],[625,111],[886,1948],[966,307],[1009,171],[1119,677],[1235,2358],[1441,185],[1736,1198],[1759,822],[1810,3370],[2029,1939],[2971,1000],[3202,911],[3360,3360],[3615,180],[3915,97],[4056,780],[5008,1421],[5211,321],[5616,1857],[9248,140],[103,5934],[288,829],[386,7238],[402,425],[415,100],[484,272],[518,583],[929,6810],[937,172],[1114,545],[1353,3391],[1404,1317],[2157,4336],[2165,378],[2392,8647],[2558,1155],[3393,709],[3839,268],[5501,620],[7530,1849],[7792,1902],[9410,9410],[12075,5970],[121,109],[273,9831],[283,987],[311,321],[312,634],[499,330],[820,1052],[859,12707],[1246,978],[1744,277],[1769,151],[1816,1062],[1996,1609],[2177,101],[2650,496],[2972,1321],[3830,172],[4878,312],[5202,533],[6698,2452],[8650,2901],[8979,152],[107,5768],[302,9261],[306,384],[322,135],[370,111],[423,465],[432,677],[953,180],[1077,116],[1104,1666],[3617,4665],[4453,272],[5110,152],[6494,115],[8755,8688],[105,1475],[122,397],[452,857],[462,274],[477,799],[655,306],[690,154],[809,385],[1248,2045],[1267,145],[1380,2797],[1545,6121],[1585,2155],[4077,370],[4713,628],[306,2293],[334,11975],[396,1307],[449,134],[492,149],[590,157],[703,1412],[755,166],[765,104],[957,138],[1208,1574],[1336,854],[2890,1118],[3149,183],[3529,119],[3620,169],[3931,2573],[4609,1573],[150,407],[452,3013],[523,10628],[557,168],[665,186],[672,293],[1041,709],[2136,128],[3534,1571],[3852,3495],[4412,1630],[5345,566],[10039,168],[73,66],[118,307],[375,1855],[386,1031],[530,1106],[575,4232],[822,833],[961,11295],[1208,1359],[1346,2204],[1618,144],[1838,378],[2029,1115],[3449,2821],[4131,103],[4680,3053],[7009,537],[68,80],[655,507],[726,5528],[784,1700],[1453,160],[3365,10987],[5140,1814],[5174,918],[5259,287],[6885,1446],[7295,352],[11687,5566],[55,51],[76,65],[111,1623],[273,311],[281,5138],[294,321],[294,356],[299,2593],[309,9716],[433,140],[550,144],[552,172],[610,148],[864,153],[1005,774],[1100,683],[1191,1916],[1216,143],[1295,505],[2963,268],[2969,938],[4391,2319],[6358,376],[12764,1230],[73,115],[237,512],[276,900],[309,8186],[388,7861],[391,1412],[394,189],[401,330],[534,134],[621,781],[726,396],[743,725],[1125,2034],[1272,111],[1547,2510],[1889,2729],[2876,277],[2885,168],[4037,9063],[4312,121],[4816,2134],[5806,332],[7764,822],[8089,115],[77,80],[101,936],[115,277],[236,7466],[270,321],[281,7594],[423,1646],[458,170],[666,1321],[1247,270],[1590,1671],[2569,895],[2965,1000],[3320,1472],[4452,3446],[288,1162],[368,923],[495,270],[507,337],[631,112],[1001,176],[1235,1210],[1678,480],[1706,842],[1895,107],[3139,2170],[3371,422],[4263,2209],[7874,6771],[105,851],[280,149],[434,112],[495,5204],[532,378],[584,287],[614,1852],[765,293],[834,2393],[896,106],[938,1726],[996,5346],[1304,844],[1313,2924],[1531,437],[1552,830],[1747,102],[2113,990],[2179,3611],[2516,1721],[2786,399],[2976,578],[3287,145],[3366,1083],[7142,479],[8426,872],[9012,2402],[9198,1720],[11772,1475],[107,277],[109,1348],[239,191],[385,599],[531,571],[578,97],[629,170],[721,7759],[947,1108],[950,357],[997,2158],[1104,6702],[1648,149],[2197,333],[2405,3198],[4044,9148],[6962,783],[32,1562],[330,451],[367,2701],[385,333],[588,129],[598,111],[669,3515],[703,3351],[817,399],[928,595],[1808,10213],[2481,769],[2653,997],[3097,451],[3171,3104],[3452,1640],[4641,306],[8850,1421],[9189,921],[109,626],[299,2259],[528,98],[707,7066],[758,154],[759,2050],[1060,357],[1145,149],[1229,190],[1408,2073],[1514,276],[1569,415],[1703,1942],[2227,132],[3317,1647],[3375,115],[5567,507],[6159,1758],[6174,3492],[10565,176],[69,274],[353,272],[355,140],[375,848],[392,134],[587,1222],[974,144],[981,533],[1000,479],[1466,933],[2071,311],[2607,2461],[2865,522],[6651,1082],[7319,340],[98,1646],[270,533],[276,4105],[285,902],[328,128],[345,2702],[568,178],[705,1057],[961,400],[1119,911],[1326,1166],[2002,8381],[2124,531],[2366,4898],[3090,3733],[3121,287],[5585,1742],[5912,4142],[7550,305],[117,829],[237,6097],[263,179],[278,381],[304,370],[306,1962],[315,181],[416,156],[486,505],[504,132],[532,723],[684,2753],[1001,168],[1059,2219],[1445,169],[1477,133],[1582,101],[1668,3056],[2636,270],[3051,7800],[3656,5694],[4492,987],[5062,307],[9691,9651],[97,1546],[168,1343],[495,621],[507,1028],[685,293],[882,10171],[1065,1065],[1349,1763],[1528,615],[1549,407],[2242,773],[2441,5022],[2795,997],[3033,3611],[3200,4041],[5849,628],[291,2963],[299,9127],[327,157],[597,147],[703,3069],[739,168],[907,1406],[958,78],[1013,724],[1168,152],[1402,691],[1473,177],[1573,1118],[1591,1822],[1635,11997],[1710,184],[2477,6748],[3011,2906],[4108,4757],[4121,1505],[4153,2523],[5273,890],[8682,804],[9845,887],[10064,4583],[119,370],[285,2670],[344,115],[384,432],[416,151],[862,145],[935,181],[1301,344],[1436,168],[2116,3864],[2192,57],[2320,307],[2343,2243],[3234,5468],[3962,2589],[4085,2646],[7819,479],[102,1606],[276,766],[367,365],[495,121],[571,985],[602,109],[639,4475],[940,2255],[1622,352],[2311,3187],[2364,55],[2365,2146],[3494,7303],[4257,99],[4889,161],[6138,1491],[7989,1688],[105,324],[194,174],[283,390],[308,190],[743,3440],[752,865],[776,3188],[855,276],[888,97],[941,347],[1100,101],[1159,2167],[1235,2785],[1309,1844],[1386,276],[1904,822],[2510,8130],[3418,752],[3449,4450],[5512,272],[5692,7066],[11151,136],[102,1563],[104,10235],[287,1294],[393,307],[401,9662],[499,1100],[516,537],[625,1563],[640,1141],[659,965],[803,918],[940,4555],[1107,566],[1719,121],[2133,102],[100,1142],[117,121],[280,12578],[320,3803],[358,189],[367,3183],[415,324],[420,1395],[704,1306],[837,4580],[930,276],[1122,1639],[1686,370],[1932,10219],[2257,1630],[2442,593],[3533,71],[5072,4357],[7209,6632],[8542,111],[11521,1849],[79,114],[102,1071],[108,621],[302,11524],[776,333],[952,709],[3007,307],[3014,132],[3309,6076],[3917,573],[7327,832],[101,1562],[122,119],[298,165],[420,5337],[497,186],[508,288],[555,191],[586,6586],[650,1736],[710,148],[932,147],[1183,1338],[1569,2105],[1588,812],[1777,1360],[2183,276],[5797,296],[273,837],[494,879],[495,1348],[760,566],[1201,1302],[1363,531],[1391,4699],[1842,2163],[2111,1768],[2406,769],[2933,3567],[3342,268],[3349,1325],[3850,3171],[8167,304],[339,829],[401,268],[503,1546],[607,741],[690,162],[693,691],[784,97],[796,1520],[961,116],[1031,8008],[1124,1731],[1376,2332],[2991,289],[3266,135],[3931,3325],[5406,1192],[121,272],[233,3854],[261,172],[263,161],[266,178],[367,432],[369,2702],[376,741],[415,289],[542,579],[564,442],[595,673],[622,556],[688,1947],[698,2130],[756,107],[1051,162],[1114,1958],[1154,598],[1314,8555],[1773,142],[2069,180],[2862,640],[3121,1511],[3470,287],[5044,766],[6910,6910],[10146,6091],[98,1071],[112,579],[320,428],[331,5076],[347,505],[444,154],[521,1665],[784,952],[913,1683],[1354,856],[2879,1687],[3821,3732],[4809,1776],[6031,179],[10728,121],[289,114],[310,311],[315,148],[331,628],[375,5580],[389,133],[535,149],[583,348],[921,103],[1072,114],[1085,357],[1552,116],[1555,1992],[2044,3078],[2483,170],[2932,145],[5889,99],[5938,101],[9413,176],[56,49],[101,921],[108,404],[116,122],[237,147],[299,467],[341,7678],[423,5225],[448,3238],[508,442],[680,131],[931,2749],[1127,415],[1153,378],[1308,3111],[1327,325],[1353,3144],[1402,1778],[1517,3058],[1549,2306],[2031,430],[2059,950],[2087,1082],[2231,3438],[2822,2835],[2935,347],[3295,788],[3681,169],[3780,7248],[6309,1166],[100,634],[102,1357],[462,268],[472,293],[966,287],[1191,1268],[1412,2189],[1993,1378],[2478,1455],[3127,450],[3228,422],[4386,2148],[294,7948],[398,163],[434,1365],[521,533],[614,99],[615,1547],[681,2687],[1163,983],[1329,531],[1701,820],[1843,1953],[2069,11719],[2454,480],[2497,1031],[3714,170],[5089,2091],[54,50],[67,84],[112,425],[291,422],[367,1005],[377,319],[457,383],[503,325],[541,1016],[587,399],[588,139],[675,2856],[979,997],[1013,1332],[1095,1602],[1555,1013],[1635,3035],[2235,119],[2607,6808],[3372,268],[3624,5037],[4696,1307],[4744,117],[5221,304],[5352,4762],[9163,2546],[9406,901],[10421,268],[104,1151],[108,115],[110,538],[110,1734],[278,10415],[350,9271],[375,1035],[401,326],[726,2757],[801,2071],[1036,8922],[1322,2662],[1642,3066],[2054,270],[2082,375],[2120,122],[3397,367],[6772,111],[116,335],[349,189],[396,122],[528,6723],[856,270],[874,480],[1033,4418],[1067,160],[1235,7125],[1858,4198],[1929,1121],[3643,417],[3783,270],[4245,3240],[6020,3619],[10824,381],[97,428],[194,160],[273,537],[276,1377],[300,122],[320,353],[402,3614],[424,138],[485,345],[497,177],[595,2292],[690,191],[713,983],[719,1044],[984,1481],[1041,1743],[1156,2170],[1314,109],[1425,136],[1680,811],[2352,1806],[2456,9235],[2606,1162],[3375,683],[441,140],[572,289],[891,1554],[1075,2811],[1314,103],[2276,480],[2822,811],[2962,2025],[3098,6432],[3490,147],[4700,132],[6643,120],[77,312],[78,692],[319,884],[350,783],[350,2484],[696,352],[776,1776],[827,972],[1063,276],[1536,116],[1651,769],[1668,592],[1972,382],[2764,1323],[3447,1378],[5168,6911],[7537,98],[7940,995],[73,69],[293,305],[336,2273],[442,3694],[490,925],[521,305],[648,4673],[871,450],[1156,1340],[1172,174],[1219,2770],[1288,2730],[1423,1423],[1753,4159],[2223,390],[2339,496],[2600,683],[4214,5262],[4668,3794],[8515,806],[8582,1692],[10305,272],[312,765],[696,559],[796,3124],[833,667],[904,1016],[963,764],[1089,354],[1236,885],[1367,347],[1991,490],[2013,1277],[2155,607],[2171,276],[2422,117],[2735,370],[3323,107],[3752,843],[7781,161],[7782,289],[8535,1457],[57,55],[273,533],[298,186],[299,574],[306,5977],[388,8986],[463,759],[573,3069],[773,1257],[813,832],[1582,2993],[3234,894],[3467,545],[4582,287],[8746,1944],[11810,1134],[76,10466],[283,9996],[294,533],[302,3115],[331,1271],[432,911],[527,132],[669,2559],[1054,4626],[1159,1186],[1466,904],[1863,2049],[1938,293],[2032,135],[2073,1996],[2208,475],[2392,8783],[3010,480],[3060,276],[4042,850],[4164,1965],[6740,3320],[11096,12377],[233,742],[278,5719],[281,2071],[302,7352],[368,2529],[426,189],[516,2134],[529,143],[529,171],[684,344],[726,1205],[858,276],[921,2191],[1125,3507],[1387,6893],[1456,115],[2121,3569],[2200,132],[4988,1822],[5168,640],[6092,764],[7748,2536],[11063,573],[103,2089],[234,180],[267,9061],[299,4770],[376,716],[495,451],[528,3525],[607,613],[765,1789],[815,276],[946,2823],[2192,50],[2361,151],[3969,289],[5643,5333],[57,54],[588,135],[738,2309],[998,185],[1156,1924],[1298,545],[1531,2071],[3344,272],[5019,1673],[5664,9771],[7649,101],[9628,988],[69,1684],[98,2308],[108,5002],[276,1812],[423,2296],[438,147],[499,6098],[531,920],[599,1418],[791,277],[796,450],[859,12734],[937,144],[1000,450],[1023,149],[1081,190],[1139,3058],[1245,795],[1680,822],[2195,5224],[3821,788],[3872,1955],[6700,467],[10980,3744],[67,11523],[79,87],[107,783],[234,188],[278,277],[305,321],[374,135],[496,4566],[548,1123],[573,3141],[611,450],[637,277],[638,142],[653,165],[836,1277],[993,1307],[1335,844],[1641,2698],[1994,276],[2271,6771],[2453,795],[2587,272],[2938,1508],[3798,811],[3964,1048],[4953,270],[5664,5379],[8333,3129],[9272,11085],[67,101],[98,851],[296,108],[298,190],[350,843],[653,168],[762,8927],[879,306],[922,272],[1067,173],[1380,2863],[1383,118],[1430,4187],[2827,2648],[3884,580],[4226,311],[4689,545],[5601,804],[6866,5482],[7799,3026],[10835,981],[70,105],[274,105],[274,806],[283,115],[288,2319],[306,886],[779,2312],[857,425],[879,494],[1246,3328],[1289,187],[1325,3748],[1795,97],[2516,109],[2806,370],[3409,1947],[4044,3446],[4624,783],[104,1038],[108,1317],[291,390],[300,100],[424,151],[586,99],[891,952],[923,293],[1232,910],[1471,1777],[1566,2868],[1570,4747],[1644,2954],[2841,12741],[3168,676],[3775,788],[6114,1937],[10113,422],[32,3439],[65,86],[269,287],[368,317],[764,116],[1044,890],[1256,1110],[1364,992],[2102,116],[2114,415],[2477,1934],[2511,115],[2616,3558],[2676,3645],[2834,2803],[3010,559],[3185,8631],[3365,3702],[3620,129],[4604,700],[4830,1870],[6089,2621],[8113,111],[86,68],[437,293],[472,68],[529,148],[606,182],[644,340],[1108,580],[1391,1070],[1537,642],[1588,667],[2176,1218],[2473,965],[2858,2523],[3195,6412],[10608,3362],[10901,7759],[111,339],[122,121],[285,378],[326,2166],[346,171],[375,883],[380,138],[532,2296],[704,2675],[752,3347],[754,5093],[766,6675],[817,553],[1141,154],[1569,2337],[2462,2822],[3047,1519],[3349,1188],[4090,1330],[5746,1187],[6054,353],[7666,400],[8087,847],[8644,7647],[10985,304],[54,49],[268,3676],[278,272],[302,370],[341,7826],[385,1066],[386,2426],[672,626],[762,4770],[773,716],[891,296],[1119,2955],[1411,505],[1559,2100],[1792,1649],[2135,3937],[2344,340],[3545,110],[3688,161],[4556,1767],[4816,3629],[4893,9013],[4985,166],[5221,1457],[6879,116],[79,71],[105,939],[294,106],[331,4407],[584,709],[586,9142],[724,875],[1423,391],[1642,1787],[1801,4139],[1805,158],[2380,941],[2956,1176],[3589,648],[4123,1283],[4592,2714],[4865,2446],[5718,3800],[6530,1118],[7545,1030],[9888,1370],[10843,307],[320,437],[333,571],[347,477],[485,3262],[555,129],[610,149],[904,7624],[1054,7116],[1068,421],[1136,582],[1424,361],[1449,1367],[1583,190],[2835,907],[3724,2953],[3759,2070],[5171,4086],[6438,121],[8124,2727],[276,2696],[866,174],[1204,168],[1376,7312],[1547,1682],[1697,12373],[1881,11285],[2229,2969],[2301,3214],[2394,1925],[2551,3661],[2887,1409],[3604,311],[3834,1158],[4741,152],[8146,537],[8518,1219],[10,1120],[102,1370],[229,181],[336,395],[347,819],[347,1281],[368,1718],[384,453],[666,407],[739,153],[910,716],[928,1004],[929,2749],[945,167],[1000,6562],[1062,1083],[1087,400],[1108,941],[1147,145],[1996,1075],[2118,376],[2245,367],[2258,538],[2832,616],[3146,692],[4110,160],[4423,894],[11236,832],[102,378],[262,190],[316,157],[348,468],[534,130],[776,1877],[869,370],[912,2396],[1483,4568],[1760,417],[1771,9258],[2488,3245],[3086,6169],[3556,330],[4748,97],[6133,7474],[6931,806],[55,52],[299,1205],[375,1588],[671,165],[684,1142],[690,160],[758,176],[904,1960],[1027,156],[1201,407],[1214,3621],[1290,752],[1329,3327],[1340,1461],[1404,10613],[1443,2282],[1471,3100],[1488,698],[1523,3303],[2004,972],[2481,607],[3057,559],[4136,425],[10989,1389],[103,3614],[262,161],[296,305],[569,179],[645,167],[704,994],[2324,111],[3903,1043],[7694,2003],[8843,2531],[77,287],[122,2396],[268,370],[432,1005],[524,2093],[858,700],[1536,370],[4626,1021],[5154,2100],[5207,2954],[9067,10983],[69,79],[71,101],[103,422],[118,274],[401,1368],[453,1241],[701,1112],[736,153],[784,3046],[866,175],[960,97],[1096,50],[1112,1305],[1133,395],[1151,936],[1329,2331],[1794,180],[1866,148],[4369,652],[4574,1334],[78,97],[195,131],[234,8186],[278,465],[285,3616],[297,174],[320,1263],[333,1070],[401,608],[402,1272],[445,190],[658,737],[711,2391],[796,6336],[942,162],[972,1544],[1012,190],[1106,994],[1189,1667],[1248,352],[1773,177],[1928,9554],[2300,146],[2410,136],[2588,480],[2622,1491],[2670,109],[5905,1937],[6500,616],[8621,1665],[80,71],[311,2977],[331,12475],[402,304],[409,145],[484,370],[531,1980],[588,173],[609,133],[622,6234],[675,430],[748,587],[779,1259],[791,6219],[814,1526],[909,910],[1000,2798],[1035,385],[1177,8806],[1938,9601],[2549,2091],[3004,972],[3467,3211],[3647,9076],[4536,1367],[4949,733],[4953,268],[97,1498],[100,804],[291,274],[776,1819],[982,383],[1123,404],[1333,1016],[1342,1105],[1362,463],[1787,357],[1833,276],[2038,1879],[2124,3429],[2132,1890],[3502,2487],[3529,98],[5249,1724],[5422,437],[6123,1720],[9466,467],[65,1196],[115,509],[285,1540],[352,1844],[452,1985],[468,2584],[486,375],[588,141],[746,131],[1090,1703],[1179,367],[1229,164],[1339,186],[2268,890],[2506,9552],[2613,1330],[2884,305],[4248,12310],[4257,1832],[5406,2650],[283,2777],[338,132],[414,173],[457,898],[510,137],[576,133],[709,370],[1010,164],[1139,611],[1368,115],[1504,176],[2407,6009],[2663,1142],[3290,9007],[3413,2226],[6373,4217],[10117,9521],[102,8201],[281,2382],[531,1839],[652,2209],[719,8672],[768,188],[799,826],[830,122],[1287,140],[1316,184],[1318,1474],[1450,2163],[1644,5312],[1757,480],[2002,3021],[2133,10045],[2805,111],[3556,1976],[4884,3058],[6047,272],[8754,2112],[9531,7538],[12035,304],[117,598],[419,116],[532,8706],[580,1055],[693,2700],[754,437],[768,157],[859,365],[889,153],[1178,571],[1201,752],[1274,817],[1657,810],[2121,103],[2197,1767],[2280,5810],[2558,109],[3942,6200],[4313,575],[5078,904],[51,1841],[304,2819],[348,730],[776,691],[1354,109],[1392,108],[1406,1328],[1501,1228],[2607,421],[2981,652],[4214,7389],[8109,115],[83,6503],[264,137],[283,5528],[286,181],[347,946],[467,995],[590,185],[657,164],[672,3433],[1117,1850],[1733,5338],[1851,305],[1860,1085],[2401,2536],[3226,111],[3582,1767],[4731,2577],[9069,307],[105,5607],[109,781],[288,102],[421,421],[607,490],[702,537],[724,615],[818,3279],[975,580],[1107,382],[1175,2617],[1426,407],[1468,115],[1696,399],[1770,175],[1935,611],[3049,3633],[3508,628],[3670,172],[5090,7534],[6093,276],[6319,415],[67,77],[77,121],[102,3345],[276,1514],[324,843],[368,521],[395,111],[532,97],[863,5386],[1032,172],[1097,276],[1248,2012],[1442,6487],[1953,4675],[2132,5162],[2199,679],[3814,810],[4146,185],[4917,340],[5273,1888],[5820,97],[84,4471],[106,3483],[117,104],[311,894],[340,556],[586,1273],[653,131],[720,826],[726,4083],[826,1101],[833,1379],[855,4704],[976,1159],[1124,7208],[1274,1187],[1375,137],[1473,171],[2073,1761],[2163,1302],[4469,2166],[4809,8669],[5402,7894],[264,135],[299,344],[391,332],[402,1675],[532,422],[532,1995],[670,5376],[839,1346],[914,2185],[1139,382],[1183,7788],[2044,788],[2496,1105],[2549,277],[2805,97],[5601,2108],[5639,480],[5996,307],[6668,1487],[10996,11114],[11058,7663],[78,71],[100,312],[281,1511],[281,3544],[383,911],[419,5935],[485,2304],[573,10009],[645,162],[684,437],[745,98],[938,276],[1127,321],[1186,3104],[1298,382],[1568,2210],[1899,376],[2330,1185],[2470,5831],[3574,1094],[3663,100],[3749,135],[4509,1101],[5516,277],[6630,272],[6701,1586],[8550,465],[116,959],[316,165],[366,3201],[385,486],[732,185],[754,277],[816,97],[862,156],[1201,9279],[1227,347],[1325,1065],[1394,1992],[1932,6024],[1941,376],[2193,5724],[2339,5827],[2499,1761],[2645,54],[3634,5655],[3732,523],[8250,2322],[8651,3795],[9220,1423],[291,2739],[401,442],[634,1499],[689,186],[694,191],[707,160],[886,1808],[1353,1068],[1738,116],[1749,4159],[1907,661],[1979,1390],[2249,333],[2612,1029],[3041,188],[3747,272],[4148,1155],[9293,4981],[83,67],[85,77],[111,1852],[122,1646],[299,109],[388,4317],[421,682],[472,1307],[651,161],[759,1801],[776,1340],[902,268],[905,175],[1008,187],[1569,616],[1694,1613],[1765,332],[1810,12488],[2202,4033],[2658,107],[3251,2416],[3350,404],[3414,367],[3682,4426],[3951,3781],[6920,1206],[10217,918],[13172,3362],[101,1879],[306,911],[527,143],[586,8814],[688,1499],[735,183],[858,1268],[998,136],[1036,761],[1199,272],[1206,6044],[1215,3917],[1361,661],[1391,4720],[1537,950],[2013,2815],[2157,2701],[2183,743],[2529,100],[3549,545],[5159,4996],[7642,8119],[7985,812],[8975,1155],[333,347],[371,172],[376,1328],[432,2143],[507,5924],[555,190],[613,861],[635,159],[704,3394],[736,166],[1228,4578],[1521,6110],[1676,4549],[1774,832],[1810,2962],[1997,169],[2214,642],[4257,1849],[278,8852],[310,115],[321,2654],[385,399],[386,1775],[652,272],[657,154],[659,770],[836,2117],[885,428],[943,572],[1342,1395],[1625,357],[1963,4128],[2608,1114],[2927,12162],[3253,5521],[3256,2602],[3287,130],[3338,453],[4656,2949],[5582,1114],[9199,172],[10522,116],[10827,112],[12969,339],[118,277],[287,683],[388,6684],[401,404],[430,2792],[521,103],[521,307],[551,84],[596,451],[656,172],[686,161],[730,770],[751,151],[891,101],[1177,7413],[1416,178],[1552,850],[1590,4091],[2503,1919],[2585,533],[3158,950],[5003,8984],[6345,1280],[7078,390],[10040,131],[105,3326],[276,1121],[423,9947],[468,348],[489,140],[523,407],[711,5103],[779,3359],[1318,7160],[1451,53],[1625,417],[2064,143],[2542,832],[2589,11113],[2683,450],[2886,117],[3363,1205],[4483,688],[5241,274],[5624,396],[80,79],[84,84],[291,442],[367,730],[383,319],[422,97],[457,2140],[669,2323],[677,5024],[716,811],[847,9797],[1081,173],[1100,995],[1353,1989],[1585,376],[3045,693],[3095,895],[4396,189],[4997,117],[5979,4277],[7196,353],[9957,117],[11807,772],[334,395],[375,949],[388,415],[448,622],[600,170],[625,289],[726,4800],[1076,149],[1119,1790],[1201,3994],[1353,333],[1744,5704],[1906,450],[1990,2656],[3319,1495],[3413,1307],[3859,743],[4926,4210],[4934,112],[65,98],[269,7470],[322,130],[324,296],[369,14721],[375,985],[401,10343],[634,533],[675,2783],[731,161],[1049,3258],[1257,1571],[1441,129],[2948,695],[3114,146],[3299,3939],[3762,153],[4609,9720],[4846,747],[4976,417],[5291,276],[5576,1663],[5655,1934],[7479,2604],[9429,2785],[10888,2182],[67,117],[82,634],[104,274],[111,101],[272,112],[273,6878],[331,12706],[366,437],[367,319],[522,2076],[554,131],[615,4888],[677,730],[700,1176],[724,1323],[779,2468],[817,855],[896,274],[954,138],[981,1715],[1147,182],[1342,7494],[1537,2603],[2619,103],[2919,2317],[3139,531],[3220,270],[3461,160],[4375,3534],[5355,3111],[7189,370],[10464,417],[13745,2534],[114,1655],[320,1254],[338,152],[432,384],[441,134],[522,531],[548,5677],[703,1206],[846,6367],[909,1029],[935,172],[1611,180],[1696,5260],[1836,972],[2848,386],[3010,417],[4781,911],[5350,121],[65,116],[102,353],[376,2662],[386,3534],[422,804],[672,304],[684,442],[763,155],[938,698],[942,156],[960,311],[1164,871],[1194,3182],[1387,856],[1641,1460],[2203,2806],[3460,115],[5210,599],[5326,832],[6505,1768],[9719,1160],[84,79],[227,132],[472,396],[501,184],[607,1589],[644,343],[795,990],[809,11177],[1128,1364],[1143,1394],[1289,133],[1404,6814],[1458,661],[1472,2205],[1762,155],[1903,4790],[2099,480],[2111,2720],[2214,5810],[2368,4718],[2778,175],[3103,274],[3470,723],[3794,1166],[4419,2514],[8136,304],[68,73],[80,69],[105,345],[118,289],[281,7002],[320,652],[323,158],[357,2643],[373,184],[386,1137],[508,1113],[738,1633],[874,3957],[1015,153],[1089,367],[1331,116],[1344,404],[1644,2189],[2159,1134],[2746,463],[3416,1083],[4075,4665],[6214,1606],[106,404],[122,305],[364,9626],[393,312],[394,149],[711,1336],[796,545],[837,2656],[851,274],[1128,667],[1177,116],[1397,2439],[1428,110],[1533,4696],[1793,340],[1800,1897],[1950,12178],[2004,724],[2879,1541],[3502,812],[3572,7204],[3617,2083],[4697,883],[5621,422],[5883,293],[9156,2346],[9334,352],[9697,115],[67,65],[72,851],[102,305],[103,442],[103,620],[264,136],[300,841],[395,2277],[472,270],[564,277],[717,137],[809,769],[896,11283],[1039,339],[1083,3032],[1111,163],[1366,118],[2190,2219],[2748,811],[2972,2049],[3343,1394],[6421,620],[7880,783],[71,114],[77,626],[287,325],[299,1809],[302,467],[375,522],[441,167],[479,4237],[666,4533],[687,160],[731,129],[797,11184],[875,3534],[1111,175],[1803,307],[1866,172],[2606,2679],[3301,100],[3724,1183],[3915,274],[4107,3370],[4148,1307],[4325,7268],[6076,1286],[6500,13096],[10194,1647],[99,110],[411,149],[462,98],[501,145],[532,11763],[667,463],[1163,2343],[1214,1170],[1244,417],[1473,132],[1590,2251],[1840,98],[1971,3036],[2874,1114],[2946,5506],[3061,1368],[3095,2775],[3097,1150],[3418,865],[3686,396],[226,135],[263,139],[350,10971],[390,116],[582,522],[631,321],[986,817],[1285,816],[1358,319],[1382,7199],[1998,995],[2071,356],[2088,1105],[2120,321],[2138,357],[2173,1984],[2478,1463],[3452,115],[4346,886],[5118,116],[5592,4949],[10871,417],[11321,1720],[195,135],[538,977],[548,467],[597,143],[691,1186],[909,6343],[943,936],[1051,187],[1126,140],[1139,2388],[1300,1170],[1906,2680],[1938,13320],[2423,1877],[3607,118],[4879,4479],[5998,13828],[331,370],[386,1226],[587,794],[718,164],[896,3831],[1551,3029],[2084,1645],[2326,3497],[2364,53],[2560,1372],[2728,1696],[2984,1074],[3023,2216],[3844,1837],[4977,895],[6279,7131],[6572,306],[15058,1038],[375,1983],[464,181],[485,307],[803,611],[842,2205],[842,6129],[1113,97],[1214,676],[1638,4081],[1765,367],[2498,1150],[2551,1101],[2794,3916],[4541,1302],[8264,400],[8907,1924],[10683,97],[11073,1647],[11227,404],[12946,324],[14247,79],[347,748],[452,2349],[584,353],[665,160],[721,12052],[1322,2275],[1326,1957],[2084,2536],[2718,3013],[3905,2643],[3969,330],[4116,5366],[4693,13060],[11437,296],[12945,1335],[15040,8789],[276,1417],[276,3901],[285,608],[310,683],[312,287],[336,4991],[557,178],[1167,152],[1300,545],[1369,417],[1679,1847],[1745,912],[2042,463],[2211,486],[2401,2609],[2516,1709],[5405,995],[5702,559],[6054,270],[9592,1378],[53,1841],[112,451],[283,2656],[292,137],[297,135],[320,5225],[375,3120],[437,111],[515,8453],[595,2662],[599,2728],[638,155],[667,1682],[756,101],[793,2062],[913,553],[1506,2572],[1753,4230],[2475,3878],[2488,1232],[3088,575],[3399,989],[3741,837],[3741,1078],[4654,6878],[5019,1440],[5155,2846],[8076,505],[14517,2358],[85,995],[278,2699],[283,14199],[336,422],[353,270],[579,122],[753,462],[880,165],[903,1457],[1048,1226],[1053,132],[1204,137],[1430,6404],[1631,148],[2149,111],[2294,180],[3670,181],[3909,277],[3993,4583],[4226,2484],[4566,430],[9994,13852],[10021,272],[86,952],[226,157],[310,1852],[366,923],[470,131],[545,566],[556,911],[667,992],[886,14612],[1119,3536],[1154,6828],[1248,382],[2013,463],[2110,5477],[2858,11322],[2933,1910],[2976,370],[3053,107],[3624,6724],[4081,941],[5843,2437],[7229,1136],[99,114],[101,396],[134,141],[227,15205],[278,353],[320,312],[362,5495],[422,272],[472,521],[475,575],[485,3358],[514,117],[528,109],[532,921],[670,1663],[719,3963],[724,1555],[738,293],[809,1315],[966,11823],[1055,1367],[1329,2756],[1434,7339],[1576,13200],[1912,107],[1962,2083],[2064,189],[3097,12544],[3317,356],[4737,274],[8715,111],[8807,307],[9440,109],[9560,10831],[68,68],[122,381],[268,2009],[295,134],[320,7873],[325,465],[368,1263],[376,2642],[514,781],[520,173],[532,344],[791,108],[951,140],[1031,1599],[1174,160],[1383,14546],[1991,595],[4004,1402],[4262,160],[9763,1714],[10412,887],[81,81],[276,1057],[364,1745],[508,2196],[529,150],[587,1328],[701,4632],[704,1106],[913,1268],[917,306],[924,4568],[984,6642],[1175,1471],[1454,453],[1504,146],[3284,701],[3445,1703],[3979,2108],[4065,176],[4836,575],[4867,345],[5369,2758],[5589,365],[6480,117],[8840,1647],[284,173],[452,1403],[588,161],[666,1244],[685,4006],[760,545],[802,168],[946,2715],[989,737],[1051,133],[1053,166],[1128,2732],[1147,130],[1157,558],[1286,1670],[1728,7677],[1740,100],[1924,1553],[2276,3671],[3210,140],[3411,3577],[3472,806],[4469,537],[5247,289],[5485,773],[6539,305],[6798,1449],[8001,693],[11024,145],[100,296],[226,137],[281,268],[379,157],[420,4058],[499,317],[500,180],[516,894],[538,3073],[658,1723],[677,5119],[684,626],[833,2050],[846,1667],[919,450],[938,693],[1181,2813],[1442,4379],[1545,6026],[1632,133],[1778,1726],[1871,179],[2453,7644],[2651,3693],[3692,2750],[6231,2046],[10684,3633],[377,494],[463,3247],[490,2346],[519,101],[715,137],[760,2666],[879,3943],[1073,146],[1254,101],[1285,1432],[1300,1890],[1575,812],[1738,894],[2999,628],[3047,1152],[3350,339],[3387,4525],[4504,1071],[7060,276],[7509,110],[10167,1949],[109,121],[235,9856],[276,3054],[278,97],[376,2010],[399,1634],[535,158],[551,68],[607,1268],[639,365],[667,386],[719,104],[814,1814],[892,4536],[1125,4249],[1147,191],[1269,139],[2231,1726],[2567,177],[2740,4356],[2764,2183],[2813,544],[2976,8439],[3173,14192],[3455,4775],[7874,5498],[70,70],[70,114],[99,2008],[172,2088],[281,1533],[312,370],[401,9451],[423,4798],[451,116],[572,104],[596,2258],[661,523],[875,276],[1896,1748],[2415,11589],[2478,773],[2533,1137],[3288,463],[4179,598],[4255,422],[4735,189],[6594,312],[133,676],[217,142],[283,451],[289,103],[291,268],[330,111],[625,425],[696,1134],[844,391],[984,2005],[1318,395],[1535,545],[2193,277],[2611,3950],[2874,7767],[2893,537],[2971,1139],[4504,1495],[10077,162],[333,1894],[385,785],[440,270],[508,570],[687,153],[705,2129],[961,1499],[1106,607],[1394,1761],[1839,6657],[2111,2913],[3052,607],[3664,2761],[4123,8129],[4681,2513],[6687,1183],[6838,558],[11123,2667],[104,4317],[105,2387],[291,5164],[377,639],[402,723],[405,186],[420,983],[503,5594],[887,854],[1051,145],[1141,150],[1248,3058],[1339,171],[1962,3185],[2304,268],[3663,102],[4933,533],[5216,140],[5928,7064],[6533,1105],[7782,287],[12853,8612],[106,105],[281,344],[289,107],[312,2379],[368,2309],[376,599],[485,422],[619,132],[826,2552],[1083,2693],[1128,992],[1426,1489],[2454,2698],[3213,311],[3244,5433],[3846,1144],[4528,1048],[5507,97],[5761,1370],[5889,2324],[7864,391],[11742,2526],[287,1647],[405,144],[434,3910],[503,2340],[531,3410],[634,894],[711,1390],[721,5983],[824,100],[840,118],[1013,3566],[1051,178],[1154,537],[1393,2128],[3811,1515],[3929,1321],[4175,1703],[4229,5694],[5106,114],[5687,1137],[7398,1209],[9294,1041],[55,647],[103,1357],[276,6130],[312,2911],[327,152],[372,164],[434,8637],[897,850],[907,1401],[1262,545],[1326,1042],[1651,3424],[1717,1103],[1909,128],[2111,10887],[2246,357],[2473,367],[3074,1173],[3318,3303],[4306,15173],[6470,2910],[12652,14589],[32,1150],[87,2340],[276,2895],[299,13080],[302,2277],[310,307],[354,7426],[508,12489],[621,97],[628,268],[1422,3697],[1459,166],[1494,14207],[1628,530],[1760,1244],[2273,107],[2595,153],[2645,53],[2923,4450],[3093,2818],[4034,2479],[4440,415],[4816,1712],[5249,11627],[7821,2489],[7829,421],[9394,1495],[10940,330],[13578,13321],[72,8139],[105,2181],[276,4220],[280,165],[304,2215],[312,747],[330,1038],[332,7680],[341,415],[610,161],[631,442],[691,725],[753,923],[801,98],[857,1368],[1037,1017],[1236,2274],[1265,4357],[1311,137],[1383,3614],[1728,2293],[1912,99],[2751,6514],[3321,8365],[4472,2559],[5301,920],[8838,2149],[9712,9712],[11076,112],[14410,854],[15260,121],[72,80],[103,1785],[299,10193],[495,353],[508,437],[587,376],[1058,832],[1140,737],[1252,1264],[1568,2117],[1834,137],[1844,1321],[2133,2277],[2159,10930],[2160,1074],[2316,3343],[2516,1365],[2965,2043],[3893,144],[3971,3610],[4201,111],[4361,562],[4976,357],[7730,1328],[8175,3937],[8629,111],[9006,556],[10620,370],[85,76],[274,533],[350,2721],[357,3356],[401,6429],[976,1092],[1129,8381],[1164,1231],[1206,13919],[1530,182],[1590,822],[1801,1341],[2362,990],[2488,3703],[2793,1173],[3516,2711],[4421,7802],[4520,925],[6787,289],[6999,137],[8332,542],[10749,116],[11418,760],[13186,1607],[57,647],[110,747],[336,652],[385,2019],[394,134],[415,829],[447,104],[509,97],[557,137],[602,977],[721,5798],[896,781],[948,168],[975,759],[1091,150],[1099,176],[1201,779],[1707,1616],[2186,9911],[2295,378],[2398,1158],[2716,910],[3218,422],[3893,148],[4103,1122],[6348,370],[6469,4053],[13161,832],[13530,7109],[51,3329],[66,121],[70,65],[304,339],[323,190],[333,1624],[385,6515],[389,178],[426,137],[453,1410],[495,14454],[532,287],[596,71],[654,190],[672,1663],[866,185],[970,114],[1068,819],[1163,3066],[1494,764],[1529,114],[1967,128],[2013,972],[2319,1294],[2726,268],[2765,5461],[3186,1984],[3419,1250],[3735,377],[3823,2250],[4335,892],[5404,98],[6477,743],[6571,545],[8266,277],[10977,172],[13491,2742],[87,83],[107,6448],[111,533],[285,4046],[296,422],[312,321],[350,106],[368,467],[390,2248],[498,169],[644,367],[666,235],[1163,1060],[1257,909],[1395,1355],[1472,6741],[1503,927],[1527,925],[1857,311],[1863,9579],[1972,2049],[2124,833],[2598,347],[2793,352],[3226,305],[3620,130],[4020,1446],[5142,620],[9486,8096],[14595,7487],[77,1254],[276,2440],[299,542],[341,1700],[483,135],[505,347],[650,5289],[658,571],[763,169],[939,110],[1095,1816],[1133,11106],[1353,1704],[1376,289],[1696,1527],[1969,1757],[2017,384],[2251,1017],[2437,1176],[2511,11520],[2513,673],[4641,367],[6152,2066],[6235,521],[7553,120],[80,85],[83,77],[285,2184],[304,268],[342,190],[375,463],[484,3485],[582,861],[657,170],[672,268],[1048,761],[1201,4769],[1411,1052],[1438,343],[1998,97],[2254,950],[2989,2308],[5434,289],[5641,1955],[6344,417],[8104,187],[8152,5724],[70,67],[226,147],[287,1389],[399,2652],[401,107],[403,190],[423,78],[774,13482],[888,1043],[931,345],[1125,345],[1141,148],[1198,276],[1450,5752],[1451,647],[1628,1219],[1963,268],[2420,140],[2771,5988],[3461,142],[3893,165],[4019,920],[4959,274],[4997,5421],[5740,300],[7498,1162],[10439,509],[11862,894],[135,140],[237,10499],[281,885],[333,2474],[467,2911],[508,8327],[760,1579],[791,5768],[810,1458],[980,2507],[1079,115],[1201,1231],[1301,1151],[1384,348],[1531,1856],[1749,2178],[1820,147],[2078,2060],[2159,11689],[2925,139],[4430,268],[8075,121],[12274,2191],[32,521],[106,12987],[107,1240],[288,370],[291,5053],[322,188],[401,108],[447,1709],[544,8865],[579,2628],[667,385],[706,104],[768,172],[931,115],[966,381],[1463,6274],[1738,7421],[5671,268],[5735,3414],[6753,2959],[7460,4266],[7688,558],[65,117],[475,382],[640,573],[658,615],[762,3924],[872,1850],[1020,154],[1039,321],[1095,1622],[1098,173],[1265,2713],[1269,177],[1300,752],[1310,1062],[1906,696],[1921,1110],[2118,820],[2645,52],[3348,13971],[3489,7408],[8820,1421],[9069,111],[10085,3511],[98,1740],[276,2873],[312,451],[333,531],[344,370],[375,3025],[421,3785],[602,509],[683,7008],[702,2589],[897,270],[928,276],[1481,566],[1487,796],[1683,1097],[1869,376],[2362,700],[2364,54],[3129,8558],[3628,4652],[3844,3111],[3906,795],[4189,1756],[4260,1509],[5456,1734],[5588,164],[8477,1999],[9114,1006],[68,1378],[83,397],[102,894],[119,2259],[236,191],[273,806],[309,180],[335,289],[340,699],[434,991],[438,181],[461,175],[527,189],[611,779],[631,1652],[718,153],[765,117],[869,339],[870,116],[966,310],[997,571],[1000,1843],[1030,1166],[1129,450],[1196,97],[1405,2493],[1553,815],[1762,160],[1815,1666],[2011,830],[2107,270],[2270,116],[2331,679],[2530,348],[2724,1039],[2979,559],[3073,467],[3271,111],[4605,1870],[4710,164],[5998,5572],[12875,15939],[13938,5642],[156,152],[186,188],[195,147],[270,307],[324,361],[421,9726],[453,468],[485,2259],[572,509],[615,743],[615,2315],[721,2360],[797,99],[938,2130],[996,321],[1014,836],[1020,186],[1230,1210],[1411,541],[1448,883],[1738,806],[1928,2444],[2094,3239],[2153,479],[3081,2960],[3613,2822],[4497,8370],[4610,538],[4899,4596],[5273,3056],[6231,894],[6641,370],[6736,2015],[7464,1499],[73,88],[98,105],[112,400],[340,655],[390,1041],[419,1439],[423,5498],[432,556],[437,114],[472,72],[485,268],[487,377],[865,407],[1036,716],[1177,2562],[1247,620],[1364,10413],[1569,356],[1840,11574],[2077,324],[2588,382],[2900,3327],[3532,9339],[3568,463],[4519,140],[6400,10008],[8038,832],[8502,7096],[299,428],[357,611],[368,4870],[388,425],[452,111],[656,168],[874,1830],[1064,185],[1394,724],[1477,166],[1644,457],[2078,1016],[2106,3926],[2221,417],[2255,705],[2356,84],[2738,2812],[2852,992],[2936,5258],[3759,5400],[4721,709],[4876,422],[5195,752],[6201,98],[8207,15702],[9402,3965],[10075,8993],[11749,9782],[14572,5266],[77,87],[104,287],[333,776],[346,178],[357,779],[386,1588],[401,84],[401,5591],[472,2880],[690,131],[743,5708],[789,276],[1099,187],[1141,147],[1182,693],[1236,15025],[1431,737],[1962,457],[2192,56],[2480,3595],[2496,2343],[3374,1683],[4440,1272],[5992,276],[6165,172],[6639,1148],[6954,2189],[8061,700],[8290,1005],[9914,837],[11471,2108],[11683,8114],[32,1841],[118,804],[226,133],[302,8480],[306,1591],[317,4743],[356,9405],[368,274],[425,102],[432,337],[434,274],[548,876],[552,183],[681,842],[685,3888],[891,465],[1037,817],[1402,496],[2106,2462],[2316,3065],[2717,103],[5897,3609],[8390,1389],[11743,2318],[13757,6591],[77,66],[262,133],[291,345],[320,1663],[430,925],[433,178],[557,166],[682,1585],[795,700],[877,979],[1000,407],[1263,307],[1267,167],[1406,2555],[1890,2888],[3657,135],[3978,2306],[4021,107],[4466,872],[5382,2091],[5946,1123],[6558,347],[6754,991],[6873,2868],[10746,960],[12464,176],[68,876],[285,344],[294,2777],[368,12487],[399,2537],[528,114],[528,4803],[532,361],[735,138],[801,3206],[857,1947],[868,4165],[996,12481],[1109,795],[1128,1092],[1300,479],[1519,6465],[1697,4981],[1747,2444],[4193,66],[4508,9007],[5813,270],[11357,2166],[12328,128],[16024,1004],[57,52],[80,117],[102,395],[234,191],[291,4703],[341,10524],[386,4047],[401,1365],[495,268],[503,442],[958,115],[1024,6337],[1431,385],[1484,4438],[1544,1016],[1555,3510],[1584,130],[1634,376],[1938,378],[2051,15840],[2540,1218],[2802,3525],[2812,907],[3572,305],[4725,325],[5731,304],[9635,4622],[11209,3446],[13027,270],[15789,146],[77,305],[109,105],[114,370],[237,10450],[289,1627],[347,1406],[436,137],[448,518],[457,1299],[493,170],[547,146],[629,134],[658,817],[665,131],[732,160],[1362,4341],[1450,382],[1574,276],[1654,305],[1834,135],[2059,1139],[2080,562],[2245,383],[2458,116],[2465,5878],[2565,6071],[2596,989],[2662,3193],[2916,1226],[3855,370],[3860,553],[5365,8607],[9880,572],[10650,4849],[14882,2820],[311,1675],[333,1257],[388,11279],[472,12297],[485,102],[532,277],[889,168],[896,2529],[953,175],[955,180],[1191,1589],[1314,903],[1412,573],[2207,5287],[2268,7967],[2549,634],[2714,4415],[2735,2443],[2900,1401],[2916,855],[3052,1400],[3783,268],[4262,140],[4389,2308],[4952,450],[5502,1221],[7182,307],[7568,1139],[10239,2737],[11315,2290],[336,97],[339,616],[472,117],[485,1011],[495,1536],[551,806],[556,2955],[837,111],[971,178],[1005,1005],[1065,1171],[1073,189],[1183,1372],[1193,3920],[1247,3585],[1265,3998],[1300,5344],[1325,4771],[1335,11494],[1863,3847],[1885,140],[2113,4090],[2222,885],[2442,1158],[2545,1395],[2751,3652],[2811,1186],[3470,381],[4809,8157],[7461,276],[7472,1370],[7672,293],[7760,7453],[9872,4041],[10789,1432],[78,592],[102,6035],[118,923],[339,396],[489,181],[555,131],[655,494],[743,1238],[752,9310],[810,1691],[826,2076],[971,186],[1005,5084],[1285,12579],[1501,793],[1843,871],[2214,2370],[2368,1469],[2421,593],[2477,949],[2516,1432],[2528,760],[2528,1105],[2641,2811],[2663,2150],[2874,1302],[3380,168],[4844,995],[5599,975],[10670,816],[10854,7824],[13813,3874],[88,88],[285,572],[317,558],[430,386],[593,580],[634,7662],[766,766],[885,706],[950,407],[1215,4408],[1322,589],[1543,816],[1657,4707],[1690,347],[1904,276],[2220,272],[2264,324],[2345,289],[2480,861],[2879,1692],[5199,312],[7741,417],[8084,178],[8382,370],[14440,421],[104,370],[288,1026],[356,272],[495,99],[548,268],[595,3017],[670,2930],[740,188],[858,815],[875,909],[949,648],[1040,130],[1144,725],[1215,3458],[1247,115],[1308,5668],[1320,3444],[1383,558],[1405,430],[1414,1179],[1475,100],[1784,304],[1925,659],[1948,886],[2473,384],[2948,348],[3471,293],[3655,14041],[6230,382],[7177,7831],[11714,941],[11753,894],[65,89],[344,2248],[366,7007],[482,156],[514,78],[675,834],[788,1259],[869,293],[882,3731],[1072,108],[1230,844],[1305,1589],[1424,335],[1838,3888],[2223,4071],[2345,422],[2466,5022],[2920,128],[3214,417],[3933,608],[4261,1520],[5094,296],[7656,4384],[7729,1087],[15458,5848],[97,101],[100,115],[310,111],[311,783],[323,187],[331,6615],[341,970],[399,486],[416,175],[416,188],[423,939],[504,163],[587,1687],[726,8684],[779,3404],[847,382],[1066,347],[1860,5817],[1894,725],[2025,382],[2128,1388],[2321,340],[2663,2108],[2834,1495],[3663,115],[4343,1947],[4383,6507],[5100,927],[7738,417],[11770,115],[15835,1004],[67,287],[273,3007],[293,108],[330,274],[389,183],[452,537],[809,12270],[909,5327],[995,6661],[1002,168],[1121,2740],[1229,158],[1248,1760],[1406,1320],[1543,4128],[1931,111],[1989,490],[2394,432],[3019,890],[3737,1071],[4020,10018],[4608,2035],[10319,168],[11458,1924],[14723,1633],[67,1263],[284,143],[465,115],[625,9939],[732,164],[754,307],[839,1539],[859,507],[943,764],[962,902],[1040,145],[1129,3488],[1138,1063],[1472,813],[1651,5721],[1831,4772],[2065,3231],[2826,8609],[3030,2957],[4079,1775],[4167,1520],[5389,357],[5775,2379],[9844,100],[11685,4479],[13431,7268],[14782,1395],[15038,626],[277,1673],[281,1554],[302,2444],[320,1370],[378,353],[420,10151],[424,152],[495,2102],[589,4308],[595,1731],[702,321],[703,2140],[728,1055],[903,2248],[1013,2416],[1059,886],[1197,272],[1274,2935],[1794,183],[2017,306],[2485,809],[2669,107],[3279,553],[3897,272],[4024,1356],[4727,422],[5561,173],[6437,5097],[6962,370],[8833,321],[16072,1948],[16368,4633],[291,312],[377,1570],[485,538],[519,783],[532,4156],[536,185],[684,4757],[684,14425],[727,370],[946,4182],[1119,884],[1251,347],[1308,15576],[1311,143],[1407,3626],[1504,165],[1549,865],[1595,2198],[1616,2675],[2062,667],[2403,10228],[2659,137],[3026,12394],[3747,361],[4710,140],[4944,3184],[5731,3274],[7490,683],[8653,4619],[11214,573],[281,4969],[337,1433],[388,277],[423,1205],[489,128],[614,1853],[997,766],[1042,3650],[1657,545],[1708,4060],[1749,4712],[1753,8133],[2002,9548],[2395,607],[2916,3394],[3309,1903],[7134,709],[8916,450],[9965,121],[11986,9735],[14575,681],[273,12775],[283,105],[307,272],[333,1292],[389,158],[422,115],[463,7495],[602,1827],[625,1071],[704,826],[743,463],[836,375],[947,1649],[1113,270],[1177,620],[1428,268],[1790,3165],[1815,4451],[2027,14363],[2081,2783],[2384,1491],[2500,8942],[2573,477],[2736,640],[2760,276],[2862,1808],[3036,997],[4174,119],[4452,1307],[5237,7416],[5748,381],[11136,851],[12976,1633],[70,6939],[104,117],[108,888],[117,2166],[233,182],[273,13998],[368,3117],[587,3025],[609,143],[646,544],[681,832],[684,4679],[770,1587],[811,3250],[827,3182],[946,1200],[1021,407],[1181,3712],[1295,385],[1311,179],[1471,716],[1537,479],[1551,2820],[1719,4696],[1875,1571],[2098,2066],[3181,170],[3498,1043],[3514,3890],[3688,165],[3714,146],[3811,442],[4434,3800],[4651,306],[5450,616],[7469,1639],[12131,2806],[13915,559],[67,111],[69,80],[84,8994],[119,564],[281,15013],[391,348],[551,114],[867,1185],[1044,1317],[1284,134],[1302,450],[1853,361],[1886,1575],[2094,1183],[2606,7220],[3244,1469],[3549,450],[5574,4760],[6416,337],[6470,1593],[8333,3653],[9607,2134],[14854,11602],[105,876],[122,289],[325,353],[331,5682],[345,293],[364,6112],[416,178],[426,132],[437,378],[477,2759],[499,912],[533,1499],[567,188],[596,65],[621,1043],[623,1227],[654,161],[701,691],[798,128],[834,2747],[885,2538],[994,733],[1000,545],[1042,724],[1073,172],[1176,925],[1201,2545],[1244,779],[1497,5613],[1913,276],[2195,101],[3572,1368],[4468,370],[5062,305],[7152,15003],[9392,7225],[12045,830],[13277,9011],[57,51],[112,12941],[236,9061],[276,3594],[294,764],[383,4747],[386,1372],[424,190],[504,140],[623,347],[625,6614],[760,643],[837,1977],[946,675],[1047,7057],[1183,6193],[1246,1460],[1300,3887],[1453,153],[1808,1604],[2236,14846],[2415,1602],[2430,122],[2706,360],[2848,1016],[3322,3384],[3413,2215],[4579,2142],[5325,101],[5708,4119],[5757,804],[7542,287],[11312,395],[11722,143],[67,293],[75,101],[334,120],[384,2723],[401,2332],[431,1668],[556,494],[588,172],[690,150],[702,97],[739,129],[762,837],[827,4673],[840,4877],[912,270],[1037,1649],[1265,1014],[1285,112],[1295,541],[1366,894],[1639,3766],[1723,1651],[1725,553],[2031,276],[2126,3867],[2179,4428],[2292,3193],[2392,12575],[2522,367],[2529,116],[2819,97],[2819,111],[3574,1791],[272,289],[294,10177],[303,167],[320,1740],[336,626],[490,2697],[593,6314],[762,304],[790,1306],[826,910],[842,3868],[951,146],[1047,9579],[1054,1802],[1058,457],[1072,101],[1178,648],[1290,661],[2327,1217],[2544,1849],[4446,1078],[5950,164],[6817,105],[281,119],[281,2463],[283,13522],[364,9367],[470,146],[477,1097],[508,856],[532,304],[827,276],[847,575],[905,189],[1117,102],[1186,385],[1354,7045],[1361,1170],[1663,344],[1903,997],[1905,836],[1913,875],[2002,4230],[2139,486],[2213,1914],[2252,921],[2432,276],[2544,2816],[3173,6642],[3365,4201],[3646,1907],[4388,1128],[4908,468],[5796,2178],[5907,1389],[6142,10599],[6526,1031],[8175,895],[8366,737],[8782,465],[12971,340],[14694,2810],[341,12983],[357,4099],[383,11952],[413,481],[422,288],[721,7468],[737,553],[897,1658],[921,321],[934,1790],[980,1495],[1163,950],[1823,7930],[2916,3327],[3567,3611],[3574,2340],[4370,16696],[4687,1437],[5275,780],[5595,2699],[6226,475],[8906,3786],[11272,304],[69,66],[83,112],[306,695],[336,344],[336,1493],[337,2539],[342,162],[354,348],[357,2253],[402,16083],[573,1206],[602,4041],[603,164],[607,1616],[679,607],[688,111],[802,180],[1035,486],[1068,1353],[1322,421],[2075,376],[2118,486],[2235,4911],[2341,842],[2413,795],[2421,833],[2949,673],[3258,720],[4098,276],[5322,348],[12867,5223],[16424,2348],[105,1803],[276,2506],[377,677],[385,941],[485,305],[530,2897],[838,407],[1030,1082],[1488,2641],[1842,2162],[1902,432],[1938,451],[2305,2612],[2324,307],[3290,2025],[4039,1150],[8530,1604],[10677,1148],[13293,2446],[14681,1356],[102,321],[278,310],[291,3012],[300,1944],[447,3002],[466,129],[505,386],[550,151],[571,276],[719,1113],[923,977],[961,14271],[1021,919],[1227,276],[1590,7625],[1606,270],[1775,716],[1848,1471],[2132,1602],[2394,7448],[2726,9157],[3093,545],[3399,827],[7502,7025],[7900,1142],[7964,1164],[7968,562],[10999,1527],[98,8706],[108,6434],[308,176],[420,2043],[503,268],[626,115],[774,1433],[819,2512],[990,1176],[1121,1926],[1290,475],[2508,376],[2619,4006],[2662,1390],[2938,296],[3156,661],[3394,587],[4388,4555],[4392,829],[4402,556],[4469,122],[5069,760],[5288,370],[7554,2091],[13229,598],[57,50],[84,72],[110,404],[114,5376],[320,345],[368,442],[374,176],[382,523],[385,650],[388,3172],[495,5091],[599,1198],[659,1241],[691,1471],[700,2188],[717,177],[769,2791],[826,1292],[984,1395],[1166,6644],[1251,1019],[1387,4487],[4180,6575],[4388,2618],[4497,10839],[6196,1328],[6463,180],[6599,2105],[6694,538],[8148,2727],[9330,152],[10809,307],[10968,1715],[14330,1666],[283,1718],[302,533],[385,623],[396,1495],[452,325],[643,382],[666,4480],[704,4060],[711,3324],[840,116],[988,339],[1206,2687],[1360,1194],[1441,156],[1993,804],[2029,16090],[2173,981],[2288,716],[2483,182],[2710,276],[3651,3347],[3706,312],[5770,12427],[7669,7363],[12847,3201],[14457,4606],[69,88],[110,804],[278,119],[294,4500],[368,305],[381,977],[522,5102],[524,1841],[606,132],[646,2117],[866,164],[958,108],[1048,1388],[1066,613],[1531,2901],[1690,386],[1972,1244],[2163,1170],[3221,97],[3597,417],[4689,566],[4761,3879],[4899,773],[6298,1365],[6775,838],[11426,304],[14521,111],[87,2458],[98,634],[278,1272],[283,274],[295,148],[295,189],[337,3216],[385,421],[572,9810],[745,2319],[950,4625],[1202,1666],[1236,111],[1389,415],[1396,138],[1432,107],[1551,276],[1560,190],[1618,180],[1642,3192],[1938,572],[2064,168],[3057,545],[3076,1139],[3080,167],[3094,4950],[3466,730],[4889,162],[10156,378],[67,73],[67,82],[105,628],[276,3096],[291,8439],[319,2125],[320,5444],[336,4083],[366,1254],[396,1283],[495,467],[573,2465],[584,12024],[621,422],[643,752],[759,795],[802,159],[996,6284],[1341,376],[1510,161],[1725,1238],[1745,345],[1802,1613],[1970,1893],[2121,8690],[2235,521],[2368,3184],[2408,741],[2832,578],[3757,68],[4623,1031],[11522,3410],[13045,1401],[77,73],[195,156],[276,2435],[426,164],[518,695],[582,599],[595,720],[695,337],[721,270],[748,2650],[749,176],[754,2538],[820,347],[869,6219],[891,3145],[916,136],[922,307],[940,1598],[1039,3801],[1107,545],[1345,2100],[1387,8029],[1991,8226],[2518,98],[2540,2935],[2663,5306],[3130,382],[3319,274],[3940,272],[4095,158],[4673,1739],[5545,272],[6137,1489],[6224,2382],[6534,352],[6688,1737],[6740,2083],[8477,1370],[10924,137],[11337,1663],[14399,709],[83,65],[121,2258],[233,175],[283,3731],[306,1815],[359,159],[375,4182],[447,2502],[462,272],[495,7862],[503,521],[769,399],[945,158],[1116,1306],[1235,1104],[1264,1017],[1560,169],[1641,5893],[1805,176],[2345,1645],[2811,2630],[3234,977],[3535,307],[3972,118],[3972,2257],[4823,1112],[5094,109],[5561,190],[6074,2151],[6370,354],[6732,121],[7553,2654],[81,85],[110,1356],[270,1814],[276,3263],[331,10111],[428,112],[562,875],[672,6035],[753,353],[837,287],[848,386],[854,693],[900,7427],[940,2775],[1030,1159],[1072,97],[1104,9047],[1424,289],[1512,360],[1919,407],[2138,10675],[2273,829],[2297,1554],[2948,432],[3241,11988],[3799,12721],[4442,575],[4923,189],[7009,122],[7276,307],[7315,3937],[12616,2903],[68,111],[107,293],[116,437],[267,177],[274,1733],[276,5667],[477,2225],[555,167],[581,145],[732,189],[917,340],[923,378],[944,321],[1085,575],[1273,111],[1285,3002],[1444,13914],[1445,141],[1817,1692],[2195,1088],[2255,1057],[2649,2545],[2921,2243],[3915,114],[4005,946],[4061,115],[4521,2901],[4639,103],[6463,149],[57,49],[299,10905],[320,6870],[325,6005],[356,927],[376,421],[377,2145],[426,136],[467,117],[485,451],[495,324],[631,4046],[705,2104],[726,7136],[921,378],[979,2809],[1074,938],[1298,407],[1372,1049],[1567,1049],[2084,3226],[2330,103],[2844,370],[3316,121],[3508,289],[3692,2170],[5249,7249],[5314,5831],[5401,104],[5813,268],[6481,442],[7135,2233],[9210,781],[10776,6561],[12851,321],[79,66],[100,2443],[118,437],[191,140],[463,386],[470,139],[523,450],[625,1046],[626,272],[650,490],[654,152],[654,160],[703,1962],[776,972],[869,111],[961,3838],[973,140],[1088,268],[1121,531],[1261,171],[1434,2910],[1928,616],[2227,185],[2334,417],[2453,2072],[2532,1460],[2569,920],[2647,796],[2716,947],[3325,477],[3445,1501],[3626,6808],[3933,4614],[4050,3739],[4650,276],[5440,1787],[5617,3644],[5688,7067],[9969,5203],[10492,6183],[12958,15561],[16041,354],[98,7249],[294,939],[333,4903],[347,2335],[463,1651],[473,177],[533,2720],[638,168],[686,157],[711,3275],[718,180],[719,305],[719,9766],[748,2386],[751,150],[765,8069],[904,276],[1123,7647],[1146,185],[1150,111],[1422,105],[1438,1179],[1443,1386],[1993,1627],[2044,2888],[2252,101],[2585,890],[2716,14044],[2794,1390],[2873,276],[2906,417],[2934,14494],[3112,102],[3128,6673],[3615,172],[3821,950],[3845,274],[3866,1065],[4050,4208],[4378,276],[4547,1426],[6282,143],[7101,480],[7913,97],[9973,99],[83,72],[87,397],[99,962],[119,97],[270,121],[286,173],[310,894],[324,3525],[472,570],[602,16562],[703,348],[720,1893],[735,131],[893,181],[905,156],[908,659],[1032,155],[1112,1509],[1259,450],[1274,693],[1337,1667],[1417,4592],[1701,623],[1805,179],[1814,312],[1907,2049],[2499,1996],[2762,347],[2893,109],[3156,3923],[3195,2775],[3246,1422],[3985,115],[4032,628],[4561,481],[4657,164],[4687,1291],[5585,335],[10380,585],[12677,432],[14424,287],[15005,9920],[107,110],[273,4775],[273,9627],[274,2379],[320,4679],[354,432],[402,3355],[495,521],[499,1357],[531,693],[539,191],[708,184],[764,121],[875,13155],[956,1019],[1220,12463],[1274,737],[1314,325],[1441,136],[1443,2169],[1773,147],[1782,829],[1848,615],[1859,376],[2370,566],[2489,1347],[2716,1136],[2883,289],[2887,1122],[2900,3785],[3467,480],[3770,170],[3959,580],[4262,153],[4985,139],[5099,824],[6517,1118],[6973,5053],[9862,1879],[11532,1949],[11707,2008],[15855,2304],[85,84],[272,1271],[276,5493],[375,562],[426,184],[446,148],[523,1062],[719,397],[725,743],[726,1043],[861,6566],[1119,360],[1295,1516],[1397,1518],[1528,430],[1573,547],[1642,1610],[2040,2439],[2045,382],[2064,157],[3645,724],[3811,7850],[4262,155],[4836,1062],[5038,348],[5899,538],[6306,2747],[6574,2251],[6866,268],[7189,422],[7706,136],[8498,2490],[11880,806],[121,4217],[236,135],[403,183],[443,161],[466,141],[468,332],[485,287],[508,312],[598,2309],[652,304],[669,6415],[702,1673],[818,8524],[819,761],[843,1493],[1413,2177],[1447,1636],[1530,166],[1592,296],[1592,451],[1651,2445],[1805,170],[2763,688],[3305,811],[4260,1291],[5110,169],[6651,1159],[7129,1038],[8257,5599],[8702,121],[8834,1083],[12950,559],[13764,5607],[15695,3078],[49,1270],[274,5300],[288,321],[289,6284],[342,157],[420,3944],[426,177],[509,370],[515,324],[548,616],[572,288],[639,5119],[653,174],[970,293],[975,1144],[1044,111],[1105,407],[1143,8847],[1235,6246],[1260,149],[1335,2057],[1355,417],[1377,16195],[1387,378],[1430,1112],[1555,1221],[1646,104],[1701,1825],[1886,1612],[1990,11896],[2073,7986],[3283,5113],[3380,136],[3736,3839],[3825,6443],[3830,180],[4699,5721],[5137,307],[5715,1370],[6142,8329],[8621,628],[8624,121],[11455,983],[12490,573],[70,1266],[84,626],[276,1620],[294,116],[384,699],[389,130],[491,128],[724,769],[831,143],[855,1171],[856,102],[995,5113],[997,490],[1024,3862],[1133,10282],[1311,138],[1753,2621],[2080,1394],[2466,1651],[3152,3241],[3442,1102],[3762,168],[4397,11297],[4713,1658],[6483,1205],[7614,772],[10098,2731],[10204,3712],[10664,2134],[13169,2281],[73,90],[467,747],[563,137],[622,367],[625,15480],[672,305],[738,17438],[826,976],[848,776],[997,276],[1047,4427],[1047,5344],[1107,407],[1387,3196],[1417,1325],[1689,1646],[1915,152],[2048,1048],[2078,1055],[2271,270],[2726,4371],[5211,783],[5870,268],[6233,890],[6609,6522],[9174,3939],[9401,6350],[50,2093],[112,272],[119,6351],[267,143],[283,4565],[320,6513],[410,184],[434,1526],[532,706],[572,305],[684,268],[695,677],[785,1692],[897,111],[944,533],[1068,2552],[1133,14954],[1170,480],[1220,2928],[1267,141],[1295,1052],[1355,450],[1431,667],[1529,9492],[1664,149],[1935,3621],[2222,118],[2690,134],[2740,673],[3882,683],[4153,99],[5526,9285],[6863,319],[10536,7894],[105,608],[195,129],[269,345],[276,4304],[285,312],[285,1976],[368,13438],[532,395],[546,137],[715,180],[1095,4821],[1157,1378],[1311,190],[1335,10770],[1686,272],[1820,174],[1988,119],[2052,3540],[2704,3430],[2814,140],[2948,659],[3007,335],[3130,865],[3181,176],[4055,8660],[6023,1401],[6107,970],[6273,813],[9793,1118],[69,71],[104,1607],[291,339],[331,6790],[333,421],[333,855],[399,1859],[421,2279],[425,339],[442,7705],[485,626],[522,2596],[532,1205],[576,128],[586,15175],[675,2427],[684,353],[726,4794],[744,272],[765,109],[931,17049],[946,3325],[1157,592],[1178,1191],[1218,1063],[1295,812],[1377,704],[1387,1348],[1418,276],[1455,1502],[1981,270],[2139,820],[2220,105],[2314,176],[2640,139],[3195,1598],[3281,357],[3613,766],[3777,854],[4222,101],[5324,8284],[6156,160],[6177,5583],[6841,6079],[7744,747],[8986,709],[10060,111],[10870,16987],[16824,14307],[17091,13546],[72,11084],[89,97],[105,105],[117,2379],[122,521],[306,1697],[322,143],[343,383],[423,77],[656,170],[675,4831],[701,1463],[856,6365],[862,154],[926,720],[1057,3138],[1057,3565],[1218,1847],[1230,842],[1236,2008],[1494,7313],[1565,168],[1987,3059],[2023,1471],[2137,161],[2776,3271],[3000,2005],[3334,5957],[3380,155],[4052,3593],[4326,978],[4670,2822],[5112,6108],[5717,375],[5741,3710],[5950,136],[6055,3385],[6508,894],[8272,1683],[9838,102],[13178,133],[13336,783],[80,344],[99,8984],[101,311],[117,1389],[266,162],[270,277],[307,307],[333,1221],[377,2955],[402,584],[404,287],[421,6859],[423,76],[459,160],[495,1263],[523,523],[554,190],[608,268],[645,175],[661,3876],[710,188],[713,1487],[810,1460],[820,505],[846,3808],[866,138],[1085,545],[1133,1700],[1262,450],[1294,537],[1461,1708],[1635,693],[1805,139],[1875,2128],[2365,347],[2365,976],[4325,9645],[5325,305],[6536,2546],[7528,10503],[14460,2721],[15155,1586],[120,1999],[280,9856],[283,1254],[294,339],[352,2648],[451,2000],[472,856],[646,2813],[707,168],[723,97],[844,2842],[862,172],[1054,2431],[1127,991],[1139,3384],[1591,1104],[1676,97],[1723,6604],[1729,2783],[1753,1746],[1896,6336],[1912,9604],[2230,376],[2272,2028],[2891,927],[2915,305],[3023,5400],[3307,1897],[3371,370],[4123,2889],[4295,2823],[4741,146],[4944,2434],[5218,304],[10493,1238],[11109,17052],[14451,3237],[32,113],[117,1185],[278,584],[291,2977],[291,6270],[296,3091],[320,2127],[333,2747],[336,396],[381,111],[414,137],[416,144],[452,2928],[472,8260],[646,1516],[765,2763],[775,13058],[776,1705],[777,153],[891,390],[960,2127],[969,181],[1057,595],[1071,97],[1143,3374],[1861,673],[1901,2589],[3255,400],[3477,10771],[3673,1639],[3869,7330],[3972,3676],[4480,148],[5538,101],[6217,1170],[6437,6900],[6468,11360],[9131,293],[10324,593],[15039,3117],[112,100],[156,2176],[320,519],[333,3161],[377,655],[396,270],[415,288],[415,1607],[523,475],[537,287],[622,340],[691,2275],[759,3166],[862,168],[966,570],[1113,692],[1154,1162],[1479,159],[1746,417],[1911,575],[2198,2279],[3040,696],[3797,2957],[4637,2947],[5412,2370],[5476,3722],[5480,111],[6896,4873],[13341,3385],[14802,1240],[195,171],[263,173],[266,159],[274,683],[388,652],[432,1028],[873,465],[900,5266],[1003,148],[1157,3730],[1201,3591],[1202,1697],[1894,1417],[2413,1443],[2668,2623],[3340,884],[3575,1134],[4095,159],[4601,310],[5202,289],[5629,5182],[5880,340],[6811,1715],[9887,268],[13362,2561],[104,15509],[111,111],[195,141],[336,2184],[345,321],[370,4990],[406,162],[459,153],[495,3629],[573,3868],[648,3182],[689,191],[844,3492],[877,1136],[1060,6249],[1065,2914],[1096,51],[1189,12593],[1210,1822],[1314,3637],[1339,142],[1426,3032],[1702,11212],[1755,691],[1979,2315],[2272,1262],[2619,270],[2770,2675],[3290,1290],[3334,1613],[3770,144],[3787,2046],[3875,1021],[3927,2143],[4240,804],[5227,1083],[5639,1170],[5820,111],[6904,8071],[7771,795],[8493,1031],[9322,3411],[11165,895],[11410,7811],[273,2387],[276,1634],[276,3956],[333,7587],[347,794],[425,288],[790,2241],[854,2362],[894,8678],[1082,1471],[1099,172],[1165,1257],[1251,6032],[1513,422],[1611,157],[1986,140],[2039,152],[2139,2612],[2173,1791],[2305,820],[2557,11270],[2606,806],[2801,357],[3063,643],[3250,848],[3470,4580],[3526,1559],[3601,2092],[3902,2379],[4893,4233],[5120,357],[5907,1593],[6630,422],[7366,304],[17355,8536],[269,579],[294,2349],[331,10320],[385,858],[431,321],[471,2843],[505,3298],[508,293],[583,2302],[593,1651],[639,5374],[643,407],[650,531],[672,2757],[675,1635],[726,1272],[1004,883],[1165,1776],[1190,188],[1299,844],[1449,941],[1547,4311],[1586,339],[1590,1811],[2126,1341],[2286,2228],[2406,607],[2484,339],[2563,2034],[2815,7379],[2981,6870],[3427,1080],[3512,272],[3922,2493],[4262,136],[4791,1646],[6493,293],[7053,2688],[8075,1378],[8343,12705],[9178,3786],[65,65],[65,102],[118,2150],[281,312],[289,3301],[294,122],[323,181],[333,1578],[385,1798],[392,183],[427,157],[445,161],[472,66],[650,11481],[774,306],[810,978],[891,1806],[922,370],[1001,181],[1040,164],[1092,1544],[1195,164],[1277,386],[1286,1889],[1306,2019],[1352,170],[1453,144],[1549,847],[1641,2276],[1890,788],[1971,1103],[2121,118],[2265,854],[2345,850],[2694,1217],[2700,12093],[2946,6387],[3010,382],[3200,14708],[3883,1649],[3970,17444],[4250,1217],[7481,1294],[8145,293],[14239,1307],[17458,17886],[70,108],[224,184],[281,564],[291,8684],[302,6959],[306,348],[320,9939],[374,175],[402,1887],[457,340],[482,128],[596,317],[596,8376],[615,463],[679,3883],[759,701],[882,1317],[1125,4444],[1208,276],[1274,3497],[1287,131],[1769,145],[1870,450],[1911,643],[2056,12607],[2096,382],[2292,276],[2999,305],[3130,5550],[3631,307],[3640,6840],[3671,1062],[3770,162],[3811,1038],[3968,11451],[4180,1944],[5502,1386],[5748,872],[6499,3393],[6609,5778],[6681,4336],[6858,395],[8374,894],[9665,144],[10644,1733],[86,82],[100,578],[276,1228],[277,305],[420,779],[467,289],[541,2929],[565,135],[660,131],[672,361],[721,2115],[726,310],[873,912],[955,173],[1114,382],[1182,1544],[1232,716],[1341,1527],[1386,1598],[1413,2182],[1518,580],[2025,5769],[2192,49],[2410,3511],[2495,1252],[2659,15389],[3026,933],[3782,1532],[3809,3645],[5517,2618],[5653,370],[8462,117],[9896,10307],[118,1718],[272,268],[291,16541],[299,4989],[306,1206],[366,1724],[493,179],[548,404],[589,976],[815,1221],[1047,12220],[1201,1678],[1314,321],[1395,2761],[1413,118],[1543,9682],[1546,305],[1719,15884],[1838,2204],[1872,562],[2044,1164],[2129,276],[2203,1495],[2268,8208],[2364,51],[2511,99],[2519,8607],[2942,1209],[3319,2310],[3646,1843],[3647,531],[3758,593],[4807,1527],[6236,1999],[7781,164],[72,122],[98,462],[107,378],[122,747],[263,189],[311,370],[332,655],[336,13465],[360,377],[376,976],[396,1378],[402,1803],[458,152],[472,79],[522,1057],[799,1893],[874,1244],[900,682],[911,1241],[1144,648],[1186,562],[1189,3347],[1273,888],[1308,5974],[1392,4453],[1657,5966],[1672,4348],[1886,648],[2137,182],[2214,1021],[2270,4847],[2278,1985],[2808,293],[3017,3237],[3188,5301],[7902,7484],[8040,3650],[9107,187],[11917,76],[12625,983],[15021,12681],[32,2112],[85,3348],[104,1857],[278,816],[278,2063],[357,1115],[368,272],[378,311],[420,2586],[421,3528],[455,159],[483,190],[485,816],[529,160],[598,1440],[648,1226],[684,8409],[762,2446],[986,276],[1054,352],[1093,1634],[1529,321],[1564,272],[1729,1867],[2344,1741],[2367,97],[2410,144],[2861,3801],[2926,1122],[3101,693],[3272,7342],[3604,12125],[4004,1187],[4556,2756],[5458,417],[8839,570],[10071,5353],[10717,559],[108,4203],[114,542],[368,857],[404,111],[490,2004],[494,384],[508,4991],[663,189],[745,103],[764,98],[784,462],[917,585],[1072,268],[1156,1778],[1248,13735],[1336,276],[1385,4523],[1450,3925],[1513,9343],[1631,140],[2013,613],[2057,2205],[2107,415],[2407,14748],[2498,12804],[3170,16342],[3457,1164],[3713,752],[3989,843],[5705,4573],[5791,593],[6075,4546],[7036,105],[9537,15857],[70,274],[80,9214],[87,6365],[100,857],[225,185],[263,187],[302,2296],[319,377],[355,150],[359,136],[394,133],[398,128],[447,830],[456,187],[551,80],[591,161],[682,1994],[754,3686],[762,764],[858,1507],[931,305],[1140,276],[1248,611],[1301,15423],[1322,2845],[1641,978],[1704,794],[1738,1955],[1815,832],[1981,395],[2037,15841],[2069,5394],[2365,2197],[2770,1031],[3386,1791],[3601,3850],[3769,3430],[3819,6610],[5990,1356],[6138,114],[8084,185],[9486,3874],[10030,6273],[13809,268],[15838,8209],[76,111],[263,180],[276,2692],[286,180],[288,806],[396,1475],[419,1623],[472,584],[485,83],[499,1378],[531,607],[573,1962],[789,2346],[960,8678],[1096,54],[1107,450],[1225,152],[1242,682],[1387,1607],[1537,1787],[1631,160],[1632,149],[1727,2894],[2094,1679],[2231,822],[2981,4968],[3571,4135],[4133,115],[7892,5418],[8308,417],[8462,4071],[9537,7798],[65,3853],[66,76],[122,307],[234,189],[278,1668],[283,521],[285,390],[306,842],[367,677],[389,165],[404,370],[423,378],[705,14386],[926,675],[1054,5391],[1631,176],[2056,480],[2090,173],[2685,890],[2893,321],[3332,9675],[4079,2488],[4713,1645],[4901,824],[5325,307],[5431,8372],[6246,1604],[6609,1789],[8771,276],[9969,1370],[10201,289],[15942,1984],[107,3839],[296,324],[531,12757],[533,2816],[670,5724],[729,110],[740,132],[753,121],[765,381],[791,395],[1354,106],[2083,306],[2192,51],[2261,12768],[2364,52],[2403,1083],[2526,417],[2840,490],[3632,592],[3978,6562],[4054,2906],[4101,6148],[4548,3910],[5151,3217],[5216,176],[5429,5429],[6426,1927],[7478,13358],[12051,2034],[16800,99],[69,65],[109,345],[110,335],[289,1378],[462,110],[472,5028],[472,10041],[520,142],[541,385],[576,129],[752,3354],[756,14367],[1074,1400],[1127,2337],[1144,3182],[1201,611],[1227,854],[1232,1660],[1246,1337],[1276,811],[1301,307],[1342,661],[1413,1526],[1537,3563],[1628,887],[1635,8350],[1805,171],[2068,162],[2231,1252],[2256,156],[2557,4668],[2630,2653],[2736,391],[2882,2372],[3334,1781],[3340,468],[3604,14158],[4047,1764],[4087,1121],[5404,2444],[6326,182],[6757,832],[7692,977],[8174,2296],[8242,1667],[8597,116],[8975,7073],[8999,615],[10767,1202],[13408,12912],[14423,107],[331,11369],[350,11959],[383,1772],[423,82],[442,112],[453,730],[538,311],[834,376],[950,450],[985,553],[1229,154],[1304,1104],[1376,274],[1513,122],[1710,132],[2136,148],[2821,673],[2939,1686],[3090,490],[3255,288],[3561,370],[3595,1377],[3705,432],[4305,11846],[4941,938],[5062,97],[6514,1974],[6860,288],[8178,5287],[9533,310],[11457,367],[11757,1378],[12387,5528],[13180,4847],[15722,4274],[114,115],[281,110],[320,521],[333,4184],[356,101],[454,116],[486,386],[595,1555],[599,1232],[704,1176],[759,276],[762,3674],[846,1105],[855,737],[886,2142],[1176,5260],[1513,1779],[1527,1478],[1535,480],[1545,2088],[1563,268],[1576,2099],[1699,115],[1745,344],[1839,8978],[2367,272],[2763,268],[3251,2199],[3493,1118],[3502,716],[3908,533],[3942,2741],[4457,4457],[5103,5103],[5629,3472],[5899,370],[6253,1280],[6282,172],[6784,417],[6830,8777],[7973,977],[10118,5265],[10478,115],[11535,1647],[69,69],[224,164],[269,1630],[354,5790],[381,1356],[414,143],[472,3543],[499,9064],[617,129],[788,480],[788,2374],[877,562],[994,13679],[1117,99],[1300,1907],[1411,385],[1450,450],[1519,463],[1594,378],[2451,2234],[2965,865],[3114,191],[3168,1927],[3604,9946],[9330,156],[13911,415],[104,11766],[109,4323],[118,2529],[269,528],[285,1346],[337,340],[364,11228],[495,15532],[516,99],[571,385],[571,925],[571,2055],[664,158],[675,1329],[717,136],[812,385],[961,2046],[1246,611],[1363,7284],[1406,2078],[1766,941],[1915,133],[2088,1231],[2138,2374],[2407,311],[2466,2441],[2612,1671],[2886,1607],[3083,6680],[3125,8921],[3403,3683],[3466,383],[3514,523],[5155,107],[5776,111],[7163,289],[10733,6575],[76,71],[103,3544],[105,999],[105,1505],[276,1207],[276,1401],[288,111],[344,121],[359,180],[491,188],[501,158],[558,99],[646,2210],[650,7747],[759,1406],[759,2953],[818,2234],[1047,6046],[1131,386],[1174,129],[1308,3776],[1310,566],[1311,133],[1392,119],[1543,6950],[1800,2545],[1800,11798],[1971,1175],[2001,156],[2083,384],[3036,4495],[3059,804],[3840,79],[4167,810],[5179,9014],[6483,8485],[6797,367],[8432,1688],[10077,161],[10909,1055],[13108,1823],[13503,1844],[65,70],[78,952],[106,5501],[276,587],[281,310],[325,2304],[339,98],[348,4843],[432,2145],[434,558],[446,131],[516,108],[552,180],[684,330],[705,793],[869,114],[878,128],[924,2293],[1009,161],[1123,370],[1157,804],[1567,854],[1844,1290],[2173,114],[2195,270],[2656,344],[2669,5348],[2799,180],[3633,522],[3645,1332],[4630,2624],[5691,417],[6360,3138],[6576,2477],[7332,5153],[7803,1272],[10160,370],[11641,11931],[13767,111],[80,77],[283,2308],[285,9123],[336,5935],[350,1850],[365,908],[366,9088],[449,189],[683,7341],[719,6351],[815,1198],[855,2914],[1044,370],[1244,1355],[1250,1595],[1300,2005],[1322,794],[1372,582],[1446,607],[1484,2217],[1649,1178],[1967,131],[2369,2721],[2422,2744],[2485,531],[2796,1143],[3084,1699],[3469,1996],[3756,357],[4103,2479],[5324,9963],[5623,1332],[6356,509],[6709,1185],[13107,3153],[71,117],[78,84],[82,272],[230,512],[306,3655],[564,268],[606,162],[735,159],[873,4006],[874,6707],[897,1148],[979,2155],[1134,15102],[1289,159],[1373,779],[1485,161],[1525,1634],[1626,277],[1681,8627],[1836,10311],[1880,180],[1990,12176],[2307,17212],[2317,116],[3557,4234],[4296,7726],[4641,383],[5039,545],[5582,3482],[6729,432],[6755,1832],[7163,1148],[7729,4072],[8487,307],[11122,2785],[18032,9920],[72,79],[104,1640],[306,3028],[336,1700],[341,114],[396,114],[472,5802],[573,383],[598,3974],[625,912],[644,934],[691,1437],[725,930],[755,176],[950,559],[961,3965],[1139,978],[1318,11327],[1436,179],[1568,385],[1719,558],[1815,886],[1839,2130],[1886,1431],[2205,7333],[2959,1916],[3797,4011],[5065,2284],[5142,3585],[108,923],[109,13860],[116,274],[120,296],[276,5272],[276,7597],[281,2259],[294,781],[300,2628],[359,129],[421,2026],[595,799],[832,1230],[1071,272],[1106,1065],[1188,3748],[1247,11459],[1254,277],[1377,2315],[1450,1469],[1516,7244],[1771,1390],[1989,1353],[2059,1953],[2098,2275],[2161,53],[2684,2785],[2800,457],[3890,417],[3966,1176],[4293,370],[4999,6983],[5261,2091],[6630,370],[7519,116],[7722,3931],[7990,276],[9551,1627],[9952,4580],[10037,6875],[14061,307],[16750,1554],[65,2763],[101,339],[281,13244],[294,1403],[304,111],[396,2402],[402,12796],[430,2124],[472,16180],[586,108],[615,1226],[650,7052],[655,5836],[814,321],[873,339],[890,2181],[909,6865],[966,1594],[1195,176],[1215,6782],[1455,658],[1524,274],[1907,1844],[2199,607],[2223,289],[2395,679],[2730,6050],[2882,2462],[3161,997],[4149,4149],[4574,382],[5256,3710],[5380,17005],[5701,1038],[6251,13541],[6549,3725],[7056,7811],[7129,3801],[7202,99],[8668,3642],[8985,1844],[10387,1341],[12422,2076],[18259,370],[110,353],[226,149],[320,2309],[324,104],[327,184],[354,2143],[362,10450],[368,117],[375,6821],[425,305],[473,184],[573,640],[598,3525],[620,305],[719,353],[832,681],[972,1884],[1001,165],[1042,7709],[1059,14865],[1251,759],[1252,571],[1298,919],[1358,507],[1440,1007],[1469,5723],[1522,168],[2153,3231],[2460,48],[2533,6665],[2670,2484],[3222,1102],[3727,4826],[4167,661],[4310,2618],[4462,1432],[4544,832],[4548,1254],[4912,580],[5507,272],[6619,1029],[8148,608],[9523,11452],[12259,115],[12833,1809],[68,83],[83,274],[85,86],[118,287],[304,783],[339,1655],[401,80],[434,4643],[449,178],[485,1887],[486,5007],[508,1317],[596,1357],[615,1388],[667,2487],[753,68],[854,1443],[881,153],[882,1078],[926,421],[961,11911],[1113,9654],[1210,391],[1514,780],[1524,856],[1525,789],[1780,1186],[1921,347],[1965,611],[2446,307],[2455,509],[2636,97],[2645,49],[3132,2170],[3309,7988],[3628,1457],[3830,168],[4414,79],[5835,276],[6480,3507],[8244,348],[10630,847],[14345,2461],[84,3177],[98,2273],[118,608],[263,176],[288,268],[317,311],[397,117],[430,833],[484,2018],[508,2102],[532,117],[583,306],[651,138],[738,5535],[853,130],[928,7197],[1000,382],[1048,18739],[1143,8123],[1170,4634],[1227,1534],[1511,122],[1671,1636],[1803,1346],[1827,272],[2211,1372],[2618,4233],[3416,5598],[3823,13611],[4452,3346],[4789,901],[5044,490],[5456,277],[5641,116],[6132,10031],[7631,1013],[7709,910],[7761,289],[7817,15940],[11784,1129],[13317,111],[13642,2219],[66,572],[122,1806],[288,397],[299,634],[343,343],[375,997],[402,3507],[452,17723],[509,109],[595,16705],[626,268],[684,519],[718,185],[721,467],[752,2163],[768,173],[939,977],[975,2167],[1049,2233],[1116,3469],[1141,149],[1195,161],[1247,378],[1348,97],[1396,147],[1404,17498],[1696,809],[2116,111],[2129,463],[2639,3896],[2711,1926],[3083,575],[3159,1016],[3557,6885],[4608,842],[6855,760],[7486,8581],[7710,103],[10125,5875],[10407,187],[12635,3516],[13340,2219],[14127,779],[83,75],[120,101],[176,352],[231,2101],[273,1827],[331,397],[412,173],[532,437],[568,171],[655,2706],[694,189],[728,2007],[766,615],[946,2573],[1046,268],[1231,1961],[1344,395],[1406,2213],[1423,2542],[1722,558],[1795,305],[2014,188],[2118,1580],[2305,486],[2320,5996],[3220,293],[3544,339],[4241,938],[4305,14764],[4327,799],[4636,692],[4727,370],[4878,112],[5079,2603],[5591,829],[5880,383],[6557,184],[6678,496],[7744,268],[8240,289],[13585,3720],[32,647],[99,1348],[269,2259],[336,3371],[339,122],[497,7267],[517,136],[544,385],[636,180],[682,496],[839,467],[892,4974],[897,122],[957,167],[1128,1601],[1520,796],[1529,97],[1784,4717],[1950,417],[1978,187],[2398,1687],[2423,1819],[2544,5594],[2592,1814],[3631,1633],[4419,3331],[4548,2181],[5539,2251],[5649,274],[6960,871],[8883,450],[9534,13996],[16325,10229],[78,1046],[109,353],[236,169],[274,6486],[300,325],[332,340],[333,3138],[341,8680],[401,381],[483,128],[486,990],[516,783],[564,272],[636,163],[684,307],[753,1151],[774,585],[779,407],[855,817],[931,97],[1044,305],[1196,116],[1354,2105],[1447,1609],[1463,1218],[1517,2634],[1657,9780],[1747,2340],[1765,340],[1935,11562],[1971,733],[2084,1658],[2297,4679],[3061,3616],[3127,357],[3555,116],[3788,1665],[3955,480],[4032,5076],[4206,17759],[4242,1060],[4593,3640],[5448,939],[7210,390],[11901,3346],[14758,10668],[67,76],[87,7890],[196,129],[233,183],[278,428],[283,1205],[287,311],[299,3012],[329,165],[364,356],[425,289],[571,7903],[611,480],[623,1336],[631,1586],[705,1138],[779,2370],[843,765],[865,1958],[866,179],[981,509],[1059,4633],[1518,994],[1536,1734],[1689,1806],[1802,559],[1950,16395],[2121,901],[2204,843],[2392,335],[2453,4115],[2577,407],[2677,644],[3400,321],[3832,385],[4093,1670],[4391,116],[5025,276],[5353,2296],[5566,2741],[5756,1830],[5873,66],[6520,2531],[7320,10116],[8518,1502],[9610,1709],[11858,272],[13345,1101],[17153,304],[65,75],[99,894],[235,128],[266,152],[299,2777],[320,2806],[347,3402],[357,450],[393,305],[401,370],[442,1273],[449,181],[478,184],[650,17488],[661,475],[732,181],[765,1389],[793,1198],[887,496],[936,270],[1012,133],[1089,1179],[1288,3384],[1489,450],[1550,159],[1691,417],[1757,978],[1900,152],[1902,3397],[1937,357],[1946,6077],[1996,693],[2342,661],[2473,2321],[2496,2906],[3131,1323],[3139,1340],[4199,545],[4342,9550],[5003,2911],[5140,13072],[5240,7563],[5520,5154],[5959,2497],[6205,2085],[6533,1610],[6727,156],[7531,417],[8797,121],[71,79],[104,692],[269,5376],[276,386],[276,1068],[321,122],[367,1102],[376,691],[434,1432],[442,901],[468,383],[508,13687],[573,842],[587,4074],[711,833],[773,399],[801,11549],[865,871],[872,1422],[943,1078],[1113,370],[1262,382],[1294,272],[1430,1136],[1433,2003],[1450,3792],[1657,1830],[1946,6591],[1965,417],[2075,490],[2203,15489],[2369,995],[2422,9317],[2532,978],[3158,3454],[3185,2785],[3503,6610],[3602,1241],[3797,4856],[3948,330],[3997,1037],[4675,1321],[4796,361],[6072,3437],[6671,3577],[6757,886],[6872,1341],[7332,2597],[7463,1043],[7553,1523],[10253,809],[67,83],[80,86],[299,16563],[302,99],[324,311],[337,1732],[434,806],[508,330],[508,361],[622,365],[666,7505],[688,270],[753,2455],[806,122],[897,272],[1029,2167],[1183,1137],[1195,141],[1264,1257],[1308,116],[1790,1005],[1932,2190],[1987,5376],[2327,319],[2627,2536],[2725,6215],[2765,17236],[3288,580],[3811,3801],[4052,2770],[4547,12507],[4698,1696],[6092,558],[7792,911],[12505,709],[13154,1594],[15917,325],[66,13102],[104,462],[108,428],[118,1142],[263,163],[320,10731],[377,2003],[452,8573],[484,6022],[494,367],[495,344],[506,165],[578,272],[608,15915],[646,3712],[723,270],[773,759],[792,134],[814,1185],[935,165],[937,181],[957,159],[1052,496],[1083,357],[1119,10121],[1163,643],[1164,407],[1239,2618],[1299,12059],[1507,743],[1601,7173],[1615,978],[2064,169],[2257,305],[3390,876],[3830,181],[3869,2812],[4281,479],[4421,3341],[4654,13749],[6466,1060],[9658,796],[9861,6065],[13812,2775],[17247,17808],[263,164],[320,465],[336,345],[336,5176],[354,4913],[375,2700],[490,930],[495,4946],[562,4183],[640,1104],[684,324],[784,572],[797,378],[881,169],[900,2061],[924,1674],[963,2758],[1244,1231],[1713,151],[1802,417],[2080,1811],[2503,1748],[3122,2546],[3549,11324],[4369,378],[4956,6082],[5485,522],[10821,9532],[11011,3553],[81,1043],[103,110],[129,676],[269,2538],[276,1488],[302,1879],[332,1384],[368,7398],[375,733],[456,175],[486,530],[524,471],[611,3205],[620,272],[621,2721],[644,354],[677,585],[682,3065],[726,287],[727,103],[820,386],[846,2005],[847,643],[960,272],[1113,335],[1394,9414],[1423,832],[1480,183],[1482,3037],[1595,877],[1672,1106],[1736,1759],[1817,522],[1856,1043],[1954,760],[2080,928],[2272,1757],[2288,2048],[2415,9788],[2447,871],[2966,84],[3020,837],[3471,324],[3681,172],[3844,772],[3964,589],[4236,7436],[4348,1401],[4791,288],[4866,17675],[4965,159],[5368,5609],[7462,875],[9565,894],[10662,12992],[11834,3328],[15828,1789],[17001,3792],[118,5508],[128,180],[171,1145],[261,180],[277,806],[288,1038],[299,4265],[312,850],[335,339],[336,4849],[343,306],[357,4533],[376,376],[390,876],[553,2155],[700,1437],[701,5210],[705,3826],[743,5620],[755,135],[800,1587],[822,12149],[854,276],[916,137],[929,14970],[1053,133],[1134,475],[1154,103],[1419,140],[1468,14244],[1497,144],[1649,975],[1749,1000],[1765,453],[1872,5485],[2111,321],[2201,12255],[2223,415],[2412,1499],[2511,112],[2692,822],[2714,1780],[2794,822],[3097,2091],[3158,1613],[3526,1957],[3816,2641],[4078,270],[4193,65],[4432,5572],[4621,1457],[5171,1633],[5200,16486],[6224,296],[6922,3232],[6983,2490],[7608,365],[7611,1509],[8704,1495],[9006,319],[9591,101],[10133,2967],[12147,1515],[16640,2215],[81,76],[83,270],[320,4493],[357,2871],[453,911],[454,851],[457,332],[574,307],[579,112],[591,168],[613,1239],[704,13884],[706,9636],[945,134],[1035,623],[1106,571],[1191,1198],[1309,357],[1375,140],[1465,1194],[1469,1906],[1545,5469],[1600,177],[2007,6111],[2011,9782],[2120,4798],[2123,306],[2600,14203],[3047,3926],[3081,1837],[3084,19042],[3096,2170],[3147,1173],[3287,178],[3392,10517],[3993,289],[4100,144],[4446,1162],[5412,1164],[6659,307],[9841,1624],[11611,1240],[12263,289],[80,111],[115,991],[277,995],[307,305],[350,2177],[361,330],[368,6066],[388,3522],[401,12128],[477,1612],[529,140],[598,109],[621,312],[658,2348],[665,172],[702,1038],[728,1692],[804,296],[855,2791],[873,395],[963,572],[1147,144],[1175,947],[1235,2205],[1256,1006],[1285,115],[1321,1746],[1393,6791],[1441,144],[1584,142],[1745,1368],[1935,779],[2059,1907],[2291,155],[2668,1899],[2685,270],[2762,2519],[2771,1469],[4000,1031],[4462,2846],[4506,7107],[4807,2328],[5652,928],[6731,1879],[9107,168],[10874,1317],[10986,747],[82,73],[114,317],[118,395],[299,425],[299,598],[320,3270],[326,12426],[331,117],[375,421],[401,119],[448,9625],[499,312],[521,1654],[644,348],[745,17145],[758,181],[947,1128],[1176,531],[1201,10892],[1248,865],[1335,15032],[1376,9801],[1385,430],[2213,3188],[2232,759],[2499,6484],[2542,1058],[2645,50],[2939,422],[3154,276],[3496,812],[3582,1671],[3952,1348],[4508,1290],[5506,673],[5852,10033],[6359,4672],[8343,3262],[9275,289],[12113,5747],[12864,4151],[13120,3867],[32,2332],[84,8327],[98,572],[101,1078],[267,7733],[305,112],[310,304],[392,141],[427,144],[505,385],[536,159],[548,324],[669,5379],[730,432],[799,1276],[844,2205],[853,154],[923,97],[931,1041],[1037,737],[1095,382],[1143,3412],[1204,143],[1404,4789],[1452,795],[1523,108],[1537,788],[1638,2786],[1689,117],[1798,276],[1970,347],[2497,19565],[2588,978],[2639,2253],[2667,305],[3516,1996],[4039,538],[4620,832],[4670,766],[4737,857],[4960,716],[6322,5698],[9135,4898],[10483,3166],[12094,462],[80,270],[121,683],[233,181],[283,1806],[339,105],[362,10499],[386,648],[428,277],[434,1721],[463,5515],[529,139],[587,1541],[591,171],[611,1259],[638,185],[721,9885],[724,2183],[745,7583],[798,148],[819,1941],[1037,3653],[1092,276],[1121,2949],[1145,152],[1178,430],[1205,442],[1247,16482],[1344,558],[1455,3410],[1473,129],[1681,407],[2029,642],[2082,2329],[2454,523],[2528,1369],[3497,347],[3933,442],[4219,6385],[5412,11098],[5857,1083],[5946,584],[6252,1948],[8088,119],[8155,556],[8341,1627],[17843,5323],[67,326],[70,84],[83,117],[104,317],[267,9861],[306,4551],[368,592],[386,1634],[399,623],[434,1745],[472,78],[477,2052],[584,115],[656,169],[694,181],[882,3485],[916,130],[924,2866],[1012,160],[1093,3133],[1144,827],[1175,1405],[1252,989],[1301,528],[1326,2548],[1335,17207],[1529,361],[1681,696],[1723,2292],[1829,3885],[2366,3967],[2401,289],[2505,16950],[2516,6979],[3210,137],[3738,3424],[3822,3800],[3998,1019],[4439,1085],[4939,4939],[5043,121],[5665,556],[5775,4283],[5914,1317],[6174,1808],[6192,289],[6442,772],[7593,2486],[7823,146],[10638,648],[13854,9889],[87,305],[104,574],[273,1944],[276,1845],[278,13539],[281,6879],[291,462],[333,3565],[496,724],[532,634],[562,2566],[572,1240],[711,2318],[717,189],[776,2513],[784,1123],[842,457],[862,138],[922,293],[1074,607],[1105,4159],[1194,992],[1393,809],[1411,2210],[1444,465],[1729,2427],[1736,2156],[1783,376],[1870,417],[2011,99],[2878,1386],[2929,2935],[2988,167],[3135,894],[3252,7546],[3418,2065],[4218,850],[4469,312],[4530,622],[5241,13014],[6528,1105],[7188,4789],[7785,575],[11599,144],[102,1011],[276,3018],[283,4201],[300,339],[300,3693],[332,5299],[390,3034],[397,1539],[401,75],[495,2757],[529,129],[600,175],[658,1207],[688,97],[730,319],[833,1682],[996,14683],[1126,152],[1314,8634],[1353,1758],[1689,6351],[1968,184],[2076,8827],[2258,851],[2331,3100],[2333,2182],[2469,10348],[2524,147],[2657,321],[2810,340],[2844,307],[3316,6045],[3318,1273],[3348,4678],[3711,562],[3821,1321],[3933,2112],[4078,111],[4487,339],[4657,144],[4690,19934],[4699,2445],[4735,183],[4848,1292],[4872,12886],[6641,268],[7089,12091],[7478,2529],[7858,1418],[8387,12822],[9789,8118],[68,117],[269,16213],[280,17275],[412,172],[438,168],[454,8164],[573,367],[629,152],[670,305],[726,521],[805,167],[877,1330],[986,399],[986,14112],[1048,5360],[1136,3305],[1192,2073],[1196,272],[1206,1666],[1242,1315],[1269,168],[1404,509],[1758,9407],[2185,3167],[2347,360],[2497,1074],[2777,425],[2892,804],[3340,677],[3927,884],[3993,12260],[4175,1501],[4782,562],[4782,1208],[5309,1469],[5786,382],[5943,13042],[6662,1623],[8731,3351],[9017,829],[9342,421],[11372,3946],[15051,5547],[114,1148],[118,1356],[304,509],[348,19826],[368,1606],[375,1031],[482,164],[496,399],[523,1823],[627,154],[642,1062],[667,1534],[853,170],[858,522],[1066,333],[1382,11635],[1542,812],[2138,19275],[2222,3787],[2485,1461],[2508,276],[2557,5519],[2657,1272],[2686,139],[2940,276],[3395,1262],[3641,305],[3712,544],[3749,148],[3868,1706],[4530,1358],[4579,14962],[6435,370],[6482,5983],[8737,1811],[9916,272],[11052,8688],[12214,1775],[14867,335],[19112,17668],[118,5303],[287,2576],[289,783],[306,1104],[314,160],[401,564],[495,4648],[504,170],[516,2000],[532,105],[532,4595],[569,170],[581,164],[586,7204],[606,128],[650,698],[678,151],[703,1299],[725,972],[753,5482],[785,1620],[863,6351],[937,155],[963,17971],[992,2128],[993,2248],[1056,174],[1133,12843],[1197,1715],[1262,2306],[1422,2633],[1537,643],[1759,811],[2011,2753],[2398,1780],[2557,3866],[2881,1927],[3023,1045],[3065,4860],[3242,276],[3588,11204],[4759,1070],[5310,20058],[5411,6383],[5643,1346],[7668,1217],[8228,16691],[10525,370],[11182,620],[13338,3171],[17196,847],[67,902],[77,688],[103,10015],[104,856],[110,305],[115,451],[269,2869],[289,8555],[293,4165],[299,415],[324,927],[371,173],[426,176],[441,5443],[451,305],[495,117],[532,1563],[596,4948],[642,643],[674,160],[733,3907],[1235,3069],[1239,376],[1308,99],[1538,522],[1704,421],[1726,909],[1793,348],[1838,6586],[1992,553],[2107,9088],[2192,55],[2320,305],[2453,1903],[3000,1083],[3452,1038],[3670,189],[3728,1118],[3770,169],[4531,8054],[5112,4297],[5883,361],[6134,417],[7171,2648],[7730,2078],[13498,304],[14754,468],[15361,1487],[15714,324],[15803,2464],[16043,2609],[16146,457],[77,277],[276,2129],[288,289],[336,1205],[397,98],[495,2006],[522,1124],[548,345],[659,337],[673,1121],[775,141],[779,661],[779,1321],[794,761],[859,3028],[958,927],[961,11132],[1125,1123],[1156,6428],[1185,122],[1236,2034],[1248,1802],[1270,53],[1406,6733],[1424,3270],[1451,56],[1454,1217],[1552,111],[1592,312],[1847,815],[1972,13712],[2064,128],[2415,7017],[2627,628],[2814,138],[2879,599],[3321,950],[3582,972],[3694,115],[4212,4234],[4635,2957],[4879,7850],[6017,12314],[6970,463],[7239,128],[7525,1179],[8401,679],[10452,1346],[10783,12145],[12329,1254],[107,6219],[263,175],[299,652],[350,15948],[370,116],[420,752],[426,182],[466,160],[565,147],[667,1767],[688,993],[706,115],[753,296],[754,1043],[795,276],[813,6782],[844,832],[871,1843],[1020,171],[1054,9282],[1191,940],[1222,1521],[1576,2162],[1671,6383],[1805,169],[1895,325],[1990,110],[2015,1757],[2043,417],[2462,691],[2614,1335],[2717,2633],[2808,11253],[2929,3497],[3387,993],[3421,1241],[3856,1922],[3872,304],[4261,8082],[4473,553],[4694,673],[4865,3642],[5100,912],[5830,5675],[6326,149],[6964,566],[8017,2147],[8419,168],[9369,382],[68,67],[69,117],[88,68],[117,326],[267,164],[291,117],[300,400],[341,1995],[345,289],[347,2904],[375,13937],[402,9157],[490,13846],[499,1039],[572,2911],[629,138],[646,2329],[738,19130],[769,477],[814,533],[853,137],[875,1614],[887,5340],[944,872],[980,9905],[1013,7644],[1193,1817],[1247,3201],[1257,10688],[1292,1136],[1335,12546],[1430,1208],[1512,332],[1529,2860],[2056,5676],[2192,54],[2639,3671],[2688,1070],[4519,138],[4645,7722],[4956,321],[7498,12366],[10149,462],[13093,5104],[16858,1356],[107,538],[114,2606],[261,135],[336,2317],[625,3345],[701,4823],[735,136],[748,1759],[795,2326],[820,613],[1141,158],[1150,339],[1482,109],[1560,152],[1707,1764],[1769,164],[1795,307],[1893,1052],[2098,1914],[2165,289],[2454,978],[2488,2087],[2639,417],[2684,2205],[3094,4369],[3132,2279],[3380,140],[3387,7025],[3598,2279],[3807,305],[4325,1254],[4923,171],[4979,1742],[8695,709],[9023,268],[13868,1604],[65,381],[74,111],[85,927],[97,1346],[98,570],[99,856],[119,1796],[373,162],[376,5506],[635,150],[666,2045],[770,2539],[836,522],[874,810],[1082,2087],[1089,360],[1244,1897],[1351,7454],[1529,116],[1765,383],[1834,155],[1851,287],[1922,2711],[1996,3129],[2348,17024],[2736,813],[2858,8659],[3999,4832],[4312,15388],[4786,3142],[4810,990],[4843,507],[5526,18733],[5748,7991],[6921,268],[8025,2229],[16458,3469],[16994,140],[66,114],[67,1508],[71,77],[276,7567],[339,1743],[375,9407],[389,153],[420,3384],[452,1709],[472,1804],[508,1603],[545,545],[752,796],[753,288],[871,382],[875,6286],[961,270],[1059,391],[1059,832],[1095,5723],[1116,2487],[1189,4132],[1398,143],[1721,1665],[1829,3784],[2365,11867],[2453,3268],[2640,177],[2837,2835],[2854,1859],[2945,276],[3253,6721],[4136,72],[4826,3143],[6157,988],[6838,3795],[7595,382],[8542,307],[12079,268],[13793,101],[13952,2379],[281,1365],[306,4076],[333,4835],[337,4513],[386,1859],[503,296],[721,634],[728,1620],[765,3213],[931,14221],[983,1830],[1488,724],[1954,2698],[2004,1776],[2286,115],[2469,11785],[2645,51],[2650,1780],[2685,289],[3738,1074],[4000,759],[4114,2882],[6052,11100],[7128,1356],[7155,111],[7943,1678],[8651,325],[8900,1999],[10001,109],[10350,2033],[11500,9480],[97,462],[121,317],[320,1474],[336,16008],[360,354],[375,1714],[395,270],[402,19550],[419,425],[614,1655],[654,139],[669,325],[719,6002],[791,8509],[793,3068],[855,769],[859,1005],[947,3193],[961,578],[1165,1138],[1176,990],[1198,1812],[1248,643],[1336,347],[1353,3133],[1362,6477],[1472,13463],[1676,890],[1739,541],[1869,2047],[1988,1151],[2032,130],[2246,450],[2261,312],[2262,1171],[2334,357],[2518,13085],[2547,399],[2559,465],[2608,9502],[3365,311],[3371,121],[3572,12193],[3873,475],[4715,276],[5616,11331],[7176,2618],[12561,6222],[13383,1887],[67,78],[115,851],[269,1523],[283,1043],[283,2006],[310,3013],[351,19183],[361,103],[421,399],[421,854],[675,3114],[728,648],[748,7769],[759,1328],[782,167],[1012,139],[1083,407],[1170,3792],[1231,545],[1328,833],[1383,103],[1394,11333],[1434,3768],[1458,796],[1693,1202],[1815,4544],[1843,1426],[1997,175],[2361,134],[2410,157],[2649,3591],[3061,5704],[3401,399],[3594,276],[3652,938],[5163,919],[5658,741],[5834,1364],[7219,558],[7234,1143],[7301,781],[7442,300],[8040,1221],[8339,1498],[8879,1000],[9031,115],[10865,1669],[13864,2331],[14511,830],[18631,598],[235,149],[273,1078],[278,2008],[280,189],[300,97],[336,10879],[345,783],[355,148],[385,463],[421,812],[436,167],[514,103],[555,152],[614,121],[696,871],[815,2474],[860,138],[900,9203],[938,1221],[965,453],[1075,743],[1083,2973],[1095,1535],[1100,268],[1169,186],[1210,2017],[1316,182],[1430,716],[1521,595],[1651,4954],[1657,1259],[1694,1000],[1915,179],[1937,417],[2198,2076],[2496,2902],[2608,1667],[2703,305],[2779,348],[2938,335],[3340,5129],[3558,2618],[3685,15628],[3893,152],[5232,1218],[5437,6981],[5582,760],[6316,289],[6924,2183],[7365,8072],[7888,132],[8159,417],[10123,10695],[72,111],[100,121],[283,310],[283,324],[291,326],[299,1668],[300,6965],[368,1043],[376,530],[383,1728],[431,2182],[499,959],[588,175],[589,1775],[602,15538],[615,1222],[646,741],[728,3138],[801,1280],[834,2783],[869,706],[882,5002],[888,6792],[958,830],[989,972],[995,18817],[1083,1139],[1119,5924],[1208,1393],[1237,162],[1248,6622],[1354,15242],[1376,1346],[1396,135],[1446,976],[1491,288],[1551,2650],[1586,274],[1644,844],[1671,3101],[1904,2823],[1920,53],[2800,3441],[3047,5233],[3468,138],[3891,3462],[4244,16058],[4396,143],[4408,2057],[4657,135],[5372,4881],[5505,2488],[5622,332],[6715,116],[7519,747],[8034,312],[8888,2816],[10152,276],[12384,783],[12829,709],[15709,7978],[32,9595],[68,77],[120,111],[122,122],[281,608],[305,509],[357,3643],[366,6005],[415,1742],[420,8874],[421,759],[495,578],[496,928],[549,147],[595,1143],[627,187],[671,164],[688,356],[886,1202],[921,5111],[922,335],[943,102],[1071,111],[1157,397],[1199,277],[1301,5722],[1326,2796],[1354,353],[1370,2387],[1446,679],[1525,3891],[1788,321],[2040,793],[2059,1939],[2128,1557],[2207,832],[2454,2276],[2487,1589],[2724,2428],[2892,5841],[2900,855],[3067,1173],[3277,2508],[4104,1949],[4456,4456],[4602,1155],[4799,3511],[6086,97],[7175,4438],[7614,321],[8954,1370],[9950,144],[11268,277],[12740,1183],[14177,272],[15581,370],[65,9947],[83,76],[294,274],[342,187],[364,103],[394,165],[401,2102],[408,164],[467,272],[531,5919],[586,14162],[588,156],[688,2046],[705,883],[818,2714],[840,311],[982,306],[1001,173],[1028,1790],[1046,6840],[1091,184],[1211,6590],[1411,667],[1419,174],[1488,811],[1613,5391],[1696,1176],[1819,673],[1910,1302],[1987,4868],[2160,920],[2235,1592],[2518,14684],[2584,348],[2685,1317],[2997,1401],[3109,1509],[3115,952],[3240,691],[3322,2253],[3380,188],[3641,272],[3874,296],[4263,2204],[4469,1007],[4904,399],[7174,97],[8256,451],[9234,1604],[10197,306],[11090,608],[11967,4769],[13366,5755],[15142,9428],[19150,815],[99,1853],[110,324],[283,16114],[337,1587],[390,289],[449,160],[462,109],[472,6057],[548,2749],[591,164],[675,650],[685,100],[705,490],[725,913],[820,385],[874,4769],[924,5013],[984,1244],[1013,10737],[1021,847],[1081,152],[1240,272],[1274,2160],[1382,12450],[1417,1188],[1442,2942],[1461,1418],[1504,178],[1784,112],[1792,3557],[1994,399],[2001,157],[2055,1049],[2138,810],[2401,3226],[2663,13025],[2990,1358],[2997,875],[3043,321],[3398,142],[3502,1923],[3631,5246],[3701,347],[4530,556],[5132,1961],[5302,8110],[10157,467],[11655,2702],[12637,3141],[14259,918],[14720,4608],[69,86],[275,186],[276,2485],[285,324],[300,3974],[321,1740],[375,7982],[404,9240],[424,175],[432,583],[535,164],[785,1055],[853,187],[961,9142],[1039,10943],[1049,4327],[1066,1620],[1240,116],[1267,159],[1309,1129],[1310,545],[1310,1373],[1468,4717],[1540,111],[1706,1822],[1891,268],[2040,2770],[2075,1687],[2475,572],[2496,1262],[2676,9620],[2817,421],[2910,116],[3014,172],[3592,806],[3693,1673],[3799,15470],[4050,3025],[4114,733],[5324,13663],[5420,7325],[6069,353],[9625,494],[11805,1786],[111,1209],[117,3702],[119,274],[269,8517],[270,107],[276,3568],[367,9419],[368,1346],[386,2329],[583,9253],[587,599],[593,4754],[599,1323],[644,383],[653,141],[679,892],[696,1343],[732,148],[769,946],[862,137],[938,2613],[1029,985],[1093,3638],[1189,3058],[1308,6144],[1336,2624],[1376,13694],[1601,941],[1800,5723],[1836,4397],[1863,3289],[1867,1703],[1960,2673],[1992,1187],[2056,1115],[2082,1452],[2119,1903],[2262,562],[2311,9852],[2764,794],[2871,19103],[3010,3074],[3146,288],[3376,1666],[3425,1826],[4475,1005],[4661,2445],[4988,11610],[4995,1760],[5326,844],[5611,779],[6326,166],[7524,890],[7632,676],[7696,709],[10865,1985],[17804,824],[32,4265],[281,1594],[285,4012],[336,3545],[366,111],[375,1448],[457,348],[499,7678],[528,99],[602,1586],[640,1058],[649,176],[665,173],[677,3414],[711,2395],[788,475],[855,522],[933,1340],[965,659],[970,268],[999,1640],[1039,558],[1096,56],[1108,2167],[1292,910],[1694,566],[1758,7982],[1808,2142],[1815,6702],[1891,10902],[2669,804],[2876,100],[3410,895],[4116,1198],[4761,3231],[4844,304],[5121,276],[5215,2048],[6049,1041],[6388,7722],[6468,13113],[7185,108],[7988,1286],[9902,537],[10648,378],[14016,936],[14527,11053],[69,2758],[80,101],[117,558],[273,17490],[285,1046],[285,1348],[296,5104],[300,356],[314,174],[316,140],[325,7313],[331,5581],[364,120],[376,1778],[390,400],[390,3518],[429,164],[499,467],[508,304],[533,1007],[639,5024],[683,8029],[749,144],[751,129],[754,620],[805,161],[853,152],[966,688],[980,13960],[993,268],[1309,2099],[1312,572],[1314,3855],[1394,1756],[1595,2495],[1680,1680],[1744,12471],[1777,2061],[1948,391],[2337,628],[2439,580],[2554,97],[2946,376],[3205,352],[3212,188],[3318,509],[3705,6459],[3908,6158],[4034,7372],[4055,6595],[4242,1469],[4844,1835],[4883,780],[5008,17846],[8427,2085],[10050,723],[10829,479],[12384,321],[14242,2949],[18412,131],[68,82],[97,2910],[100,1209],[102,13048],[107,117],[108,344],[110,121],[111,2085],[261,189],[276,4706],[378,1294],[390,277],[420,7337],[472,80],[548,378],[615,2952],[622,3165],[633,188],[678,184],[738,344],[741,1239],[753,1271],[928,430],[943,117],[976,904],[990,858],[1066,1541],[1104,16015],[1125,1421],[1199,1263],[1361,3192],[1480,148],[1519,4341],[1565,135],[1646,620],[1754,136],[1802,357],[1908,950],[1909,185],[2165,2034],[2254,407],[2265,399],[2619,102],[2887,779],[2979,3404],[3007,305],[3019,311],[3316,829],[3317,8685],[3322,3944],[3370,352],[3749,151],[3992,8484],[4389,888],[5407,1409],[5589,10325],[7479,2738],[7509,304],[7858,1198],[9754,2185],[10157,6016],[10859,18902],[12914,116],[14769,1209],[16000,556],[104,15568],[107,330],[108,1357],[269,2112],[273,104],[344,397],[357,1259],[388,5496],[397,1879],[421,2170],[421,3394],[423,325],[434,339],[463,9213],[485,5124],[546,169],[587,587],[588,147],[625,1209],[633,169],[661,1259],[691,2047],[869,816],[957,190],[1203,187],[1216,138],[1248,1487],[1288,3944],[1453,173],[1513,117],[1668,608],[1678,382],[1719,3581],[1950,7399],[2343,3611],[2421,920],[2476,2022],[2497,7500],[2683,6839],[2717,11327],[2772,276],[3564,626],[3664,1469],[3791,793],[4075,13230],[4965,163],[5211,268],[5570,4558],[5707,383],[6777,115],[7669,890],[8340,18148],[8928,2089],[9611,2761],[13695,13695],[15014,146],[17396,5293],[18889,850],[86,3145],[119,3803],[276,3225],[326,1421],[333,992],[375,5364],[378,462],[391,2205],[401,885],[401,2559],[420,8283],[499,11442],[503,19604],[531,1418],[548,2458],[568,155],[598,1491],[614,10170],[623,646],[638,165],[672,20411],[686,165],[688,3702],[752,7491],[943,7062],[1089,365],[1090,6443],[1112,2213],[1246,352],[1322,1507],[1322,2427],[1326,4011],[1376,7258],[1385,1924],[1488,1561],[1672,1405],[1881,16341],[1904,811],[1938,324],[2165,5409],[2642,3785],[2685,850],[2714,811],[2796,8604],[2817,720],[2966,65],[2976,1646],[3287,190],[3400,977],[3444,5097],[3457,779],[3562,930],[3890,357],[3942,7035],[4219,1446],[4388,3639],[4592,2870],[4823,599],[5159,4949],[5927,4297],[6013,1902],[6989,1796],[7239,138],[8561,1658],[9121,370],[9652,2376],[10190,1100],[11539,3848],[103,901],[105,1892],[268,268],[269,2196],[277,20681],[299,1851],[302,2502],[319,4266],[333,1393],[334,2191],[337,367],[381,321],[401,289],[533,1786],[536,177],[547,19544],[554,184],[580,5906],[599,430],[600,149],[655,16835],[725,2128],[733,3109],[788,3923],[790,376],[913,4839],[936,9306],[985,276],[1020,181],[1046,783],[1068,595],[1119,3735],[1136,938],[1204,178],[1305,11071],[1364,1548],[1406,1703],[1413,312],[1458,382],[1471,17970],[1482,533],[1524,105],[1601,1152],[1811,1144],[1827,578],[2335,276],[2371,3388],[2435,761],[2493,1656],[2987,2066],[3173,2088],[4030,276],[4857,1194],[5148,1029],[6340,12810],[7280,97],[7706,168],[7823,135],[7937,1867],[8650,105],[9003,1290],[10405,1280],[12243,20357],[13818,104],[14191,14326],[17528,12137],[21240,171],[67,8932],[69,85],[111,912],[276,4082],[281,9309],[292,155],[299,572],[299,6810],[345,11604],[397,122],[423,80],[465,381],[465,16506],[472,272],[485,3855],[489,149],[490,4367],[551,683],[602,1630],[661,611],[682,2397],[709,6903],[809,3659],[839,103],[975,858],[1066,1877],[1071,5114],[1091,141],[1216,175],[1406,7208],[1432,101],[1450,1787],[1527,3566],[1872,5631],[1931,102],[1956,652],[2029,4299],[2053,376],[2062,10836],[2120,7901],[2202,5980],[2337,2112],[2368,1060],[2921,545],[3264,268],[3284,3038],[3323,324],[3670,165],[4132,479],[5261,14571],[6638,14434],[8503,276],[10097,115],[10648,293],[13604,4141],[263,167],[269,7977],[273,270],[278,8525],[302,10317],[360,365],[368,11246],[388,395],[395,1072],[467,396],[482,170],[509,111],[515,12388],[571,2675],[613,682],[703,13309],[765,509],[796,3354],[822,701],[830,101],[833,2839],[943,83],[995,111],[1228,13303],[1273,2304],[1357,538],[1477,136],[1631,129],[1664,167],[1745,324],[1934,5005],[1954,417],[1997,135],[2039,173],[2044,3231],[2153,1395],[2213,6060],[2313,141],[2630,544],[2685,324],[2921,4237],[3452,305],[3947,276],[4057,1703],[4406,1623],[4994,306],[5209,795],[5519,1065],[5643,307],[5726,321],[5953,1173],[6608,9032],[6856,3222],[6922,5803],[7171,11857],[7302,1913],[7359,276],[7424,783],[8624,6284],[10125,4572],[12663,161],[15797,1721],[18690,13245],[76,67],[77,274],[79,572],[104,17719],[269,1266],[300,3807],[333,743],[333,1692],[345,114],[378,99],[392,171],[401,8451],[403,162],[465,108],[485,723],[513,150],[586,1673],[593,1019],[611,788],[640,4575],[672,4028],[687,163],[721,843],[753,13841],[761,909],[934,1179],[962,121],[1037,1763],[1067,188],[1112,1705],[1113,307],[1124,562],[1139,1487],[1157,1263],[1164,559],[1494,274],[1847,743],[1972,20238],[2159,3124],[2229,2229],[2318,700],[2410,145],[3330,417],[3416,1134],[4281,1373],[4440,1554],[4526,311],[4903,1534],[5350,397],[5665,7114],[5676,2621],[5741,1262],[5758,272],[6719,370],[7210,841],[7550,97],[7784,5917],[9189,3622],[11720,4266],[11775,1280],[13208,165],[13970,608],[14899,10778],[108,3839],[153,676],[274,102],[283,608],[296,890],[299,1357],[317,509],[333,2514],[357,6191],[402,1853],[420,450],[430,1397],[503,8637],[515,952],[522,4690],[629,185],[639,3414],[711,1557],[762,17082],[928,2713],[937,170],[940,6078],[980,5028],[1084,11296],[1105,559],[1119,8302],[1197,270],[1246,3087],[1462,53],[1468,829],[1591,832],[1591,11288],[1697,5554],[1871,173],[1905,691],[1968,177],[2098,1466],[2116,570],[2330,12147],[2462,1771],[2562,103],[2642,976],[2790,122],[3245,920],[3563,417],[3909,442],[4202,276],[4204,11565],[4329,384],[4824,8006],[8258,8319],[8965,450],[12531,12531],[12794,321],[13031,673],[79,936],[194,170],[266,153],[278,324],[278,960],[300,4453],[305,97],[320,268],[320,6610],[330,307],[376,933],[402,404],[419,5354],[427,147],[495,1011],[577,173],[588,159],[595,4016],[622,319],[681,1210],[766,2010],[844,7485],[846,2902],[894,311],[1055,276],[1059,1822],[1070,4759],[1075,1293],[1388,4856],[1413,122],[1450,475],[1537,611],[1584,168],[1608,172],[1642,1107],[1820,132],[2164,2242],[2310,1606],[2567,180],[3061,564],[3081,1931],[3236,1717],[3316,400],[3748,1390],[3808,676],[3812,1011],[3844,108],[4153,4381],[4173,9219],[4221,7007],[4339,3192],[5539,2618],[5845,1509],[7777,659],[7889,1063],[10329,6947],[10936,270],[11647,509],[11845,1555],[16121,4195],[16566,1039],[17473,3203],[17477,132],[32,6851],[97,11210],[109,381],[112,422],[299,14171],[302,6296],[339,2444],[386,716],[399,682],[405,170],[454,270],[490,531],[519,268],[563,171],[590,176],[596,121],[677,360],[705,1176],[741,2329],[763,162],[801,19443],[865,847],[886,306],[924,5977],[943,939],[984,2577],[1146,189],[1170,1469],[1183,1484],[1214,3356],[1264,1776],[1284,128],[1405,1923],[1421,921],[1569,521],[1612,7439],[1614,3406],[1652,110],[1772,3505],[2421,385],[2456,1273],[3014,137],[3761,1872],[3990,276],[5507,422],[6378,5383],[6564,4011],[7133,422],[7310,109],[8452,1475],[8553,348],[11818,21229],[15351,17054],[16610,17067],[20253,16481],[21281,19527],[57,2644],[119,1554],[236,175],[269,288],[288,105],[293,109],[293,2290],[302,13512],[312,1317],[321,2615],[335,307],[349,179],[650,2792],[693,761],[721,12541],[773,1696],[788,1115],[815,1418],[938,861],[948,136],[1054,788],[1170,752],[1214,6752],[1960,10399],[2107,296],[2239,3551],[2358,384],[2408,276],[2486,111],[2736,457],[2736,1412],[2840,766],[2965,523],[3582,3850],[3879,566],[3922,1775],[4120,450],[5827,773],[7785,1579],[8761,1191],[8833,894],[9957,5781],[10932,2283],[11966,3643],[13720,111],[14528,5540],[276,1306],[278,765],[305,1508],[321,274],[341,277],[350,6767],[353,307],[375,1281],[383,3705],[386,1208],[388,1151],[399,2676],[402,960],[430,1478],[463,13013],[472,335],[495,4760],[507,432],[527,151],[571,1449],[657,176],[743,1183],[791,13680],[893,134],[906,2093],[1047,545],[1059,5600],[1087,2980],[1146,167],[1163,3745],[1167,188],[1359,724],[1483,432],[1543,856],[1543,8829],[1739,14437],[1753,950],[1829,947],[1901,528],[1940,2576],[2128,1118],[2132,1781],[2467,2005],[2919,2699],[2933,4299],[2984,607],[2999,521],[3267,8130],[3314,347],[3422,575],[3469,1131],[3471,121],[3635,1118],[3799,3629],[4240,1378],[4245,3789],[5447,1156],[7230,5882],[7608,343],[9427,3393],[12811,390],[18221,781],[20541,3393],[21511,939],[72,84],[110,103],[273,1007],[278,3433],[294,344],[333,2040],[341,13184],[361,1294],[368,5224],[408,160],[514,936],[556,340],[573,1210],[625,353],[677,5374],[677,5831],[684,2274],[721,7413],[769,1103],[796,4384],[855,795],[882,2115],[938,1187],[961,381],[1047,8662],[1054,643],[1129,357],[1194,795],[1257,13034],[1292,887],[1434,6661],[1575,385],[1708,1418],[1919,382],[1928,103],[2002,407],[2007,1541],[2182,268],[2252,293],[2284,417],[2332,97],[2470,332],[2656,1835],[2690,132],[2725,1518],[2777,121],[3023,2700],[3241,11861],[3603,1038],[3656,6443],[3927,468],[4123,114],[4252,399],[5171,2868],[5452,752],[5452,5951],[5476,2484],[5614,7602],[5629,17320],[6516,270],[6944,4432],[7643,276],[7821,3108],[8220,347],[8715,2127],[9878,2091],[12329,304],[13026,104],[14384,3910],[17053,1142],[19076,7551],[19557,1461],[20680,2377],[107,843],[114,296],[117,2184],[237,15955],[264,149],[269,293],[288,2592],[292,160],[336,564],[421,4457],[423,4136],[434,98],[485,9214],[486,812],[527,147],[545,480],[571,11174],[586,2071],[602,11016],[613,2852],[622,383],[636,139],[650,16298],[684,6590],[686,129],[705,1589],[868,105],[879,7517],[1027,128],[1057,1778],[1163,9310],[1178,2004],[1274,1638],[1354,3111],[1429,289],[1529,1944],[1643,12410],[1686,270],[1764,1252],[1793,453],[1912,988],[2192,52],[2393,3644],[2438,815],[2535,5744],[2546,1007],[2585,1185],[2608,5484],[2612,1957],[3073,747],[3107,1955],[3364,480],[3413,1088],[3450,268],[3580,115],[3631,305],[3797,4114],[4417,417],[5253,384],[5749,4573],[5939,5939],[6467,13758],[6659,305],[6989,268],[8017,2416],[10119,1563],[10289,919],[10727,400],[11449,7195],[11805,1007],[12571,391],[13127,3205],[18320,850],[263,169],[296,4643],[298,185],[320,1724],[350,1572],[377,1741],[508,79],[625,18892],[626,97],[636,151],[645,170],[660,137],[711,2199],[842,2785],[857,304],[923,2678],[931,1475],[943,396],[960,10218],[980,16567],[984,1781],[1047,3359],[1128,276],[1141,156],[1146,144],[1298,1021],[1300,611],[1404,111],[1417,276],[1693,10027],[1875,14168],[1923,2835],[2059,2246],[2102,837],[2292,1390],[2365,7933],[2473,332],[2526,2015],[2650,1763],[2691,2047],[2736,2842],[2963,747],[2971,865],[3121,296],[3181,149],[3194,154],[3250,571],[3361,276],[3390,3518],[3499,2181],[3684,367],[4653,399],[5089,564],[5164,122],[5672,422],[5776,289],[6094,1306],[8246,628],[8358,1658],[9357,18666],[12791,1136],[70,66],[98,2008],[234,9716],[264,148],[280,15816],[291,3450],[296,1715],[331,1087],[336,959],[430,3562],[459,128],[484,1365],[508,82],[514,1046],[548,79],[554,171],[638,158],[683,4487],[910,2748],[924,3755],[980,21318],[1058,2842],[1112,276],[1127,1582],[1129,479],[1176,3394],[1220,304],[1285,1586],[1353,421],[1557,2508],[1584,132],[1668,1495],[1752,185],[1860,352],[1920,647],[1958,417],[1981,108],[1987,1809],[2084,105],[2544,1779],[2891,1421],[2919,2296],[3281,788],[3400,533],[3503,11242],[3780,1630],[3811,960],[4086,3446],[4972,575],[5140,9715],[5478,1021],[6122,76],[6894,13934],[7021,6591],[7138,2428],[8236,2837],[8274,13133],[9185,1686],[9521,2346],[9646,425],[17849,99],[49,1883],[84,73],[99,2317],[269,850],[270,305],[276,2023],[323,159],[351,348],[353,305],[386,7237],[477,1144],[548,451],[670,1785],[703,2465],[710,152],[717,173],[792,183],[813,3351],[842,2189],[900,376],[931,1254],[943,107],[1079,116],[1089,383],[1118,875],[1130,5052],[1171,3440],[1228,1703],[1298,865],[1299,457],[1429,270],[1434,6491],[1595,1124],[1614,2405],[1617,168],[1657,575],[1815,2205],[1872,3095],[1909,142],[2021,144],[2087,6716],[2211,1137],[2333,12423],[2383,306],[2454,5893],[2676,1193],[2774,912],[3418,1395],[4557,579],[4903,2624],[5043,6814],[5388,1767],[5620,648],[6178,272],[6554,2883],[6736,3873],[8046,804],[8674,404],[9944,1217],[11950,287],[16512,16208],[99,99],[152,145],[190,140],[269,3137],[292,185],[319,3584],[333,658],[333,854],[337,8626],[386,819],[421,615],[460,168],[495,3663],[514,118],[788,357],[855,5464],[886,2281],[891,111],[931,876],[1053,182],[1054,4718],[1130,407],[1186,827],[1189,2434],[1222,276],[1335,832],[1342,450],[1374,348],[1382,114],[1459,175],[1472,4018],[1483,348],[1504,188],[1518,3247],[1584,171],[1853,121],[2118,623],[2272,4712],[2333,116],[2631,927],[2690,129],[2776,1633],[2794,2406],[3020,8609],[3030,5098],[3080,177],[3220,114],[3428,5289],[3485,628],[3526,3542],[3530,9980],[3562,2697],[4013,4269],[4209,6323],[4600,175],[4763,2766],[5445,897],[5952,5952],[6598,375],[10474,2134],[12070,1456],[13705,12570],[14712,531],[65,103],[68,114],[105,5016],[108,3932],[115,97],[276,1702],[276,6067],[283,11292],[308,155],[334,100],[336,9659],[341,437],[376,724],[404,105],[431,121],[456,165],[508,2699],[509,121],[516,102],[583,9489],[587,819],[677,4973],[685,15172],[803,2586],[833,780],[982,2074],[1005,3442],[1096,52],[1106,827],[1172,167],[1183,553],[1377,490],[1404,1456],[1693,2083],[1802,3203],[1972,2648],[2290,400],[2365,815],[2467,9772],[2671,20861],[3136,422],[3366,357],[3437,4068],[3558,14233],[3664,11344],[4342,1231],[5200,1283],[6430,15156],[7047,97],[9234,14206],[10408,7187],[11886,1806],[13314,952],[13707,7685],[13831,344],[13956,390],[15037,15092],[22119,4862],[68,69],[99,268],[112,1508],[288,1539],[361,98],[374,148],[385,3005],[415,1949],[423,121],[487,2157],[516,13480],[522,759],[557,173],[584,1700],[589,599],[598,100],[701,2946],[784,307],[812,276],[814,1389],[855,2711],[921,268],[1056,185],[1089,343],[1281,1144],[1300,1760],[1494,3752],[1611,184],[1681,1469],[1815,640],[1836,3565],[1897,696],[1901,6081],[1912,115],[2092,2040],[2256,128],[2286,103],[2345,1148],[2653,2870],[2841,4941],[2852,1589],[2995,2324],[3200,116],[3320,21943],[3458,1230],[3597,14738],[4012,2901],[4385,10695],[4443,272],[4591,1221],[4702,16957],[4979,9913],[5046,268],[5126,528],[5617,553],[6392,1654],[6594,2166],[6846,890],[9541,12590],[9798,97],[11717,6439],[119,3963],[267,6672],[294,19741],[301,181],[326,3697],[333,1991],[351,367],[367,7798],[374,129],[385,1514],[388,1456],[496,2815],[503,339],[509,101],[639,1454],[654,168],[753,15320],[766,1405],[801,885],[826,3343],[869,110],[893,174],[908,965],[1123,304],[1126,190],[1278,141],[1316,188],[1396,169],[1426,417],[1468,116],[1536,783],[1552,289],[1701,2408],[1737,276],[2087,562],[2141,531],[2230,2613],[2476,875],[2516,108],[2688,2564],[2890,4842],[2919,6161],[2971,2043],[2988,170],[3186,6183],[3641,1742],[3703,920],[3799,272],[3915,3042],[4055,19592],[4409,505],[4951,1498],[6343,6343],[6507,4219],[6692,2031],[6838,118],[7083,326],[7386,7805],[8367,890],[8841,1397],[10479,321],[13063,268],[14567,111],[15096,10505],[21522,1869],[67,2880],[82,79],[99,317],[99,1508],[273,3555],[276,4017],[283,344],[311,442],[328,136],[366,1185],[375,1193],[480,4358],[501,136],[514,1853],[532,312],[679,1559],[696,1610],[716,530],[727,5594],[755,179],[963,4678],[1020,128],[1089,585],[1104,2785],[1105,847],[1127,395],[1280,1724],[1284,129],[1477,144],[1606,99],[1693,1697],[1815,20825],[2011,310],[2094,3166],[2132,6065],[2330,3697],[2345,1658],[2519,7046],[2612,477],[2839,9559],[2861,307],[3010,5817],[3342,109],[3468,155],[3501,276],[3626,421],[3626,1933],[3785,587],[4052,895],[4054,1062],[5504,115],[5590,305],[6259,1031],[6835,111],[6883,98],[7758,417],[8872,417],[8888,8115],[10808,3853],[13853,180],[14167,21245],[300,335],[306,1423],[319,1119],[333,2265],[336,1646],[336,2258],[339,289],[345,5053],[367,7680],[394,150],[425,105],[463,1544],[551,1539],[677,1454],[684,296],[755,152],[769,920],[798,129],[826,2514],[903,107],[909,1574],[924,4765],[935,175],[1089,2863],[1095,950],[1246,5769],[1256,385],[1302,1114],[1311,150],[1388,1737],[1405,1521],[1429,10033],[1466,997],[1733,2340],[1851,2248],[1954,480],[1980,1692],[2771,611],[2997,5291],[3152,522],[3221,120],[3522,2041],[3526,972],[6931,3878],[7150,73],[8047,321],[9744,268],[9973,3525],[16022,2918],[18498,4546],[22140,1639],[80,84],[80,626],[109,296],[265,140],[274,2721],[276,1829],[283,6513],[300,1785],[325,335],[325,2184],[333,2691],[333,3286],[352,1982],[361,121],[402,7895],[452,3348],[452,12814],[463,1519],[701,376],[721,8762],[726,816],[757,104],[822,1340],[926,2823],[932,133],[984,1310],[1021,643],[1058,1706],[1133,8397],[1159,892],[1316,136],[1494,339],[1546,3034],[1801,10623],[1850,307],[1933,580],[2042,1306],[2207,1666],[2300,142],[2458,272],[2598,12620],[3044,111],[3202,1433],[3421,348],[3908,2667],[4249,2257],[4700,189],[4753,908],[5919,6128],[6645,2705],[8359,128],[8850,4444],[9132,634],[9250,6153],[9320,9320],[9444,1449],[10913,304],[11326,289],[13110,2026],[13131,1640],[13762,312],[14725,977],[17388,796],[103,6728],[268,107],[272,296],[276,2010],[291,317],[351,9167],[356,2428],[375,4233],[386,3179],[458,145],[509,10898],[553,2809],[611,2325],[614,287],[621,115],[654,173],[822,1056],[975,1045],[984,3105],[1098,129],[1100,1440],[1119,7896],[1395,417],[1595,1165],[1725,522],[1747,10368],[1983,276],[1987,1797],[2327,3414],[2450,1574],[2453,5274],[2577,8627],[2979,417],[3067,352],[3082,276],[3173,1000],[3288,2513],[3320,832],[3516,14362],[3602,507],[3806,1867],[3883,1178],[4119,463],[4293,115],[4378,811],[4400,611],[5144,1723],[5447,8736],[5471,2339],[5671,101],[5842,116],[6430,6750],[6999,144],[7684,115],[7831,1343],[7932,1499],[8037,330],[9337,340],[9607,335],[12603,397],[14324,5671],[18426,1114],[97,97],[99,1786],[119,1809],[283,4371],[299,688],[306,8425],[312,442],[381,311],[472,14226],[495,1675],[748,3826],[759,430],[834,2427],[947,1792],[1092,698],[1157,2703],[1183,8073],[1266,2679],[1322,1063],[1361,3923],[1387,293],[1518,815],[1568,1452],[1832,1857],[2091,288],[2131,407],[2190,1948],[2352,15036],[2413,276],[2528,1302],[2565,1850],[2582,780],[2663,2889],[2881,3488],[3152,2852],[3234,422],[3249,2900],[3678,276],[4199,4853],[4228,528],[4393,1062],[4671,6267],[5591,2500],[5653,115],[6132,8485],[6461,2461],[6483,10031],[6593,3618],[6726,1187],[6873,9648],[7266,925],[7917,1947],[8111,804],[8497,152],[9564,1142],[10123,335],[10441,4678],[10500,1129],[11374,2991],[12953,357],[13521,307],[105,3768],[276,2171],[299,1918],[300,2768],[422,330],[446,138],[456,182],[625,437],[625,6439],[663,146],[762,558],[784,692],[875,376],[1048,3748],[1143,1811],[1154,783],[1190,149],[1214,7978],[1537,1953],[1551,7439],[1687,4411],[1799,3843],[1901,378],[1939,2545],[2045,1115],[2205,3917],[2383,1102],[2486,628],[2550,1778],[2669,99],[2807,1341],[2942,3341],[3063,475],[3140,661],[3154,875],[3170,4775],[3173,3732],[3263,276],[3466,343],[3953,451],[4163,4635],[4246,809],[4651,383],[5398,136],[5890,382],[5929,858],[6078,997],[8340,10597],[10426,13312],[12660,1678],[17139,5207],[83,396],[110,634],[117,296],[276,7237],[300,1440],[347,496],[375,6395],[384,13847],[386,4150],[388,1853],[401,3592],[487,908],[522,877],[531,615],[532,467],[536,168],[561,173],[721,2149],[741,2210],[747,101],[773,1864],[791,370],[840,120],[855,1238],[877,522],[1108,1144],[1269,144],[1287,160],[1327,1977],[1375,142],[1512,585],[1584,178],[1625,1105],[1794,137],[1877,673],[1912,6233],[2116,608],[2307,9905],[2316,2676],[2590,2590],[2921,12389],[2979,450],[3295,3732],[3535,381],[3577,759],[3632,3091],[3647,1112],[3846,4885],[4086,2226],[4154,1107],[4222,451],[4392,21179],[4723,16751],[5086,272],[5503,2931],[6544,20964],[6570,1341],[6956,18792],[10492,6415],[12066,960],[13521,305],[20051,5386],[69,781],[78,79],[86,73],[105,2803],[294,11576],[333,562],[357,978],[388,17860],[434,2177],[438,158],[450,475],[452,3002],[514,4136],[545,1062],[551,83],[555,187],[589,385],[596,2127],[622,494],[753,11084],[760,475],[779,611],[785,2525],[830,1185],[846,545],[874,382],[930,11756],[966,7895],[1141,138],[1177,3386],[1296,5958],[1353,3638],[1383,885],[1504,149],[1895,8313],[1931,626],[2229,2265],[2333,330],[2473,12090],[2669,2816],[2920,139],[3035,3369],[3228,538],[3432,3833],[3458,2614],[3626,992],[3812,7258],[4180,2679],[4551,332],[4556,1946],[5060,19974],[5529,709],[6429,747],[7734,523],[8339,274],[8468,1092],[8730,4397],[8788,1320],[9114,1110],[9699,353],[11567,1627],[12078,1307],[12622,395],[12782,344],[12990,2570],[14803,592],[16355,13272],[19085,673],[20463,17945],[66,79],[77,111],[104,8655],[121,268],[274,104],[276,658],[276,6424],[320,4968],[328,154],[333,3816],[337,7446],[416,167],[486,946],[503,1849],[514,772],[542,7220],[579,1852],[586,2561],[715,172],[739,159],[790,716],[878,136],[887,3825],[924,4076],[1001,134],[1052,3296],[1057,724],[1065,2062],[1197,538],[1208,3644],[1299,5766],[1333,1574],[1397,793],[1484,1250],[1528,766],[1591,573],[1847,1418],[1994,553],[2119,2916],[2192,53],[2286,12163],[2344,1570],[2412,1582],[2465,13129],[2776,312],[2793,12929],[2969,1341],[3375,2562],[3648,287],[3819,15820],[3922,776],[4243,100],[4258,1401],[4260,1731],[4834,2000],[4995,3808],[5681,1115],[6236,804],[6556,1457],[6582,138],[6943,1013],[7432,321],[7481,117],[8265,146],[8575,8575],[8696,10045],[11786,289],[12665,1889],[12782,1835],[12820,288],[14647,2416],[17309,8397],[19086,1654],[19140,1369],[32,5844],[68,296],[112,2308],[122,12823],[285,857],[319,507],[341,1554],[341,8594],[356,2446],[397,5778],[421,4394],[428,1317],[515,120],[637,117],[646,1239],[672,4317],[745,12650],[746,170],[776,562],[954,131],[1017,855],[1037,691],[1077,3801],[1204,167],[1214,1691],[1392,20767],[1403,404],[1525,1859],[1525,4863],[1625,950],[1671,3540],[1686,97],[1719,339],[1774,3458],[1808,832],[1838,8433],[1965,6026],[2084,4648],[2286,339],[2364,49],[2381,115],[2409,1683],[2488,1758],[2498,2112],[3221,912],[3510,2815],[3526,1671],[3757,84],[4388,7457],[4421,437],[4773,14205],[4958,11745],[4988,2954],[5428,347],[5574,381],[5734,1665],[5756,475],[5775,2226],[6132,1205],[6453,1162],[7524,592],[7592,288],[7995,1142],[8131,1852],[8308,357],[8668,6927],[9071,1421],[9668,376],[13699,897],[14481,5607],[17080,4870],[17223,13719],[17899,9963],[83,9707],[87,8672],[100,370],[102,5169],[102,6341],[103,850],[274,6455],[277,804],[304,1100],[350,1709],[378,901],[388,17624],[390,1947],[421,5178],[422,370],[450,1298],[466,145],[499,10110],[505,1578],[516,404],[522,1090],[548,330],[548,922],[643,575],[661,3290],[719,1995],[758,183],[784,2963],[834,553],[931,3518],[996,1603],[1076,129],[1178,766],[1182,376],[1206,842],[1353,2060],[1371,10864],[1383,102],[1488,2714],[1517,1021],[1527,2216],[1537,1610],[1601,724],[1676,312],[1679,2700],[1706,1104],[1937,407],[2316,10646],[2486,105],[2528,2162],[2685,422],[2794,2956],[3081,1627],[3120,1364],[3269,115],[3311,886],[3603,1931],[3706,122],[3819,465],[4060,1557],[4201,1949],[4261,16293],[4431,274],[4790,1532],[5181,2462],[5352,3124],[5823,2727],[8900,1370],[9886,383],[9934,1173],[10615,304],[11712,479],[11983,983],[13489,109],[16027,1235],[19231,12009],[20487,747],[22524,167],[32,2105],[71,69],[85,80],[109,324],[269,960],[287,2928],[291,1087],[323,139],[341,10392],[343,6640],[357,2276],[368,621],[495,1887],[545,3192],[554,188],[567,144],[573,1235],[583,1741],[599,2183],[600,160],[622,332],[704,12210],[745,119],[846,1602],[982,384],[1005,556],[1049,2328],[1068,347],[1085,1579],[1117,806],[1211,272],[1232,1521],[1268,1817],[1376,1493],[1394,773],[1466,773],[1482,2846],[1487,1085],[1595,773],[1701,1256],[1972,5510],[2041,12641],[2047,673],[2242,2076],[2482,312],[2939,17768],[3044,4185],[3047,18497],[3704,17421],[4034,4853],[4713,1665],[4959,345],[5761,19555],[6309,1042],[6468,2306],[6667,4384],[6999,132],[7774,1187],[7898,822],[7939,417],[10461,704],[11386,3460],[12748,8447],[16484,14360],[17935,5498],[77,3694],[106,1205],[291,2296],[293,901],[319,354],[331,404],[404,4151],[465,528],[485,856],[495,1046],[532,396],[553,1055],[579,120],[650,2540],[691,2087],[692,538],[752,13125],[753,11279],[790,615],[865,5988],[1054,1337],[1445,172],[1488,2234],[1599,673],[1754,181],[1830,1021],[1929,531],[2041,2191],[2200,144],[2330,101],[2365,6072],[2441,769],[2554,2360],[2812,541],[3035,743],[3707,892],[3783,442],[3789,691],[4095,157],[4502,921],[4586,537],[5041,404],[5206,14433],[5582,3066],[5786,545],[5869,1257],[6174,1822],[6851,1999],[7110,417],[7239,129],[7481,4277],[7926,109],[8276,1563],[8279,10094],[8435,9941],[8533,773],[8686,422],[9160,1107],[9236,307],[9978,832],[10719,3360],[11473,378],[11909,381],[14412,6732],[16768,1378],[66,66],[98,4595],[115,396],[122,11755],[194,178],[276,3605],[300,1768],[333,1386],[360,348],[377,7426],[401,2296],[474,155],[518,354],[520,183],[545,4230],[596,1087],[646,1659],[648,1559],[672,2622],[700,1219],[711,4020],[838,475],[868,22477],[873,277],[895,6716],[937,185],[1096,57],[1123,353],[1129,417],[1147,156],[1189,2761],[1294,270],[1308,3874],[1314,579],[1468,4797],[1606,324],[1840,304],[1984,988],[2104,2062],[2123,1741],[2205,1822],[2277,2085],[2409,2022],[2415,1259],[2903,5312],[3002,1370],[3257,166],[4115,1338],[4169,3100],[4527,902],[4868,97],[5261,6503],[5403,307],[5631,2852],[5830,17882],[5948,184],[6002,890],[7207,1207],[7386,2928],[7406,1478],[8256,747],[8603,5923],[8807,111],[12920,304],[18820,1804],[19040,960],[68,97],[71,270],[72,75],[79,112],[98,747],[102,7470],[103,538],[274,977],[276,2575],[314,131],[336,1508],[376,2078],[384,348],[385,658],[428,304],[432,2955],[495,272],[505,1245],[544,4233],[593,2353],[673,463],[702,8342],[725,1037],[751,135],[758,149],[785,827],[1008,145],[1049,22307],[1060,417],[1130,6211],[1349,2305],[1468,300],[1513,311],[1528,490],[1708,1585],[1912,311],[2286,1283],[2422,1592],[2429,5711],[2516,3720],[2533,2495],[2565,843],[2734,1370],[3181,159],[3447,335],[3714,129],[3797,10710],[3893,161],[3930,22360],[4149,682],[4442,6680],[4671,10366],[4805,20361],[5281,141],[5703,3731],[6220,1298],[6378,4803],[6591,1532],[7128,324],[8481,1633],[8917,115],[9643,147],[10885,382],[14289,3640],[17948,20088],[19371,1491],[20723,8106],[87,104],[287,304],[375,20399],[423,68],[495,13994],[528,5383],[605,158],[625,2259],[650,376],[704,23138],[733,399],[738,317],[795,733],[872,806],[950,643],[999,100],[1076,188],[1117,321],[1124,522],[1163,523],[1201,6121],[1206,1104],[1228,6111],[1252,562],[1462,50],[1544,811],[1565,136],[1574,1778],[1776,18793],[1802,4772],[2128,2916],[2173,277],[2195,3477],[2251,9149],[2262,1013],[3186,3042],[3498,7787],[3612,3104],[3640,115],[3809,5623],[4064,804],[4086,1155],[4137,1045],[4180,103],[4335,1118],[4884,3336],[4963,804],[5075,1317],[5227,810],[6453,5300],[6519,105],[7092,450],[7172,1959],[8407,988],[13466,4294],[15396,1540],[16123,2204],[21962,17015],[97,903],[119,104],[281,856],[297,151],[350,19168],[386,1884],[388,3137],[412,175],[423,2753],[495,652],[544,1171],[553,1137],[596,4203],[610,137],[705,933],[774,2863],[785,2662],[844,15285],[847,3032],[1145,180],[1342,611],[1431,956],[1581,148],[1583,177],[1726,276],[1781,696],[1962,23034],[1990,4046],[2133,119],[2232,4085],[2496,9582],[3994,1520],[4792,1646],[4896,399],[5008,9849],[5638,1645],[5698,357],[7184,806],[7231,276],[7414,110],[8421,3571],[11724,386],[11840,1094],[14245,164],[16789,804],[65,107],[86,111],[276,486],[294,10281],[299,7067],[329,172],[331,2869],[336,425],[376,1386],[388,2332],[406,149],[415,107],[430,571],[434,4165],[472,97],[472,105],[485,12211],[581,157],[584,579],[651,186],[669,2349],[697,152],[792,142],[797,7454],[803,5893],[859,730],[920,11121],[1037,3129],[1046,272],[1314,268],[1454,1358],[1459,130],[1544,648],[1610,3957],[1707,2946],[1773,158],[1781,450],[1800,2012],[1848,385],[2173,1645],[2239,102],[2350,187],[2549,7538],[2644,48],[2794,4894],[2832,1562],[3628,1041],[3655,844],[4008,830],[4333,386],[4396,176],[4427,11593],[4452,1155],[4720,2062],[4937,347],[5071,8329],[5862,97],[6331,407],[6519,422],[7379,1402],[7520,1673],[8257,4795],[8452,608],[8531,272],[8939,578],[9531,10012],[10828,479],[11516,6274],[12127,530],[12414,4453],[12922,98],[13722,4072],[15787,628],[18519,12343],[112,304],[119,287],[119,437],[302,4453],[385,5626],[397,1709],[452,1721],[467,114],[486,2625],[532,2308],[579,307],[590,172],[591,170],[631,4381],[729,425],[858,743],[891,287],[917,4812],[946,3025],[980,688],[1070,4456],[1189,17785],[1240,4847],[1274,3892],[1322,2292],[1374,332],[1374,6280],[1387,579],[1483,3755],[1543,11637],[1766,276],[1792,1075],[1927,871],[2044,2028],[2178,479],[2243,417],[2251,1884],[2265,1333],[2288,1192],[2323,305],[2483,142],[2676,1971],[2683,1107],[3021,523],[3583,276],[3762,160],[4131,2085],[4163,2232],[4225,1877],[4388,1208],[4519,188],[5251,6576],[5309,1369],[5313,1586],[5595,2332],[6205,626],[7593,390],[7996,608],[8113,97],[9758,1134],[10241,17273],[11118,306],[11482,422],[11923,1879],[14273,2015],[14484,304],[87,1026],[100,397],[105,106],[116,856],[288,1142],[430,2688],[437,289],[447,108],[472,82],[494,306],[499,2332],[508,289],[510,163],[588,157],[596,1072],[661,1863],[667,276],[677,4475],[818,9411],[961,6968],[1114,450],[1151,6522],[1219,5220],[1220,21533],[1468,4990],[1517,407],[1702,2959],[1781,566],[1815,813],[1847,1198],[1969,1107],[2200,156],[2223,100],[2317,783],[2350,161],[2361,191],[2384,272],[2549,465],[2606,829],[2737,463],[2814,148],[2878,2373],[2887,2634],[3174,3566],[3219,15415],[3221,1272],[3353,276],[3371,1356],[3655,4544],[4039,293],[4508,4459],[4749,305],[5406,3883],[5450,5711],[5628,1863],[5638,1665],[6711,306],[6715,1850],[7797,450],[9397,780],[9734,116],[10148,559],[10953,3183],[11024,144],[15160,307],[18117,296],[21114,4150],[69,70],[102,5225],[276,2896],[297,143],[304,105],[305,533],[312,851],[336,521],[336,11843],[340,1102],[342,151],[344,5677],[348,337],[368,115],[383,1371],[478,186],[586,885],[615,2909],[784,5997],[796,1259],[801,5353],[1019,1093],[1088,270],[1126,168],[1236,837],[1242,1983],[1246,918],[1305,599],[1481,545],[1729,793],[1776,20320],[1784,1809],[1799,4363],[1890,3231],[1980,7969],[2045,480],[2179,523],[2264,1297],[2352,14777],[2412,1786],[2496,1746],[2620,276],[3102,1252],[3150,463],[3311,640],[3756,417],[4037,1200],[4054,2902],[4160,347],[4243,296],[4369,747],[4602,1307],[5145,628],[5412,11562],[5546,422],[5561,169],[5830,4794],[6798,2494],[8871,4294],[8970,1959],[9893,417],[10071,2883],[10479,116],[11365,1006],[13061,1527],[16271,7086],[21765,3007],[85,73],[100,353],[100,422],[112,326],[114,1263],[121,1785],[350,3111],[354,2245],[357,4553],[421,705],[423,111],[434,4021],[445,152],[509,2429],[544,496],[548,857],[617,177],[648,1567],[706,108],[759,925],[773,892],[791,15239],[818,2476],[846,1062],[924,8425],[953,154],[966,21068],[987,97],[1009,152],[1066,1323],[1107,1746],[1159,386],[1197,1297],[1201,2825],[1229,168],[1267,140],[1361,6046],[1471,496],[1503,465],[1680,1193],[1711,111],[1737,2712],[1792,1108],[1906,566],[1925,348],[1957,795],[2345,628],[2403,450],[2469,1768],[2558,288],[2585,2942],[2607,1393],[2626,1519],[2704,2055],[2801,919],[3095,4183],[3340,4711],[3822,2969],[3858,276],[4107,21940],[4129,766],[4163,1561],[4407,1779],[4610,339],[4687,2731],[4768,1785],[4995,661],[5352,6026],[5545,422],[5681,8197],[6231,12215],[6532,2055],[7580,289],[7889,1574],[8853,1571],[10452,1273],[12414,5015],[15437,2816],[15829,340],[16675,421],[281,570],[295,145],[333,2042],[350,101],[375,675],[375,1171],[375,1242],[383,8608],[395,105],[396,293],[401,451],[401,1205],[425,415],[453,2202],[545,450],[547,165],[611,611],[625,12819],[645,158],[672,451],[702,2768],[719,5496],[749,185],[753,274],[766,2359],[775,165],[790,463],[909,276],[1081,145],[1186,2927],[1290,545],[1351,378],[1354,8762],[1377,822],[1572,272],[1908,7116],[1970,386],[2023,2087],[2087,909],[2281,640],[2351,647],[2493,2812],[2796,7473],[2948,332],[3129,679],[3136,370],[3304,306],[4018,2017],[4961,6148],[5041,14951],[5379,10610],[6125,457],[6400,344],[6866,442],[7793,2429],[8847,1394],[9637,7907],[11410,2806],[12298,396],[14624,104],[32,3172],[70,88],[72,65],[76,3757],[84,270],[98,381],[112,109],[269,1562],[269,1718],[276,3060],[281,2323],[285,8647],[311,422],[388,912],[485,652],[486,741],[499,12245],[545,1602],[548,5482],[593,2164],[675,5055],[677,8261],[795,1636],[829,104],[869,404],[1017,1245],[1143,1590],[1199,16601],[1285,11619],[1286,2371],[1332,920],[1555,580],[1631,133],[1904,421],[1923,693],[2016,1164],[2121,1832],[2153,871],[2307,5677],[2344,332],[2490,5430],[2728,599],[2796,9767],[2987,1226],[3000,1927],[3090,766],[3098,3231],[3197,1922],[3234,1786],[3391,2461],[3902,3262],[3902,17513],[4364,4277],[4389,2304],[4530,4973],[4878,5825],[5441,538],[6133,7104],[6177,7257],[6224,2507],[6483,608],[6884,5286],[8253,1333],[8736,385],[8748,1221],[8954,1999],[9022,421],[11527,417],[12385,22753],[16109,2066],[17348,1520],[19738,390],[72,82],[98,521],[299,8655],[305,1421],[319,934],[347,3547],[368,1185],[401,521],[401,3448],[423,71],[499,3196],[507,367],[548,723],[586,558],[594,155],[615,693],[681,19464],[840,15866],[963,2444],[1047,661],[1093,1859],[1103,276],[1299,832],[1422,370],[1444,395],[1528,725],[2087,7556],[2265,6868],[2553,352],[2558,620],[3000,1426],[3023,3437],[3290,810],[3806,1372],[3845,578],[3908,116],[4043,270],[4647,6503],[4817,4375],[4837,14599],[5028,102],[6528,2983],[6593,20156],[8282,927],[8470,1706],[8792,8516],[9611,1691],[10017,1487],[10742,7364],[11478,390],[11677,894],[20513,1187],[67,66],[79,98],[107,339],[117,936],[276,595],[276,6260],[276,7243],[299,824],[312,5414],[315,188],[333,3552],[373,145],[428,305],[436,170],[486,1227],[501,177],[522,2022],[532,1142],[711,3528],[976,1097],[1018,184],[1035,646],[1068,2668],[1183,8812],[1201,642],[1232,9119],[1257,276],[1261,185],[1267,169],[1360,386],[1459,185],[1482,10177],[1547,1589],[1584,133],[1620,505],[1845,1847],[1901,2928],[1909,153],[1979,9875],[1988,3053],[2076,1682],[2157,2522],[2315,5892],[2409,496],[2480,1400],[2577,1302],[2616,920],[2774,578],[2838,3833],[2862,3492],[2900,2445],[2916,2445],[3086,5006],[3128,339],[3298,12692],[3365,274],[3373,883],[3603,272],[5570,5270],[5928,2763],[6033,8927],[7286,2780],[7579,1730],[7615,1924],[8223,272],[8287,10679],[8703,4271],[10471,3776],[11231,14688],[12523,642],[15054,4775],[16545,12666],[17651,111],[21649,164],[65,572],[122,2763],[235,140],[261,183],[263,162],[268,102],[281,626],[299,816],[319,4461],[351,507],[378,2500],[490,892],[526,12888],[596,296],[596,888],[616,1495],[621,888],[623,613],[679,333],[782,149],[803,523],[864,144],[867,2507],[925,276],[1193,4310],[1340,812],[1431,941],[1529,2196],[1613,2906],[1693,6273],[1736,1735],[1829,6477],[1850,1007],[2084,628],[2235,110],[2342,2902],[2668,3285],[2921,20847],[3110,117],[3168,865],[3270,1209],[3271,122],[3418,2680],[3618,796],[3668,161],[4014,450],[4039,2112],[4235,65],[4720,3132],[4970,382],[5927,3346],[6691,422],[7037,3346],[7101,5079],[8778,709],[11115,347],[11970,1959],[11972,922],[12106,8002],[13367,1839],[14110,339],[15496,824],[15623,10636],[18319,2017],[69,3853],[80,97],[108,1011],[114,330],[122,706],[237,136],[276,463],[276,3009],[283,923],[288,850],[291,5719],[318,169],[357,420],[381,3091],[424,147],[424,165],[432,18544],[508,268],[510,129],[514,4877],[557,152],[625,4800],[752,566],[900,2715],[934,4812],[961,1586],[1019,276],[1056,165],[1075,682],[1075,2326],[1155,307],[1199,5781],[1206,22116],[1247,10101],[1354,2177],[1380,5910],[1635,1075],[1638,376],[1703,1501],[1710,181],[1753,19675],[1863,382],[1944,370],[1988,339],[2044,4132],[2403,2246],[2608,2603],[2764,599],[2877,6782],[3095,1872],[3344,97],[3373,1372],[4663,4036],[4955,2148],[5000,938],[6363,3782],[6585,332],[6636,370],[6956,15624],[6993,1142],[7219,9314],[7692,270],[9878,10012],[14305,1947],[15182,164],[19123,103],[19227,185],[20524,15599],[21268,7790],[68,277],[68,14012],[102,3215],[110,105],[111,84],[117,1294],[276,1894],[283,2340],[294,117],[300,111],[347,623],[366,16663],[375,3761],[395,3754],[424,166],[434,999],[513,163],[579,102],[616,2196],[617,187],[649,170],[701,992],[704,6739],[717,169],[727,20684],[753,6684],[784,277],[790,2126],[871,950],[1037,17708],[1045,1048],[1047,3404],[1059,2083],[1171,553],[1195,163],[1238,725],[1270,49],[1318,1294],[1343,661],[1382,293],[1387,3870],[1394,421],[1569,2631],[1572,109],[1689,6392],[1800,3347],[1915,185],[1929,2391],[1997,155],[2021,176],[2042,593],[2076,6877],[2098,496],[2103,3593],[2422,1879],[2447,2325],[2886,415],[2975,1094],[3048,6744],[3226,307],[3238,487],[3656,3068],[3917,813],[4067,1297],[4270,2318],[4557,804],[5558,1645],[5570,770],[5806,340],[6378,7269],[6460,747],[6536,1043],[6570,1974],[6775,417],[7478,952],[8599,276],[9585,348],[12034,876],[12278,1658],[14245,176],[21984,370],[83,97],[103,324],[105,1160],[109,103],[112,765],[194,167],[270,397],[323,188],[333,2164],[336,8007],[337,644],[345,533],[357,3289],[382,545],[419,120],[423,118],[451,304],[510,167],[514,77],[514,325],[516,4691],[520,140],[603,150],[607,698],[634,442],[652,321],[655,3573],[689,129],[713,3087],[801,13313],[853,172],[888,270],[965,730],[970,747],[1020,174],[1046,370],[1119,7806],[1145,158],[1165,1030],[1206,9987],[1211,97],[1302,2045],[1302,2163],[1322,819],[1349,2139],[1375,158],[1580,505],[1609,1165],[1759,276],[1829,741],[1912,1123],[2400,1298],[2453,7574],[2468,382],[2805,305],[3398,161],[3532,810],[3877,3766],[3952,6973],[4084,1128],[4201,988],[4535,376],[4557,1142],[4776,2434],[4846,289],[5218,321],[5335,399],[6197,559],[6442,18845],[6497,9660],[6542,21041],[7301,274],[8188,370],[8786,10784],[9177,422],[9527,17905],[9830,415],[9989,1785],[10204,2813],[10251,99],[11397,370],[12079,101],[12598,559],[13714,268],[14583,6839],[15033,2443],[15846,5757],[15931,105],[15967,310],[17915,108],[21647,24004],[66,451],[67,6057],[82,111],[112,1185],[207,137],[273,1294],[285,851],[285,23233],[291,1209],[299,592],[333,3510],[337,384],[421,3521],[425,22023],[436,135],[485,6614],[498,128],[548,8299],[571,1651],[573,4981],[588,153],[625,65],[625,14382],[738,4732],[800,1433],[897,1505],[1030,8536],[1031,3822],[1068,2158],[1195,162],[1201,1897],[1314,16904],[1377,8622],[1488,716],[1713,189],[1744,1493],[1766,593],[1811,1452],[1839,376],[1920,52],[2010,276],[2028,1907],[2499,2169],[2892,598],[3126,289],[3130,17654],[3186,991],[3429,795],[3655,12689],[4225,2646],[4315,5357],[4487,1721],[4664,3650],[5184,593],[5259,2298],[5963,1244],[6646,2494],[7070,912],[8430,4634],[8730,5097],[10026,22562],[10095,376],[11086,2377],[11155,779],[12523,479],[12685,558],[14934,780],[15946,5941],[17372,3643],[22801,2315],[102,6439],[283,952],[302,103],[333,1242],[336,6116],[368,1853],[373,156],[413,148],[495,307],[514,76],[582,933],[596,2633],[610,168],[626,307],[665,166],[675,2155],[684,3870],[719,1796],[753,723],[846,16635],[863,1630],[864,5613],[869,465],[869,15112],[877,1992],[976,276],[984,1231],[995,122],[1013,2445],[1066,2183],[1140,399],[1189,643],[1205,272],[1214,17931],[1236,1791],[1355,407],[1396,177],[1407,5292],[1613,450],[1632,171],[1641,3087],[1657,450],[1701,1893],[1750,523],[1783,3782],[1808,3492],[1845,276],[1905,817],[1919,545],[2089,1209],[2195,14958],[2235,108],[2287,647],[2486,97],[2547,276],[2707,276],[2774,11361],[3033,1965],[3208,9319],[3363,97],[4103,1373],[4281,1122],[4509,1016],[4720,2445],[5695,17767],[5728,6363],[6468,3328],[6554,2085],[7212,3472],[7536,384],[8294,10523],[9569,1273],[10407,168],[10717,357],[11011,7479],[11486,3330],[11982,370],[12226,274],[12339,1151],[12787,11723],[13701,1191],[14365,1343],[15041,463],[69,100],[78,307],[98,2209],[101,1947],[122,353],[263,165],[294,7356],[302,7828],[308,181],[324,4803],[360,19966],[433,158],[459,134],[467,960],[472,1837],[485,13978],[522,3678],[598,1985],[639,585],[650,7564],[706,2085],[773,833],[789,2004],[835,268],[868,10898],[869,16784],[896,381],[929,1835],[936,101],[1019,1037],[1057,276],[1057,799],[1264,822],[1309,1487],[1327,339],[1426,1000],[1430,1660],[1450,480],[1580,385],[1614,3305],[1823,357],[1830,566],[1840,3274],[1856,272],[1890,2028],[1929,1585],[1948,2219],[1950,23935],[1979,5892],[2029,8469],[2056,1965],[2119,1328],[2384,6965],[2403,15026],[2510,276],[2861,824],[2938,10282],[3076,407],[3137,304],[3171,1226],[3381,989],[3400,901],[3804,9104],[4231,1221],[4836,1579],[5866,804],[5963,12269],[6028,9627],[6231,872],[6344,559],[6447,183],[6772,307],[6861,21474],[7383,761],[7449,3876],[8638,19508],[8925,304],[9163,2664],[9181,1499],[9199,189],[9764,107],[9862,4136],[10441,462],[11592,2534],[12633,19091],[12923,593],[16313,121],[66,277],[103,1142],[108,288],[114,1474],[115,107],[117,1254],[281,1389],[336,692],[336,1796],[356,270],[377,14017],[388,952],[401,288],[402,324],[415,442],[452,5699],[453,1102],[472,361],[499,462],[508,1797],[514,82],[571,3276],[573,340],[609,156],[626,270],[669,307],[700,858],[752,545],[855,3193],[873,98],[912,307],[1051,152],[1095,417],[1174,191],[1192,496],[1237,171],[1357,272],[1359,10112],[1432,6045],[1484,3652],[1778,2130],[1790,494],[1870,559],[1932,844],[2029,3876],[2069,141],[2082,2117],[2146,2841],[2207,22363],[2277,1779],[2560,3102],[2921,1760],[3123,8554],[3177,83],[3295,450],[3349,385],[3457,1085],[3457,4237],[3499,1254],[3636,3247],[3929,1244],[4259,14930],[4292,4464],[4988,6740],[5356,276],[5448,1422],[5884,5884],[6598,2318],[6615,683],[6649,1041],[6821,769],[8093,5228],[8374,2270],[8511,370],[8636,391],[9872,105],[10584,2185],[10806,101],[12050,305],[12436,1985],[20510,162],[83,1271],[98,1142],[112,894],[112,6145],[114,437],[269,1297],[276,599],[277,533],[278,1606],[283,5677],[287,3676],[289,102],[293,117],[294,2444],[312,1607],[319,4539],[320,14633],[333,6567],[342,174],[345,305],[357,1910],[401,1562],[402,3585],[434,8053],[472,1272],[472,1718],[508,584],[522,562],[584,2269],[640,20919],[646,1431],[655,655],[665,138],[810,575],[810,796],[863,7586],[886,842],[893,165],[961,3960],[973,155],[1073,139],[1134,3488],[1305,276],[1392,100],[1403,307],[1423,2877],[1667,417],[1677,829],[1827,378],[1875,8090],[1882,772],[1939,1298],[2011,13378],[2398,2575],[2440,1991],[2518,109],[2532,643],[2623,376],[2640,188],[2680,1244],[2826,2105],[2891,97],[2987,875],[3061,3663],[3152,1692],[3646,6527],[3805,3393],[4509,1551],[4656,20027],[5427,11846],[5616,288],[5866,1627],[5927,1155],[6649,1959],[6701,113],[7674,988],[8030,592],[8961,1699],[9238,783],[11507,3438],[12237,6184],[15082,272],[15688,3229],[16230,1785],[17725,16444],[17953,1208],[21307,1518],[24671,2534],[51,2093],[103,960],[273,598],[299,396],[314,170],[317,293],[320,4870],[354,2866],[357,1105],[357,2599],[432,351],[447,19547],[470,137],[510,154],[548,2515],[548,4595],[597,134],[621,117],[622,7232],[643,1259],[673,1207],[765,6903],[991,1647],[1047,6523],[1048,2216],[1064,145],[1068,376],[1184,145],[1382,7586],[1387,5722],[1410,1179],[1482,4021],[1527,1219],[1551,10646],[1614,1333],[1781,545],[1784,15529],[1838,683],[1908,450],[1962,11008],[2071,296],[2236,296],[2356,83],[2364,50],[2376,312],[2456,288],[3105,417],[3599,553],[3602,468],[3736,10227],[3819,2819],[3881,1019],[3957,1610],[4602,2226],[5075,11399],[5110,164],[5448,2127],[5463,1388],[5558,1665],[5687,1372],[6100,480],[6445,1194],[7465,272],[7554,395],[7771,2326],[8766,463],[9289,114],[9369,480],[9448,978],[9595,396],[9897,2957],[11578,1775],[11733,7019],[13483,12280],[14189,4445],[14526,391],[16572,2228],[17387,3448],[21375,17863],[100,104],[114,381],[235,3039],[333,1836],[347,1798],[395,804],[452,2091],[459,170],[485,77],[495,4794],[499,6590],[505,1029],[507,383],[532,425],[602,12163],[704,595],[728,599],[732,177],[760,1622],[846,950],[882,5864],[891,14925],[897,274],[913,5292],[1040,142],[1088,121],[1096,55],[1104,5878],[1141,157],[1178,490],[1182,691],[1183,2393],[1186,976],[1246,357],[1351,17240],[1382,10932],[1451,51],[1483,4765],[1485,184],[1561,4361],[1564,15136],[1575,1367],[1610,1021],[1611,134],[1668,335],[1753,1290],[1800,2766],[1860,6496],[1988,2894],[2044,950],[2062,5274],[2094,7486],[2271,381],[2802,4803],[2921,1520],[3062,276],[3103,5984],[3286,1692],[3513,1183],[3532,1487],[3601,3327],[4234,1775],[4257,274],[4261,7321],[4295,376],[4334,13662],[4406,310],[4520,4367],[4565,1562],[5201,3403],[5261,10686],[5617,1193],[6060,7135],[6731,8810],[7083,110],[9473,566],[10393,10393],[11567,804],[11901,2226],[12332,1356],[12370,86],[13563,1094],[14878,910],[15578,111],[18323,109],[19128,97],[20936,7635],[78,5839],[85,67],[98,324],[98,564],[98,2296],[108,2678],[268,9864],[270,356],[276,10202],[291,3932],[341,1718],[415,353],[426,185],[431,5113],[468,659],[495,442],[505,1017],[530,22708],[542,112],[551,894],[557,149],[621,116],[622,448],[696,2312],[741,2625],[803,6496],[879,8067],[923,533],[969,154],[999,7609],[1121,3324],[1140,14334],[1178,691],[1304,1210],[1308,1856],[1504,171],[1576,7416],[1591,844],[1625,713],[1635,1811],[1640,537],[1760,19223],[1785,268],[1844,2088],[1864,1507],[1960,3017],[2099,611],[2125,367],[2133,312],[2157,1371],[2305,2408],[2368,1395],[2553,1085],[2555,1756],[2815,7860],[2827,19100],[2834,10962],[2835,541],[3507,116],[3907,1437],[3940,370],[3971,6126],[4308,1555],[4552,8225],[4820,3393],[4864,399],[5181,1253],[5319,276],[5966,136],[6018,2960],[6749,3965],[7162,2693],[7456,97],[8954,4510],[9019,2073],[9229,428],[9887,747],[9960,2219],[11412,450],[12399,289],[13167,1185],[16981,3465],[114,1627],[263,178],[287,1947],[288,98],[291,572],[299,404],[300,13924],[302,4423],[310,1990],[329,140],[373,167],[402,6098],[402,12609],[431,10308],[461,160],[485,8642],[491,135],[499,8409],[508,11306],[548,1536],[645,176],[672,425],[682,3023],[700,386],[754,1809],[877,276],[995,119],[1045,985],[1061,3276],[1104,8118],[1121,1417],[1228,463],[1261,154],[1318,10038],[1354,4564],[1468,11046],[1488,2713],[1513,3786],[1585,682],[1671,3553],[1681,1870],[1708,2959],[1760,2045],[1823,6323],[1882,1539],[1971,544],[2173,9539],[2330,97],[2407,109],[2638,386],[2676,1408],[2801,559],[2871,382],[2943,4007],[2981,1724],[3150,2288],[3373,769],[3392,15914],[3503,2091],[3582,1957],[3856,3258],[4452,6452],[4811,463],[4950,116],[5441,370],[5546,289],[6014,616],[6373,109],[6572,367],[7011,268],[8246,270],[8441,3802],[9290,162],[9783,5593],[10278,2824],[10386,5984],[10387,1218],[10660,1155],[11582,847],[14480,319],[15101,1633],[20668,324],[21348,6333],[68,76],[77,77],[84,378],[109,9240],[122,321],[263,191],[268,307],[276,589],[288,11576],[294,19891],[306,2245],[347,421],[401,5304],[415,105],[425,2089],[428,1365],[447,1633],[452,2102],[484,462],[530,486],[554,144],[623,10978],[743,3701],[744,977],[840,8328],[954,167],[990,276],[997,997],[1075,848],[1150,307],[1166,724],[1205,105],[1326,4856],[1451,50],[1553,615],[1684,1038],[1933,1501],[1954,3384],[1993,2250],[2032,148],[2190,2189],[2635,310],[2718,112],[2815,4004],[2856,276],[2988,179],[3097,2259],[3332,1518],[3603,5300],[3930,18686],[4077,2443],[4299,2872],[4867,3270],[5011,8944],[5313,1947],[5730,272],[6164,276],[6842,99],[8455,3450],[9638,537],[11440,5296],[12787,5156],[13770,894],[14677,4060],[14819,317],[14916,578],[15641,3776],[19539,3643],[52,1841],[106,2363],[115,5970],[119,397],[263,168],[274,117],[281,422],[281,5002],[285,8783],[368,16644],[385,430],[386,827],[402,1594],[403,171],[407,1129],[428,9240],[457,2877],[463,1055],[463,2316],[472,12736],[475,566],[499,5093],[528,19344],[573,2736],[590,134],[667,1559],[717,175],[731,138],[765,1684],[791,404],[816,114],[882,16727],[887,3209],[896,8317],[917,1179],[933,833],[966,528],[1044,277],[1108,759],[1119,5167],[1125,1272],[1215,3351],[1442,16667],[1482,112],[1484,3419],[1567,14839],[1672,673],[1687,2662],[1710,133],[1784,105],[1792,975],[1896,1535],[1950,1535],[2107,3201],[2214,479],[2295,23792],[2431,2163],[2599,810],[2982,2146],[3127,559],[3265,122],[3589,930],[3659,761],[3953,370],[5089,876],[6446,4348],[7985,667],[8150,901],[11937,1902],[13856,5297],[15694,2376],[21486,6270],[21910,13207],[108,1474],[263,174],[280,154],[285,16911],[298,183],[300,287],[321,2576],[337,453],[462,270],[495,1668],[508,959],[589,5631],[598,99],[672,6728],[678,146],[688,1786],[700,276],[703,3311],[741,1516],[748,2013],[845,136],[873,8049],[886,384],[922,268],[930,2164],[931,2463],[1061,989],[1114,1170],[1365,272],[1531,3348],[1635,928],[1642,950],[1725,3406],[1753,1337],[1837,105],[1896,661],[1978,153],[2381,2360],[2438,276],[2585,3629],[2602,3065],[2723,1241],[3015,14404],[3208,4346],[3357,391],[4547,1259],[4592,1723],[4647,378],[4772,1170],[4915,2158],[5038,306],[5620,4119],[5846,789],[6152,3800],[6313,22970],[6990,2387],[7078,121],[7266,1922],[7328,268],[7382,766],[7746,1508],[7900,4467],[8131,296],[9058,417],[9074,1122],[11357,312],[12021,272],[14056,272],[15473,361],[19893,6690],[25213,1364],[65,104],[71,65],[76,79],[97,1554],[110,2184],[164,3356],[236,141],[268,4271],[276,2940],[300,304],[316,137],[333,2575],[337,383],[345,381],[345,538],[357,5469],[361,116],[364,116],[370,2283],[375,1670],[397,119],[423,293],[423,339],[485,5759],[490,3237],[506,178],[579,304],[583,4539],[732,186],[796,407],[830,120],[834,615],[846,17861],[887,6274],[921,311],[963,117],[1096,3329],[1104,22656],[1250,1070],[1262,2370],[1310,450],[1310,2872],[1326,399],[1566,101],[1614,2748],[1696,385],[1738,100],[1783,1544],[1858,2228],[1901,9465],[1938,4770],[2048,8285],[2159,3618],[2195,304],[2239,7356],[2241,486],[2293,1674],[2328,6584],[2338,2570],[2415,1244],[2464,2395],[2551,4824],[2875,3484],[2889,111],[3046,305],[3208,1058],[3498,344],[3502,667],[3593,2675],[3651,2246],[4009,1245],[4024,272],[4259,13133],[4551,367],[4574,1085],[4890,276],[5117,1741],[5284,818],[5388,3440],[5756,1907],[5891,1280],[6374,417],[6844,18444],[7174,289],[7253,5095],[7402,274],[7456,3042],[9470,904],[9742,79],[9744,6441],[10680,115],[12748,17736],[14085,2465],[14353,933],[18921,1041],[19502,16286],[19542,901],[21103,598],[21357,12670],[22704,1779],[76,83],[80,723],[80,3262],[86,80],[118,324],[273,415],[302,8336],[319,337],[401,706],[434,21695],[490,2710],[515,381],[523,918],[590,166],[596,2739],[634,268],[669,305],[858,1408],[875,1118],[1066,1687],[1077,894],[1078,111],[1123,1254],[1193,2675],[1235,16059],[1283,339],[1392,109],[1448,1315],[1451,52],[1511,107],[1656,1996],[1669,628],[1704,819],[1836,6900],[1900,138],[1900,167],[1901,537],[1919,475],[1981,9781],[1982,357],[1994,3815],[2107,289],[2140,2142],[2186,12751],[2205,1104],[2333,100],[2347,556],[2841,16908],[2888,3278],[3186,6847],[3266,155],[3558,17630],[3798,5006],[3933,1959],[4097,65],[4332,450],[4365,5361],[4837,14997],[5378,289],[5381,1958],[5667,11781],[5834,822],[6251,3795],[7523,1786],[8222,2705],[8230,2586],[9679,293],[9840,13739],[10189,531],[12893,274],[13074,941],[20062,1811],[77,84],[102,6602],[113,113],[195,128],[269,335],[269,395],[273,17241],[285,652],[285,7483],[291,2631],[292,153],[300,2790],[312,404],[341,335],[341,16361],[347,1052],[401,310],[431,98],[459,191],[465,370],[503,14917],[532,82],[532,564],[658,1836],[672,97],[720,2408],[721,116],[774,340],[834,276],[846,382],[940,3595],[958,21754],[960,111],[984,2634],[1000,1310],[1163,1787],[1186,496],[1191,2642],[1295,375],[1392,107],[1484,1807],[1555,2169],[1871,144],[1991,3825],[1996,3653],[2098,1031],[2124,4241],[2174,53],[2713,2023],[2797,1241],[2875,3424],[2926,11924],[2989,3371],[3300,783],[3380,133],[3508,1658],[3578,304],[3764,3764],[3872,2209],[4269,3820],[4689,382],[4738,892],[4961,2922],[5921,111],[6106,10189],[6149,793],[6282,145],[6976,1063],[6980,463],[8217,559],[8320,691],[8481,14082],[11774,2721],[12371,1370],[12613,3810],[13009,1520],[15909,352],[17130,1832],[18732,370],[24922,583],[10,9],[80,425],[128,352],[276,5541],[279,141],[305,2360],[306,9015],[321,120],[336,688],[381,422],[386,3841],[460,159],[462,521],[530,711],[532,2273],[579,1768],[595,2773],[642,1622],[675,2809],[677,9577],[728,2525],[753,268],[756,272],[844,3208],[891,105],[926,4894],[947,1763],[972,4229],[1067,184],[1101,2092],[1125,1439],[1195,128],[1247,3213],[1451,57],[1583,175],[1639,357],[1737,7845],[1757,382],[1870,1613],[1894,6688],[2231,1855],[2232,496],[2291,154],[2326,3132],[2407,21151],[2731,1926],[3019,321],[3130,22991],[3595,9777],[3992,796],[4085,2055],[4690,1341],[4879,2702],[4918,115],[5565,1155],[5622,2321],[5662,875],[6112,7136],[6372,3482],[6505,321],[6873,4148],[8037,288],[8366,817],[8935,9875],[9402,1540],[11526,347],[20503,17222],[20803,17548],[22828,111],[23305,268],[66,82],[276,3598],[278,3715],[285,17873],[295,135],[310,305],[399,5679],[409,148],[430,650],[447,1572],[493,189],[495,14682],[532,1071],[548,293],[556,583],[601,153],[625,97],[640,3492],[646,3163],[675,6906],[696,847],[696,2603],[790,18132],[812,1559],[827,827],[837,5415],[854,16190],[877,3849],[888,307],[921,1852],[940,1101],[1064,148],[1130,871],[1134,1369],[1268,2716],[1416,170],[1528,2004],[1566,14585],[1704,2822],[1831,4358],[2026,4858],[2045,545],[2246,1802],[2338,2337],[2390,1105],[2669,311],[2757,115],[2972,1134],[3098,357],[3174,1775],[3325,1914],[3502,1112],[3875,865],[4099,352],[4288,780],[5108,399],[5718,3926],[6168,1369],[7857,793],[8282,119],[9628,2298],[10683,2428],[11193,847],[11717,4493],[11860,1814],[13481,1370],[15049,1627],[19193,326],[21671,781],[74,82],[83,304],[87,353],[99,1356],[103,397],[117,1007],[276,2760],[276,4785],[281,8942],[299,9123],[319,10044],[330,270],[337,3976],[368,2112],[396,268],[402,422],[461,141],[514,1856],[522,2551],[593,759],[669,102],[672,277],[675,941],[691,276],[725,580],[760,382],[774,4812],[785,2075],[789,430],[842,306],[966,2507],[1028,585],[1047,11927],[1125,10038],[1141,141],[1177,321],[1215,1644],[1229,178],[1247,13417],[1284,186],[1358,468],[1485,169],[1560,140],[1792,21666],[1916,276],[2016,1748],[2056,8627],[2110,2766],[2132,3307],[2160,582],[2207,391],[2287,53],[2441,14833],[2602,3343],[2699,117],[2833,622],[2926,2049],[2999,3270],[3526,2026],[3593,1031],[3665,947],[4037,8533],[4329,306],[4421,296],[4480,132],[4541,1107],[4889,163],[4960,3633],[5195,3205],[5467,53],[5776,10694],[5877,276],[6172,1155],[6261,311],[7365,1499],[9059,306],[9994,22384],[10660,3346],[10697,10697],[12828,9542],[13734,467],[17959,8164],[22021,14352],[71,378],[122,1456],[273,339],[276,3837],[276,4127],[285,1835],[320,2529],[368,2127],[373,191],[375,2494],[388,4505],[423,83],[423,1832],[431,22454],[467,10062],[514,83],[516,107],[523,2343],[542,24517],[567,129],[575,545],[579,305],[582,2176],[682,855],[737,1305],[779,1535],[791,103],[882,25151],[1028,448],[1074,1778],[1109,1208],[1180,187],[1182,2019],[1189,1000],[1254,830],[1294,5768],[1312,465],[1392,4131],[1404,307],[1448,733],[1456,104],[1542,386],[1703,815],[1911,1170],[1919,450],[1932,832],[2159,2643],[2257,112],[2270,270],[2271,105],[2409,4536],[2500,12465],[2533,1136],[2688,3748],[2989,287],[3200,894],[3400,11307],[3476,2863],[3729,399],[3989,17613],[4326,19242],[4687,1188],[4797,1271],[5013,1371],[5336,1346],[6177,1546],[6197,382],[7996,289],[8236,3869],[9101,847],[9226,9672],[10297,321],[11850,2733],[14240,10696],[16403,6339],[16579,5223],[16937,4436],[19628,950],[109,2034],[121,2191],[276,2450],[276,8280],[276,11767],[287,119],[302,2609],[341,519],[410,129],[484,2150],[494,1179],[520,166],[548,2356],[558,121],[565,188],[593,4106],[602,4525],[617,143],[623,375],[639,965],[672,82],[685,1593],[704,1075],[719,3807],[720,1276],[751,136],[791,9781],[802,176],[846,1369],[868,18853],[869,17933],[961,4512],[969,167],[970,437],[971,129],[1000,1298],[1005,622],[1087,1947],[1137,716],[1139,4634],[1193,2618],[1278,190],[1382,103],[1423,1202],[1426,357],[1463,2130],[1468,110],[1477,187],[1482,7356],[1577,169],[1580,1431],[1799,3456],[1842,1115],[1872,582],[1901,1038],[2041,112],[2071,4337],[2159,4459],[2261,11156],[2301,1115],[2420,164],[2453,6850],[2482,2744],[2503,2312],[2540,6437],[2646,833],[2771,1481],[3052,6628],[3436,11760],[3494,3494],[4167,1164],[4191,642],[4375,9135],[5200,3460],[6654,6654],[7524,3056],[7706,140],[8497,148],[9023,304],[9942,659],[11209,1307],[12826,3569],[16905,335],[17553,457],[17637,3012],[19926,3071],[20173,2531],[22174,879],[72,270],[102,747],[234,15956],[276,2899],[288,1317],[291,3180],[299,335],[312,509],[320,14382],[327,172],[347,820],[374,145],[386,1128],[406,160],[420,1487],[451,268],[472,121],[472,274],[508,97],[571,1567],[631,1078],[640,2140],[693,4687],[721,3960],[757,2910],[766,914],[784,390],[793,1291],[797,21225],[842,1815],[859,432],[869,3839],[869,4708],[869,6448],[912,105],[1008,175],[1030,8789],[1037,907],[1042,4546],[1059,1815],[1082,615],[1107,2621],[1114,407],[1250,1778],[1270,52],[1296,4843],[1311,185],[1335,15487],[1376,3709],[1406,3167],[1494,1853],[1512,659],[1531,9539],[1664,152],[1736,5674],[1798,20158],[1830,661],[2039,176],[2072,6730],[2080,522],[2119,2462],[2289,1913],[2381,509],[2401,14520],[2446,305],[2467,1870],[2496,566],[2598,1679],[2856,4367],[2930,102],[3250,972],[3360,2451],[3657,144],[3747,422],[3883,5987],[3952,1940],[4462,118],[4698,682],[5168,2190],[5500,545],[6090,480],[6402,463],[6585,659],[6722,106],[7054,3782],[7322,642],[7397,465],[7530,829],[7619,189],[7921,276],[7941,883],[8708,370],[10146,14689],[10386,781],[14886,417],[15941,921],[16437,960],[17392,1940],[18198,3622],[18591,321],[66,381],[83,19606],[85,68],[99,404],[102,2250],[107,747],[119,353],[276,276],[276,748],[276,5940],[283,2304],[287,2086],[294,783],[319,982],[333,648],[333,759],[333,972],[348,14473],[350,3037],[375,1277],[386,3402],[432,1662],[447,2114],[462,3810],[513,165],[515,4028],[635,177],[650,582],[661,3354],[727,8238],[855,1324],[885,2296],[886,1104],[926,4182],[940,9204],[989,743],[1029,1029],[1039,830],[1117,10971],[1157,442],[1176,1264],[1232,5673],[1314,894],[1342,5676],[1402,4036],[1656,7799],[1700,23468],[1725,1165],[1834,139],[1932,2017],[2110,2005],[2231,4000],[2281,5972],[2338,988],[2380,1761],[2409,1031],[2488,1430],[2557,1941],[3051,20870],[3127,3088],[3146,1879],[3192,417],[3850,1016],[3908,764],[3991,115],[4218,1148],[4243,3292],[4414,84],[4594,5069],[5314,699],[5412,1244],[6028,12100],[8120,1143],[8258,1822],[8470,1808],[8835,884],[9753,7408],[10025,983],[10030,7333],[10190,397],[11158,1007],[12037,422],[13050,376],[15115,2618],[15546,700],[15922,938],[17442,10498],[19656,321],[20130,391],[21709,122],[98,652],[99,1743],[105,1148],[263,166],[273,16998],[278,1071],[285,3346],[292,174],[294,103],[350,109],[385,3162],[431,118],[434,2562],[435,13403],[467,1043],[490,817],[496,716],[507,3466],[514,112],[548,13115],[613,895],[616,287],[625,310],[627,189],[658,976],[661,2025],[690,134],[717,159],[840,843],[858,2916],[890,101],[1006,276],[1017,276],[1017,1811],[1123,3730],[1164,450],[1192,2499],[1232,1128],[1304,842],[1513,2942],[1518,4438],[1554,122],[1576,3808],[1581,169],[1759,5291],[1860,4058],[1869,1521],[2013,385],[2064,179],[2260,276],[2706,585],[2734,111],[2933,1953],[3220,18229],[3514,3873],[3985,620],[4259,112],[4343,783],[4429,851],[4617,13577],[4761,1939],[5160,1007],[5365,9737],[6217,1489],[6607,9501],[6695,417],[7129,274],[7763,6015],[7866,10244],[7902,1666],[8224,2053],[9121,422],[9221,772],[9958,9354],[10733,2500],[12116,18598],[14161,14161],[16538,2243],[21854,2643],[72,923],[78,451],[116,8327],[119,5002],[273,683],[320,4868],[326,109],[333,1108],[350,2561],[364,2270],[375,607],[381,830],[447,1586],[475,983],[551,7150],[753,584],[761,1726],[772,339],[791,415],[882,885],[924,1483],[945,137],[958,75],[1123,268],[1140,941],[1157,7363],[1177,3585],[1225,181],[1232,1364],[1245,769],[1372,10981],[1394,3781],[1481,1115],[1494,12203],[1504,144],[1591,4665],[1739,1188],[1787,1958],[1842,3066],[1900,155],[2044,1298],[2137,189],[2177,116],[2359,1951],[2412,2387],[2435,276],[2510,1682],[2650,1528],[2701,1374],[2875,1390],[2897,376],[3191,2065],[3241,571],[3661,6674],[3736,107],[3771,1321],[3969,5093],[4018,12792],[4082,276],[4112,10849],[4267,2599],[4400,480],[4873,417],[4880,2856],[4926,2679],[5250,858],[5846,490],[5857,3454],[5997,116],[6470,1389],[8965,357],[9549,5662],[9962,5258],[10542,376],[10765,822],[11014,2182],[11338,376],[12550,1389],[13527,422],[13770,783],[15875,5185],[17003,479],[49,2364],[104,1536],[112,397],[112,4770],[190,188],[273,105],[273,1209],[273,2115],[284,168],[288,628],[304,1283],[312,1297],[341,2363],[378,296],[399,2840],[402,921],[420,810],[434,14815],[480,566],[586,1745],[658,1379],[666,17975],[670,830],[686,186],[689,177],[820,812],[904,7093],[944,2228],[961,1786],[979,10765],[992,743],[1024,115],[1070,5669],[1124,9832],[1125,1474],[1177,370],[1200,276],[1256,10978],[1309,545],[1313,2843],[1318,6098],[1354,521],[1382,16653],[1471,1386],[1479,181],[1524,3148],[1616,1649],[1681,1489],[1703,580],[1740,116],[1823,559],[1836,5097],[1939,3591],[2783,276],[2919,304],[2974,1941],[3035,25559],[3252,7933],[3390,288],[3513,737],[3628,1665],[3747,1540],[3749,155],[3814,1757],[3818,1031],[3997,6724],[4158,4467],[4619,822],[4748,304],[4757,268],[4951,12175],[5144,463],[5302,824],[5371,1226],[5448,98],[5473,1036],[5823,1645],[5909,276],[5953,2586],[6816,463],[7568,17501],[7595,475],[9840,5768],[10474,2570],[11059,357],[12482,268],[13734,3752],[15429,442],[17401,2515],[19375,1142],[22472,13019],[23038,378],[65,2296],[110,100],[110,116],[281,8153],[294,3555],[366,608],[368,2379],[388,307],[414,160],[434,783],[438,161],[461,182],[499,335],[508,117],[523,2325],[558,117],[584,18627],[622,982],[635,142],[725,2164],[765,2679],[816,270],[826,2646],[857,307],[858,8617],[880,181],[887,562],[907,1194],[994,7937],[1042,1257],[1098,141],[1127,12810],[1134,382],[1176,3236],[1178,1305],[1236,14924],[1300,643],[1305,1097],[1310,983],[1408,19073],[1432,97],[1501,815],[1517,2825],[1569,5293],[1610,1489],[1619,111],[1692,1063],[1697,832],[1747,13464],[1780,4892],[1784,14479],[1832,370],[1901,293],[1916,7033],[2129,4835],[2205,9960],[2214,788],[2382,353],[2787,14077],[2851,918],[2881,450],[2989,578],[3027,812],[3147,2586],[3194,138],[3468,190],[3496,667],[3675,347],[3881,766],[3953,2443],[3978,3066],[3979,101],[4097,80],[4121,111],[4272,276],[4394,7061],[4508,16895],[4610,9683],[4690,892],[4741,153],[5210,1112],[5401,4925],[5592,1946],[5695,11132],[5702,352],[6580,6071],[6910,2578],[8367,521],[10133,324],[11970,2150],[12356,8062],[12863,810],[14661,941],[16744,2250],[18465,2740],[19885,114],[32,1803],[66,289],[67,3543],[69,339],[70,82],[80,2414],[102,3433],[106,117],[110,608],[122,578],[268,4297],[288,3545],[312,1645],[324,112],[333,276],[333,2697],[336,2274],[336,10709],[358,140],[376,1292],[401,9707],[409,146],[432,367],[458,181],[462,1640],[499,1809],[508,1456],[544,2914],[596,428],[651,190],[658,4166],[664,163],[675,1766],[753,18257],[813,3458],[943,1439],[1039,2597],[1046,115],[1080,8424],[1085,7978],[1100,18473],[1199,3371],[1200,2078],[1248,3563],[1264,836],[1418,2696],[1625,760],[1719,400],[1769,130],[1787,847],[1911,3211],[2133,558],[2178,357],[2412,1007],[2621,8375],[2656,6258],[2676,11325],[2858,977],[3048,6283],[3127,417],[3363,1851],[3481,272],[3562,2815],[3799,19025],[3952,12102],[3972,558],[3984,344],[5101,3239],[5252,6687],[5688,9374],[5858,1733],[6162,116],[6295,353],[7929,9672],[8264,1786],[8866,2744],[10685,3450],[11078,296],[13112,2388],[13943,2127],[14325,7805],[14985,2507],[18559,709],[21284,1857],[21631,1658],[22045,465],[84,7571],[111,626],[115,8752],[188,8413],[194,165],[280,22087],[307,268],[340,337],[361,9065],[429,177],[434,3632],[452,10357],[455,165],[460,175],[486,3708],[503,112],[508,1266],[595,276],[661,760],[695,770],[759,463],[760,838],[795,2540],[815,2510],[816,885],[826,1393],[837,608],[842,3069],[857,17607],[893,183],[920,3188],[924,1241],[1009,153],[1017,2764],[1088,2656],[1175,724],[1314,15543],[1382,6247],[1417,1191],[1466,3159],[1537,2246],[1644,9739],[1659,7244],[1744,8007],[1948,5600],[1987,598],[1987,616],[2002,2906],[2120,1715],[2307,1995],[2368,12858],[2456,2114],[2571,1087],[2676,5623],[2832,274],[3004,3004],[3025,276],[3385,716],[3403,759],[3479,2477],[3538,2092],[3675,386],[3845,428],[3979,2250],[3981,1007],[4223,475],[4460,5515],[4527,274],[4620,5326],[4630,3606],[4639,101],[4884,1613],[5099,3988],[5155,122],[5207,6741],[5241,10348],[5450,9557],[5458,559],[5991,102],[6196,7929],[6597,1864],[6639,344],[6757,7198],[6759,1294],[6999,164],[7581,479],[8478,566],[9460,4419],[9821,13053],[9824,2317],[9902,2928],[11397,422],[12581,3174],[13294,121],[13853,132],[15572,7062],[19602,1743],[21002,1041],[68,2908],[85,115],[99,115],[100,277],[102,6939],[273,7991],[277,977],[285,1853],[289,927],[299,616],[299,851],[300,538],[302,1603],[306,1299],[319,965],[365,1241],[393,3346],[436,147],[493,178],[530,3125],[531,1598],[639,8261],[642,566],[666,420],[666,788],[711,711],[711,4701],[753,3471],[788,2043],[857,289],[858,1379],[869,17564],[886,1206],[948,180],[1077,2000],[1117,683],[1195,175],[1207,385],[1314,2000],[1364,1390],[1397,1933],[1404,1720],[1461,1198],[1491,1422],[1494,21251],[1529,995],[1582,272],[1648,153],[1753,871],[1913,19682],[2012,450],[2103,276],[2158,2750],[2334,2588],[2464,4554],[2635,21019],[2888,3347],[3009,276],[3212,156],[3336,871],[3375,6586],[3705,5567],[3749,150],[3798,6169],[3845,706],[4059,673],[4284,2228],[4355,1666],[4596,769],[4694,4694],[4965,191],[5050,796],[5313,988],[5319,2696],[5461,9195],[5701,1849],[7182,305],[7332,558],[7529,3643],[8511,538],[9606,3143],[10120,307],[11058,14272],[11584,830],[11629,11173],[12661,21938],[16198,9320],[18715,304],[20094,479],[22533,116],[24918,9200],[25365,12416],[26053,4436],[72,83],[98,1046],[101,1791],[102,339],[235,154],[274,335],[277,111],[281,1432],[320,15460],[324,18041],[336,616],[336,1071],[336,2893],[347,276],[350,3722],[366,344],[367,2125],[367,11161],[411,141],[443,160],[463,1836],[464,132],[470,169],[472,20001],[495,396],[582,2366],[614,270],[631,17390],[651,164],[675,3798],[677,6640],[698,1541],[726,11292],[966,18464],[1005,15961],[1044,538],[1083,1816],[1116,3516],[1123,1627],[1170,2163],[1298,4159],[1412,9633],[1416,175],[1578,1921],[1730,4363],[1735,1567],[1783,1315],[1804,115],[1897,566],[2139,623],[2256,139],[2398,490],[2484,115],[2534,3576],[2608,8459],[2999,304],[3247,673],[3563,3214],[3636,766],[3707,5388],[3925,417],[3969,296],[4228,109],[4241,698],[4607,611],[4995,450],[5163,407],[7302,376],[7725,5243],[8897,10340],[9089,422],[9326,385],[10258,1475],[11337,764],[16042,9801],[16989,2559],[20035,140],[20937,4544],[24193,15183],[77,76],[99,1734],[106,287],[117,3958],[274,2000],[276,12524],[276,12830],[281,465],[395,370],[423,1594],[430,900],[446,135],[454,13278],[520,191],[523,13004],[532,5935],[553,737],[589,833],[591,163],[593,9213],[596,1317],[617,181],[625,67],[703,1202],[719,1071],[738,5187],[752,3058],[753,442],[754,2869],[773,992],[829,2340],[836,817],[858,990],[858,2900],[885,312],[909,6000],[981,5853],[1066,1819],[1147,169],[1329,1016],[1336,399],[1378,268],[1531,1160],[1547,3267],[1557,276],[1697,1230],[1723,3867],[1758,477],[1760,407],[1810,2918],[1856,117],[1919,479],[1991,599],[2126,1138],[2239,98],[2334,480],[2531,566],[2560,910],[2577,479],[2723,348],[2770,3930],[2810,360],[2814,12888],[2888,417],[2971,950],[3260,541],[3298,553],[3325,811],[3506,1792],[3587,276],[3652,1527],[3670,147],[3773,276],[3816,3437],[4185,521],[4339,696],[4346,2219],[4436,1589],[4460,599],[4480,151],[4557,621],[4745,724],[4845,793],[4857,4892],[4908,556],[5323,384],[5400,1767],[5579,462],[5579,1734],[5990,5223],[6069,709],[6615,1240],[6715,772],[6787,317],[7092,545],[7402,16760],[7437,780],[7580,111],[8145,2664],[8511,270],[8624,923],[8808,14943],[9248,132],[10719,21713],[11064,5941],[11869,1746],[13046,842],[13639,1949],[13660,3213],[14960,480],[15625,5709],[15957,78],[16168,1048],[16178,6982],[17691,960],[17746,1779],[20974,991],[21531,1533],[71,83],[107,15183],[112,378],[115,12921],[119,2806],[276,705],[283,634],[289,442],[291,10903],[335,4818],[336,1142],[367,884],[385,1367],[386,2210],[386,2618],[393,105],[401,4046],[421,2462],[499,3585],[521,5015],[523,1259],[533,120],[571,1138],[573,898],[587,2867],[633,153],[655,556],[737,490],[753,1071],[753,14952],[756,8573],[896,6418],[921,806],[976,667],[994,4615],[1042,1360],[1054,2918],[1061,811],[1072,709],[1082,2405],[1108,1534],[1129,407],[1182,2078],[1230,3576],[1248,2545],[1310,382],[1404,19881],[1657,566],[2138,352],[2294,128],[2361,178],[2628,901],[2704,1717],[2800,26217],[2830,270],[2897,276],[2950,48],[2983,611],[2991,850],[3279,593],[3708,761],[3845,7490],[4081,2856],[4254,399],[4897,399],[5083,1774],[5161,5895],[5741,2766],[5773,6256],[6405,122],[6598,1182],[6712,3569],[8188,272],[8992,19947],[9051,381],[9984,537],[12252,1645],[16967,463],[20703,628],[87,270],[119,538],[122,370],[188,136],[278,17219],[283,1606],[285,268],[299,14682],[333,1330],[432,3442],[472,24807],[495,14171],[499,2008],[516,2928],[575,407],[586,1856],[587,1877],[631,816],[652,326],[685,18432],[703,1058],[703,3141],[707,133],[711,1994],[784,79],[812,2740],[820,375],[829,939],[853,171],[873,1852],[891,277],[899,191],[924,432],[940,7091],[975,2624],[1042,895],[1088,105],[1095,18718],[1108,486],[1121,1585],[1125,289],[1156,4676],[1210,3208],[1246,8326],[1388,276],[1438,2556],[1537,1105],[1595,2242],[1661,851],[1668,18390],[1689,119],[1770,132],[1798,5750],[1837,1852],[1850,1499],[1865,51],[1870,357],[2002,3488],[2076,1589],[2110,1613],[2136,4418],[2203,3450],[2208,3356],[2263,12913],[2271,288],[2338,3988],[2482,564],[2704,1567],[2714,276],[2745,289],[2751,1446],[2825,611],[2877,12300],[2889,97],[3108,1347],[3184,450],[3221,296],[3398,132],[3627,3437],[3756,2479],[3769,1437],[3981,1786],[4075,457],[4084,562],[4406,2703],[4421,270],[4554,1191],[4867,1853],[4884,2643],[5075,621],[5776,20984],[5823,2803],[6355,417],[6388,1131],[6695,407],[6720,1495],[6722,2913],[6766,450],[7509,3274],[8404,69],[8831,553],[9000,1980],[9146,304],[9456,111],[9569,3303],[10859,8208],[10903,2228],[11837,1281],[13202,7851],[15164,287],[15808,1910],[16104,487],[18330,3581],[23818,11113],[32,15589],[71,82],[72,73],[77,519],[274,1947],[276,1397],[278,6590],[281,6116],[281,6219],[291,121],[320,25743],[327,187],[361,321],[375,1166],[383,859],[463,2042],[472,3071],[497,169],[533,1832],[548,465],[571,930],[571,1430],[573,5168],[587,1049],[596,289],[646,1575],[652,293],[704,673],[726,25765],[728,1934],[740,152],[870,100],[995,305],[1031,4434],[1039,8129],[1047,4267],[1089,859],[1106,648],[1110,276],[1114,643],[1124,1330],[1270,56],[1382,119],[1426,2693],[1513,451],[1543,2515],[1585,875],[1608,148],[1610,788],[1753,1355],[1944,118],[1989,421],[2133,465],[2214,13870],[2278,2720],[2301,1164],[2352,21363],[2358,306],[2368,2871],[2573,854],[2611,1226],[2676,2630],[2750,1055],[2764,819],[2821,589],[2826,837],[2905,1607],[2920,25278],[2981,1630],[2997,8329],[3005,2772],[3006,16638],[3252,4105],[3366,475],[4342,1469],[4663,1914],[4671,6561],[5003,1700],[5422,312],[5448,12322],[5457,781],[5538,533],[6172,2226],[6237,804],[6975,13496],[7133,115],[8157,8157],[8497,165],[8881,4435],[9382,9208],[10660,2226],[11706,3745],[11749,4008],[12106,2046],[13062,3717],[13191,370],[13696,1185],[14887,475],[15243,1835],[18160,1789],[21897,2379],[25298,7587],[50,471],[68,66],[87,105],[98,709],[269,4510],[276,544],[276,1573],[277,2250],[305,111],[330,390],[364,99],[375,1635],[392,148],[420,1458],[447,3037],[451,783],[466,175],[474,132],[486,855],[495,1044],[506,145],[532,67],[551,66],[572,116],[650,759],[655,448],[672,437],[672,2273],[682,587],[764,101],[897,2576],[914,956],[963,2455],[1095,8776],[1106,743],[1170,1355],[1178,1847],[1314,293],[1358,917],[1361,5278],[1392,103],[1479,141],[1494,3555],[1542,667],[1775,693],[1820,169],[1836,3138],[1907,3021],[2041,3697],[2043,919],[2262,1030],[2297,521],[2372,3423],[2412,400],[2447,696],[2466,6859],[2490,2791],[2627,2181],[2854,1634],[2858,533],[3114,161],[3123,1044],[3244,8955],[3340,2797],[3379,399],[3400,10610],[3637,121],[3844,936],[3865,2114],[4158,397],[4204,5293],[4260,3410],[4888,22118],[5118,4832],[5120,417],[5565,7603],[5849,1658],[5849,3609],[6183,121],[6197,480],[6861,144],[6996,9883],[7054,6015],[9448,950],[9838,783],[10085,128],[10367,20980],[11068,1293],[12117,1217],[12997,2571],[15941,3622],[18522,462],[19740,4990],[86,67],[98,396],[100,2515],[103,851],[278,688],[281,578],[283,6319],[287,121],[287,962],[289,1663],[301,128],[309,15956],[333,2795],[333,6935],[357,6367],[384,16828],[648,1401],[667,1418],[726,6116],[739,180],[748,386],[753,304],[764,3642],[765,3853],[766,5078],[784,3786],[796,11689],[818,477],[851,122],[947,3166],[1042,399],[1049,720],[1150,335],[1153,3974],[1169,159],[1335,2800],[1352,184],[1363,817],[1369,1613],[1391,1534],[1458,480],[1609,1855],[1610,1321],[1639,6546],[1759,421],[1773,178],[1795,112],[1831,2374],[1881,272],[1896,3105],[1927,559],[1979,13049],[2022,1257],[2116,18262],[2186,1007],[2305,399],[2858,25208],[2874,3231],[3055,465],[3168,1823],[3206,397],[3315,276],[3376,2785],[3806,1137],[4556,3816],[4687,1219],[5021,1250],[5460,3946],[5525,1173],[5558,628],[5921,293],[5990,5332],[6090,382],[7188,11945],[7270,2202],[9323,417],[9442,25069],[9986,659],[10029,9941],[10050,307],[10101,274],[10577,370],[10912,733],[12435,451],[13120,7409],[14853,5547],[15399,479],[17616,304],[73,100],[107,2283],[112,107],[118,922],[168,417],[277,304],[298,184],[316,183],[319,2302],[320,361],[320,1718],[334,1582],[341,3730],[347,975],[367,583],[381,270],[420,8809],[460,174],[486,7227],[487,934],[497,175],[532,1100],[555,184],[584,425],[586,14477],[591,189],[672,353],[677,8661],[717,144],[752,450],[758,162],[820,682],[830,2340],[845,171],[879,8801],[888,289],[899,185],[917,332],[931,1368],[956,3031],[961,344],[1030,1884],[1037,14833],[1111,160],[1112,2555],[1139,779],[1182,21800],[1236,1887],[1295,682],[1405,2336],[1412,6037],[1426,643],[1488,14600],[1506,6346],[1567,2216],[1649,3331],[1748,559],[1805,161],[1896,2545],[1904,5291],[1971,776],[2128,2900],[2186,1786],[2231,5125],[2268,3056],[2288,5357],[2300,176],[2336,6152],[2340,903],[2342,10366],[2520,2486],[2606,6506],[2611,1252],[2683,1170],[2764,4233],[2805,307],[3003,21974],[3081,1959],[3173,1958],[3197,1293],[3690,2820],[3714,133],[3892,276],[3968,522],[4504,608],[4508,23349],[4598,1603],[5008,3042],[5072,3542],[5075,6814],[5774,141],[6123,111],[6123,533],[6152,11121],[6237,390],[6539,894],[6659,9374],[7095,857],[7101,12713],[7916,983],[8466,1115],[8506,276],[9302,988],[10649,978],[14177,370],[14255,268],[15751,1844],[16061,1041],[16964,462],[17486,425],[20931,13974],[72,5780],[75,69],[83,1421],[278,2428],[285,921],[285,5303],[296,1160],[319,24602],[333,913],[337,2855],[368,1254],[368,5093],[368,8310],[376,347],[401,1389],[409,4418],[487,1241],[497,143],[514,2739],[533,962],[548,10155],[644,22274],[647,49],[704,1268],[730,20868],[790,2378],[815,3267],[820,2625],[840,2703],[844,1706],[926,3484],[934,384],[1035,375],[1095,2099],[1227,399],[1301,4868],[1351,11009],[1357,353],[1392,13085],[1418,994],[1443,679],[1444,1026],[1451,54],[1452,3800],[1501,580],[1529,3787],[1536,321],[1540,2105],[1550,148],[1625,1870],[1759,716],[1776,854],[1829,3403],[1869,276],[1996,562],[2129,430],[2161,50],[2193,1857],[2581,799],[2883,465],[3096,276],[3122,97],[3123,8453],[3208,2017],[3222,306],[3441,2083],[3514,11269],[3663,683],[3675,385],[3747,2209],[3842,2355],[4003,146],[4228,14949],[4284,103],[4373,21648],[4532,806],[5155,20762],[5194,3731],[5247,1297],[5624,1421],[5672,2086],[5772,1306],[5795,901],[5902,140],[6580,312],[6580,1088],[6695,919],[7069,3764],[7391,417],[7496,479],[8074,1856],[8674,8779],[8936,2824],[9116,1031],[10007,276],[10642,752],[11592,12689],[13558,1940],[15918,156],[16192,391],[16944,1910],[17213,463],[23181,26821],[103,3483],[262,165],[263,171],[266,167],[270,764],[275,130],[277,2009],[283,902],[283,1421],[285,24799],[287,428],[287,995],[294,4901],[306,5060],[321,105],[368,816],[376,761],[378,1786],[430,3139],[437,274],[452,8049],[460,183],[474,154],[495,4529],[573,2142],[587,2513],[601,154],[621,1803],[625,688],[653,132],[675,3086],[732,156],[735,129],[753,395],[811,530],[826,276],[868,120],[869,428],[909,14824],[943,21313],[966,1205],[973,153],[975,1183],[1037,7910],[1097,615],[1119,6543],[1156,812],[1235,2017],[1251,3469],[1272,105],[1339,187],[1382,13418],[1386,1082],[1395,450],[1436,151],[1513,289],[1523,97],[1529,533],[1592,902],[1711,97],[1749,2388],[1755,737],[2110,611],[2120,121],[2165,1348],[2214,10268],[2311,899],[2366,1926],[2401,6153],[2431,2221],[2585,2868],[2711,693],[2714,1446],[2797,348],[2938,3450],[3027,347],[3173,978],[3287,169],[3316,1499],[3400,116],[3603,2589],[3736,102],[3832,347],[4078,1043],[4608,8631],[4741,145],[5072,1101],[5200,13576],[5346,3958],[5758,109],[6508,509],[6637,56],[6780,360],[7773,370],[7803,5032],[9125,1706],[9512,2871],[9551,1493],[9637,16423],[9665,160],[10073,422],[10582,1087],[10608,829],[10743,2226],[11478,15017],[11821,5583],[11862,977],[12358,3217],[12773,391],[13957,6248],[14138,140],[14197,1924],[16731,1041],[19538,2980],[22235,7904],[27440,2034],[70,2273],[97,119],[107,396],[234,26268],[266,171],[276,5347],[287,272],[289,12868],[289,15543],[291,1879],[296,1734],[341,3803],[368,353],[375,4904],[385,3293],[463,580],[486,1336],[495,564],[499,1077],[507,351],[507,7976],[514,86],[528,7269],[551,98],[666,12710],[670,6833],[672,422],[679,6176],[711,7507],[721,9367],[743,3171],[745,9618],[752,13141],[813,3141],[913,1250],[1106,2315],[1150,5015],[1194,1758],[1211,1150],[1382,404],[1385,780],[1401,997],[1609,1075],[1641,611],[1744,305],[1991,2623],[2059,2680],[2124,2170],[2138,14248],[2138,16157],[2207,2219],[2262,1992],[2282,1683],[2361,152],[2392,114],[2406,3478],[2473,453],[2598,6072],[2771,1535],[3068,276],[3109,11121],[3219,611],[3286,2047],[3336,1114],[3344,287],[3471,330],[3572,11045],[3573,23894],[3612,3327],[3918,88],[3970,27739],[3978,417],[4240,3435],[4242,2306],[4688,479],[4722,3926],[5219,992],[5827,522],[5948,180],[6866,272],[6958,6085],[7155,5224],[7192,4384],[7566,1310],[7995,2108],[9151,890],[9212,2281],[10617,2479],[10972,2046],[12482,116],[18349,3059],[20237,3561],[21596,356],[21985,832],[54,1841],[83,428],[86,304],[87,1113],[119,1044],[276,904],[276,1859],[299,4012],[299,5651],[337,1902],[341,5496],[354,4767],[366,897],[375,1108],[385,3791],[410,158],[434,3720],[453,583],[579,287],[605,168],[621,378],[708,177],[717,150],[738,7954],[748,720],[748,1443],[776,1926],[864,132],[879,3912],[907,2617],[942,175],[960,288],[962,2166],[1000,1244],[1069,179],[1137,938],[1150,876],[1206,1058],[1229,169],[1231,450],[1292,928],[1330,2613],[1376,8408],[1417,3065],[1429,296],[1493,4803],[1612,941],[1636,994],[1702,399],[1896,611],[1896,1083],[1899,12515],[1942,2439],[2132,1937],[2160,24117],[2425,779],[2431,357],[2450,276],[2476,1016],[2701,1241],[3149,163],[3187,766],[3489,304],[3663,1990],[3834,1687],[3950,276],[4085,5183],[4364,105],[4531,18017],[5249,100],[5400,2156],[5565,2226],[5856,417],[5897,5076],[5990,370],[6440,11371],[6465,743],[6852,8313],[6976,1574],[7563,357],[7760,98],[7840,910],[8122,3431],[8917,608],[8947,976],[8965,559],[9679,361],[10111,105],[11722,142],[12175,939],[12365,837],[13106,1850],[13391,1786],[14972,2790],[15043,2112],[15765,977],[80,68],[87,288],[89,111],[115,634],[274,1879],[276,1364],[283,921],[284,169],[291,270],[331,5781],[333,1037],[336,4595],[343,1217],[350,872],[360,884],[362,134],[365,2866],[375,975],[375,2848],[385,3353],[386,794],[388,570],[392,172],[420,8326],[432,1925],[472,2008],[486,1282],[495,2379],[522,1118],[537,422],[587,3225],[679,2650],[684,1655],[759,582],[816,804],[820,541],[846,788],[891,7698],[926,3826],[929,467],[947,7827],[958,2589],[997,1636],[1037,8449],[1056,164],[1088,115],[1123,923],[1123,21303],[1180,166],[1200,1328],[1228,3240],[1272,268],[1337,1164],[1382,9454],[1385,5826],[1395,357],[1527,2070],[1569,13619],[1838,3013],[1840,272],[1901,4991],[2153,2634],[2236,442],[2311,8126],[2333,1785],[2352,652],[2401,628],[2648,352],[2710,1620],[2775,1377],[2781,53],[2817,675],[2822,2812],[2900,1388],[2971,1746],[3050,4990],[3235,1217],[3307,1000],[3651,5162],[3660,390],[3736,287],[3833,347],[3998,743],[4067,324],[4256,1776],[4355,2205],[4867,12245],[4983,276],[5013,2522],[5155,637],[5253,306],[5259,988],[5461,5274],[5708,827],[6020,1164],[6342,592],[6535,3569],[7829,1030],[9554,111],[10148,417],[10274,5626],[10649,382],[11128,272],[11416,22568],[11538,353],[13368,117],[14359,8557],[16356,952],[17211,153],[17647,584],[20075,2903],[24288,311],[24369,832],[65,16679],[77,97],[82,76],[82,422],[97,1797],[266,145],[277,537],[280,8929],[281,837],[308,160],[326,396],[330,1515],[333,2642],[337,1741],[381,683],[434,13001],[463,817],[463,1109],[499,305],[499,1043],[499,6160],[503,287],[514,6122],[522,3118],[532,652],[532,1357],[545,611],[602,105],[602,462],[613,1983],[636,190],[640,6246],[675,4848],[684,2332],[703,1235],[721,5339],[754,2310],[755,177],[784,9496],[833,1055],[836,607],[843,876],[882,13269],[900,1573],[950,1105],[980,816],[980,5074],[994,2362],[1072,10397],[1144,691],[1236,293],[1270,54],[1288,4237],[1305,743],[1322,580],[1379,907],[1382,107],[1393,276],[1394,593],[1528,593],[1555,7941],[1561,12236],[1600,183],[1765,348],[1810,14516],[1843,1060],[1987,5164],[2014,184],[2235,5997],[2263,2284],[2528,2284],[2560,5102],[2613,1889],[2640,178],[2685,317],[2884,97],[3061,344],[3190,1703],[3246,1007],[3308,1136],[3338,911],[3377,276],[3655,21362],[3716,417],[4088,347],[4242,2824],[4364,22635],[4786,3331],[5275,1651],[5848,1932],[5849,1645],[5927,2942],[6835,15991],[6922,289],[7212,3180],[7762,312],[7900,2889],[8645,1121],[9071,4444],[9510,1062],[10220,288],[11386,781],[13106,806],[13310,1115],[14532,1924],[14577,592],[15611,3621],[18061,509],[19458,804],[19823,2204],[20454,4901],[20986,12822],[22751,467],[24711,18222],[70,79],[80,2034],[86,6743],[261,177],[295,140],[296,1603],[299,1940],[303,144],[320,2546],[323,147],[339,111],[404,570],[424,167],[476,173],[495,330],[532,288],[548,9849],[551,86],[595,4892],[607,1390],[614,105],[620,442],[698,276],[703,1472],[754,2112],[832,2954],[855,2393],[932,187],[947,992],[957,184],[978,417],[982,340],[993,1356],[1054,1613],[1093,3391],[1117,2597],[1166,399],[1170,566],[1265,1367],[1376,4265],[1401,7382],[1411,1659],[1430,1574],[1450,22349],[1553,724],[1608,153],[1672,421],[1771,276],[1800,1787],[1912,5590],[1937,611],[2042,276],[2076,1253],[2131,3032],[2241,385],[2286,2360],[2307,1740],[2338,11172],[2347,2863],[2476,5234],[2496,1481],[2671,20316],[2828,430],[2832,462],[2944,172],[2979,3359],[3171,1101],[3216,348],[3287,164],[3337,3337],[3493,3143],[3528,276],[4688,1678],[4806,764],[5404,4511],[5427,26009],[5628,20652],[5698,417],[5777,304],[5793,1666],[5823,628],[5870,18981],[6089,1114],[6896,1907],[6936,12567],[7586,122],[8339,6347],[8426,993],[10622,558],[10843,305],[10866,1062],[11728,559],[12577,6248],[13430,268],[13808,4491],[15363,2529],[16014,2980],[17676,1814],[18901,395],[24606,950],[52,2093],[84,88],[112,121],[114,117],[115,1554],[119,1700],[136,643],[147,185],[234,28256],[278,2181],[331,13211],[357,1114],[384,337],[386,496],[419,8342],[435,14615],[486,1708],[493,146],[499,415],[508,1623],[532,22587],[533,107],[603,184],[639,4973],[650,333],[721,1403],[728,2845],[732,178],[738,324],[761,2130],[765,12187],[784,1665],[818,7290],[820,1029],[857,305],[882,7468],[892,1031],[928,1101],[943,579],[945,138],[976,3612],[977,1209],[984,1298],[1011,272],[1266,102],[1274,933],[1278,133],[1310,407],[1468,304],[1568,1739],[1708,1198],[1729,675],[1810,4432],[1842,2545],[1869,875],[1922,773],[2041,3115],[2102,112],[2118,1256],[2120,296],[2154,6724],[2229,6895],[2238,1049],[2342,1487],[2386,2617],[2398,1541],[2483,180],[2486,270],[2604,2835],[2849,4479],[2883,3262],[2905,111],[2976,22401],[3074,2586],[3125,276],[3126,122],[3256,1330],[3283,2444],[3542,1016],[3746,56],[4078,18366],[4136,1779],[4185,116],[4186,160],[4191,1409],[4215,960],[4236,986],[5043,311],[5098,1250],[5112,6066],[5222,450],[5382,10686],[5422,305],[5587,1218],[6637,55],[7155,8139],[7823,138],[8374,321],[9530,2209],[10426,26682],[10561,304],[11382,2465],[11625,480],[13105,1832],[14220,376],[19252,21720],[22312,1837],[26087,1774],[32,572],[85,66],[98,370],[104,850],[109,6035],[114,2869],[226,139],[235,187],[276,1227],[278,3694],[281,2310],[285,922],[319,17643],[325,442],[333,1090],[360,677],[384,655],[385,725],[396,335],[407,2028],[472,353],[536,167],[576,175],[589,3388],[644,911],[669,999],[708,173],[761,2070],[782,147],[791,396],[832,2614],[879,583],[886,2534],[892,1397],[896,345],[905,178],[961,3007],[1068,766],[1083,559],[1128,1541],[1180,181],[1304,16493],[1314,2273],[1340,780],[1382,15856],[1387,11089],[1392,15697],[1480,164],[1481,382],[1494,1700],[1566,9367],[1573,766],[1763,1108],[1825,385],[1886,571],[1935,5872],[1937,1579],[2055,5630],[2279,531],[2338,415],[2353,773],[2380,13246],[2409,1466],[2453,15262],[2607,1518],[2684,7198],[2762,386],[2938,12753],[2972,2005],[2999,345],[3028,348],[3120,2421],[3184,3231],[3194,183],[3197,4327],[3233,48],[3246,293],[3288,2199],[3570,276],[3589,1057],[3706,296],[3778,769],[3933,7454],[3965,312],[3993,7073],[4021,2816],[4179,13263],[4332,4138],[4417,7911],[4483,9570],[4610,5225],[4625,417],[4690,1527],[4826,4826],[5288,538],[6154,24306],[6401,5502],[6605,938],[7543,480],[7561,3348],[7975,270],[8834,25628],[9128,19060],[9236,12576],[9851,2908],[10752,928],[11440,18310],[12890,15563],[14817,479],[16551,933],[17821,1779],[18065,17683],[21728,376],[24267,1341],[79,108],[108,616],[116,118],[281,8162],[294,18015],[331,570],[335,11459],[375,2914],[401,921],[403,169],[422,121],[423,936],[423,1745],[434,20489],[461,166],[463,1019],[472,317],[472,579],[490,667],[595,1207],[596,5724],[603,13058],[631,108],[656,184],[666,12597],[673,1323],[684,1271],[702,437],[756,99],[806,117],[868,325],[869,19708],[893,155],[917,2863],[922,305],[977,1515],[982,332],[989,1004],[1019,1328],[1039,806],[1054,1298],[1150,4453],[1183,1257],[1206,3617],[1257,11447],[1318,24477],[1417,3624],[1430,24852],[1506,1286],[1511,6153],[1574,2626],[1595,1264],[1690,541],[1696,1461],[1726,3782],[1738,25369],[1776,15490],[1905,711],[1928,17259],[1932,10019],[1981,2166],[2198,7872],[2214,1830],[2223,7097],[2385,19102],[2412,4512],[2518,12965],[2544,4379],[2684,5878],[2800,1644],[2881,479],[2921,871],[3008,84],[3018,276],[3019,3091],[3059,1142],[3078,1021],[3094,110],[3187,2812],[3282,480],[3299,5127],[3306,2134],[3321,1262],[3553,2711],[3799,107],[3825,938],[3860,1227],[3922,582],[3976,583],[4648,121],[4710,172],[4941,7161],[4958,15544],[5019,4043],[5507,370],[5561,137],[6155,450],[6406,270],[6626,116],[7138,442],[7144,276],[7149,422],[7177,2049],[7439,1872],[7543,382],[7667,16638],[8279,3115],[8614,399],[8803,2707],[9311,307],[9893,450],[11213,10811],[12261,345],[15738,897],[15918,180],[18164,752],[21901,2015],[68,18382],[82,442],[268,422],[276,531],[276,1049],[276,3858],[280,19452],[283,2310],[285,7663],[288,1100],[333,333],[350,10357],[383,6381],[421,3590],[463,4093],[469,163],[490,4699],[508,1633],[527,139],[596,1804],[602,3965],[683,293],[726,442],[766,4036],[793,1418],[855,682],[859,556],[1005,3610],[1047,566],[1058,813],[1090,1933],[1205,1142],[1247,1684],[1308,102],[1395,1115],[1453,174],[1468,13605],[1635,8400],[1748,847],[1759,1092],[1762,159],[1774,1412],[1842,950],[1908,3873],[2067,167],[2190,457],[2294,156],[2384,3974],[2470,695],[2595,183],[3043,7870],[3069,6411],[3135,321],[3141,457],[3295,865],[3654,276],[3848,3052],[3931,1864],[4204,3012],[4306,480],[4521,4901],[4575,2800],[4872,760],[4904,1651],[4929,173],[5008,10218],[5158,5113],[5546,97],[6297,1408],[6987,8328],[7464,1007],[7737,1055],[8796,1949],[8863,2877],[9039,102],[9592,370],[9808,293],[10894,1112],[11496,760],[12613,2134],[12944,912],[13364,3442],[13579,114],[15184,272],[15450,1192],[16310,661],[16742,330],[20177,390],[21677,2877],[22167,3730],[23497,310],[102,312],[108,692],[112,1317],[273,3180],[275,191],[276,7346],[278,572],[288,467],[300,3676],[319,453],[354,340],[375,12118],[384,859],[422,111],[432,383],[434,7742],[495,105],[499,2112],[522,3825],[530,820],[586,3795],[604,155],[640,7642],[650,14253],[734,135],[766,1542],[779,1823],[826,716],[869,425],[877,1598],[910,1128],[940,3553],[1083,661],[1106,530],[1108,858],[1130,6596],[1166,2147],[1168,146],[1323,2048],[1334,176],[1482,3007],[1494,304],[1513,3392],[1577,183],[1621,187],[1689,1592],[1762,135],[1783,1328],[1784,8053],[1860,918],[2004,1037],[2015,382],[2042,1138],[2056,2766],[2063,709],[2072,3599],[2075,1055],[2311,1397],[2332,115],[2371,720],[2820,2959],[2979,838],[3017,531],[3060,1006],[3067,417],[3219,3124],[3287,167],[3324,5139],[3750,1019],[3798,15657],[3835,2086],[4221,12423],[4518,918],[4597,566],[4607,2761],[5082,6366],[5118,288],[5154,3859],[5352,3563],[5410,523],[5433,1939],[5441,97],[5789,417],[5929,1118],[6189,1740],[6663,8865],[6843,107],[7167,2041],[7757,479],[8041,2142],[8094,2371],[8160,1369],[9211,417],[10408,152],[12911,276],[13817,575],[15912,101],[16906,2040],[17111,2846],[18386,1944],[19548,9596],[19988,1574],[20236,415],[21373,6199],[28470,6718],[67,816],[68,1209],[98,4156],[105,6263],[109,3274],[154,752],[273,17728],[304,4413],[317,120],[330,5759],[333,2533],[341,296],[350,921],[367,655],[370,709],[375,1612],[375,4728],[388,7852],[397,3740],[421,914],[451,1603],[455,147],[472,76],[495,69],[495,83],[519,4413],[532,317],[557,132],[561,172],[640,3311],[666,6367],[690,190],[727,11172],[745,829],[796,4909],[797,16679],[816,2278],[836,1452],[886,3185],[891,2790],[975,1649],[1028,911],[1108,1045],[1157,1627],[1183,3166],[1210,2358],[1214,1321],[1250,12367],[1261,136],[1262,643],[1318,1686],[1395,480],[1407,562],[1525,1836],[1681,1688],[1923,1571],[2019,276],[2029,1667],[2075,700],[2115,97],[2124,2279],[2147,858],[2151,100],[2174,57],[2323,307],[2464,4662],[2540,1092],[2558,339],[2663,5489],[2734,14303],[2736,2057],[2833,4022],[2903,2189],[3055,422],[3057,1321],[3185,457],[3235,1358],[3514,1870],[3548,2244],[3845,1797],[4095,132],[4250,5119],[4363,57],[4541,4628],[4972,407],[5009,4743],[5247,850],[5400,1559],[5622,1570],[5819,21979],[6289,5478],[6700,3862],[6884,2618],[7771,1792],[7855,3639],[8461,370],[8598,1837],[8979,148],[9851,121],[11312,2538],[12406,12406],[13391,1715],[18893,404],[19636,305],[23363,2046],[24808,5337],[25684,2622],[25991,18571],[78,73],[86,101],[99,850],[104,13841],[112,652],[300,542],[322,167],[325,304],[333,1521],[375,1136],[377,644],[385,3824],[386,920],[432,494],[456,140],[486,667],[530,623],[563,138],[587,2078],[602,19479],[603,145],[615,6094],[629,183],[644,556],[679,938],[826,3045],[826,6387],[858,6532],[909,7855],[980,24334],[982,367],[986,811],[1020,169],[1044,324],[1045,904],[1066,3138],[1093,3144],[1133,378],[1139,1290],[1203,152],[1211,558],[1275,1144],[1340,1589],[1406,1595],[1450,643],[1513,3053],[1592,1294],[1598,607],[1672,1672],[1693,842],[1749,8365],[1993,5293],[2004,430],[2055,2070],[2058,1571],[2079,5234],[2154,2372],[2211,820],[2276,6209],[2427,276],[2436,2048],[2464,1074],[2464,2737],[2596,2216],[2875,2956],[2925,191],[3054,276],[3127,15149],[3529,12966],[3564,2085],[3648,1977],[3686,843],[3700,378],[3909,115],[4164,6647],[4172,276],[4566,9555],[4895,370],[4923,146],[4977,2437],[5003,1043],[5193,2910],[5219,875],[5463,8062],[5592,2756],[5690,399],[5793,5972],[6075,1933],[6197,417],[6345,837],[6987,1955],[7082,116],[7724,11408],[8498,1347],[9591,270],[10040,164],[10685,4703],[10870,16232],[12141,923],[12452,2576],[14813,442],[15556,709],[15588,4669],[16251,1297],[16317,4832],[17066,747],[22038,2903],[22997,5415],[23828,521],[27430,3849],[65,465],[65,6947],[66,2757],[75,277],[80,304],[84,66],[86,288],[104,592],[135,6621],[235,135],[237,19543],[281,977],[293,304],[299,6012],[306,2862],[324,4499],[324,11250],[333,3750],[338,145],[342,138],[361,312],[398,152],[402,370],[420,357],[428,960],[429,172],[487,365],[525,151],[581,146],[613,1578],[741,679],[759,5375],[762,111],[784,84],[789,571],[891,82],[896,120],[896,616],[926,541],[943,2402],[1037,679],[1039,1785],[1065,496],[1154,3960],[1183,1687],[1196,467],[1252,8956],[1285,2349],[1414,911],[1466,3849],[1477,161],[1535,523],[1632,157],[1694,545],[1694,611],[1697,25170],[1777,562],[1783,2042],[2011,1985],[2111,7700],[2138,25648],[2194,276],[2307,390],[2346,815],[2378,276],[2436,2288],[2516,2993],[2559,1562],[2571,14152],[2574,160],[2619,2429],[2644,56],[2753,890],[2909,2870],[3094,107],[3154,682],[3253,7473],[3311,2017],[3363,465],[3416,3905],[3645,940],[3723,430],[3782,9024],[3856,1293],[4018,18857],[4296,877],[4311,9024],[4398,367],[4417,357],[4437,12981],[4582,936],[4601,116],[5083,13799],[5410,10142],[5587,1926],[5645,11042],[5702,760],[6383,9149],[6637,52],[6695,357],[6853,811],[7513,101],[7540,2865],[8108,56],[8113,307],[8266,321],[8561,289],[8868,276],[8885,843],[9835,6981],[10384,747],[10814,1208],[10976,1226],[11741,122],[13130,673],[13257,3472],[14160,538],[14972,10696],[15565,188],[15570,3362],[16489,1999],[18361,3346],[18687,3715],[18896,1804],[24710,2959],[71,73],[80,1887],[104,1160],[105,1940],[109,1887],[263,160],[276,7637],[285,1779],[315,172],[317,1526],[331,2112],[333,3142],[335,305],[375,593],[428,2134],[451,509],[498,143],[523,611],[523,978],[582,399],[602,1562],[691,938],[702,1640],[728,992],[753,467],[753,851],[753,970],[791,12982],[842,844],[889,171],[903,521],[926,799],[950,847],[997,3665],[1011,8573],[1052,3169],[1201,1906],[1256,347],[1262,475],[1306,2042],[1424,8779],[1469,545],[1528,20350],[1551,909],[1592,11252],[1606,3776],[1671,3444],[1689,12823],[1774,7333],[1885,144],[1924,858],[1983,3298],[2047,858],[2062,812],[2222,10523],[2367,305],[2487,1756],[2509,191],[2520,19289],[2552,1848],[2567,184],[2809,607],[2838,12405],[2878,1679],[2987,2042],[3103,26731],[3107,116],[3354,4445],[3365,1563],[3486,276],[3575,357],[3586,1649],[3655,20823],[3951,3951],[4830,3336],[4875,780],[5077,3532],[5097,14088],[5126,804],[5537,1016],[5557,276],[5625,2676],[5974,400],[6807,1409],[7500,1397],[7792,10364],[8088,5113],[8435,115],[8719,14268],[8727,1097],[9815,1291],[9992,4570],[10942,8405],[12601,779],[13425,149],[13947,3053],[16555,12271],[16577,276],[17209,296],[17653,2676],[18834,693],[19394,11977],[19514,4578],[21987,533],[26326,985],[111,272],[115,99],[339,894],[367,924],[375,1333],[395,268],[397,396],[401,325],[422,1607],[426,145],[430,972],[472,3958],[484,1100],[495,2606],[495,9127],[548,2889],[551,12540],[571,1232],[596,1655],[603,165],[650,5520],[664,168],[665,156],[686,168],[753,5496],[803,1982],[812,463],[822,1143],[851,105],[907,1019],[1020,135],[1031,9173],[1068,1019],[1075,3109],[1165,3797],[1166,5071],[1214,14679],[1222,1221],[1282,385],[1447,2169],[1541,4411],[1614,276],[1705,1557],[1827,305],[1960,799],[2139,1187],[2261,5590],[2322,8001],[2440,2696],[2456,1515],[2608,3307],[2630,1103],[2832,5528],[3033,2178],[3118,276],[3156,2647],[3271,781],[3287,144],[3332,3332],[3420,5651],[3768,83],[3940,381],[4018,1412],[4075,8530],[4148,2226],[4223,14613],[4331,854],[4332,4054],[4425,608],[4470,84],[4707,6099],[4745,5973],[5261,9394],[5475,1000],[5953,3671],[5981,6839],[6385,1226],[6532,1764],[6564,2865],[7162,559],[7449,407],[7973,321],[7973,894],[8049,310],[8364,1586],[9216,11622],[9297,479],[9671,272],[9984,2928],[11227,22121],[11235,13728],[11461,5015],[11513,5752],[12508,628],[12851,783],[15009,111],[17743,537],[17872,422],[17910,10715],[18090,3571],[20395,2523],[26523,3465],[70,293],[87,4471],[115,6689],[183,148],[217,132],[277,3270],[312,4271],[333,1238],[354,908],[367,2143],[402,16422],[454,1150],[486,1017],[595,12634],[615,1219],[625,584],[652,305],[661,1460],[684,404],[684,2757],[702,13464],[765,19199],[795,1683],[796,3992],[864,169],[926,7216],[946,2386],[961,5535],[1014,4924],[1023,164],[1029,5430],[1038,902],[1039,829],[1146,190],[1194,1386],[1201,1115],[1248,357],[1287,164],[1314,927],[1383,10523],[1407,4839],[1479,149],[1487,1343],[1488,2113],[1494,13682],[1530,171],[1555,2282],[1560,154],[1591,3492],[1611,160],[1713,164],[1722,2046],[1742,108],[1822,306],[1991,1379],[2040,2070],[2044,5307],[2116,8877],[2138,14547],[2220,305],[2331,496],[2336,1257],[2377,2652],[2400,3454],[2456,27336],[2605,806],[2685,97],[2896,376],[2913,2816],[2952,1670],[2990,7616],[3003,22759],[3124,3124],[3167,530],[3567,1458],[3572,26104],[3575,2643],[3628,628],[3666,272],[3796,6547],[3875,1369],[3979,11928],[4007,818],[4097,69],[4396,185],[4630,1143],[4879,939],[4904,2441],[5922,276],[6015,8645],[6083,1228],[6657,376],[6784,407],[6817,4089],[6858,2593],[7494,696],[7523,6045],[7561,2592],[8103,760],[9060,716],[9071,2911],[9896,991],[10647,7552],[10919,385],[11759,1272],[12265,997],[12383,289],[12892,1557],[13208,163],[16671,352],[17652,1162],[17713,106],[18846,467],[19611,537],[20214,1949],[29294,13171],[104,1357],[114,578],[285,528],[320,3274],[324,3569],[324,6283],[333,1538],[356,542],[357,475],[359,173],[388,1887],[407,12886],[496,759],[522,3410],[539,137],[556,7806],[579,542],[599,1207],[625,572],[666,1259],[680,148],[684,17682],[697,184],[719,970],[728,815],[760,3774],[790,854],[813,4544],[863,18154],[866,186],[873,3776],[904,1074],[928,6674],[929,2631],[995,101],[1013,10244],[1020,145],[1066,766],[1117,2009],[1139,566],[1166,7545],[1181,2897],[1197,1317],[1200,1016],[1235,3311],[1244,1781],[1419,141],[1442,13152],[1591,6411],[1616,541],[1719,5158],[1938,5124],[1962,391],[2044,1535],[2066,1257],[2104,2427],[2159,4909],[2189,6052],[2231,1388],[2268,353],[2296,321],[2415,575],[2469,1273],[2486,268],[2497,3820],[2608,3058],[2703,307],[2812,8827],[2838,3314],[3033,2742],[3087,643],[3098,3454],[3706,321],[3756,2588],[3993,1645],[4007,1221],[4042,370],[4162,822],[4334,2721],[4659,4670],[4723,1613],[4833,1241],[5078,7123],[5175,883],[5323,12305],[5361,23713],[5746,1400],[5905,1244],[5943,5111],[6089,950],[6308,84],[6405,114],[6453,4925],[6525,5580],[6714,1302],[6811,107],[6993,22747],[7095,13174],[8041,391],[8097,276],[8429,1310],[8600,187],[8710,479],[8831,9859],[9210,3932],[9488,764],[9879,3362],[10727,116],[10835,837],[11241,1437],[11584,17517],[15745,496],[16590,573],[21672,901],[24195,6229],[24771,9743],[25118,2538],[25901,28162],[28384,1045],[68,79],[83,634],[86,538],[119,970],[137,865],[276,7955],[281,17495],[283,1743],[287,4141],[300,764],[336,12361],[336,21624],[388,2337],[401,356],[401,18046],[437,305],[461,163],[486,793],[522,992],[532,4277],[622,659],[685,4043],[715,129],[728,3841],[743,2315],[765,1593],[820,646],[841,1370],[872,1515],[874,871],[907,5022],[940,386],[954,141],[954,142],[1013,3431],[1031,276],[1159,2405],[1285,6767],[1322,700],[1327,1879],[1478,276],[1576,1337],[1584,181],[1591,13462],[1680,2697],[1748,407],[1800,3611],[1815,6037],[1843,1823],[1881,8256],[2011,118],[2256,137],[2262,1330],[2549,8680],[2692,276],[2760,463],[2821,1780],[3011,1746],[3112,110],[3129,1092],[3529,1294],[3578,115],[3653,3588],[4322,1124],[4582,26930],[5171,14989],[5352,788],[5546,1658],[5558,1658],[5569,399],[6524,12776],[6851,109],[6911,1210],[7165,5124],[7175,2053],[7299,340],[7566,1114],[7633,268],[7853,110],[9182,4336],[9658,13044],[11019,376],[11286,4643],[11880,1539],[13057,1726],[14551,977],[15011,335],[15641,6144],[15831,2015],[16664,1809],[17235,3464],[18212,307],[18317,571],[19763,300],[22358,12519],[25159,2538],[27664,5669],[66,111],[74,11823],[75,6448],[78,8876],[83,353],[99,442],[102,7345],[110,1162],[112,1536],[266,188],[285,960],[285,3446],[291,1100],[291,9479],[318,184],[322,138],[322,177],[340,2522],[350,1603],[357,1579],[364,5145],[375,28782],[381,97],[421,3267],[438,174],[453,18658],[472,8294],[490,737],[491,169],[495,465],[514,88],[522,2514],[548,1809],[557,140],[601,176],[617,147],[621,293],[661,21690],[665,155],[672,5934],[673,4709],[690,133],[691,1252],[735,150],[752,352],[754,7490],[807,188],[827,1401],[837,305],[838,760],[913,949],[980,1675],[1053,141],[1137,276],[1163,420],[1167,183],[1195,178],[1265,3750],[1314,10440],[1316,146],[1366,764],[1384,1587],[1397,276],[1426,2818],[1436,173],[1474,709],[1484,4596],[1792,3129],[1793,384],[1808,4665],[1896,24804],[1937,450],[1967,168],[2032,167],[2185,2339],[2193,100],[2207,1058],[2236,1370],[2260,877],[2294,153],[2397,3129],[2520,2382],[2602,5485],[2734,307],[2988,132],[3092,858],[3160,16785],[3283,117],[3284,541],[3296,276],[3381,930],[3554,1527],[3562,725],[3707,595],[3757,65],[3816,3759],[3936,691],[4018,2190],[4179,21440],[4218,97],[4284,339],[4635,3967],[4826,2234],[5098,2487],[5520,1807],[5542,2435],[5554,832],[5716,10599],[6119,752],[6132,1594],[6334,16055],[6807,1122],[6835,17891],[7178,399],[7750,12068],[8095,3229],[9090,3132],[9562,3677],[10296,2696],[10506,1812],[10609,293],[11166,4786],[11441,556],[12070,7408],[12452,2768],[12806,6354],[12990,824],[13430,960],[14783,432],[15215,3448],[15537,1779],[16732,121],[17069,8002],[21453,4553],[22892,19706],[24660,9634],[25435,13064],[26752,390],[27268,1062],[66,270],[98,437],[283,1368],[300,901],[319,622],[319,695],[323,149],[333,1548],[350,110],[353,104],[383,1512],[399,5464],[404,289],[433,152],[454,3002],[508,428],[514,13786],[524,3335],[548,519],[572,370],[599,496],[625,7328],[672,442],[679,2026],[683,16650],[684,390],[691,2748],[715,155],[720,820],[720,1144],[814,7609],[844,4544],[855,5430],[869,304],[910,3859],[972,7842],[986,5942],[989,2953],[1051,141],[1070,562],[1183,1541],[1219,1016],[1230,13462],[1302,545],[1316,178],[1354,7366],[1386,1553],[1388,399],[1394,7303],[1433,583],[1532,854],[1561,496],[1578,759],[1748,479],[1770,129],[1865,53],[1896,1115],[1912,3874],[1913,724],[2016,3619],[2048,2436],[2096,2693],[2139,2408],[2161,647],[2251,8388],[2257,1888],[2280,3611],[2284,752],[2286,1857],[2436,3781],[2606,1539],[2827,475],[3019,592],[3076,1021],[3330,357],[3455,24591],[3482,417],[3756,1122],[3794,2841],[4039,268],[4385,5579],[4436,2062],[4492,2091],[4494,14607],[4717,100],[5072,2092],[5271,2289],[5388,1559],[5660,4583],[5883,114],[6236,634],[6299,1640],[6805,357],[6986,26488],[6989,451],[7343,12203],[7370,1310],[7611,4495],[8266,2576],[8352,822],[8383,1029],[8430,382],[8617,1764],[9125,886],[10102,5212],[10189,13210],[10649,480],[11737,737],[12103,2519],[12103,25610],[15160,305],[15396,277],[16632,15613],[17061,370],[17217,3521],[19153,1884],[21414,5744],[21745,1386],[23287,5723],[26373,13562],[32,2226],[65,111],[101,117],[269,17989],[310,321],[325,6705],[333,2750],[345,3512],[356,1254],[368,6263],[378,112],[385,1256],[386,11499],[401,4024],[406,184],[407,1062],[414,150],[434,8336],[437,326],[468,360],[472,1858],[483,129],[494,1902],[496,817],[521,121],[572,311],[579,107],[596,105],[596,584],[596,921],[687,167],[705,1386],[719,105],[719,1931],[745,101],[872,1491],[929,15395],[1005,6779],[1104,4346],[1143,4875],[1246,5477],[1268,15132],[1273,3870],[1310,1890],[1363,386],[1370,122],[1392,806],[1444,14598],[1450,9502],[1482,11888],[1488,2931],[1506,1394],[1518,4434],[1703,1506],[1810,19180],[1811,1017],[1894,276],[1896,2871],[1963,528],[1974,854],[2246,559],[2352,5415],[2722,276],[2726,121],[2988,187],[2989,16139],[3003,7394],[3033,1535],[3162,276],[3194,133],[3390,3034],[3658,883],[4037,1388],[4166,276],[4417,2801],[4548,1142],[4725,9364],[4828,268],[4923,132],[5077,10679],[5260,9346],[5302,1949],[5457,11477],[5458,357],[5636,7717],[6560,811],[6697,270],[6787,400],[7046,13505],[7354,357],[7523,5111],[7840,701],[8265,128],[8598,1603],[8962,1789],[9028,384],[9140,1170],[9194,4860],[9982,109],[10712,1926],[11103,2959],[11389,2104],[11578,2242],[11655,288],[12664,186],[16016,5363],[16164,370],[19064,287],[20482,1250],[23962,1290],[24605,15897],[24916,809],[25322,1209],[27611,20743],[70,83],[100,14435],[103,361],[119,2034],[269,4595],[296,2930],[299,17166],[320,339],[331,1700],[351,2157],[358,186],[367,4747],[402,4317],[422,344],[434,1582],[485,987],[498,160],[513,157],[536,169],[562,2353],[570,830],[613,562],[614,3180],[648,691],[672,1976],[698,1687],[711,9597],[759,4623],[826,1208],[833,3698],[861,1097],[868,6493],[869,1263],[943,99],[985,1057],[1085,2049],[1091,178],[1340,2061],[1344,2655],[1382,995],[1412,1210],[1426,450],[1453,177],[1502,5220],[1580,667],[1670,1839],[1700,293],[1800,2343],[1802,450],[1823,661],[1853,293],[1893,682],[1896,1487],[1979,2513],[1991,1899],[2175,432],[2203,3910],[2426,347],[2487,2700],[2495,1250],[2498,312],[2520,117],[2750,10104],[3004,1013],[3044,1041],[3081,3787],[3136,3393],[3641,723],[3657,188],[3819,12360],[4450,589],[4652,1142],[4702,117],[4850,103],[4886,2377],[5085,276],[5103,4939],[5563,3045],[5573,6654],[5640,545],[5701,296],[5791,12587],[5812,941],[6060,3521],[6147,357],[6292,122],[6370,659],[6580,6488],[7133,370],[7498,325],[7724,344],[8074,2310],[8285,2288],[8598,305],[9151,3056],[9246,9246],[10221,121],[10408,172],[10738,822],[10783,353],[11018,9932],[11556,7221],[12762,26887],[13726,2824],[15967,2744],[16226,396],[17367,830],[17614,159],[18191,312],[19358,317],[20041,559],[21344,2868],[24622,1100],[67,304],[83,533],[86,425],[115,923],[117,287],[140,141],[276,6918],[281,4265],[299,29662],[321,321],[336,14070],[350,1652],[367,4336],[368,851],[375,1075],[383,5952],[401,584],[415,304],[428,3810],[434,3037],[472,339],[472,1931],[514,3111],[539,15408],[551,77],[586,16219],[614,15514],[614,16661],[625,2860],[666,3404],[666,3887],[673,1598],[675,3275],[688,268],[699,2866],[704,3785],[730,556],[744,24122],[745,12768],[762,3632],[797,10329],[826,1386],[868,395],[873,1346],[887,1875],[904,6196],[943,100],[943,17894],[961,1700],[994,1761],[1057,3291],[1065,3305],[1133,829],[1134,545],[1136,5154],[1157,3301],[1226,276],[1277,1239],[1322,990],[1348,120],[1364,276],[1366,1273],[1458,4813],[1473,155],[1474,370],[1557,595],[1639,1062],[1648,166],[1687,673],[1829,505],[1861,780],[1912,6744],[1928,105],[1979,1443],[1994,1324],[2071,1593],[2082,385],[2120,274],[2123,340],[2123,9504],[2164,595],[2372,4778],[2466,773],[2490,522],[2619,9636],[2672,3511],[2685,111],[2972,6203],[3073,2283],[3245,3245],[3279,522],[3334,523],[4055,118],[4357,3327],[4451,2903],[4466,977],[4828,1271],[5325,1307],[5505,8245],[5636,1036],[5898,347],[6008,692],[6409,340],[6461,276],[6574,2618],[6637,54],[7092,382],[7172,465],[7430,1118],[8217,357],[8226,1049],[8692,3036],[8754,4614],[9792,425],[11933,276],[12138,370],[13750,311],[14347,1551],[16250,2283],[16736,270],[16771,15796],[16845,422],[17866,18935],[18085,1889],[20022,12891],[22181,521],[23932,3354],[26409,10106],[70,3137],[80,121],[110,1046],[278,17776],[291,10613],[292,162],[296,112],[320,2250],[336,16382],[388,1857],[401,542],[401,5853],[430,580],[478,168],[496,737],[503,9683],[508,5002],[509,995],[523,760],[524,3329],[544,6871],[551,103],[556,2145],[581,148],[598,3676],[602,927],[625,3433],[661,1060],[695,319],[728,12473],[738,345],[769,587],[782,156],[820,667],[869,5404],[943,27915],[977,939],[982,585],[1037,5631],[1133,912],[1134,450],[1160,8110],[1174,172],[1201,1610],[1235,20641],[1236,16764],[1244,752],[1257,2750],[1265,7919],[1314,1827],[1326,1408],[1351,8328],[1374,10342],[1396,133],[1447,1614],[1472,1104],[1512,556],[1569,8762],[1624,276],[1782,6575],[1836,1103],[1887,277],[1905,1144],[1938,361],[1952,6792],[1994,9404],[2010,8730],[2056,2028],[2070,3946],[2110,1080],[2204,830],[2280,1667],[2390,3184],[2400,950],[2416,5669],[2518,2562],[2683,661],[2761,12507],[3098,21340],[3183,348],[3363,339],[3607,321],[3647,2033],[3661,3767],[3778,1372],[3901,276],[4075,640],[4437,270],[4475,507],[4511,5138],[4749,8354],[4806,5586],[4964,2676],[4970,417],[5047,901],[5379,321],[5504,3803],[5608,417],[5614,907],[5758,274],[5953,2253],[6131,545],[6597,1867],[6843,1947],[7464,2387],[7694,2955],[9132,29970],[9478,566],[10420,276],[10582,20274],[11048,6066],[11234,578],[11368,781],[11508,335],[12141,19049],[12252,628],[12676,1495],[13497,3864],[13513,733],[13767,307],[13793,307],[14094,289],[14484,1148],[14567,97],[15168,15168],[15371,1185],[17364,26470],[17456,2104],[18428,921],[25325,6930],[28686,781],[29003,1493],[71,13088],[99,1715],[99,2699],[104,6684],[106,115],[108,2283],[119,1266],[206,177],[234,182],[273,927],[276,1198],[285,1368],[311,305],[319,2245],[359,132],[364,772],[364,3722],[390,1627],[401,22125],[425,5462],[430,1922],[455,128],[469,159],[472,1148],[495,2471],[497,160],[615,5772],[655,583],[671,170],[683,4757],[705,1090],[706,17361],[729,3460],[753,876],[753,5780],[869,345],[879,2245],[888,28804],[913,985],[927,6256],[966,270],[985,910],[1000,1105],[1044,1495],[1052,1336],[1097,724],[1264,817],[1287,178],[1290,450],[1451,55],[1482,110],[1503,2018],[1507,276],[1656,1092],[1663,1623],[1709,1949],[1725,385],[1741,1371],[1890,1298],[2001,184],[2007,1055],[2173,116],[2176,2128],[2344,965],[2352,634],[2469,12591],[2503,2162],[2560,386],[2862,844],[2996,48],[2997,877],[2999,422],[3243,7799],[3363,116],[3642,991],[4329,332],[4574,5079],[4741,140],[4880,276],[5038,367],[5123,3794],[5480,9714],[6168,2005],[6237,1100],[6662,120],[6697,370],[6698,5800],[7301,960],[7497,6792],[7694,7806],[7777,306],[8011,10227],[8018,643],[8628,8284],[8630,2868],[10143,2182],[10977,176],[11535,537],[13093,4583],[13434,15344],[13535,114],[14254,307],[18118,747],[18545,4271],[20484,4493],[23831,3965],[26404,451],[32,330],[84,13687],[100,102],[112,1837],[114,100],[114,598],[227,136],[235,146],[263,182],[276,946],[276,3773],[276,5054],[277,5095],[289,2960],[289,8928],[296,1283],[312,2041],[320,287],[337,8612],[341,27131],[357,788],[368,13564],[375,8601],[376,720],[423,829],[431,5397],[432,2003],[434,10317],[437,960],[463,2164],[532,12361],[544,3154],[569,137],[584,1663],[611,5391],[646,682],[658,815],[658,1362],[683,11089],[702,462],[738,14308],[748,711],[759,3239],[762,102],[789,3275],[862,165],[877,1430],[891,274],[926,1609],[932,156],[940,3086],[1000,1667],[1011,274],[1081,182],[1107,1259],[1151,6840],[1228,3789],[1274,16808],[1300,1083],[1306,743],[1309,1469],[1357,111],[1387,23683],[1423,4577],[1450,4634],[1518,3822],[1525,1055],[1569,9753],[1592,2633],[1774,886],[1864,858],[1891,2739],[1903,6570],[2013,1735],[2023,1764],[2039,145],[2102,616],[2178,559],[2261,12553],[2381,100],[2400,8335],[2627,1254],[2646,691],[2979,3986],[3010,713],[3061,706],[3160,2135],[3264,4847],[3276,2815],[3466,354],[3522,4271],[3759,4556],[3909,960],[3959,759],[3979,5941],[4122,2783],[4164,2388],[4243,509],[4370,102],[4483,422],[4493,277],[4574,357],[4594,779],[4761,2370],[4972,919],[5171,7720],[5247,97],[5441,422],[5848,23912],[6273,886],[6287,5778],[6549,2155],[7149,894],[7179,2437],[7972,272],[8394,421],[8833,2000],[8931,3442],[9271,628],[10925,1041],[11073,356],[12158,531],[12399,317],[12880,2017],[12915,7907],[14796,1297],[14871,587],[15806,696],[19889,3857],[20033,5671],[20886,8583],[20900,1639],[21113,12267],[24002,6144],[26111,3433],[26279,1389],[83,7594],[104,1297],[276,2201],[281,521],[283,2628],[285,1155],[306,9337],[321,311],[321,2523],[345,747],[353,3932],[368,824],[388,117],[413,152],[421,580],[421,8102],[430,1013],[442,274],[472,345],[496,615],[508,305],[599,691],[616,288],[631,10636],[651,133],[677,2855],[708,171],[724,1116],[755,183],[768,185],[774,1902],[785,646],[814,6082],[863,1888],[877,1394],[928,463],[943,78],[944,2592],[1000,1259],[1037,7650],[1038,5711],[1045,1764],[1057,1198],[1099,189],[1118,1839],[1206,1210],[1277,385],[1286,1461],[1293,1449],[1309,6046],[1348,287],[1382,287],[1398,133],[1423,6037],[1468,21585],[1496,173],[1506,2213],[1543,10734],[1562,270],[1578,623],[1620,385],[1686,115],[1739,5102],[1784,8336],[1792,3849],[1829,6465],[1894,430],[1897,1870],[1901,804],[2058,2010],[2075,1019],[2077,330],[2084,6153],[2094,3424],[2107,98],[2193,1607],[2205,886],[2265,1137],[2333,344],[2341,1202],[2425,22238],[2503,3879],[2564,1964],[2725,4175],[2757,1740],[2962,7317],[3013,901],[3017,1547],[3035,759],[3041,156],[3103,2593],[3266,174],[3607,1814],[3638,2697],[3663,7220],[3666,296],[3701,276],[3972,98],[4013,811],[4199,2479],[4299,417],[4706,615],[4720,5022],[4807,2373],[4972,21881],[5149,1907],[5429,1542],[5543,3769],[5890,2419],[5899,270],[6539,97],[7219,5153],[7486,759],[7493,773],[7941,1670],[8123,1394],[8194,811],[8279,312],[8385,523],[8502,2258],[8778,2248],[8900,4510],[10017,9353],[10384,451],[11197,747],[12496,6208],[12846,1043],[13620,2980],[13702,114],[14857,5997],[15358,3477],[16295,385],[16997,345],[17106,20121],[17641,1297],[18072,68],[20446,2089],[23419,8491],[24650,2429],[25619,2867],[25872,12772],[28608,16830],[100,13115],[104,1271],[235,173],[273,27709],[291,19006],[310,3292],[319,10795],[341,2727],[356,8690],[368,18934],[375,1957],[375,29564],[391,859],[402,11253],[430,815],[434,300],[448,800],[484,4775],[485,579],[491,142],[522,990],[573,1104],[577,163],[595,3794],[625,15729],[672,7895],[684,69],[684,816],[725,1264],[753,1858],[764,9454],[765,321],[842,21916],[869,2894],[874,2012],[891,1851],[895,1589],[909,11499],[931,15949],[951,128],[975,956],[978,407],[1000,919],[1017,2490],[1095,12038],[1123,608],[1154,22815],[1219,541],[1388,486],[1506,1466],[1542,2625],[1575,956],[1606,304],[1622,10024],[1634,789],[1655,1240],[1667,950],[1697,2017],[1794,191],[1798,376],[1859,789],[1910,10366],[1941,276],[2223,13520],[2225,593],[2297,26999],[2305,623],[2343,417],[2352,1536],[2353,979],[2458,335],[2639,6209],[2649,1781],[2668,490],[2689,119],[2938,304],[3019,116],[3093,643],[3100,3036],[3213,111],[3304,453],[3496,2625],[3555,304],[3655,5326],[3655,9097],[3721,339],[3778,883],[3969,305],[4204,4801],[4284,115],[4296,1124],[4325,3622],[4466,988],[4618,3994],[4956,3637],[4998,417],[5202,97],[5236,340],[5265,3571],[5375,2249],[5673,1253],[5700,12814],[5757,105],[5814,6021],[5889,8045],[6175,339],[6313,2086],[6520,1000],[6741,457],[6750,941],[7070,18997],[7171,4821],[7405,276],[7497,114],[7688,5153],[7699,417],[7945,761],[8282,13361],[8444,8724],[8808,983],[8866,310],[9132,7538],[9946,747],[10221,105],[12057,2461],[12517,9156],[14318,765],[14481,17121],[16602,180],[18326,12753],[18456,479],[20283,1827],[25338,8349],[74,528],[77,1348],[102,272],[107,12451],[121,100],[195,150],[267,141],[285,781],[285,1607],[285,7242],[343,5024],[367,1438],[388,1071],[401,68],[458,179],[468,453],[479,4909],[499,289],[654,144],[705,599],[753,3857],[790,599],[791,1044],[834,820],[882,5586],[887,1306],[891,289],[900,2050],[933,1683],[934,306],[1058,1591],[1089,12314],[1099,135],[1104,27561],[1131,3025],[1154,977],[1175,693],[1193,5517],[1197,850],[1228,2346],[1237,163],[1287,189],[1314,8928],[1387,1630],[1422,119],[1535,2370],[1537,1262],[1642,3281],[1668,5332],[1669,116],[1693,6129],[1700,538],[1703,1228],[1704,1836],[1721,305],[1731,1761],[1759,2823],[1933,1703],[1950,25969],[2022,276],[2024,30155],[2059,788],[2211,17432],[2224,608],[2239,122],[2315,276],[2388,7416],[2466,1397],[2508,22469],[2867,1118],[2882,724],[3053,98],[3120,1771],[3120,13996],[3565,276],[3677,6176],[3710,8904],[3809,7611],[4095,140],[4229,938],[4235,83],[4399,16802],[4636,1804],[4749,509],[4796,9993],[5021,2373],[5472,12437],[5665,1454],[5692,161],[5990,9267],[6089,11924],[6168,450],[6399,1191],[6784,357],[6996,111],[7365,400],[7437,2441],[7503,2312],[7718,3465],[7997,531],[8058,118],[8124,608],[8150,307],[8443,2048],[8603,13836],[8900,4588],[9152,599],[9813,5414],[11201,128],[11712,2015],[12889,156],[13517,7896],[13692,475],[14337,3225],[14894,89],[15103,11638],[15147,5246],[15231,7268],[15640,788],[16546,288],[16992,616],[19417,9405],[21564,976],[22339,1540],[84,959],[99,288],[108,2507],[116,345],[117,11884],[118,5332],[235,29337],[276,3723],[278,6232],[291,3371],[333,724],[333,3254],[333,7316],[343,1358],[434,8750],[463,725],[495,425],[514,370],[532,1663],[541,3163],[587,1819],[614,5780],[725,989],[733,3251],[773,667],[788,919],[792,140],[793,2156],[803,12916],[832,367],[840,109],[846,1115],[917,3238],[945,132],[968,268],[976,1923],[994,10868],[1037,2055],[1182,544],[1216,185],[1235,1235],[1248,1105],[1256,1227],[1308,8096],[1314,4672],[1366,2086],[1376,2250],[1423,573],[1672,8351],[1690,505],[1706,17958],[1711,13403],[1738,115],[1749,1694],[1832,1607],[1927,450],[1960,1226],[2126,9346],[2142,306],[2193,18475],[2198,1238],[2348,3410],[2352,465],[2390,2131],[2401,16084],[2447,1965],[2447,3192],[2518,119],[2544,11307],[2600,4136],[2706,332],[2799,143],[2815,1388],[2979,5597],[2995,509],[3010,523],[3030,12191],[3272,2073],[3666,533],[3883,496],[4281,1678],[4311,820],[4451,2687],[4654,114],[4829,5164],[4898,3967],[5604,812],[5883,2209],[6143,128],[6206,1594],[6884,2770],[7614,927],[8402,357],[8521,31080],[8906,8705],[8906,9343],[9440,1038],[9511,593],[11737,30833],[11842,400],[13746,465],[13797,2358],[14234,304],[14251,1029],[15054,9257],[15560,538],[15950,479],[16013,760],[18212,305],[18253,780],[20293,1606],[20691,1475],[21282,1604],[24395,24954],[25842,10018],[66,73],[77,68],[101,272],[119,4760],[283,330],[299,9662],[306,1932],[320,6098],[375,834],[375,1994],[401,69],[467,307],[516,16185],[532,1044],[609,184],[613,544],[625,117],[625,339],[625,1686],[654,175],[684,4046],[745,21744],[784,1655],[832,898],[844,17926],[877,2809],[886,844],[892,1118],[896,5984],[904,3129],[910,276],[927,274],[929,528],[996,1789],[1033,156],[1042,1364],[1064,141],[1101,1092],[1104,9030],[1105,4404],[1119,5910],[1156,820],[1193,693],[1231,407],[1276,2616],[1361,1746],[1385,571],[1412,2207],[1423,28032],[1488,2792],[1522,157],[1576,29376],[1622,417],[1671,2711],[1723,1305],[1739,386],[1834,156],[1836,4367],[1863,4869],[1908,9936],[1954,3944],[1992,1013],[1995,370],[2056,2680],[2080,1946],[2185,430],[2234,6021],[2276,2253],[2318,276],[2324,112],[2335,682],[2378,1367],[2406,926],[2485,2620],[2518,18441],[2569,1074],[2631,109],[2714,5669],[2717,105],[2717,1809],[2810,10325],[2956,4411],[3023,3542],[3171,2909],[3220,2209],[3469,14362],[3517,4089],[3572,28506],[3856,3698],[3897,101],[3933,1346],[4210,400],[4247,2260],[4364,8045],[4608,2189],[4720,3612],[4798,422],[4817,3588],[4892,1595],[5485,1992],[5491,1634],[5693,21732],[6755,111],[6855,382],[6902,422],[7547,3697],[7761,542],[8628,107],[8755,11611],[9016,1307],[10353,919],[10417,3964],[11013,541],[11162,2008],[11461,3642],[12184,1563],[12202,827],[12569,2609],[15575,2269],[16396,1499],[18393,611],[19679,991],[21213,1418],[21502,19952],[22024,6950],[22165,19744],[24875,5642],[27959,16956],[56,1841],[84,117],[103,104],[105,17766],[117,1786],[121,287],[121,1403],[273,11210],[276,3242],[276,6279],[285,17019],[295,185],[306,6190],[315,163],[320,3464],[338,189],[352,1487],[364,2500],[366,15622],[368,13405],[375,3486],[377,583],[381,3435],[393,3702],[430,2494],[434,11888],[490,430],[522,716],[535,138],[548,1623],[550,174],[562,914],[574,5682],[596,781],[596,14079],[620,378],[625,6005],[643,1781],[650,8185],[670,28963],[695,1483],[719,7676],[748,6521],[763,175],[765,326],[816,116],[822,10511],[891,268],[928,1016],[986,930],[994,883],[1001,188],[1042,1601],[1070,276],[1136,2100],[1151,683],[1156,4036],[1194,976],[1214,2599],[1235,2057],[1274,582],[1335,7864],[1404,988],[1412,20615],[1454,365],[1481,1085],[1528,1847],[1538,1555],[1591,842],[1612,4592],[1705,276],[1720,272],[1728,3505],[1776,562],[1851,345],[1917,26767],[1933,1228],[1982,5598],[2246,3482],[2258,97],[2258,115],[2540,3444],[2602,910],[2624,1914],[2877,573],[2997,1136],[3054,530],[3250,276],[3367,8837],[3398,136],[3515,1715],[3823,2108],[3848,2984],[4044,390],[4257,2679],[4340,152],[4383,9226],[5389,696],[5405,381],[5806,965],[5806,2321],[5943,1421],[5995,276],[6224,2112],[6460,116],[6462,2479],[6667,357],[6749,3988],[6835,17892],[6986,29454],[7083,111],[7158,857],[7580,3232],[7666,1786],[8075,104],[8473,417],[9078,3905],[10372,18136],[10455,3732],[10520,720],[10621,7487],[10746,268],[12644,415],[13538,354],[15207,29018],[15390,3709],[15853,22634],[20221,2215],[21166,1789],[23408,1356],[27231,437],[70,73],[72,11267],[102,1740],[104,1148],[108,390],[132,9710],[270,274],[270,8752],[277,397],[278,3622],[280,187],[294,304],[298,175],[299,6684],[312,293],[336,13994],[337,12293],[344,1162],[393,2166],[394,144],[397,2576],[401,317],[451,324],[463,11401],[472,69],[486,6311],[487,494],[495,1851],[514,7422],[532,851],[608,289],[629,150],[654,130],[669,1652],[680,188],[738,1505],[766,1521],[790,724],[791,312],[794,976],[819,1683],[958,112],[1010,181],[1021,871],[1047,2312],[1219,938],[1247,21836],[1248,5663],[1292,522],[1323,1138],[1336,795],[1340,599],[1376,5780],[1426,1953],[1429,268],[1450,1907],[1517,6220],[1518,1703],[1521,1817],[1537,3710],[1569,12128],[1642,3957],[1684,274],[1702,276],[1743,890],[1744,9474],[1774,28813],[1800,1062],[1932,7485],[1965,2088],[2029,25185],[2041,10094],[2110,865],[2254,919],[2346,593],[2381,12434],[2400,2343],[2470,13732],[2486,1456],[2699,4071],[2731,386],[2774,312],[2802,7269],[2838,276],[2918,1520],[2932,144],[2938,4703],[3041,128],[3069,886],[3095,1400],[3271,117],[3285,25261],[3682,365],[3692,1534],[3903,2941],[4031,978],[4435,339],[4517,6388],[4574,918],[4607,1000],[4781,1179],[4909,2643],[5060,1932],[5331,14418],[6732,397],[7058,643],[7280,272],[7455,3262],[7521,2576],[8222,3977],[8428,4433],[8659,628],[8919,1811],[10123,5802],[11439,829],[12299,2910],[12351,1547],[13334,304],[13608,17727],[16328,352],[16386,307],[17435,6793],[17814,97],[18451,1594],[27512,1100],[28011,1254],[29064,9467],[31392,25423],[84,856],[89,272],[109,3561],[121,1294],[263,131],[285,6487],[294,1582],[299,2628],[302,311],[319,908],[346,136],[347,1580],[350,4794],[364,22518],[393,764],[399,4704],[402,467],[419,4691],[428,268],[486,1052],[495,80],[498,153],[500,150],[522,698],[611,2973],[652,268],[663,169],[675,1274],[683,5074],[720,2494],[726,5329],[730,487],[748,1551],[765,462],[855,2315],[861,826],[872,3910],[886,11008],[999,537],[1077,1389],[1092,1192],[1097,10169],[1107,865],[1215,681],[1226,2260],[1233,128],[1242,9497],[1268,809],[1269,138],[1403,1011],[1418,3187],[1447,766],[1450,2531],[1623,268],[1625,11313],[1671,2201],[1689,462],[1697,4665],[1736,587],[1781,3378],[1843,417],[2028,357],[2087,3920],[2128,1226],[2131,1290],[2261,8837],[2330,9428],[2333,806],[2559,300],[2601,370],[2649,1298],[2658,988],[3051,23150],[3076,611],[3188,701],[3403,7641],[3884,759],[4561,136],[4788,276],[5044,3276],[5159,6078],[5209,1903],[5260,10926],[5277,6809],[5420,709],[5595,876],[5716,892],[5891,6439],[5891,20383],[5899,115],[6214,335],[6246,26065],[6601,325],[6681,2701],[6949,981],[6987,13678],[7078,804],[7219,102],[7932,400],[8407,325],[8519,875],[8558,3188],[8959,8079],[9052,2615],[9106,847],[9132,10012],[9848,1160],[10339,2538],[11259,764],[12371,1999],[13089,15280],[13424,1297],[13571,462],[14701,4053],[14885,109],[14947,683],[15651,1495],[16205,9041],[19541,417],[20299,21886],[20514,559],[20656,7109],[24135,376],[26271,3810],[28403,268],[28822,25952],[32,4110],[70,76],[103,105],[135,97],[273,1038],[276,5995],[276,9515],[285,876],[337,19393],[356,806],[368,1077],[375,1052],[378,107],[384,23208],[385,4181],[385,4540],[399,3973],[401,977],[413,150],[432,3610],[480,450],[487,2866],[499,1011],[522,6907],[523,3087],[532,692],[532,2757],[547,162],[548,6513],[595,759],[597,132],[640,19753],[650,5755],[651,170],[654,177],[684,584],[696,3359],[704,1437],[732,143],[759,6110],[763,174],[775,164],[779,2012],[832,813],[840,100],[869,19084],[888,2790],[929,428],[930,743],[934,340],[1036,14331],[1065,1437],[1078,3046],[1097,858],[1097,18056],[1170,1083],[1268,2225],[1289,176],[1294,1440],[1537,4053],[1542,1052],[1573,276],[1574,5683],[1592,98],[1628,1502],[1747,98],[1934,2696],[1935,2012],[1978,155],[2065,417],[2153,1613],[2190,573],[2268,289],[2334,1688],[2392,6705],[2503,661],[2611,1855],[2693,382],[2718,9743],[2736,12792],[2771,7017],[2852,376],[2933,950],[3057,1115],[3084,21655],[3184,417],[3356,796],[3911,761],[3969,1271],[4116,2953],[4191,479],[4226,326],[4389,1043],[4810,700],[4842,1320],[4959,6909],[6124,23661],[6129,2017],[6147,407],[6264,21802],[6559,276],[6634,450],[6637,53],[6667,24683],[6719,268],[6774,289],[6807,479],[6996,304],[7524,3091],[7610,276],[7763,3782],[8970,117],[8988,13500],[9170,764],[10064,5104],[10708,97],[11140,1505],[11634,6705],[13180,10062],[14000,97],[14518,1999],[15011,11330],[15201,23928],[17017,10763],[17557,18858],[18038,2614],[20164,3878],[21659,1191],[25697,116],[26633,1931],[27057,18656],[28933,3956],[29011,168],[31058,3709],[31462,4270],[77,422],[104,970],[109,1515],[114,1505],[118,1959],[276,2707],[281,13119],[283,9718],[291,1378],[294,5984],[300,9374],[302,24048],[304,103],[311,608],[320,23224],[331,27111],[336,621],[336,5704],[343,360],[375,7452],[377,5924],[378,118],[385,956],[391,453],[402,465],[463,2602],[473,188],[491,168],[515,288],[516,339],[530,22178],[532,330],[535,151],[539,135],[548,65],[593,3332],[613,946],[621,538],[684,326],[711,4002],[719,425],[773,2022],[784,3137],[790,2380],[802,191],[812,1198],[822,530],[869,396],[899,156],[940,994],[961,325],[1156,1379],[1251,13146],[1251,15736],[1258,147],[1361,6432],[1376,2304],[1401,276],[1444,15299],[1525,3911],[1657,11620],[1671,6942],[1741,911],[1747,9674],[1781,3058],[1829,3816],[1912,3392],[1979,12392],[1988,422],[2197,12925],[2246,475],[2289,682],[2331,2331],[2409,1756],[2449,2812],[2466,463],[2544,4963],[2566,30748],[2626,1682],[2802,115],[2891,4444],[2986,5121],[3208,16778],[3221,305],[3363,118],[3468,141],[3506,795],[3563,357],[3576,306],[3737,119],[3811,268],[3908,7613],[4224,1092],[4237,3808],[4260,1502],[4505,272],[5090,2673],[5131,276],[5487,79],[5520,2217],[5667,463],[5787,4968],[5793,13217],[5938,422],[6213,4217],[6383,20205],[6446,1848],[6460,268],[6572,383],[6693,1816],[6759,21583],[6975,7281],[7108,2622],[7332,7097],[7666,894],[7688,7097],[8060,530],[8079,2875],[8122,3092],[8331,1085],[9146,12555],[9261,628],[9266,307],[9326,716],[10182,10182],[10302,8118],[10513,4958],[11182,10220],[13327,1112],[13397,2980],[13489,1038],[13717,5074],[14875,276],[15818,370],[15966,304],[16138,422],[16327,370],[16664,4143],[17258,3464],[18861,3803],[19995,9470],[22046,30316],[22098,1593],[26653,3513],[27018,804],[27776,11294],[77,67],[98,296],[117,537],[239,488],[276,6473],[287,804],[291,6012],[312,296],[320,317],[333,667],[345,4413],[375,1245],[386,1068],[415,2248],[430,490],[493,139],[499,922],[514,1785],[542,4185],[548,1041],[571,648],[602,116],[675,2551],[684,317],[695,354],[728,2292],[733,7384],[737,6139],[745,1745],[784,396],[842,1697],[846,3454],[855,12596],[857,6792],[887,2055],[976,716],[1039,10291],[1104,14734],[1163,2065],[1164,865],[1194,1393],[1264,399],[1295,2329],[1352,140],[1376,390],[1411,1431],[1412,1059],[1451,49],[1458,2028],[1462,54],[1469,1481],[1494,122],[1536,97],[1580,1052],[1642,2621],[1701,1282],[1704,376],[1738,509],[1747,13224],[1838,268],[1867,587],[1962,28998],[2002,2666],[2025,480],[2076,3692],[2165,2702],[2184,100],[2201,2158],[2262,7208],[2277,15514],[2330,10726],[2392,2622],[2392,7826],[2415,475],[2426,3691],[2621,3021],[2669,1627],[2795,2231],[2875,1238],[3036,1723],[3095,1484],[3123,572],[3319,10903],[3320,2017],[3514,6707],[3554,5860],[3647,2213],[3705,5831],[3782,3193],[3931,14262],[4032,304],[4035,1389],[4116,3239],[4208,1138],[4265,1142],[4329,340],[4340,161],[4349,643],[4406,2667],[4437,3393],[4541,1939],[4605,4237],[4965,173],[4997,1422],[5538,270],[5543,773],[5775,3346],[6052,1591],[6536,1663],[6565,276],[6594,1185],[6714,2162],[6749,1949],[8012,2620],[8799,580],[9364,122],[9403,276],[9643,168],[10104,673],[10826,1357],[11120,110],[11599,148],[13720,307],[15085,696],[15565,152],[16572,1100],[17066,2283],[17358,3525],[19392,3265],[19745,531],[20070,28586],[20224,5193],[20546,422],[20809,1307],[22963,709],[23074,890],[23703,673],[24309,6082],[24555,2209],[24734,1811],[25397,1948],[25511,1731],[29711,1742],[31188,4212],[65,1640],[72,20686],[82,83],[105,317],[273,2571],[276,5829],[288,270],[296,608],[312,621],[331,987],[331,21104],[336,1044],[337,332],[353,268],[375,4145],[375,12504],[421,9360],[450,2370],[508,4911],[508,7122],[532,3709],[586,4423],[597,145],[598,114],[615,1227],[683,9629],[762,120],[779,838],[795,2626],[801,1511],[804,1007],[826,2242],[833,276],[837,381],[869,395],[916,183],[934,367],[956,4112],[1021,479],[1088,7603],[1121,1861],[1163,5069],[1165,1547],[1205,6258],[1210,1202],[1244,1907],[1248,5337],[1283,18955],[1293,9555],[1326,701],[1329,3104],[1414,2863],[1566,6708],[1566,7366],[1666,306],[1686,400],[1697,4577],[1784,4165],[1792,7872],[1802,919],[1912,1586],[1951,4079],[1997,145],[2057,3617],[2178,450],[2179,6752],[2211,623],[2252,22896],[2295,6767],[2307,5028],[2316,22414],[2549,14512],[2697,3276],[2794,2118],[2858,558],[2861,312],[3081,1944],[3363,2150],[3489,628],[3507,12501],[3597,357],[3694,121],[3699,276],[3818,1250],[3922,1112],[4298,272],[4332,357],[4447,850],[4610,3752],[4763,1489],[4828,114],[4997,1294],[5261,117],[5823,1658],[5991,1985],[6360,276],[6385,1388],[6405,1699],[6531,1663],[6637,49],[7512,1926],[7892,26797],[8240,97],[8729,3012],[8930,1186],[9183,330],[9477,566],[9848,345],[10705,480],[10792,289],[11078,24575],[11537,901],[11821,8372],[15074,2993],[17539,1240],[17686,24809],[19630,293],[20798,116],[26669,3635],[80,1011],[85,71],[108,3754],[117,442],[122,272],[188,7745],[276,6329],[276,11790],[281,2006],[281,2304],[283,688],[292,169],[296,2589],[310,24629],[325,10645],[336,12843],[336,31648],[350,2323],[368,415],[377,982],[386,3691],[396,324],[399,3090],[430,809],[462,305],[486,599],[487,583],[495,66],[514,25530],[518,468],[533,1209],[575,450],[580,6268],[594,160],[596,69],[634,97],[650,21183],[654,157],[669,1582],[672,25157],[728,925],[818,904],[873,3002],[921,98],[928,854],[944,2562],[963,2941],[975,2393],[980,5677],[989,1090],[1048,1446],[1133,10286],[1137,496],[1164,696],[1199,706],[1202,457],[1218,858],[1241,1384],[1248,1262],[1262,2902],[1273,370],[1429,22253],[1564,13278],[1713,168],[1720,2151],[1787,559],[1870,407],[1890,2531],[1911,2162],[1912,781],[1913,1103],[1938,20871],[1972,2599],[2016,8526],[2075,6111],[2207,2017],[2401,100],[2516,20947],[2615,747],[2627,1457],[2756,3779],[2944,187],[2948,4515],[3089,558],[3131,17898],[3236,1320],[3292,1307],[3470,1977],[3738,6872],[3872,27425],[4129,1541],[4204,14515],[4309,2048],[4657,132],[5123,2773],[5688,1148],[5773,107],[5950,178],[6060,5315],[6074,4801],[6152,2022],[6247,829],[6844,27472],[6966,155],[7188,6366],[7329,353],[7510,837],[7748,628],[7803,370],[7829,3142],[8065,1013],[8561,4125],[8623,370],[9055,648],[9345,827],[9814,1872],[9879,422],[10005,783],[10968,533],[11467,450],[11745,21239],[13675,841],[13839,1150],[14695,8009],[15235,3362],[15276,1338],[15824,1290],[16386,111],[17235,4775],[17885,102],[18413,1043],[19873,3357],[20222,830],[20553,145],[22752,370],[23826,5462],[29894,2379],[67,425],[73,26919],[79,6122],[79,15386],[110,115],[110,952],[112,353],[117,1317],[118,1554],[122,6847],[276,4030],[281,3483],[302,4901],[319,1483],[320,706],[336,1150],[353,105],[362,128],[384,6538],[402,2546],[454,2446],[463,2076],[466,161],[486,1109],[486,5684],[493,184],[495,397],[508,13997],[524,1883],[532,584],[532,1046],[532,16046],[554,129],[579,538],[615,854],[627,160],[650,3588],[672,1853],[684,1718],[725,1165],[726,2274],[730,730],[741,1659],[835,7420],[869,7662],[882,422],[885,1797],[927,321],[947,1624],[1034,164],[1035,3314],[1065,2326],[1091,140],[1104,2684],[1165,1017],[1170,475],[1178,1061],[1214,2045],[1231,13081],[1366,105],[1390,2416],[1394,3849],[1412,844],[1446,1386],[1469,407],[1651,773],[1664,156],[1705,6368],[1749,1105],[1800,661],[1973,152],[1990,5997],[2011,23452],[2014,160],[2195,97],[2195,3508],[2214,2526],[2233,3964],[2305,1580],[2334,1122],[2359,8730],[2410,129],[2511,830],[2595,146],[2789,276],[2812,2104],[2875,822],[2919,9465],[3061,23565],[3086,15657],[3140,566],[3186,11520],[3193,276],[3301,19386],[3564,2910],[3629,1142],[3662,7573],[3816,3023],[3967,1250],[4017,646],[4069,21376],[4079,10505],[4100,149],[4129,1687],[4171,304],[4231,1402],[4338,399],[4473,1336],[4846,2283],[4909,480],[4919,1884],[4962,76],[5171,5246],[5309,5723],[5378,4027],[5458,847],[5688,9932],[5691,559],[6292,4271],[6433,5550],[6497,26242],[6533,3925],[6536,117],[6589,1580],[6599,2337],[6710,30434],[6845,3360],[7480,854],[7942,925],[8279,2910],[8444,1622],[9518,361],[10569,1737],[10572,11361],[11584,509],[11585,533],[12159,6263],[12396,1164],[12639,886],[12944,626],[13587,1768],[14170,10137],[14753,1539],[16123,378],[17141,865],[18304,370],[19097,4494],[20710,1343],[21464,1999],[21591,103],[22625,2273],[22773,2191],[26224,9629],[27640,5600],[28831,4857],[30806,390],[77,71],[83,843],[104,339],[108,324],[114,2656],[118,115],[283,620],[283,6086],[320,1498],[333,10147],[386,6151],[395,3115],[423,66],[423,18675],[434,4963],[437,268],[440,538],[460,162],[469,157],[483,175],[532,345],[532,1113],[532,14070],[613,2094],[614,558],[615,1336],[650,1557],[675,759],[692,3477],[705,667],[719,24986],[780,530],[798,144],[832,2017],[840,110],[846,25312],[873,105],[917,911],[939,2196],[961,1540],[1071,692],[1083,2825],[1095,575],[1171,986],[1172,146],[1175,1913],[1197,97],[1214,382],[1301,1835],[1328,587],[1374,3726],[1513,2085],[1531,10428],[1553,11006],[1572,111],[1584,185],[1591,23212],[1704,490],[1719,20505],[1747,107],[1753,1343],[1753,1958],[1839,1726],[1891,381],[1896,788],[1993,616],[2039,159],[2079,399],[2104,793],[2165,7954],[2185,1872],[2207,2189],[2270,747],[2442,1894],[2473,340],[2489,13470],[2510,1418],[2608,2045],[2674,1207],[2714,7029],[2838,5936],[3202,1570],[3226,268],[3301,1491],[3332,1926],[3347,4384],[3582,3521],[3604,8680],[3706,2382],[3762,178],[3871,276],[3886,376],[3894,3539],[3948,2656],[4170,268],[4191,2634],[4260,2731],[4314,348],[4436,276],[4492,324],[4530,4266],[4854,3951],[4859,26929],[4959,781],[5217,2960],[5420,437],[5487,69],[5587,3748],[5608,1122],[5712,1090],[5715,1999],[5726,311],[5824,1693],[6364,673],[6637,50],[6937,2642],[7050,1112],[7356,2721],[7645,696],[8824,705],[8943,84],[9390,13440],[9936,1395],[10075,509],[11257,1715],[11813,116],[12173,307],[12527,103],[14448,575],[15824,2871],[16226,1421],[16455,6574],[17734,2244],[18660,25446],[19200,307],[19956,2250],[20426,4071],[22142,1263],[23092,23384],[24491,307],[28350,2083],[30190,26540],[32207,4309],[70,3345],[77,117],[87,111],[100,293],[104,3715],[107,274],[112,5907],[236,22088],[267,191],[276,7308],[281,816],[320,781],[337,2470],[339,15567],[347,7765],[353,1088],[365,3755],[384,9169],[385,4040],[386,5460],[388,8751],[389,156],[421,3816],[421,6404],[432,3828],[434,3924],[468,924],[472,7483],[496,1367],[548,5304],[571,430],[571,8247],[584,2655],[613,2625],[629,184],[635,133],[651,188],[725,8198],[743,907],[793,2424],[800,12666],[801,3229],[812,1239],[840,1403],[859,3165],[863,3262],[891,1718],[898,730],[950,1535],[958,2086],[1037,1328],[1042,7545],[1065,6041],[1069,180],[1073,173],[1092,2838],[1127,7594],[1127,13619],[1151,2910],[1183,593],[1245,2426],[1248,5418],[1274,2234],[1300,3925],[1316,144],[1335,6420],[1357,115],[1374,4250],[1379,2477],[1408,19678],[1423,1808],[1430,1013],[1471,276],[1472,1412],[1488,18343],[1494,856],[1513,28319],[1578,486],[1725,276],[1784,1365],[1867,675],[2087,599],[2132,2766],[2326,5530],[2359,276],[2474,4974],[2548,4309],[2602,16569],[2653,1186],[2667,370],[2792,376],[2862,2358],[2862,13340],[2883,8554],[2905,1422],[3002,330],[3048,2816],[3098,559],[3187,2822],[3212,139],[3219,1134],[3279,1137],[3284,1437],[3307,523],[3416,11907],[3489,10931]],"version":3}

In [ ]:
%%writefile kani_advanced.py
"""Kani Advanced: Kani's own transformer, pretrained from scratch on Kaggle's free GPUs.

The architecture and tokenizer are exactly the local Kani's (kelus/model.py, data/big-v1/tokenizer.json):
pre-LayerNorm blocks, rotary positions, a 4x GELU feed-forward and tied embeddings, so the exported
kani-advanced.npz loads straight into `kelus`. No pretrained weights: it learns from public text,
FineWeb-Edu and Cosmopedia v2, a new slice of it every Kaggle session.

    python kani_advanced.py prepare --settings settings.json    # download and tokenize this session's text
    torchrun --nnodes=1 --nproc_per_node=2 --master_addr=127.0.0.1 --master_port=29500 kani_advanced.py train --settings settings.json
    python kani_advanced.py sample --settings settings.json --prompt "The moon is"
    python kani_advanced.py export --settings settings.json     # kani-advanced.npz from checkpoint.pt
    python kani_advanced.py retokenize --settings settings.json --checkpoint old/checkpoint.pt --old-tokenizer old.json
    python kani_advanced.py grams --tests DIR --out contamination.npy   # the test sets' 13-word runs, never to train on
    python kani_advanced.py grow --settings settings.json --base kani-advanced.npz --progress progress.json   # a deeper run

Multilingual runs (BPE pattern version 3) give each source a `share` of the session's tokens, a `filter` that keeps
documents in their language's script, and a held-out `validation_file`; `retokenize` moves an English checkpoint
to the multilingual vocabulary, keeping everything it learned. Shares also mix English with code. A source's
`revision` reads another branch of its dataset, such as Hugging Face's parquet copy of a JSON dataset
('refs/convert/parquet'), and its `column` names the field holding the text ('text' unless set; 'code' for code).
"""
import functools
import argparse
import hashlib
from contextlib import nullcontext
import glob
import itertools
import json
import math
import os
from pathlib import Path
import re
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PAD, BOS, EOS, USER, ASSISTANT = range(256, 261)

LATIN, HANGUL = "A-Za-z\u00c0-\u00d6\u00d8-\u00f6\u00f8-\u024f\u1e00-\u1eff", "\uac00-\ud7a3\u1100-\u11ff\u3130-\u318f"
KANA, HAN = "\u3040-\u309f\u30a0-\u30ff\u31f0-\u31ff", "\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff"
PATTERNS = {2: re.compile(r"```[A-Za-z]*| ?[A-Za-z]+| ?[0-9]+|\s+|[^A-Za-z0-9\s]"),
            3: re.compile(f"```[A-Za-z]*| ?[{LATIN}]+| ?[{HANGUL}]+|[{KANA}]+|[{HAN}]+| ?[0-9]+|\\s+|[^\\s]")}

DEFAULTS = {
    'run_dir': '/kaggle/working/kani-advanced',
    'input_root': '/kaggle/input',
    'data_dir': '/tmp/kani-advanced-data',
    'tokenizer': 'kani_tokenizer.json',
    'width': 1024, 'layers': 16, 'heads': 16, 'context': 1024,
    'target_tokens': 4_000_000_000,
    'session_tokens': 1_200_000_000,
    'batch_tokens': 262_144,
    'micro_batch': 4,
    'lr': 4e-4, 'min_lr_ratio': 0.1, 'warmup_tokens': 100_000_000, 'decay_fraction': 0.2,
    'weight_decay': 0.1,
    'hours': 10.0,
    'checkpoint_minutes': 30,
    'eval_every': 250, 'eval_batches': 16,
    'validation_documents': 4000,
    'sources': [
        {'repo': 'HuggingFaceFW/fineweb-edu', 'prefix': 'sample/10BT/', 'per_session': 1},
        {'repo': 'HuggingFaceTB/smollm-corpus', 'prefix': 'cosmopedia-v2/', 'per_session': 1},
    ],
    'gradient_checkpointing': False,
    'compile': False,
    'seed': 2041,
    'continue_from_previous': False,
    'workers': 0,
    'rewarm_tokens': 50_000_000,
    'decontaminate': False,
}


def load_settings(path):
    settings = {**DEFAULTS, **json.loads(Path(path).read_text())}
    w, h = settings['width'], settings['heads']

    if not (16 <= settings['context'] <= 2048 and 8 <= w <= 1024 and 1 <= settings['layers'] <= 24
            and w % h == 0 and (w // h) % 2 == 0):
        raise SystemExit('Kani needs context <= 2048, width <= 1024, layers <= 24, and an even head width')
    return settings




class Tokenizer:
    """kelus.tokenizer.BPETokenizer (versions 2 and 3), with an unbounded per-piece cache for speed."""

    def __init__(self, merges, version=2):
        self.version, self.pattern = version, PATTERNS[version]
        self.merges = [tuple(pair) for pair in merges]
        self.pieces = [bytes([i]) for i in range(256)] + [b''] * 5
        self.ranks = {}
        for rank, (a, b) in enumerate(self.merges):
            self.ranks[(a, b)] = (rank, len(self.pieces))
            self.pieces.append(self.pieces[a] + self.pieces[b])
        self.vocab_size = len(self.pieces)
        self.cache = {}

    @classmethod
    def load(cls, path):
        value = json.loads(Path(path).read_text())
        if value.get('kind') != 'bpe' or value.get('version') not in PATTERNS:
            raise SystemExit(f'{path} is not a Kani BPE tokenizer (version 2 or 3)')
        return cls(value['merges'], value['version'])

    def encode_piece(self, piece):
        cached = self.cache.get(piece)
        if cached is not None:
            return cached
        sequence = list(piece.encode('utf-8'))
        while len(sequence) > 1:
            best = None
            for pair in zip(sequence, sequence[1:]):
                rank = self.ranks.get(pair)
                if rank is not None and (best is None or rank < best[0]):
                    best = (rank, pair)
            if best is None:
                break
            (_, token), (a, b) = best
            merged, i = [], 0
            while i < len(sequence):
                if i + 1 < len(sequence) and sequence[i] == a and sequence[i + 1] == b:
                    merged.append(token)
                    i += 2
                else:
                    merged.append(sequence[i])
                    i += 1
            sequence = merged
        if len(self.cache) > 4_000_000:
            self.cache.clear()
        self.cache[piece] = sequence
        return sequence

    def encode(self, text):
        return [token for piece in self.pattern.findall(text) for token in self.encode_piece(piece)]

    def decode(self, tokens):
        return b''.join(self.pieces[t] for t in tokens if 0 <= t < self.vocab_size).decode('utf-8', errors='replace')

    def to_dict(self):
        return {'kind': 'bpe', 'merges': [list(pair) for pair in self.merges], 'version': self.version}


_worker = None


def fast_tokenizer(path):
    """The Rust tokenizer (kaggle/kani_tokenize: the same tokens, ~4.5x faster), or None when it is not installed."""
    try:
        from kani_tokenize import Tokenizer as Rust  # kaggle/kani_tokenize's source folder alone imports, but has no Tokenizer
    except ImportError:
        return None
    value = json.loads(Path(path).read_text())
    return Rust([tuple(pair) for pair in value['merges']], value['version'])


def _start_worker(path, banned=None):
    global _worker, _banned
    _worker = fast_tokenizer(path) or Tokenizer.load(path)
    _banned = np.load(banned) if banned else np.zeros(0, dtype=np.uint64)


# The test sets Kani is measured on (kelus/standard_eval.py: HellaSwag, ARC, PIQA, Winogrande, LAMBADA) are never
# trained on: a document sharing a run of 13 words with one of them is dropped before tokenizing, as GPT-3 did.
# contamination.npy (sorted hashes of those runs) comes from the `grams` command; prepare looks for it in run_dir, then
# beside an attached checkpoint, and stops when `decontaminate` is set and it is missing.
GRAM, _WORDS = 13, re.compile(r'[a-z0-9]+')
_banned = np.zeros(0, dtype=np.uint64)


@functools.lru_cache(maxsize=1 << 20)
def _word_hash(word):
    return int.from_bytes(hashlib.blake2b(word.encode(), digest_size=8).digest(), 'little')


def gram_hashes(text):
    """A 64-bit hash of every run of 13 lower-cased words (letters and digits only) in `text`."""
    words = _WORDS.findall(text.lower())
    count = len(words) - GRAM + 1
    if count < 1:
        return np.zeros(0, dtype=np.uint64)
    ids = np.fromiter(map(_word_hash, words), dtype=np.uint64, count=len(words))
    out = np.zeros(count, dtype=np.uint64)
    for k in range(GRAM):  # wraps at 2**64, the same on every machine
        out = out * np.uint64(1099511628211) + ids[k:k + count]
    return out


def contaminated(text, banned):
    grams = gram_hashes(text)
    if not len(grams) or not len(banned):
        return False
    at = np.minimum(np.searchsorted(banned, grams), len(banned) - 1)
    return bool((banned[at] == grams).any())


def benchmark_texts(folder):
    """Every question with each of its answers, from kelus.standard_eval's test files in `folder`."""
    folder = Path(folder)
    lines = lambda name: [json.loads(line) for line in (folder / name).read_text(encoding='utf-8').splitlines() if line.strip()]
    for row in lines('hellaswag_val.jsonl'):
        yield from (f"{row['activity_label']}: {row['ctx']} {ending}" for ending in row['endings'])
    for name in ('arc_easy_test.jsonl', 'arc_challenge_test.jsonl'):
        for row in lines(name):
            yield from (f"{row['question']} {choice}" for choice in row['choices']['text'])
    for row in lines('piqa_valid.jsonl'):
        yield from (f"{row['goal']} {row['sol1']}", f"{row['goal']} {row['sol2']}")
    for row in lines('winogrande_val.jsonl'):
        yield from (row['sentence'].replace('_', row['option1']), row['sentence'].replace('_', row['option2']))
    for row in lines('lambada_test.jsonl'):
        yield row['text']


def write_grams(folder, out):
    grams = np.unique(np.concatenate([gram_hashes(text) for text in benchmark_texts(folder)]))
    np.save(out, grams)
    print(f'{len(grams):,} distinct 13-word runs from the test sets in {folder} -> {out}')



PAIRS = ('们們 这這 说說 时時 国國 会會 来來 对對 发發 后後 个個 为為 学學 经經 动動 现現 机機 开開 关關 长長 问問 见見 进進 实實 两兩 车車 东東 门門 '
         '应應 电電 气氣 头頭 从從 书書 难難 爱愛 让讓 认認 识識 读讀 写寫 听聽 买買 卖賣 钱錢 银銀 医醫 药藥 话話 语語 请請 谢謝 体體 题題 华華 业業 '
         '产產 网網 线線 结結 给給 红紅 绿綠 鸟鳥 鱼魚 马馬 龙龍 风風 飞飛 云雲 万萬 与與 区區 级級 历歷 战戰 农農 观觀 欢歡 乐樂 节節 质質 资資 '
         '务務 处處 员員 种種 总總 统統 系係 组組 织織 设設 计計 论論 议議 记記 讲講 该該 报報 场場 还還 运運 远遠 选選 边邊 导導 '
         '将將 岁歲 热熱 图圖 团團 园園 圆圓 广廣 庆慶 张張 强強 归歸 当當 录錄 彻徹 忆憶 态態 怀懷 恶惡 悬懸 惊驚 愿願 戏戲 户戶').split()
SIMPLIFIED, TRADITIONAL = {p[0] for p in PAIRS}, {p[1] for p in PAIRS}


def in_script(text, kind):
    """True when most of the document's letters are in the script `kind` (FineWeb-2's language labels are not perfect)."""
    letters = [c for c in text[:5000] if c.isalpha()]
    if not letters:
        return False
    if kind == 'hangul':
        return sum('\uac00' <= c <= '\ud7a3' for c in letters) >= 0.6 * len(letters)
    kana = sum('\u3040' <= c <= '\u30ff' for c in letters)
    han = sum('\u4e00' <= c <= '\u9fff' for c in letters)
    if kind == 'kana':
        return kana >= 20 and kana + han >= 0.6 * len(letters)
    simplified, traditional = sum(c in SIMPLIFIED for c in text[:5000]), sum(c in TRADITIONAL for c in text[:5000])
    wanted = simplified if kind == 'simplified' else traditional
    return (kana == 0 and han >= 0.6 * len(letters) and simplified + traditional >= 3
            and wanted >= 4 * (simplified + traditional - wanted))


def encode_documents(texts, kind=None):
    """[BOS] text [EOS] for every document (in the script `kind`, when given), as kelus's raw corpora store them."""
    if len(_banned):
        texts = [text for text in texts if text and not contaminated(text, _banned)]
    if not isinstance(_worker, Tokenizer):
        if kind is not None:
            texts = [text for text in texts if text and in_script(text, kind)]
        return np.frombuffer(_worker.encode_documents(texts), dtype='<u2')
    out = []
    for text in texts:
        if text and (kind is None or in_script(text, kind)):
            out.append(BOS)
            out.extend(_worker.encode(text))
            out.append(EOS)
    return np.asarray(out, dtype='<u2')




def find_file(settings, name):
    """run_dir/<name>, else the same file in an attached earlier output of this notebook (or a dataset). With several
    attached, the one beside the progress.json that has seen the most tokens: the latest session's."""
    run = Path(settings['run_dir'])
    if (run / name).exists():
        return run / name
    root = Path(settings['input_root'])
    # In a folder named like run_dir (a notebook's output), or loose at a dataset's top level (files uploaded as they are).
    patterns = [f'{"*/" * depth}{run.name}/{name}' for depth in (1, 2, 3)] + [f'{"*/" * depth}{name}' for depth in (1, 2, 3)]
    found = list(dict.fromkeys(path for pattern in patterns for path in (sorted(root.glob(pattern)) if root.is_dir() else [])))

    def tokens(path):
        try:
            return json.loads((path.parent / 'progress.json').read_text())['tokens_seen']
        except (OSError, ValueError, KeyError):
            return -1
    return max(found, key=tokens) if found else None


def read_progress(settings):
    path = find_file(settings, 'progress.json')
    return json.loads(path.read_text()) if path else {'session': 0, 'tokens_seen': 0, 'step': 0}


def source_files(source):
    """Sorted data files of one source: Hugging Face parquet files, or local ones ('local' glob) for testing."""
    if 'local' in source:
        return sorted(glob.glob(source['local']))
    from huggingface_hub import HfApi
    files = HfApi().list_repo_files(source['repo'], repo_type='dataset', revision=source.get('revision'))
    return sorted(f for f in files if f.startswith(source['prefix']) and f.endswith('.parquet'))


def fetch(source, name, folder):
    if 'local' in source:
        return Path(name)
    from huggingface_hub import hf_hub_download

    return Path(hf_hub_download(source['repo'], name, repo_type='dataset', revision=source.get('revision'), local_dir=folder))


def parquet_texts(path, documents, column='text'):
    """Lists of up to `documents` texts from a parquet file's text column, read lazily."""
    import pyarrow.parquet as pq
    for batch in pq.ParquetFile(path).iter_batches(columns=[column], batch_size=documents):
        yield batch.column(column).to_pylist()


def multilingual(settings):
    return any('share' in source for source in settings['sources'])


def source_key(source):
    return source.get('local') or f"{source['repo']}/{source.get('prefix', '')}"


def start_file(source, session, cursors):
    """The number of the first file of `source` that a session reads: the one after the last file read before (progress
    'next_file', which both this script and kani_tpu.py keep), else session x per_session + offset. Numbers wrap around
    the source's files."""
    key = source_key(source)
    return cursors[key] if key in cursors else session * source['per_session'] + source.get('offset', 0)


def data_files(settings):
    """Each source's training files, and the held-out validation file(s).

    English runs hold out the first source's last file. Multilingual runs hold out a file of every source (its
    `validation_file`, else its last file) and can start a source `offset` files in, so two sources reading the
    same files (Simplified and Traditional Chinese) read different ones.
    """
    lists, validation = [], [] if multilingual(settings) else None
    for index, source in enumerate(settings['sources']):
        files = source_files(source)
        if not multilingual(settings):
            if index == 0:
                validation, files = files[-1], files[:-1]
        else:
            held = source.get('validation_file') or files[-1]
            validation.append((index, held))
            files = [name for name in files if name != held]
        lists.append(files)
    return lists, validation


def session_plan(settings, session, cursors=None):
    """(source index, file) pairs to tokenize this session, the held-out validation file(s), and where the next
    session starts in each source."""
    lists, validation = data_files(settings)
    plan, after = [], {}
    for index, (source, files) in enumerate(zip(settings['sources'], lists)):
        start = start_file(source, session, cursors or {})
        plan += [(index, files[(start + j) % len(files)]) for j in range(source['per_session'])]
        after[source_key(source)] = start + source['per_session']
    return plan, validation, after


def validation_tokens(settings, validation, pool, folder):
    """The held-out documents' tokens: validation_documents in all, shared between the held-out files."""
    held_out = validation if multilingual(settings) else [(0, validation)]
    parts = []
    for index, name in held_out:
        source = settings['sources'][index]
        path = fetch(source, name, folder)
        encode = functools.partial(encode_documents, kind=source.get('filter'))
        wanted, texts = settings['validation_documents'] // len(held_out), []
        for batch in parquet_texts(path, 2000, source.get('column', 'text')):
            texts += [text for text in batch if source.get('filter') is None or in_script(text, source['filter'])]
            if len(texts) >= wanted:
                break
        texts = texts[:wanted]
        parts += pool.map(encode, [texts[i:i + 250] for i in range(0, len(texts), 250)])
        if 'local' not in source:
            path.unlink(missing_ok=True)
    return np.concatenate(parts)


def tokenize_file(source, name, out, limit, pool, workers, folder):
    """Append one parquet file's documents to `out` as tokens; returns the tokens written (about `limit` at most)."""
    path = fetch(source, name, folder)
    reader, written = parquet_texts(path, 500, source.get('column', 'text')), 0
    encode = functools.partial(encode_documents, kind=source.get('filter'))
    with open(out, 'ab') as stream:

        while written < limit and (group := list(itertools.islice(reader, 4 * workers))):
            for tokens in pool.map(encode, group):
                tokens.tofile(stream)
                written += len(tokens)
    if 'local' not in source:
        path.unlink(missing_ok=True)
    return written


def attached(settings):
    """The folders under input_root, two levels deep, to show what Kaggle mounted."""
    root = Path(settings['input_root'])
    if not root.is_dir():
        return [f'{root} does not exist']
    return sorted(str(p.relative_to(root)) for p in root.glob('*/*')) or sorted(p.name for p in root.iterdir()) or ['(nothing attached)']


def prepare(settings):
    from multiprocessing import Pool
    checkpoint = find_file(settings, 'checkpoint.pt')
    print('attached inputs:', ', '.join(attached(settings)[:20]), flush=True)
    Tokenizer.load(settings['tokenizer'])
    print('tokenizing with', 'Rust (kani_tokenize)' if fast_tokenizer(settings['tokenizer']) else 'Python', flush=True)
    if checkpoint:
        print(f'continuing from {checkpoint}', flush=True)
    elif settings['continue_from_previous']:
        raise SystemExit('No earlier checkpoint.pt found, so this session would start over from zero. '
                         'Attach this notebook\'s latest output (Add Input → Your Work → Notebooks), '
                         'or set continue_from_previous to False to begin a new run.')
    else:
        print('no earlier checkpoint: starting a NEW run from zero', flush=True)
    banned = find_file(settings, 'contamination.npy')
    if banned:
        print(f'dropping documents that share 13 words with the test sets ({len(np.load(banned)):,} runs, {banned})', flush=True)
        run = Path(settings['run_dir'])
        if banned.parent != run:  # it travels on with this session's output
            run.mkdir(parents=True, exist_ok=True)
            banned = Path(shutil.copy(banned, run / 'contamination.npy'))
    elif settings['decontaminate']:
        raise SystemExit('decontaminate is on but no contamination.npy was found: attach the dataset that holds it '
                         '(made by "kani_advanced.py grams") beside the checkpoint.')
    progress = read_progress(settings)
    session, out = progress['session'], Path(settings['data_dir'])
    plan, validation, after = session_plan(settings, session, progress.get('next_file'))
    key = {'session': session, 'plan': plan, 'validation': validation, 'tokenizer': settings['tokenizer'],
           'session_tokens': settings['session_tokens'], 'validation_documents': settings['validation_documents'],
           'decontaminated': bool(banned)}
    key = json.loads(json.dumps(key))
    if (out / 'manifest.json').exists() and json.loads((out / 'manifest.json').read_text()).get('key') == key:
        print(f'session {session}: {out} is already prepared')
        return
    out.mkdir(parents=True, exist_ok=True)
    for name in ('manifest.json', 'train.bin', 'validation.bin'):
        (out / name).unlink(missing_ok=True)
    folder, started, workers = str(out / 'downloads'), time.time(), settings['workers'] or os.cpu_count()
    print(f'session {session}: {", ".join(name for _, name in plan)}', flush=True)
    with Pool(workers, initializer=_start_worker, initargs=(settings['tokenizer'], banned and str(banned))) as pool:
        validation_tokens(settings, validation, pool, folder).tofile(out / 'validation.bin')
        total = 0
        for index, name in plan:
            source = settings['sources'][index]

            share = (int(settings['session_tokens'] * source['share'] / source['per_session']) if multilingual(settings)
                     else settings['session_tokens'] // len(plan))
            written = tokenize_file(source, name, out / 'train.bin', share, pool, workers, folder)
            total += written
            print(f'  {name}: {written:,} tokens ({time.time() - started:.0f}s)', flush=True)
    validation_count = (out / 'validation.bin').stat().st_size // 2
    (out / 'manifest.json').write_text(json.dumps({'key': key, 'train_tokens': total, 'validation_tokens': validation_count,
                                                   'next_file': after}, indent=1))
    print(f'{total:,} training tokens and {validation_count:,} validation tokens in {time.time() - started:.0f}s')




def rope_tables(context, head_width):
    """kelus.model's rotary tables: angle = position * 10000^(-i/half), repeated for both halves."""
    half = head_width // 2
    angles = np.arange(context)[:, None] * 10000.0 ** (-np.arange(half) / half)
    cos = np.concatenate([np.cos(angles)] * 2, axis=1).astype(np.float32)
    sin = np.concatenate([np.sin(angles)] * 2, axis=1).astype(np.float32)
    return torch.from_numpy(cos), torch.from_numpy(sin)


def rotate(x, cos, sin):
    half = x.shape[-1] // 2
    return x * cos + torch.cat([-x[..., half:], x[..., :half]], dim=-1) * sin


class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(width, eps=1e-5), nn.LayerNorm(width, eps=1e-5)
        self.q, self.k, self.v, self.projection = (nn.Linear(width, width, bias=False) for _ in range(4))
        self.up, self.down = nn.Linear(width, 4 * width, bias=False), nn.Linear(4 * width, width, bias=False)

    def forward(self, x, cos, sin):
        b, t, d = x.shape
        h = self.norm1(x)
        q, k, v = (layer(h).view(b, t, self.heads, d // self.heads).transpose(1, 2) for layer in (self.q, self.k, self.v))
        q, k = rotate(q, cos[:t], sin[:t]).to(v.dtype), rotate(k, cos[:t], sin[:t]).to(v.dtype)
        mixed = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(b, t, d)
        x = x + self.projection(mixed)
        return x + self.down(F.gelu(self.up(self.norm2(x)), approximate='tanh'))


class Kani(nn.Module):
    def __init__(self, vocab, width, layers, heads, context, checkpointing=False):
        super().__init__()
        self.context, self.checkpointing = context, checkpointing
        self.token = nn.Embedding(vocab, width)
        self.blocks = nn.ModuleList(Block(width, heads) for _ in range(layers))
        self.final = nn.LayerNorm(width, eps=1e-5)
        cos, sin = rope_tables(context, width // heads)
        self.register_buffer('cos', cos, persistent=False)
        self.register_buffer('sin', sin, persistent=False)
        for name, p in self.named_parameters():
            if p.dim() == 2:
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * layers) if name.endswith(('projection.weight', 'down.weight')) else 0.02)

    def forward(self, ids):
        x = self.token(ids)
        for block in self.blocks:
            if self.checkpointing and self.training:
                x = torch.utils.checkpoint.checkpoint(block, x, self.cos, self.sin, use_reentrant=False)
            else:
                x = block(x, self.cos, self.sin)
        return F.linear(self.final(x), self.token.weight)


def build(settings, vocab):
    return Kani(vocab, settings['width'], settings['layers'], settings['heads'], settings['context'], settings['gradient_checkpointing'])


def kani_parameters(net):
    """The model's weights under kelus.model's names and layouts (matrices are input x output)."""
    state = {k: v.detach().float().cpu().numpy() for k, v in net.state_dict().items()}
    values = {'token': state['token.weight'], 'final.gain': state['final.weight'], 'final.bias': state['final.bias']}
    for i in range(len(net.blocks)):
        p = f'blocks.{i}.'
        for norm in ('norm1', 'norm2'):
            values[f'block{i}.{norm}.gain'] = state[p + norm + '.weight']
            values[f'block{i}.{norm}.bias'] = state[p + norm + '.bias']
        for matrix in ('q', 'k', 'v', 'projection', 'up', 'down'):
            values[f'block{i}.{matrix}'] = np.ascontiguousarray(state[p + matrix + '.weight'].T)
    return values


def export(net, settings, tokenizer, progress, path):
    """A checkpoint in kelus's format ("kelus-b1" version 1), loadable by kelus.model.Model.load."""
    config = {'context': settings['context'], 'width': settings['width'], 'heads': settings['heads'],
              'layers': settings['layers'], 'seed': settings['seed'], 'vocab_size': tokenizer.vocab_size,
              'positions': 'rope', 'tie_embeddings': True, 'dropout': 0.0}
    metadata = {'display_name': 'Kani Advanced', 'steps': progress['step'], 'tokens_seen': progress['tokens_seen'],
                'sessions': progress['session'], 'stage': 'pretraining (raw text); needs instruction tuning before chat',
                'data': [s.get('repo', s.get('local')) for s in settings['sources']], 'trained_on': 'Kaggle (kaggle/kani_advanced.py)'}
    header = {'format': 'kelus-b1', 'version': 1, 'name': 'Kani Seal 3', 'config': config,
              'tokenizer': tokenizer.to_dict(), 'metadata': metadata}
    values = {'parameter.' + k: v for k, v in kani_parameters(net).items()}
    temp = Path(str(path) + '.tmp.npz')
    np.savez(temp, header=np.array(json.dumps(header)), **values)
    temp.replace(path)
    print(f'exported {path} ({sum(p.numel() for p in net.parameters()):,} parameters, '
          f'{progress["tokens_seen"] / 1e9:.2f}B tokens seen)', flush=True)


@torch.no_grad()
def generate(net, tokenizer, prompt, tokens=60, temperature=0.8, top_k=40, seed=0):
    device = next(net.parameters()).device
    ids = [BOS] + tokenizer.encode(prompt)
    generator = torch.Generator(device=device).manual_seed(seed)
    for _ in range(tokens):
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = net(torch.tensor([ids[-net.context:]], device=device))[0, -1].float()
        logits[[PAD, BOS, USER, ASSISTANT]] = -float('inf')
        values, indices = logits.topk(top_k)
        token = int(indices[torch.multinomial((values / temperature).softmax(-1), 1, generator=generator)])
        if token == EOS:
            break
        ids.append(token)
    return tokenizer.decode(ids[1:])


PROMPTS = ['The capital of France is', 'Photosynthesis is the process by which', 'In 1969, astronauts',
           'To sort a list in Python, you can', 'The main causes of the First World War were']

PROMPTS_MULTILINGUAL = PROMPTS[:3] + ['Der Himmel ist blau, weil', 'La capitale de la France est', 'El agua hierve a',
                                      'O Brasil é o maior país', '한국의 수도는', '日本で一番高い山は', '中国的首都是', '台灣最高的山是']




def learning_rate(settings, tokens, rewarm_from=None):
    """Warmup, then constant, then a linear decay over the last `decay_fraction` of target_tokens. After a retokenize
    (a fresh optimizer from `rewarm_from` tokens) the rate ramps up again over `rewarm_tokens`."""
    rate = schedule(settings, tokens)
    if rewarm_from is not None and tokens - rewarm_from < settings['rewarm_tokens']:
        rate *= (tokens - rewarm_from + settings['batch_tokens']) / settings['rewarm_tokens']
    return rate


def schedule(settings, tokens):
    peak, target = settings['lr'], settings['target_tokens']
    if tokens < settings['warmup_tokens']:
        return peak * (tokens + settings['batch_tokens']) / settings['warmup_tokens']
    decay_start = target * (1 - settings['decay_fraction'])
    if tokens < decay_start:
        return peak
    done = min(1.0, (tokens - decay_start) / max(target - decay_start, 1))
    return peak * (1 - done * (1 - settings['min_lr_ratio']))


def windows(stream, count, context, rng):
    starts = rng.integers(0, len(stream) - context - 1, size=count)
    blocks = torch.from_numpy(np.stack([stream[s:s + context + 1] for s in starts]).astype(np.int64))
    return blocks[:, :-1], blocks[:, 1:]


class Blocks:
    """Training text in a shuffled order without repeats: every context-long block of the session's text is read once
    before any block comes back (random windows re-read about a third of the text and never reach another third).
    The position follows the steps taken this session, so a restarted run continues where it stopped."""

    def __init__(self, stream, context, micro_batch, seed, session, rank, world, done_micro_steps):
        self.stream, self.context, self.micro_batch = stream, context, micro_batch
        self.count = (len(stream) - 1) // context
        self.seed, self.session, self.rank, self.world = seed, session, rank, world
        self.position = done_micro_steps * micro_batch * world
        self.order, self.epoch = None, -1

    def block(self, index):
        epoch, within = divmod(index, self.count)
        if epoch != self.epoch:
            self.order, self.epoch = np.random.default_rng([self.seed, self.session, epoch]).permutation(self.count), epoch
        start = int(self.order[within]) * self.context
        return self.stream[start:start + self.context + 1]

    def next(self):
        first = self.position + self.rank * self.micro_batch
        self.position += self.world * self.micro_batch
        blocks = torch.from_numpy(np.stack([self.block(first + j) for j in range(self.micro_batch)]).astype(np.int64))
        return blocks[:, :-1], blocks[:, 1:]


@torch.no_grad()
def validate(net, stream, settings, device):
    net.eval()
    rng, total = np.random.default_rng(0), 0.0
    for _ in range(settings['eval_batches']):
        x, y = windows(stream, settings['micro_batch'], settings['context'], rng)
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = net(x.to(device))
        total += F.cross_entropy(logits.float().flatten(0, 1), y.to(device).flatten()).item()
    net.train()
    return total / settings['eval_batches']


def save_checkpoint(settings, net, opt, scaler, progress):
    run = Path(settings['run_dir'])
    temp = run / 'checkpoint.tmp'
    torch.save({'net': net.state_dict(), 'opt': opt and opt.state_dict(), 'scaler': scaler.state_dict(), 'progress': progress,
                'settings': settings}, temp)
    temp.replace(run / 'checkpoint.pt')
    (run / 'progress.json').write_text(json.dumps(progress, indent=1))


def train(settings):
    distributed = int(os.environ.get('WORLD_SIZE', '1')) > 1
    cuda = torch.cuda.is_available()
    if distributed:
        torch.distributed.init_process_group('nccl' if cuda else 'gloo')
    rank = torch.distributed.get_rank() if distributed else 0
    world = torch.distributed.get_world_size() if distributed else 1
    local = int(os.environ.get('LOCAL_RANK', '0'))
    device = torch.device('cuda', local) if cuda else torch.device('cpu')
    if cuda:
        torch.cuda.set_device(device)
        torch.backends.cuda.matmul.allow_tf32 = True
    main = rank == 0
    per_step = world * settings['micro_batch'] * settings['context']
    if settings['batch_tokens'] % per_step:
        raise SystemExit(f"batch_tokens must be a multiple of GPUs x micro_batch x context = {per_step}")
    accumulate = settings['batch_tokens'] // per_step

    tokenizer = Tokenizer.load(settings['tokenizer'])
    data = Path(settings['data_dir'])
    if not (data / 'manifest.json').exists():
        raise SystemExit(f'No prepared text in {data}: run "prepare" first (it may have been interrupted)')
    manifest = json.loads((data / 'manifest.json').read_text())
    prepared_session = manifest['key']['session']
    stream = np.memmap(data / 'train.bin', dtype='<u2', mode='r')
    held_out = np.memmap(data / 'validation.bin', dtype='<u2', mode='r')
    run = Path(settings['run_dir'])
    run.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(settings['seed'])
    net = build(settings, tokenizer.vocab_size).to(device)
    decay = [p for n, p in net.named_parameters() if p.dim() == 2]
    no_decay = [p for n, p in net.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{'params': decay, 'weight_decay': settings['weight_decay']}, {'params': no_decay, 'weight_decay': 0.0}],
                            lr=settings['lr'], betas=(0.9, 0.95), eps=1e-8, fused=cuda)
    scaler = torch.amp.GradScaler('cuda', enabled=cuda)
    progress = {'session': 0, 'tokens_seen': 0, 'step': 0, 'history': []}
    checkpoint = find_file(settings, 'checkpoint.pt')
    if checkpoint:
        state = torch.load(checkpoint, map_location='cpu', weights_only=True)
        for key in ('width', 'layers', 'heads', 'context'):
            if state['settings'][key] != settings[key]:
                raise SystemExit(f'{checkpoint} has {key}={state["settings"][key]}; keep the model settings or use a new run_dir')
        if state['net']['token.weight'].shape[0] != tokenizer.vocab_size:
            raise SystemExit(f'{checkpoint} has a {state["net"]["token.weight"].shape[0]:,}-token vocabulary, the tokenizer '
                             f'{tokenizer.vocab_size:,}: run retokenize first')
        net.load_state_dict(state['net'])
        if state['opt'] is not None:
            opt.load_state_dict(state['opt'])
            scaler.load_state_dict(state['scaler'])
        progress = state['progress']
        if main:
            print(f'resumed from {checkpoint}: session {progress["session"]}, step {progress["step"]:,}, '
                  f'{progress["tokens_seen"] / 1e9:.2f}B tokens', flush=True)
    model = nn.parallel.DistributedDataParallel(net, device_ids=[local] if cuda else None) if distributed else net
    forward = torch.compile(model) if settings['compile'] else model
    if main and prepared_session != progress['session']:
        print(f'! the prepared text is for session {prepared_session}, not {progress["session"]}: run prepare first '
              'for fresh text (training continues on the old text)', flush=True)
    if main:
        print(json.dumps({'parameters': sum(p.numel() for p in net.parameters()), 'gpus': world, 'device': str(device),
                          'session': progress['session'], 'train_tokens_available': len(stream),
                          'tokens_per_step': settings['batch_tokens'], 'accumulate': accumulate}), flush=True)


    session_start = progress['history'][-1]['step'] if progress.get('history') else 0
    blocks = Blocks(stream, settings['context'], settings['micro_batch'], settings['seed'], progress['session'], rank, world,
                    max(0, progress['step'] - session_start) * accumulate)
    started = last_checkpoint = time.time()
    budget, stop, stopping = settings['hours'] * 3600, torch.zeros(1, device=device), False
    total, count, window = torch.zeros((), device=device), 0, time.time()
    validation_loss = None
    while not stopping:
        lr = learning_rate(settings, progress['tokens_seen'], progress.get('rewarm_from'))
        for group in opt.param_groups:
            group['lr'] = lr
        for micro in range(accumulate):
            x, y = blocks.next()
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            sync = nullcontext() if not distributed or micro == accumulate - 1 else model.no_sync()
            with sync:
                with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                    logits = forward(x)
                loss = F.cross_entropy(logits.float().flatten(0, 1), y.flatten()) / accumulate
                scaler.scale(loss).backward()
            total += loss.detach()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        progress['step'] += 1
        progress['tokens_seen'] += settings['batch_tokens']
        count += 1

        if progress['step'] % 10 == 0:
            done = time.time() - started > budget or progress['tokens_seen'] >= settings['target_tokens']
            stop.fill_(float(done))
            if distributed:
                torch.distributed.broadcast(stop, 0)
            stopping = bool(stop.item())
        if main and (progress['step'] % settings['eval_every'] == 0 or stopping):
            validation_loss = validate(net, held_out, settings, device)
        if main and (progress['step'] % 20 == 0 or stopping):
            rate = count * settings['batch_tokens'] / (time.time() - window)
            status = {'step': progress['step'], 'tokens': f"{progress['tokens_seen'] / 1e9:.3f}B", 'loss': round(total.item() / count, 4),
                      'validation_loss': validation_loss and round(validation_loss, 4), 'lr': f'{lr:.2e}', 'tokens_per_second': round(rate),
                      'hours_left': round(max(0.0, budget - (time.time() - started)) / 3600, 2)}
            if progress['step'] == 20 and cuda:
                status['gpu_memory_gb'] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 1)
            print(json.dumps(status), flush=True)
            (run / 'status.json').write_text(json.dumps({**status, 'state': 'running', 'updated': time.time()}))
            total, count, window = torch.zeros((), device=device), 0, time.time()
        if main and time.time() - last_checkpoint > settings['checkpoint_minutes'] * 60:
            save_checkpoint(settings, net, opt, scaler, progress)
            last_checkpoint = time.time()
            print(f'checkpoint at step {progress["step"]:,}', flush=True)

    if main:
        progress['history'].append({'session': progress['session'], 'step': progress['step'], 'tokens_seen': progress['tokens_seen'],
                                    'validation_loss': validation_loss, 'hours': round((time.time() - started) / 3600, 2)})
        if prepared_session == progress['session'] and 'next_file' in manifest:
            progress['next_file'] = manifest['next_file']
        progress['session'] += 1
        save_checkpoint(settings, net, opt, scaler, progress)
        export(net, settings, tokenizer, progress, run / 'kani-advanced.npz')
        complete = progress['tokens_seen'] >= settings['target_tokens']
        (run / 'status.json').write_text(json.dumps({'step': progress['step'], 'tokens_seen': progress['tokens_seen'],
                                                     'validation_loss': validation_loss, 'state': 'complete' if complete else 'stopped'}))
        net.eval()
        prompts = PROMPTS_MULTILINGUAL if tokenizer.version == 3 else PROMPTS
        samples = {prompt: generate(net, tokenizer, prompt) for prompt in prompts}
        (run / 'samples.json').write_text(json.dumps(samples, indent=1, ensure_ascii=False))
        for prompt, text in samples.items():
            print(f'--- {text}', flush=True)
        print(('target reached: ' if complete else 'session done: ') + f'{progress["tokens_seen"] / 1e9:.2f}B of '
              f'{settings["target_tokens"] / 1e9:.1f}B tokens, validation loss {validation_loss:.4f}', flush=True)
    if distributed:
        torch.distributed.destroy_process_group()


def retokenize(settings, checkpoint, old_path):
    """Move `checkpoint` (trained with the tokenizer at `old_path`) to settings['tokenizer'], as this run's start.

    The blocks keep everything they learned. Each new token's embedding (also the output layer, which is tied)
    starts as the mean of the old embeddings of the old tokens that spell it, so known words keep their meaning.
    The optimizer starts afresh, so training warms the learning rate up again (rewarm_tokens).
    """
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    old, new = Tokenizer.load(old_path), Tokenizer.load(settings['tokenizer'])
    table = state['net']['token.weight']
    if table.shape[0] != old.vocab_size:
        raise SystemExit(f'{checkpoint} does not use {old_path}')
    rows = []
    for token in range(new.vocab_size):
        if token < 261:
            rows.append(table[token])
            continue
        piece = new.pieces[token]
        try:
            ids = old.encode(piece.decode('utf-8'))
        except UnicodeDecodeError:
            ids = list(piece)
        rows.append(table[ids].mean(0))
    state['net']['token.weight'] = torch.stack(rows)
    progress = {**state['progress'], 'rewarm_from': state['progress']['tokens_seen'], 'retokenized_from': str(checkpoint)}
    run = Path(settings['run_dir'])
    run.mkdir(parents=True, exist_ok=True)
    torch.save({'net': state['net'], 'opt': None, 'scaler': None, 'progress': progress,
                'settings': {**state['settings'], 'tokenizer': settings['tokenizer']}}, run / 'checkpoint.tmp')
    (run / 'checkpoint.tmp').replace(run / 'checkpoint.pt')
    (run / 'progress.json').write_text(json.dumps(progress, indent=1))
    print(f'retokenized {checkpoint} ({old.vocab_size:,} tokens) to {new.vocab_size:,} tokens in {run}: step {progress["step"]:,}, '
          f'{progress["tokens_seen"] / 1e9:.2f}B tokens', flush=True)




def grow(settings, base, progress):
    """Start a deeper run from a trained Kani (`base`, kelus's npz format): its blocks in order, with a copy of every
    k-th block inserted after it until there are settings['layers'] blocks. A copy starts with its attention projection
    and feed-forward `down` at zero, so it adds nothing and the grown model is exactly the base (LLaMA Pro's block
    expansion); training then grows the copies into new layers. The optimizer starts fresh: train() ramps the
    learning rate up again over rewarm_tokens from `progress['rewarm_from']`."""
    with np.load(base) as z:
        header, values = json.loads(str(z['header'])), {k.removeprefix('parameter.'): z[k] for k in z.files if k != 'header'}
    config, layers = header['config'], settings['layers']
    for key in ('width', 'heads', 'context'):
        if config[key] != settings[key]:
            raise SystemExit(f'{base} has {key}={config[key]}, the settings {settings[key]}')
    extra, old = layers - config['layers'], config['layers']
    if not 0 < extra <= old or old % extra:
        raise SystemExit(f'grow adds 1 to {old} blocks, a whole fraction of {old}; asked for {extra}')
    every, order = old // extra, []
    for i in range(old):
        order.append((i, False))
        if i % every == every - 1:
            order.append((i, True))
    state = {'token.weight': values['token'], 'final.weight': values['final.gain'], 'final.bias': values['final.bias']}
    for j, (i, copy) in enumerate(order):
        for norm in ('norm1', 'norm2'):
            state[f'blocks.{j}.{norm}.weight'] = values[f'block{i}.{norm}.gain']
            state[f'blocks.{j}.{norm}.bias'] = values[f'block{i}.{norm}.bias']
        for matrix in ('q', 'k', 'v', 'projection', 'up', 'down'):
            weight = np.ascontiguousarray(values[f'block{i}.{matrix}'].T)
            state[f'blocks.{j}.{matrix}.weight'] = np.zeros_like(weight) if copy and matrix in ('projection', 'down') else weight
    net = build(settings, values['token'].shape[0])
    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in state.items()})
    progress = {**progress, 'rewarm_from': progress['tokens_seen'],
                'grown_from': {'checkpoint': str(base), 'layers': old, 'tokens_seen': progress['tokens_seen'], 'copied_blocks': [i for i, c in order if c]}}
    run = Path(settings['run_dir'])
    run.mkdir(parents=True, exist_ok=True)
    save_checkpoint(settings, net, None, torch.amp.GradScaler('cuda', enabled=False), progress)
    print(f'grew {base} from {old} to {layers} blocks ({sum(p.numel() for p in net.parameters()):,} parameters); '
          f'copies after blocks {[i for i, c in order if c]}', flush=True)
    return net


def load_trained(settings):
    checkpoint = find_file(settings, 'checkpoint.pt')
    if checkpoint is None:
        raise SystemExit('No checkpoint.pt yet; train first.')
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    tokenizer = Tokenizer.load(settings['tokenizer'])
    net = build({**settings, **{k: state['settings'][k] for k in ('width', 'layers', 'heads', 'context')}}, tokenizer.vocab_size)
    net.load_state_dict(state['net'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return net.to(device).eval(), tokenizer, state


def main():
    import sys
    if 'ipykernel' in sys.modules:
        print('This is the training script, not the notebook. On Kaggle use File → Import Notebook with '
              'kaggle/kani-advanced.ipynb, which writes this file and runs it.')
        return
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('command', choices=['prepare', 'train', 'sample', 'export', 'retokenize', 'grams', 'grow'])
    parser.add_argument('--settings', default='settings.json')
    parser.add_argument('--prompt', action='append', help='sample: text to continue (repeatable)')
    parser.add_argument('--tokens', type=int, default=80)
    parser.add_argument('--checkpoint', help='retokenize: the checkpoint.pt to move to the new vocabulary')
    parser.add_argument('--old-tokenizer', help="retokenize: that checkpoint's tokenizer")
    parser.add_argument('--tests', help="grams: the folder of kelus.standard_eval's test files")
    parser.add_argument('--out', help='grams: where to write contamination.npy')
    parser.add_argument('--base', help='grow: the trained kani-advanced.npz to grow')
    parser.add_argument('--progress', help="grow: the base's progress.json (session, step, tokens_seen, next_file)")
    args = parser.parse_args()
    if args.command == 'grams':
        if not (args.tests and args.out):
            raise SystemExit('grams needs --tests and --out')
        write_grams(args.tests, args.out)
        return
    settings = load_settings(args.settings)
    if args.command == 'prepare':
        prepare(settings)
    elif args.command == 'train':
        train(settings)
    elif args.command == 'grow':
        if not (args.base and args.progress):
            raise SystemExit('grow needs --base and --progress')
        grow(settings, args.base, json.loads(Path(args.progress).read_text()))
    elif args.command == 'retokenize':
        if not (args.checkpoint and args.old_tokenizer):
            raise SystemExit('retokenize needs --checkpoint and --old-tokenizer')
        retokenize(settings, args.checkpoint, args.old_tokenizer)
    elif args.command == 'export':
        net, tokenizer, state = load_trained(settings)
        Path(settings['run_dir']).mkdir(parents=True, exist_ok=True)
        export(net, {**settings, **state['settings'], 'run_dir': settings['run_dir']}, tokenizer, state['progress'],
               Path(settings['run_dir']) / 'kani-advanced.npz')
    else:
        net, tokenizer, _ = load_trained(settings)
        for i, prompt in enumerate(args.prompt or (PROMPTS_MULTILINGUAL if tokenizer.version == 3 else PROMPTS)):
            print(f'--- {generate(net, tokenizer, prompt, tokens=args.tokens, seed=i)}')


if __name__ == '__main__':
    main()

## This session's text
Downloads one new file per language and turns them into Kani's tokens (about 10 minutes).

In [ ]:
!python kani_advanced.py prepare --settings settings.json
# A failed prepare (for example no earlier checkpoint attached) stops the notebook here, not with errors further down.
import kani_advanced
from pathlib import Path
if not (Path(kani_advanced.load_settings('settings.json')['data_dir']) / 'manifest.json').exists():
    raise RuntimeError('Stopped before training: read the message just above.')

## Train

A log line every 20 steps. `loss` and `validation_loss` should fall over the session and across sessions.
The first line shows `gpu_memory_gb`; if a run stops with "CUDA out of memory", set `micro_batch` to 2.

In [ ]:
gpu_count = max(1, torch.cuda.device_count())
!torchrun --nnodes=1 --nproc_per_node={gpu_count} --master_addr=127.0.0.1 --master_port=29500 kani_advanced.py train --settings settings.json

## How it writes now

In [ ]:
progress = json.load(open('kani-multilingual/progress.json'))
for session in progress['history']:
    print(session)
for prompt, text in json.load(open('kani-multilingual/samples.json')).items():
    print('\n---', text)

## Bring it to Kani

Download `kani-multilingual/kani-advanced.npz` from the Output tab and put it in `KelusModelB1/checkpoints/`.
It is a raw-text model: it continues text but does not chat yet. Chat tuning (Kani's conversations, web
search and reading data) and the capability tests then run on the Mac, before it replaces the current Kani.